# Learning to read, and learning to talk

Built from commit `0e95f25`. No pretrained weights, no gradients, no GPU, and no dataset attached - this
notebook goes and gets everything it learns from.

Four stages, each measured on material it has not seen:

1. **Read the world.** 228 encyclopedia pages, kept as sentences.
2. **Learn to speak.** A sparse variable-order sequence memory hears them once each. Measured by how much of
   a held-out sentence it would have predicted.
3. **Learn to read.** The hand-written reader knows four sentence shapes; the pairs it is confident about are
   used as seeds, the corpus is scanned for other ways those same pairs are phrased, and what comes back is a
   table of phrasings nobody wrote down. 3 rounds, each bootstrapping on the last. Measured as facts
   recovered from held-out sentences the hand-written reader gets nothing out of.
4. **Learn to talk.** Real conversation - interviews, testimony, dialogue, quoted speech - harvested from
   Wikisource, Wikiquote and Project Gutenberg. What is learned is how a turn *opens*, counted separately for
   openings that answer a question and openings that continue a statement.

Then the whole thing is run end to end: it is asked who somebody is and to write an essay, with the internet
on, and both the answer and its reasoning trace are printed.

In [ ]:
import os, sys, subprocess, time
for d in ("tests", "chat", "brain"):
    os.makedirs(f"/kaggle/working/neuromoe/{d}", exist_ok=True)
os.makedirs("/kaggle/working/artifacts", exist_ok=True)
print(sys.version)

In [ ]:
%%writefile /kaggle/working/neuromoe/neuromoe.py
"""NeuroMoE: a growing, gradient-free sparse mixture of experts.

No backprop and no pretrained weights anywhere. The model starts empty and grows as it is used:

  SparseExpansion  fixed random sparse wiring + k-winners-take-all (fly mushroom body / dentate gyrus). Similar
                   inputs share active units, different inputs barely overlap. Never trained.
  ExpertPool       experts are assemblies with input synapses (a prototype over the sparse code) and output synapses.
                   Routing is lateral inhibition: the best-matching experts win. When nothing matches well enough a
                   new expert is born (neurogenesis) wired to exactly the active units (synaptogenesis); when the
                   winner predicts wrong, the match threshold is raised above it and the search continues (ARTMAP
                   match tracking), so exceptions get their own expert instead of overwriting old ones. Winners drift
                   toward their inputs (Hebbian instar: grow synapses to active units, weaken the rest), weak synapses
                   are pruned, experts that stop winning die.
  WorkingMemory    one-shot associative store written by an error-correcting local rule
                   (M += beta (v - M k) k^T / |k|): the value is present when it is written, so no gradient is needed.
                   Sparse keys are nearly orthogonal, so capacity scales with the code size, not with training. A
                   fixed "time cell" code per step lets it bind items to positions (serial recall, n-back).

Every update is local: it uses only the activity on either side of a synapse plus a target that is physically
present at that moment (a label, the next symbol, a stored value). Nothing is differentiated.
"""
import math

import torch


def _generator(seed):
    return torch.Generator().manual_seed(int(seed))


class SparseExpansion:
    """d_in -> n units, each summing `fan_in` randomly chosen inputs with fixed positive weights; exactly k fire.
    Positive, distinct weights make the code deterministic (no ties). An optional homeostatic bias equalises how often
    each unit fires, so no unit is dead - a local rule on each unit's own firing rate."""

    def __init__(self, d_in, n=2048, k=32, fan_in=8, seed=0, homeostasis=0.0):
        g = _generator(seed)
        self.d_in, self.n, self.k = d_in, n, k
        # every input must reach at least 2k units, or a lone active input (a symbol) cannot fill the k winners and
        # the rest are ties at zero - the same filler units for every symbol, which makes keys collide
        fan_in = max(fan_in, math.ceil(2 * k * d_in / n))
        # connections spread evenly over inputs, then shuffled: no input is under-represented by chance
        wiring = torch.arange(n * fan_in) % d_in
        self.idx = wiring[torch.randperm(n * fan_in, generator=g)].reshape(n, fan_in)
        self.w = 0.5 + torch.rand(n, fan_in, generator=g)
        self.bias = torch.zeros(n)
        self.rate = torch.full((n,), k / n)
        self.homeostasis = homeostasis

    @torch.no_grad()
    def __call__(self, x):
        """x (B, d_in) -> binary code (B, n) with exactly k ones."""
        drive = (x[:, self.idx] * self.w).sum(-1) - self.bias
        code = torch.zeros(x.shape[0], self.n)
        code.scatter_(1, drive.topk(self.k, dim=1).indices, 1.0)
        if self.homeostasis:
            self.rate.mul_(0.99).add_(0.01 * code.mean(0))
            self.bias.add_(self.homeostasis * (self.rate - self.k / self.n))
        return code


class ExpertPool:
    """Neurogenesis mixture of experts over binary sparse codes.

    match(e, x) = (P_e . x) / |x| in [0, 1]: the share of the input's active units that expert e is wired to.
    labels="exact": classification; ARTMAP match tracking recruits a new expert whenever the accepted one predicts
    the wrong label. labels="stochastic": next-symbol prediction; an expert splits (a new, more specific expert is
    born) only when it has seen enough evidence and still gave the observed symbol low probability - so contexts are
    refined exactly where the data demands it (variable-order memory)."""

    def __init__(self, n_code, n_out, vigilance=0.7, k_route=3, temperature=0.05, beta=0.2, prune=0.05,
                 max_experts=4096, labels="exact", split_prob=0.1, split_evidence=4, smoothing=0.1, trust=0.0):
        self.n_code, self.n_out = n_code, n_out
        self.vigilance, self.k_route, self.temperature = vigilance, k_route, temperature
        self.beta, self.prune, self.max_experts = beta, prune, max_experts
        self.labels, self.split_prob, self.split_evidence, self.smoothing = labels, split_prob, split_evidence, smoothing
        # trust > 0: an expert's vote is scaled by evidence / (evidence + trust), so a newborn expert resting on one
        # example (possibly a mislabelled one) is heard weakly until other examples confirm it
        self.trust = trust
        self.P = torch.zeros(0, n_code)  # input synapses (prototype strengths)
        self.O = torch.zeros(0, n_out)   # output synapses (association counts)
        self.last_win = torch.zeros(0)
        self.step = 0
        self.born = self.died = 0

    def __len__(self):
        return self.P.shape[0]

    @torch.no_grad()
    def match(self, code):
        return self.P @ code / code.sum().clamp_min(1)

    @torch.no_grad()
    def predict(self, code, return_match=False):
        """Distribution over outputs: the top k_route experts vote, weighted by match (lateral inhibition)."""
        if not len(self):
            p = torch.full((self.n_out,), 1.0 / self.n_out)
            return (p, 0.0) if return_match else p
        m = self.match(code)
        top = m.topk(min(self.k_route, len(self))).indices
        w = self._weights(m[top], top)
        votes = self.O[top] + self.smoothing
        p = (w[:, None] * votes / votes.sum(1, keepdim=True)).sum(0)
        return (p, m.max().item()) if return_match else p

    def _weights(self, match, idx):
        w = torch.softmax(match / self.temperature, -1)
        if self.trust:
            evidence = self.O[idx].sum(-1)
            w = w * evidence / (evidence + self.trust)
            w = w / w.sum(-1, keepdim=True).clamp_min(1e-12)
        return w

    @torch.no_grad()
    def predict_batch(self, codes):
        """Vectorised predict for many inputs at once: (distributions (B, n_out), best match (B,))."""
        B = codes.shape[0]
        if not len(self):
            return torch.full((B, self.n_out), 1.0 / self.n_out), torch.zeros(B)
        m = codes @ self.P.T / codes.sum(1, keepdim=True).clamp_min(1)
        top = m.topk(min(self.k_route, len(self)), dim=1)
        w = self._weights(top.values, top.indices)
        votes = self.O[top.indices] + self.smoothing
        return (w[..., None] * votes / votes.sum(-1, keepdim=True)).sum(1), m.max(1).values

    @torch.no_grad()
    def _birth(self, code, y):
        if len(self) >= self.max_experts:  # full: the longest-idle expert dies first
            self._kill(self.last_win.argmin())
        self.P = torch.cat([self.P, code[None]])
        o = torch.zeros(1, self.n_out)
        o[0, y] = 1.0
        self.O = torch.cat([self.O, o])
        self.last_win = torch.cat([self.last_win, torch.tensor([float(self.step)])])
        self.born += 1
        return len(self) - 1

    @torch.no_grad()
    def _kill(self, e):
        keep = torch.ones(len(self), dtype=torch.bool)
        keep[e] = False
        self.P, self.O, self.last_win = self.P[keep], self.O[keep], self.last_win[keep]
        self.died += 1

    @torch.no_grad()
    def _reinforce(self, e, code, y):
        """Hebbian instar with synaptic turnover: grow toward active units, weaken the rest, prune weak synapses."""
        self.P[e] += self.beta * (code - self.P[e])
        self.P[e][self.P[e] < self.prune] = 0.0
        self.O[e, y] += 1.0
        self.last_win[e] = float(self.step)

    @torch.no_grad()
    def learn(self, code, y):
        """One labelled example, seen once. Returns the expert that learned it."""
        self.step += 1
        if not len(self):
            return self._birth(code, y)
        m = self.match(code)
        rho = self.vigilance
        for e in m.argsort(descending=True).tolist():
            if m[e] < rho:
                break  # nothing left clears the (possibly raised) vigilance
            if self.labels == "exact":
                if self.O[e].argmax().item() == y:
                    self._reinforce(e, code, y)
                    return e
                rho = m[e].item() + 1e-6  # match tracking: this expert is wrong here, demand a closer one
                continue
            # stochastic: accept the best match unless it has confidently learned that y is unlikely here
            counts = self.O[e]
            p_y = (counts[y] + self.smoothing) / (counts.sum() + self.smoothing * self.n_out)
            if counts.sum() >= self.split_evidence and p_y < self.split_prob and m[e] < 0.999:
                break  # a surprise in a context it knows well: split, give this context its own expert
            self._reinforce(e, code, y)
            return e
        return self._birth(code, y)

    def synapses(self):
        return int((self.P > 0).sum()) + int((self.O > 0).sum())


class WorkingMemory:
    """One-shot associative store: M (n_value x n_key), written M += beta (v - M k) k^T / |k|. After a write with
    beta = 1 the key recalls its value exactly; earlier bindings are disturbed only through shared active units.
    `decay` < 1 lets old items fade (a leaky buffer); 1 keeps everything until overwritten."""

    def __init__(self, n_key, n_value, beta=1.0, decay=1.0):
        self.M = torch.zeros(n_value, n_key)
        self.beta, self.decay = beta, decay

    @torch.no_grad()
    def write(self, key, value):
        if self.decay < 1.0:
            self.M.mul_(self.decay)
        self.M += self.beta * torch.outer(value - self.M @ key, key) / key.sum().clamp_min(1)

    @torch.no_grad()
    def read(self, key):
        return self.M @ key

    def clear(self):
        self.M.zero_()

    def synapses(self):
        return int((self.M != 0).sum())


class TimeCells:
    """A fixed sparse code for every time step (hippocampal time cells): step t has its own random k-subset of n
    units, independent of content, so items can be bound to *when* they arrived and replayed in order."""

    def __init__(self, n=2048, k=32, seed=1, max_steps=4096):
        g = _generator(seed)
        self.codes = torch.zeros(max_steps, n)
        for t in range(max_steps):
            self.codes[t, torch.randperm(n, generator=g)[:k]] = 1.0

    def __call__(self, t):
        return self.codes[t]


class NeuroMoE:
    """The assembled model for a symbolic or vector stream: expansion (+ optional context trace) -> expert pool,
    with a working memory alongside. `context` > 0 feeds a decaying trace of recent inputs into the expansion, so the
    same symbol after different histories lands on different experts."""

    def __init__(self, d_in, n_out, n=2048, k=32, fan_in=8, context=0.0, context_decay=0.5, seed=0, **pool):
        self.d_in, self.context, self.context_decay = d_in, context, context_decay
        self.expand = SparseExpansion(d_in * (2 if context else 1), n=n, k=k, fan_in=fan_in, seed=seed)
        self.pool = ExpertPool(n, n_out, **pool)
        self.trace = torch.zeros(d_in)

    @torch.no_grad()
    def encode(self, x):
        """x (d_in,) -> sparse code (n,). With context on, the current input is concatenated with the trace."""
        inp = torch.cat([x, self.context * self.trace]) if self.context else x
        return self.expand(inp[None])[0]

    @torch.no_grad()
    def advance(self, x):
        if self.context:
            self.trace.mul_(self.context_decay).add_(x)

    def reset(self):
        self.trace.zero_()

    def footprint(self):
        """Stored synapses (nonzero weights) - the model's actual size, which grows with use."""
        return self.pool.synapses()

In [ ]:
%%writefile /kaggle/working/neuromoe/cogmap.py
"""CognitiveMap: pseudospatial memory. Items are placed in a low-dimensional latent space by their *relations*, not by
how they look, so geometry carries meaning and relations that were never observed can be computed.

  place neurons   a population tiling the latent space; an item's location is a bump of activity over them, and
                  content is bound to location (Huber: a place cell is a what+where conjunction), so memories can be
                  retrieved *by position* ("what is two steps above X?")
  relations       each relation is a displacement ("taller than" = one step along some direction), learned from use
  one-shot        a new item is placed the moment its first relation arrives: position = known item + displacement
  re-anchoring    a fact that links two separate islands of knowledge shifts the whole smaller island so the fact
                  holds, then merges them (remapping: known memories adapted into one frame)
  consolidation   every observed relation is kept as an episode; replaying episodes applies a local delta rule that
                  moves both items (and the relation vector) toward consistency - scattered facts settle into one map,
                  and a mislabelled fact is outvoted by its neighbours instead of breaking a chain

No gradients: the only update is error = (pos[b] - pos[a]) - d_rel, applied to the three quantities involved.
"""
import math
import random

import torch


class PlaceCells:
    """Place neurons with Gaussian fields on a regular lattice over [-span, span]^dim."""

    def __init__(self, dim=2, per_dim=48, span=24.0, width=0.6, max_cells=20000, seed=0, modules=8):
        self.dim, self.span = dim, span
        self.width = width * (2 * span / (per_dim - 1))
        if dim <= 2 and per_dim ** dim <= max_cells:   # one population, tiling the space directly
            axes = torch.linspace(-span, span, per_dim)
            grids = torch.meshgrid(*[axes] * dim, indexing="ij")
            self.centers = torch.stack([g.reshape(-1) for g in grids], 1)
            self.projections = [torch.eye(dim)]
            self.tiles = [self.centers]
        else:
            # Scattering cells through four or more dimensions leaves them hopelessly far apart - the same
            # 20,000 cells that tile a plane finely are lost in 4-D, and a bump moved one step lands between
            # them. Entorhinal cortex does not have this problem because it does not tile the full space: grid
            # modules each code a low-dimensional projection, and position is recovered from their conjunction.
            # So each module here tiles its own random plane through the space, finely, and an item's code is
            # what all the modules say at once.
            g = torch.Generator().manual_seed(seed)
            side = max(8, int((max_cells / modules) ** 0.5))
            axes = torch.linspace(-span, span, side)
            grid = torch.stack([x.reshape(-1) for x in torch.meshgrid(axes, axes, indexing="ij")], 1)
            self.projections, self.tiles = [], []
            for m in range(modules):
                q, _ = torch.linalg.qr(torch.randn(dim, 2, generator=g))
                self.projections.append(q)
                self.tiles.append(grid)
            self.centers = torch.cat([grid] * modules)     # centres, module by module
            self.width = width * (2 * span / (side - 1))

    def encode(self, pos):
        """Activity of every place neuron for positions (..., dim): each module responds to its own projection."""
        parts = []
        for proj, tile in zip(self.projections, self.tiles):
            p = pos @ proj if proj.shape[0] == pos.shape[-1] else pos
            d2 = ((p[..., None, :] - tile) ** 2).sum(-1)
            parts.append(torch.exp(-d2 / (2 * self.width ** 2)))
        return torch.cat(parts, dim=-1)

    def flow(self, bump, displacement):
        """Path integration: activity moves to the cells whose fields sit one displacement away.

        Each module shifts within its own plane, by the displacement projected into that plane, which is how a
        population of grid modules keeps track of movement without anyone computing a coordinate.
        """
        out, offset = [], 0
        for proj, tile in zip(self.projections, self.tiles):
            n = tile.shape[0]
            here = bump[offset:offset + n]
            offset += n
            step = displacement @ proj if proj.shape[0] == displacement.shape[-1] else displacement
            active = here > here.max() * 0.02
            if not bool(active.any()):
                out.append(here)
                continue
            d2 = torch.cdist(tile, tile[active] + step) ** 2
            out.append(torch.exp(-d2 / (2 * self.width ** 2)) @ here[active])
        moved = torch.cat(out)
        return moved / moved.sum().clamp_min(1e-9)


class CognitiveMap:
    def __init__(self, dim=2, lr=0.5, rel_lr=0.1, replay=30, robust=None, place=None, seed=0,
                 transform=False, mat_lr=0.05, anchor_targets=False):
        self.dim, self.lr, self.rel_lr, self.replay = dim, lr, rel_lr, replay
        # robust: the push from any one fact saturates at this size, so a single contradicted fact is outvoted as an
        # outlier instead of dragging its whole neighbourhood (a quadratic rule spreads every error)
        self.robust = robust
        self.rng = random.Random(seed)
        self.gen = torch.Generator().manual_seed(seed)
        self.names, self.index = [], {}
        self.pos = torch.zeros(0, dim)
        self.placed = torch.zeros(0, dtype=torch.bool)
        # An item does not have a position, it has a region it is probably in. One fact puts something roughly
        # somewhere; five agreeing facts pin it; a contradicted one goes vague again rather than moving to a
        # confident wrong place. The number is a radius in map units, and retrieval widens its bump by it - so
        # uncertainty is in the representation rather than bolted on afterwards as a score.
        self.uncertainty = torch.zeros(0)
        self.island = []  # which connected island of knowledge each item belongs to
        self.members = {}  # island -> its items, kept as facts arrive so merging need not scan
        self.rel = {}
        # A relation as a *transformation* rather than a step. A displacement says everything standing in
        # this relation is the same move away, which is right for "taller" and impossible for "is in
        # the country" - one vector cannot send every city to its own country. A matrix can: it may
        # project, rotate or collapse, so many things can be sent to one without the map tearing.
        self.transform, self.mat_lr = transform, mat_lr
        # Whether a relation may move the things it points at. Sending many things to one needs a
        # transformation that loses information, and the same loss lets it satisfy every fact by
        # squashing its targets together - perfectly learned and perfectly useless. Anchoring the
        # targets says: the country is where its own facts put it; the relation must reach it.
        self.anchor_targets = anchor_targets
        self.mat = {}
        self._bulk = False
        self._centre_cache = None
        self._centre_at = 0
        self.centroid_every = 64
        self.episodes = []
        self.weights = []
        self.repel = None
        self.place = place or PlaceCells(dim=dim)
        self._islands = 0

    # ---------------------------------------------------------------- learning
    def _item(self, name):
        if name not in self.index:
            self.index[name] = len(self.names)
            self.names.append(name)
            self.pos = torch.cat([self.pos, torch.zeros(1, self.dim)])
            self.placed = torch.cat([self.placed, torch.tensor([False])])
            self.uncertainty = torch.cat([self.uncertainty, torch.tensor([1.5])])   # unheard of: wide open
            self.island.append(-1)
        return self.index[name]

    def _relation(self, rel):
        if rel not in self.rel:
            # a new relation gets its own direction, as orthogonal as possible to the ones already known
            d = torch.randn(self.dim, generator=self.gen)
            for other in self.rel.values():
                d = d - (d @ other) / (other @ other) * other
            if d.norm() < 1e-3:
                d = torch.randn(self.dim, generator=self.gen)
            self.rel[rel] = d / d.norm()
        return self.rel[rel]

    def _fresh_spot(self):
        """An unconnected fact starts its own island, a few steps from the known map, until something links it."""
        self._islands += 1
        direction = torch.randn(self.dim, generator=self.gen)
        return self._centre() + 4.0 * direction / direction.norm()

    def _centre(self):
        """The centroid, but reused for a while when placing a corpus.

        A fresh island only needs somewhere roughly beside what is already known, so the exact centre does not
        matter - but computing it averages every placed item, and over the tens of thousands of islands a corpus
        produces that alone dominates the build. Incremental placement keeps the exact value; bulk placement
        refreshes it every `centroid_every` islands."""
        if not self._bulk:
            return self._centroid()
        if self._centre_cache is None or self._islands - self._centre_at >= self.centroid_every:
            self._centre_cache = self._centroid()
            self._centre_at = self._islands
        return self._centre_cache

    def _centroid(self):
        return self.pos[self.placed].mean(0) if bool(self.placed.any()) else torch.zeros(self.dim)

    def recenter(self):
        """Shift the whole map so its centre sits in the middle of the place-cell lattice. A global shift changes
        no relation; it keeps the map inside the region the place neurons cover (place cells remap to the space in
        use)."""
        if bool(self.placed.any()):
            self.pos[self.placed] -= self._centroid()

    def remap(self, scale=4.0, eps=1e-6):
        """Spread the map back out over the space the place neurons cover, without disturbing any fact.

        Consolidation alone has a trivial solution: put everything in one spot and every relation at zero length,
        which satisfies "a is like b" perfectly and says nothing. Relaxation drifts toward it whenever undirected
        facts outnumber directed ones. The escape is that positions only ever mean anything *relative* to each
        other: applying one linear map T to every position and every relation vector leaves every displacement
        exactly as satisfied as it was, because (x@T - y@T) = (x-y)@T. So the map is free to be re-expanded to
        unit spread along every axis - a remapping to the enclosure, not a correction - and the collapsing
        direction is removed on each pass. What survives is the structure that the facts actually pin down."""
        live = self.placed.nonzero().squeeze(1)
        if len(live) < 2:
            return
        self.pos[live] -= self.pos[live].mean(0)
        X = self.pos[live]
        C = (X.T @ X) / len(live) + eps * torch.eye(self.dim, device=X.device, dtype=X.dtype)
        evals, evecs = torch.linalg.eigh(C)
        T = evecs @ torch.diag(scale * evals.clamp_min(eps).rsqrt()) @ evecs.T  # C^(-1/2), symmetric
        self.pos = self.pos @ T
        for k in self.rel:
            self.rel[k] = self.rel[k] @ T

    def downscale(self, factor=0.98, floor=0.05):
        """Sleep's other job: scale everything down a little, and drop what falls through.

        The synaptic homeostasis account of sleep is that waking drives synapses up across the board and sleep
        scales them back proportionally, so relative strengths survive while the accumulated floor of weak,
        accidental associations does not - the signal is kept and the noise is renormalised away. Here the
        analogue is uniform: multiply every position and every relation vector by the same factor, which changes
        no relation at all (a scale is the same linear map applied to both sides), then forget the episodes whose
        residual is so large that they never fitted anything. Unlike inhibition this pushes nothing apart; it
        removes what was never holding together."""
        self.pos *= factor
        for k in self.rel:
            self.rel[k] = self.rel[k] * factor
        if floor and self.episodes:
            keep, kept_w = [], []
            for (a, moves, b), w in zip(self.episodes, self.weights or [1.0] * len(self.episodes)):
                d = self._disp(moves) if moves else torch.zeros(self.dim, device=self.pos.device)
                if float(((self.pos[b] - self.pos[a]) - d).norm()) <= floor or w >= 1.0:
                    keep.append((a, moves, b))
                    kept_w.append(w)
            dropped = len(self.episodes) - len(keep)
            if dropped and len(keep) > 10:
                self.episodes, self.weights = keep, kept_w
            return dropped
        return 0

    def separate_relations(self, strength=0.5, floor=0.15):
        """Keep different relations pointing in different directions.

        They are *made* orthogonal, and then consolidation is free to drag them together: whatever direction
        best fits one relation's facts also partly fits another's, and nothing pushes back. Measured on the
        built map, HasPrerequisite, MotivatedByGoal and HasFirstSubevent had collapsed onto one another at a
        cosine of 1.00, and DerivedFrom onto IsA at 0.99.

        That is precisely why the relation labels came out wrong. If two relations are the same direction then
        stepping along either lands in the same place, and which name gets attached to the answer is a coin
        toss - "a doctor is part of a medical practitioner" is that coin landing badly.

        Distinct relations need distinct codes, and nothing in the map was keeping them distinct. This is
        lateral inhibition again, applied to the directions rather than to the things: relations that have
        grown too alike push each other apart.
        """
        names = sorted(self.rel)
        if len(names) < 2:
            return 0.0
        M = torch.stack([self.rel[k] for k in names])
        norms = M.norm(dim=1, keepdim=True).clamp_min(1e-9)
        U = M / norms
        cos = U @ U.T
        cos.fill_diagonal_(0.0)
        # push each relation away from the others in proportion to how far it has drifted onto them
        push = (cos.abs() * torch.sign(cos)) @ U
        U = U - strength * push / max(1, len(names) - 1)
        U = U / U.norm(dim=1, keepdim=True).clamp_min(1e-9)
        for i, k in enumerate(names):
            self.rel[k] = U[i] * norms[i].clamp_min(floor)
        after = (U @ U.T)
        after.fill_diagonal_(0.0)
        return float(after.abs().mean())

    def inhibit(self, pairs=4096, margin=1.0, strength=0.2, repel=None):
        """Lateral inhibition: sample random pairs of items and push apart any that have crowded closer than
        `margin`. This is what keeps the map from collapsing, and it is the same kind of rule as everything else
        here - local, pairwise, error-driven, no gradient and no global decomposition.

        It is needed because "a is like b" is satisfied perfectly by putting a and b in the same spot, so a map
        told mostly about likeness drifts toward a single point that says nothing. Two items being *distinct* is
        itself information the facts never state, and inhibition is how the population asserts it: neighbouring
        fields that fire together for everything separate until they don't. Only crowded pairs are touched, so
        items the facts genuinely place together stay together - the push is a floor, not a spring."""
        live = self.placed.to(self.pos.device).nonzero().squeeze(1)
        if len(live) < 2:
            return 0.0
        repel = None if repel is None else repel.to(self.pos.device)
        # the map's own generator lives on the CPU; off it, fall back to the global stream
        g = self.gen if self.pos.device.type == "cpu" else None
        pick = lambda: live[torch.randint(len(live), (pairs,), generator=g, device=self.pos.device)]
        i, j = pick(), pick()
        if repel is not None and len(repel):  # pairs a fact says are unlike: always pushed, never sampled away
            i = torch.cat([i, repel[:, 0]])
            j = torch.cat([j, repel[:, 1]])
        keep = i != j
        i, j = i[keep], j[keep]
        diff = self.pos[i] - self.pos[j]
        dist = diff.norm(dim=1)
        crowded = dist < margin
        if not bool(crowded.any()):
            return 0.0
        i, j, diff, dist = i[crowded], j[crowded], diff[crowded], dist[crowded]
        # a random direction for pairs sitting exactly on top of each other, which have no direction to push along
        rand = torch.randn(diff.shape, generator=g, device=self.pos.device)
        unit = torch.where(dist[:, None] > 1e-6, diff / dist[:, None].clamp_min(1e-6),
                           rand / rand.norm(dim=1, keepdim=True).clamp_min(1e-6))
        push = strength * (margin - dist)[:, None] * unit
        self.pos.index_add_(0, i, 0.5 * push)
        self.pos.index_add_(0, j, -0.5 * push)
        return float(crowded.float().mean())

    def _matrix(self, rel):
        """The transformation for a relation, starting as identity so it begins as a pure displacement and only
        departs from that as far as the facts push it."""
        if rel not in self.mat:
            self.mat[rel] = torch.eye(self.dim)
        return self.mat[rel]

    def predict(self, a_pos, rel):
        """Where a relation says you land, starting from here."""
        if self.transform:
            return a_pos @ self._matrix(rel).T + self._relation(rel)
        return a_pos + self._relation(rel)

    def _learn_transform(self, ia, rel, ib, lr):
        """One fact, learned locally.

        error = where b is, minus where the relation said b would be. The matrix is corrected by that error
        times the input it was applied to - the delta rule for a linear associator, which is Hebbian with a sign
        and needs no backward pass. The two items move as well, b toward the prediction and a along the
        transposed weights, which is what a reciprocal connection does in a network.
        """
        M = self._matrix(rel)
        d = self._relation(rel)
        a, b = self.pos[ia], self.pos[ib]
        err = b - (a @ M.T + d)
        if self.robust:
            err = err * min(1.0, self.robust / (float(err.norm()) + 1e-9))
        scale = float(a @ a) + 1e-6
        self.mat[rel] = M + self.mat_lr * torch.outer(err, a) / scale
        step = d + self.rel_lr * err
        # the same floor the displacement path has always had. Without it the offset shrinks toward nothing,
        # which fits every fact and means nothing - the collapse this map has had to be defended from at every
        # level. Leaving it out is why the transformation lost even with its matrix held at the identity.
        self.rel[rel] = step if float(step.norm()) > 0.2 else d
        if not self.anchor_targets:
            self.pos[ib] = b - 0.5 * lr * err
        # the division by the input's size belongs to the matrix update, which has to be scale-free;
        # applying it here as well shrank every position step by the square of the map's spread, and
        # with the matrix frozen at the identity that alone made this path lose to the one it copies
        self.pos[ia] = a + (1.0 if self.anchor_targets else 0.5) * lr * (err @ M)

    def _disp(self, moves):
        """Displacement of a combination of moves, e.g. (("east", 1), ("north", 2))."""
        return sum((count * self.rel[r] for r, count in moves), torch.zeros(self.dim))

    def _consolidate(self, a, moves, b, lr):
        moves = ((moves, 1),) if isinstance(moves, str) else moves
        if self.transform and len(moves) == 1 and moves[0][1] == 1:
            return self._learn_transform(a, moves[0][0], b, lr)
        d = self._disp(moves)
        err = (self.pos[b] - self.pos[a]) - d
        if self.robust:
            err = err * min(1.0, self.robust / (float(err.norm()) + 1e-9))
        self.pos[a] += 0.5 * lr * err
        self.pos[b] -= 0.5 * lr * err
        if self.rel_lr and len(moves) == 1 and moves[0][1] == 1:  # single-step facts teach the relation itself
            rel = moves[0][0]
            step = self.rel_lr * err
            self.rel[rel] = d + step if (d + step).norm() > 0.2 else d

    def observe(self, a, rel, b, count=1):
        """One fact: `b` lies `count` `rel`-steps from `a` (a='Bob', rel='height', b='Alice': Alice is taller)."""
        self.observe_moves(a, ((rel, count),), b)

    def distinguish(self, pairs):
        """Facts that say two things are *unlike* (antonyms, "is distinct from"). Co-locating them would be the
        opposite of what the fact says, and a displacement pointing one way would be just as wrong, so they are
        recorded as pairs that inhibition always pushes apart."""
        idx = [(self._item(a), self._item(b)) for a, b in pairs]
        self.repel = torch.tensor(idx, dtype=torch.long) if idx else None
        return self

    def observe_moves(self, a, moves, b, weight=1.0):
        """One fact made of several moves: b = a + sum(count * relation), e.g. 2 north and 1 east."""
        moves = tuple((r, c) for r, c in moves if c)
        for r, _ in moves:
            self._relation(r)
        ia, ib, d = self._item(a), self._item(b), self._disp(moves)
        if not self.placed[ia] and not self.placed[ib]:
            self.pos[ia] = self._fresh_spot()
            self.pos[ib] = self.pos[ia] + d
            self.island[ia] = self.island[ib] = self._islands
            self.members[self._islands] = [ia] if ia == ib else [ia, ib]
        elif not self.placed[ib]:
            self.pos[ib] = self.pos[ia] + d
            self.island[ib] = self.island[ia]
            self.members.setdefault(self.island[ia], []).append(ib)
        elif not self.placed[ia]:
            self.pos[ia] = self.pos[ib] - d
            self.island[ia] = self.island[ib]
            self.members.setdefault(self.island[ib], []).append(ia)
        elif self.island[ia] != self.island[ib]:
            self._reanchor(ia, ib, d)
        else:
            self._consolidate(ia, moves, ib, self.lr)
        self.placed[ia] = self.placed[ib] = True
        self.episodes.append((ia, moves, ib))
        self.weights.append(float(weight))
        for _ in range(self.replay):  # replay: consolidation over remembered episodes
            ea, er, eb = self.episodes[self.rng.randrange(len(self.episodes))]
            self._consolidate(ea, er, eb, self.lr)

    def attract(self, a, b, weight=1.0):
        """A symmetric fact ("x is like y"): no direction, just co-location. A displacement that means "is like"
        would be wrong pointing either way, so the fact is recorded as a zero displacement and the same delta rule
        pulls the two together. Directed relations keep their norm-floored directions, so the map still spreads
        out instead of collapsing to a point."""
        self.observe_moves(a, (), b, weight=weight)

    def observe_bulk(self, edges, progress=None):
        """Place many facts at once. The per-fact replay that `observe_moves` does is what makes one-shot learning
        settle immediately; over a corpus it is both unaffordable and unnecessary, because `sleep()` replays every
        episode in one batched pass afterwards. Each edge is (a, moves, b); moves=() means attract."""
        edges = list(edges)
        fresh = []
        for edge in edges:      # every name the corpus will mention, allocated once
            for name in (edge[0], edge[2]):
                if name not in self.index:
                    self.index[name] = len(self.names) + len(fresh)
                    fresh.append(name)
        if fresh:
            self.names.extend(fresh)
            self.pos = torch.cat([self.pos, torch.zeros(len(fresh), self.dim)])
            self.placed = torch.cat([self.placed, torch.zeros(len(fresh), dtype=torch.bool)])
            self.uncertainty = torch.cat([self.uncertainty, torch.full((len(fresh),), 1.5)])
            self.island.extend([-1] * len(fresh))
        replay, self.replay = self.replay, 0
        self._bulk, self._centre_cache = True, None
        try:
            for i, edge in enumerate(edges):
                a, moves, b = edge[0], edge[1], edge[2]
                self.observe_moves(a, moves, b, weight=edge[3] if len(edge) > 3 else 1.0)
                if progress and i and i % progress == 0:
                    print(f"  placed {i:,} facts, {len(self.names):,} items, {self._islands:,} islands", flush=True)
        finally:
            self.replay = replay
            self._bulk = False
        return self

    def _reanchor(self, ia, ib, d):
        """Two islands just got linked: move the smaller one rigidly so the new fact holds, and merge them."""
        ka, kb = self.island[ia], self.island[ib]   # read the ids before anything overwrites them
        A = self.members.setdefault(ka, [ia])
        B = self.members.setdefault(kb, [ib])
        shift = (self.pos[ia] + d) - self.pos[ib]
        if len(B) <= len(A):
            self.pos[B] += shift
            keep, gone, move = ka, kb, B
        else:
            self.pos[A] -= shift
            keep, gone, move = kb, ka, A
        for i in move:
            self.island[i] = keep
        self.members[keep] = self.members.get(keep, []) + move
        self.members.pop(gone, None)

    def consolidate_transform(self, rounds=80, lr=0.15, target_rate=0.15, tolerance=0.5,
                              device=None, verbose=False, inhibit=True, margin=1.0,
                              strength=0.2, pairs=8192, shrink=0.0):
        """Consolidation for relations held as transformations, over a whole corpus at once.

        The same rule as `_learn_transform`, applied to every fact of a relation together instead of one at a
        time: the matrix is corrected by the errors times the inputs they were applied to, which batched is one
        matrix product. `target_rate` is how far a relation may move the things it points at - zero pins them
        entirely, which is what stops a many-to-one relation from satisfying itself by squashing its targets,
        and a little above zero lets a thing that is only ever a target still settle.
        """
        if not self.episodes:
            return
        home = self.pos.device
        if device is not None:
            self.pos = self.pos.to(device)
            self.rel = {k: v.to(device) for k, v in self.rel.items()}
            self.mat = {k: v.to(device) for k, v in self.mat.items()}
            self.placed = self.placed.to(device)
            if self.repel is not None:
                self.repel = self.repel.to(device)

        by_rel = {}
        for ia, moves, ib in self.episodes:
            if len(moves) != 1 or moves[0][1] != 1:
                continue
            by_rel.setdefault(moves[0][0], ([], []))
            by_rel[moves[0][0]][0].append(ia)
            by_rel[moves[0][0]][1].append(ib)
        grouped = {r: (torch.tensor(a, device=self.pos.device), torch.tensor(b, device=self.pos.device))
                   for r, (a, b) in by_rel.items() if len(a) > 1}
        if verbose:
            print(f"  {len(grouped)} relations, {sum(len(a) for a, _ in grouped.values()):,} single-step facts",
                  flush=True)

        # Inhibition's margin has to mean something relative to how spread out the map actually is. Given a
        # fixed margin of 1 on a map whose items sit 0.2 apart, every pair counts as crowded and the whole map
        # is pushed outward every round - which is exactly what happened: it expanded to a spread of ninety.
        live = self.placed.nonzero().squeeze(1)[:600]
        if len(live) > 2 and margin:
            typical = float(torch.cdist(self.pos[live], self.pos[live]).median())
            margin = margin * max(typical, 1e-6)
            if verbose:
                print(f"  inhibition margin scaled to {margin:.4f} for a map spaced {typical:.4f}", flush=True)

        for step in range(rounds):
            total = count = 0.0
            # every item is pulled once per fact it takes part in, so a thing in a hundred thousand facts
            # would be kicked a hundred thousand times a round. What each item wants is the *average* of what
            # its facts ask of it, which is what the displacement path has always computed.
            delta = torch.zeros_like(self.pos)
            hits = torch.zeros(len(self.pos), 1, device=self.pos.device)
            for rel, (ia, ib) in grouped.items():
                M, d = self._matrix(rel).to(self.pos.device), self._relation(rel).to(self.pos.device)
                A, B = self.pos[ia], self.pos[ib]
                err = B - (A @ M.T + d)
                w = torch.clamp(tolerance / (err.norm(dim=1) + 1e-9), max=1.0)[:, None]
                err = err * w
                total += float(err.norm(dim=1).sum())
                count += len(ia)
                scale = float((A * A).sum()) + 1e-6
                new_M = M + self.mat_lr * (err.T @ A) / scale
                if shrink:
                    # A full matrix is four thousand numbers per relation against a few thousand facts, and
                    # on held-out facts every setting of it did worse than a plain step. Pulling it back
                    # toward the identity each round says: be a displacement unless the facts insist
                    # otherwise, and pay for every departure.
                    new_M = (1.0 - shrink) * new_M + shrink * torch.eye(
                        self.dim, device=self.pos.device, dtype=new_M.dtype)
                self.mat[rel] = new_M
                nd = d + self.rel_lr * err.mean(0)
                self.rel[rel] = nd if float(nd.norm()) > 0.2 else d
                ones = torch.ones(len(ia), 1, device=self.pos.device)
                delta.index_add_(0, ia, err @ M)
                hits.index_add_(0, ia, ones)
                if target_rate:
                    delta.index_add_(0, ib, -target_rate * err)
                    hits.index_add_(0, ib, ones * target_rate)
            self.pos += lr * delta / hits.clamp_min(1.0)
            if inhibit:
                self.inhibit(pairs=pairs, margin=margin, strength=strength, repel=self.repel)
            if verbose and step % max(1, rounds // 10) == 0:
                print(f"  round {step:4d}/{rounds}  residual {total / max(1, count):.4f}  "
                      f"spread {float(self.pos[self.placed].std()):.3f}", flush=True)

        if device is not None:
            self.pos = self.pos.to(home)
            self.rel = {k: v.to(home) for k, v in self.rel.items()}
            self.mat = {k: v.to(home) for k, v in self.mat.items()}
            self.placed = self.placed.to(home)
            if self.repel is not None:
                self.repel = self.repel.to(home)

    def sleep(self, rounds=200, damping=0.5, tolerance=0.5, device=None, verbose=False, spread=None, drift=0.0,
              inhibit=None, margin=1.0, strength=0.2, pairs=4096, schema=0.0, shy=0.0,
              rel_floor=0.0, separate=0.0):
        """Offline consolidation: every remembered episode is replayed at once (like sleep ripples) and every item
        moves toward where its neighbours, taken together, say it should be. Local - each item hears only its own
        neighbours - but the network relaxes as a whole, so a region misplaced by one bad fact is pulled back as a
        unit. Facts that disagree with the consensus by more than `tolerance` are down-weighted (robust: a lone
        mislabelled fact is outvoted rather than averaged in). Relation vectors settle to the average displacement
        of the facts that carry them."""
        if not self.episodes:
            return
        self.recenter()
        home = self.pos.device
        if device is not None:  # the relaxation is pure batched tensor work; it runs anywhere
            self.pos = self.pos.to(device)
            self.rel = {k: v.to(device) for k, v in self.rel.items()}
            self.placed = self.placed.to(device)
            if self.repel is not None:
                self.repel = self.repel.to(device)
        a = torch.tensor([e[0] for e in self.episodes], device=self.pos.device)
        b = torch.tensor([e[2] for e in self.episodes], device=self.pos.device)
        rels = sorted(self.rel)
        C = torch.zeros(len(self.episodes), len(rels), device=self.pos.device)  # how many steps of each relation every episode spans
        for i, (_, moves, _) in enumerate(self.episodes):
            for rname, count in moves:
                C[i, rels.index(rname)] += count
        fact_w = torch.tensor(self.weights or [1.0] * len(self.episodes), device=self.pos.device)[:, None]
        unit = (C.abs().sum(1) == 1) & (C.max(1).values == 1)  # plain single-step facts
        r = C.argmax(1)
        for _ in range(rounds):
            D = C @ torch.stack([self.rel[k] for k in rels])
            res = (self.pos[b] - self.pos[a]) - D
            w = torch.clamp(tolerance / (res.norm(dim=1) + 1e-9), max=1.0)[:, None] * fact_w  # Huber x fact
            if schema:  # what already fits the structure is taken up faster (schema-consistent memory)
                w = w * torch.exp(-schema * res.norm(dim=1))[:, None]
            num = torch.zeros_like(self.pos)
            den = torch.zeros(len(self.pos), 1, device=self.pos.device)
            num.index_add_(0, b, w * (self.pos[a] + D))
            num.index_add_(0, a, w * (self.pos[b] - D))
            den.index_add_(0, b, w)
            den.index_add_(0, a, w)
            has = den[:, 0] > 0
            self.pos[has] = (1 - damping) * self.pos[has] + damping * num[has] / den[has]
            if self.rel_lr:
                disp = self.pos[b] - self.pos[a]
                for k, name in enumerate(rels):
                    sel = unit & (r == k)
                    if not bool(sel.any()):
                        continue
                    avg = (w[sel] * disp[sel]).sum(0) / w[sel].sum().clamp_min(1e-9)
                    self.rel[name] = (1 - self.rel_lr) * self.rel[name] + self.rel_lr * avg
                    if rel_floor:
                        # a relation that means something has to be a real step. Facts that disagree about a
                        # relation are best fitted by shrinking it to nothing - which fits well and says nothing,
                        # the same trade the whole map makes about likeness - so hold it to a minimum length.
                        n = self.rel[name].norm()
                        if 0 < float(n) < rel_floor:
                            self.rel[name] = self.rel[name] * (rel_floor / n)
            if inhibit:
                self.inhibit(pairs=pairs, margin=margin, strength=strength, repel=self.repel)
            if separate:
                self.separate_relations(strength=separate)
            if shy:
                self.downscale(factor=1.0 - shy, floor=0.0)
            elif spread:
                self.remap(scale=spread)
            if verbose and _ % max(1, rounds // 10) == 0:
                print(f"  sleep {_:4d}/{rounds}  residual {float(res.norm(dim=1).mean()):.4f}", flush=True)
        self._settle_uncertainty(a, b, res, drift)
        if device is not None:
            self.pos = self.pos.to(home)
            self.rel = {k: v.to(home) for k, v in self.rel.items()}
            self.placed = self.placed.to(home)
            if self.repel is not None:
                self.repel = self.repel.to(home)

    def _settle_uncertainty(self, a, b, res, drift=0.0):
        """After replay, how sure each item's position is - and a little drift, because real codes drift.

        The residual is how far each fact is from being satisfied where things currently sit. An item whose
        facts all agree ends up with a small residual and a tight position; one caught between facts that
        disagree stays wide, which is the honest state of affairs rather than a confident average of two
        incompatible claims.

        The drift is not noise for its own sake. Hippocampal and cortical codes for the same thing move over
        days while behaviour stays put, because what is stable is the *relations* rather than the coordinates.
        Anything here that depended on an absolute position would break under it, and nothing does: every
        answer is a difference, and consolidation re-solves the differences after each nudge.
        """
        if not len(self.episodes) or not len(self.uncertainty):
            return
        # Where would each of this item's facts put it, on its own? Facts that agree propose the same place and
        # the item is pinned; facts that disagree propose different places and it is not. Measuring it this way
        # rather than by the residual matters, because when relations are free to shrink a contradiction can be
        # "satisfied" by making the relation mean less - the residual goes quiet and the disagreement is still
        # there in where the facts point.
        D = torch.stack([self._disp(moves) for _, moves, _ in self.episodes]).to(self.pos.device)
        implied_b, implied_a = self.pos[a] + D, self.pos[b] - D
        count = torch.zeros(len(self.pos), device=self.pos.device)
        mean = torch.zeros_like(self.pos)
        count.index_add_(0, b, torch.ones(len(D), device=self.pos.device))
        count.index_add_(0, a, torch.ones(len(D), device=self.pos.device))
        mean.index_add_(0, b, implied_b)
        mean.index_add_(0, a, implied_a)
        seen = count > 0
        mean[seen] = mean[seen] / count[seen][:, None]
        var = torch.zeros(len(self.pos), device=self.pos.device)
        var.index_add_(0, b, (implied_b - mean[b]).norm(dim=1))
        var.index_add_(0, a, (implied_a - mean[a]).norm(dim=1))
        disagreement = torch.zeros_like(var)
        disagreement[seen] = var[seen] / count[seen]
        # one fact leaves you placed but unpinned, so evidence counts too - and it counts as 1/sqrt(n)
        settled = disagreement[seen] + 0.9 / count[seen].clamp_min(1.0).sqrt()
        u = self.uncertainty.to(self.pos.device)
        u[seen] = 0.4 * u[seen] + 0.6 * settled
        u = u.clamp(0.05, 3.0)
        self.uncertainty = u.cpu() if self.pos.device.type != "cpu" else u
        if drift:
            step = torch.randn(self.pos.shape, generator=self.gen, device=self.pos.device) if self.gen.device \
                == self.pos.device else torch.randn(self.pos.shape, device=self.pos.device)
            self.pos[self.placed] += drift * step[self.placed]

    def spread_of(self, name):
        """How wide this item's position is - what a retrieval of it should be blurred by."""
        if name not in self.index or not len(self.uncertainty):
            return 1.0
        return float(self.uncertainty[self.index[name]])

    # ---------------------------------------------------------------- reasoning
    def knows(self, name):
        return name in self.index and bool(self.placed[self.index[name]])

    def steps(self, a, rel, b):
        """How many `rel`-steps b lies from a (positive: b is `rel` of a). The magnitude doubles as confidence -
        which is why far-apart items are compared more reliably (the symbolic distance effect)."""
        d = self.rel[rel]
        return float((self.pos[self.index[b]] - self.pos[self.index[a]]) @ d / (d @ d))

    def at(self, a, moves, top=3):
        """Items nearest to the position reached from `a` by the given moves [(rel, count), ...] - retrieval by
        location through the place neurons, not by any stored link."""
        target = self.pos[self.index[a]].clone()
        for rel, count in moves:
            target += count * self.rel[rel]
        return self.nearest(target, top)

    def nearest(self, target, top=3):
        """Place-cell readout: the items whose place codes overlap most with the code of the target position. For
        Gaussian place fields tiling the space, that overlap falls monotonically with distance (the product of two
        Gaussian bumps integrates to a Gaussian of their separation), so ranking by distance is the exact readout -
        and it keeps working at the lattice's edge, where individual place cells run out."""
        live = self.placed.nonzero().squeeze(1)
        if not len(live):
            return []
        dist = (self.pos[live] - target).norm(dim=1)
        order = dist.argsort()[:top]
        return [(self.names[int(live[i])], float(dist[i])) for i in order]

    def between(self, a, b, rel):
        """Items whose position along `rel` lies strictly between a and b."""
        lo, hi = sorted((0.0, self.steps(a, rel, b)))
        return [n for n in self.names if n not in (a, b) and self.knows(n) and lo + 0.5 < self.steps(a, rel, n) < hi - 0.5]

    def ranking(self, rel):
        """Every placed item ordered along `rel`."""
        d = self.rel[rel]
        live = [n for n in self.names if self.knows(n)]
        return sorted(live, key=lambda n: float(self.pos[self.index[n]] @ d))

    def footprint(self):
        return int(self.placed.sum()) * self.dim + len(self.rel) * self.dim

In [ ]:
%%writefile /kaggle/working/neuromoe/chat/language.py
"""Controlled English for NeuroMoE's cognitive map. A hand-written parser - no language model - that turns a fixed set
of sentence patterns into facts ("teach") and questions ("ask"). Anything else is refused rather than guessed at.

Facts      Alice is taller than Bob.          The cafe is north of the park.      Dan is 2 blocks east of Eve.
           Bob is shorter than Alice.         The bank is north-east of the park. Cara is more careful than Dan.
Questions  Is Alice taller than Cara?         Who is taller, Alice or Dan?        Who is the tallest?
           Rank everyone by height.           Who is between Alice and Dan?       How much taller is Alice than Dan?
           What is 2 blocks north of the park?  Is the bank north of the cafe?    Where is the cafe relative to the bank?
           Tell me about Alice.
Commands   sleep | what do you know | forget everything | help
"""
import re

NUMBERS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
           "ten": 10, "a": 1, "an": 1}
NUM = r"(?:\d+|one|two|three|four|five|six|seven|eight|nine|ten|a|an)"

# comparative -> (axis, sign). The axis is a dimension of its own; sign says which way along it the word points.
COMPARATIVES = {
    "taller": ("height", 1), "shorter": ("height", -1), "older": ("age", 1), "younger": ("age", -1),
    "bigger": ("size", 1), "larger": ("size", 1), "smaller": ("size", -1), "heavier": ("weight", 1),
    "lighter": ("weight", -1), "faster": ("speed", 1), "slower": ("speed", -1), "richer": ("wealth", 1),
    "poorer": ("wealth", -1), "stronger": ("strength", 1), "weaker": ("strength", -1), "smarter": ("smarts", 1),
    "better": ("quality", 1), "worse": ("quality", -1), "hotter": ("heat", 1), "colder": ("heat", -1),
    "warmer": ("heat", 1), "cooler": ("heat", -1), "louder": ("loudness", 1), "quieter": ("loudness", -1),
    "happier": ("happiness", 1), "sadder": ("happiness", -1), "cheaper": ("price", -1), "longer": ("length", 1),
}
SUPERLATIVE_EXCEPTIONS = {"best": "better", "worst": "worse"}
NOUNS = {"height": "height", "age": "age", "size": "size", "weight": "weight", "speed": "speed", "wealth": "wealth",
         "strength": "strength", "price": "price", "length": "length", "heat": "heat", "temperature": "heat",
         "quality": "quality", "loudness": "loudness", "happiness": "happiness"}

# spatial words -> (axis, sign). Every spatial axis lives in one shared space, so routes may mix them.
# Left/right are read as on a map (right = east); in front/behind and above/below get axes of their own.
DIRECTIONS = {"north": ("north", 1), "south": ("north", -1), "east": ("east", 1), "west": ("east", -1),
              "above": ("up", 1), "below": ("up", -1), "under": ("up", -1), "over": ("up", 1),
              "left": ("east", -1), "right": ("east", 1), "behind": ("front", -1), "in front": ("front", 1)}
SPATIAL_AXES = ("north", "east", "up", "front")
SAY_AXIS = {"north": ("north", "south"), "east": ("east", "west"), "up": ("above", "below"),
            "front": ("in front", "behind")}


def space_of(axis):
    return "place" if axis in SPATIAL_AXES else axis


def positive_word(axis):
    """The word for one step up an axis: 'height' -> 'taller', 'careful' -> 'more careful'."""
    for word, (a, s) in COMPARATIVES.items():
        if a == axis and s == 1:
            return word
    return axis if axis.endswith("er") else "more " + axis


CONTRACTIONS = {"'s": " is", "'re": " are", "'m": " am", "'ve": " have", "'ll": " will", "n't": " not",
                "'d": " would"}
LEADING = re.compile(r"^(?:so|well|ok(?:ay)?|right|and|but|then|anyway|hmm+|oh|yeah|actually|i think|"
                     r"i reckon|i guess|i'd say)[,\s]+", re.I)


def expand(text):
    """"he's the smartest" is a sentence. Refusing it because of an apostrophe is not a principled position.

    Contractions and the little words people start sentences with carry no content for this system, and the
    grammar it does understand is hiding behind them - so both come off before anything is parsed.
    """
    out = text
    for short, long_ in CONTRACTIONS.items():
        out = re.sub(re.escape(short) + r"\b", long_, out, flags=re.I)
    previous = None
    while previous != out:                     # "so, well, who is the laziest" - strip them all
        previous = out
        out = LEADING.sub("", out).lstrip()
    return out


def clean(text):
    text = expand(text.strip()).strip().rstrip("?.!;").strip()
    return re.sub(r"\s+", " ", text)


def sentences(text):
    """Split a message into sentences, so a whole story can be taught in one go."""
    return [s.strip() for s in re.split(r"(?<=[.!?;])\s+|\n+", text) if clean(s)]


POSSESSIVE = r"(?:the|a|an|my|your|his|her|our|their|its)"


def name(text):
    """'the Cafe' -> ('cafe', 'the Cafe'): a lookup key and how to say it.

    "my dog Rex" is Rex: when a phrase introduces something by category and then names it, the name is what the
    speaker will use next time, so that is what it gets filed under.
    """
    shown = text.strip().strip(",").strip()
    key = re.sub(r"^" + POSSESSIVE + r"\s+", "", shown.lower()).strip()
    words = shown.split()
    # "my dog Rex" -> Rex, but "the Cafe" stays as it is: it takes a category word in between for the last word
    # to be a name being introduced rather than part of the thing's own name
    if len(words) > 2 and words[-1][:1].isupper() and not words[-2][:1].isupper():
        key, shown = words[-1].lower(), words[-1]
    return key, shown


def comparative(word):
    """Known comparatives, plus any unknown '-er' word as a dimension of its own (so 'nicer' just works)."""
    w = word.lower()
    if w in COMPARATIVES:
        return COMPARATIVES[w]
    if len(w) > 4 and w.endswith("er") and w not in ("other", "never", "under", "over", "either", "rather"):
        return (w, 1)
    return None


def superlative(word):
    w = word.lower()
    if w in SUPERLATIVE_EXCEPTIONS:
        return comparative(SUPERLATIVE_EXCEPTIONS[w])
    if len(w) > 4 and w.endswith("est"):
        stem = w[:-3]
        for guess in (stem + "er", stem[:-1] + "ier" if stem.endswith("i") else None, stem + "r"):
            if guess and guess in COMPARATIVES:
                return COMPARATIVES[guess]
        return comparative(stem + "er")
    return None


def count(text):
    if not text:
        return 1
    text = text.strip().lower()
    return int(text) if text.isdigit() else NUMBERS.get(text)


COMPASS = r"(north|south|east|west)(?:[- ]?(north|south|east|west))?"
_ONE = r"(?:" + NUM + r"\s+)?(?:(?:blocks?|steps?|miles?|km|meters?|units?|streets?)\s+)?(?:north|south|east|west)(?:[- ]?(?:north|south|east|west))?"
MOVES = r"(" + _ONE + r"(?:(?:\s+and\s+|,\s*)" + _ONE + r")*)"


def moves(text):
    """'2 blocks north-east' / 'north' / '3 north and 1 east' -> [(axis, count), ...] or None."""
    total = {}
    for part in re.split(r"\s+and\s+|,\s*", text.strip().lower()):
        m = re.fullmatch(r"(?:(" + NUM + r")\s+)?(?:(?:blocks?|steps?|miles?|km|meters?|units?|streets?)\s+)?"
                         + COMPASS, part.strip())
        if not m:
            return None
        n = count(m.group(1))
        for word in filter(None, (m.group(2), m.group(3))):
            axis, sign = DIRECTIONS[word]
            total[axis] = total.get(axis, 0) + sign * n
    return [(a, c) for a, c in total.items() if c] or None


def _cmp(word):
    """A comparative phrase ('taller', 'more careful', 'less careful') -> (axis, sign, word) or None."""
    m = re.fullmatch(r"(more|less) (\w+)", word, re.I)
    if m:
        return m.group(2).lower(), 1 if m.group(1).lower() == "more" else -1, word.lower()
    c = comparative(word)
    return (c[0], c[1], word.lower()) if c else None


SPATIAL_WORDS = "|".join(sorted(DIRECTIONS, key=len, reverse=True))
HEDGE = r"(?:a bit |much |slightly |a lot |way |far |even )?"


PARENS = re.compile(r"\([^)]*\)")
CLAUSE = re.compile(r"[;:]|,\s+(?:which|who|where|and is|although|though|while)\b")
CATEGORY = re.compile(r"^(.{2,60}?)\s+(?:is|was|are|were)\s+(?:an?|the)\s+"
                      r"((?:[a-z][a-z'-]*)(?:\s+[a-z][a-z'-]*){0,3})", re.I)
# a phrase that stops mid-way is not the name of anything: "genre of popular" and "synthesis of a" were filling
# the map with things no page will ever be about, and every one of them wasted a reading
DANGLING = ("of", "a", "an", "the", "and", "or", "in", "to", "for", "with", "from", "by", "as", "that", "which")
GENERIC = {"type", "kind", "form", "sort", "number", "member", "part", "group", "term", "word", "name",
           "example", "variety", "one", "thing", "way", "series", "set", "range", "list", "lists"}


def phrase(text):
    """Trim a captured noun phrase back to something that could be the name of a thing, or None."""
    words = re.sub(r"\s+", " ", text.strip().strip(",.;:")).split()
    while words and words[-1].lower() in DANGLING:
        words.pop()                     # "genre of popular" -> "genre of" -> "genre"
    while words and words[0].lower() in ("a", "an", "the"):
        words.pop(0)
    if not words or len(" ".join(words)) < 3:
        return None
    if words[-1].lower() in GENERIC and len(words) == 1:
        return None                     # "a type", "a form": true of everything, and about nothing
    if len(words) > 4:
        words = words[:4]              # a name is short; anything longer is a sentence being mistaken for one
        while words and words[-1].lower() in DANGLING:
            words.pop()                # trimming before the cut is not enough: the cut can leave a new dangle
    return " ".join(words) if words else None
PLACE_IN = re.compile(r"\b(?:is|was)\s+(?:an?|the)?\s*[a-z' -]*\b(?:in|from|of)\s+"
                      r"([A-Z][\w'-]*(?:\s+[A-Z][\w'-]*){0,2})")
DID = re.compile(r"^(.{2,50}?)\s+(?:has |had |have )?(released|recorded|starred in|founded|created|wrote|"
                 r"developed|originated in|began in|emerged (?:in|from)|performed|appeared in|won|led|"
                 r"pioneered|popularised|popularized|influenced|enabled|provided|borrows from|"
                 r"consists of|combines|includes|features|uses|evolved from|developed from|spread to|"
                 r"became|grew out of|drew on|is rooted in|is based on|is used in|is played on)"
                 r"\s+(?:an?|the|his|her|their)?\s*"
                 r"([A-Za-z][\w' -]{2,40})", re.I)
ROLE_AS = re.compile(r"^(.{2,60}?)\s+(?:is|was)\s+(?:best )?known (?:as|for)\s+(?:an?|the|his|her)?\s*"
                     r"((?:[a-z][a-z'-]*)(?:\s+[a-z][a-z'-]*){0,2})", re.I)


def prose_facts(sentence, limit=2):
    """Facts out of ordinary written English - what an encyclopedia sentence is actually claiming.

    The conversational patterns are deliberately strict: a person saying "Rex is older than Max" means exactly
    that, and guessing would be worse than refusing. Prose is different - it is not addressed to this system,
    it is long, and the parts it asserts are buried in apposition and parentheses. So the reading path strips
    the decoration and takes only the two claim shapes that survive it: what a thing *is*, and where it is.

    Everything this produces is marked as read rather than told, and is encoded more weakly, because a sentence
    that happened to fit a pattern is much weaker evidence than a person telling you something.
    """
    s = PARENS.sub(" ", sentence)
    s = CLAUSE.split(s)[0].strip()
    # a pronoun subject is the page's subject; keeping it lets the body of an article contribute, which is
    # where nearly everything beyond "X is a Y" actually lives
    s = re.sub(r"^(It|They|This|These|He|She)\b", lambda mm: mm.group(1).lower(), s)
    s = re.sub(r"\s+", " ", s)
    out = []
    m = CATEGORY.match(s)
    if m:
        subject = phrase(m.group(1))
        # "an American actor and rapper" is two claims, not one: a coordination is a list of categories, and
        # keeping them separate is what later lets one of them be the link to something else
        for part in re.split(r"\band\b|,", m.group(2)):
            category = phrase(part)
            if subject and category and len(subject) <= 60 and subject.lower() != category.lower():
                out.append({"kind": "fact", "a": name(subject), "b": name(category),
                            "moves": [("kind", 1)], "word": "a kind of", "how": "read"})
    m = ROLE_AS.match(s)
    if m and len(out) < limit:
        who, role = phrase(m.group(1)), phrase(m.group(2))
        if who and role:
            out.append({"kind": "fact", "a": name(who), "b": name(role),
                        "moves": [("kind", 1)], "word": "known as", "how": "read"})
    m = PLACE_IN.search(s)
    if m and out and len(out) < limit:
        where = phrase(m.group(1))
        if where and where.lower() != out[0]["a"][0]:
            out.append({"kind": "fact", "a": out[0]["a"], "b": name(where),
                        "moves": [("origin", 1)], "word": "from", "how": "read"})
    # what a thing does or made: the relations that actually connect people to fields, which categories alone
    # never will - "Will Smith released an album" is what puts him anywhere near music
    m = DID.search(s)
    if m and len(out) < limit and not re.search(r"\b(is|was|are|were)\b", m.group(1)):
        who, what = phrase(m.group(1)), phrase(re.split(r"\band\b", m.group(3))[0])
        if who and what and who.lower() != what.lower():
            out.append({"kind": "fact", "a": name(who), "b": name(what),
                        "moves": [("involved", 1)], "word": m.group(2).lower(), "how": "read"})
    return out[:limit]


def parse(text):
    """-> dict with 'kind' in {fact, ask, command, unknown} and whatever that kind needs."""
    t = clean(text)
    low = t.lower()

    if low in ("sleep", "consolidate", "go to sleep", "rest", "dream"):
        return {"kind": "command", "do": "sleep"}
    if low in ("forget everything", "reset", "clear", "start over"):
        return {"kind": "command", "do": "reset"}
    if low in ("what do you know", "show the map", "show map", "status", "what do you remember", "summary"):
        return {"kind": "command", "do": "summary"}
    if low in ("help", "what can i say", "examples", "what can you do"):
        return {"kind": "command", "do": "help"}

    # questions -------------------------------------------------------------------------------------------
    m = re.fullmatch(r"is (.+?) " + HEDGE + r"((?:more|less) \w+|\w+) than (.+)", t, re.I)
    if m and _cmp(m.group(2)):
        axis, sign, word = _cmp(m.group(2))
        return {"kind": "ask", "q": "compare", "a": name(m.group(1)), "b": name(m.group(3)), "axis": axis,
                "sign": sign, "word": word}
    m = re.fullmatch(r"is (.+?) (?:directly |exactly )?" + MOVES + r" of (.+)", t, re.I)
    if m and moves(m.group(2)):
        exact = bool(re.match(r"\s*" + NUM + r"\s", m.group(2), re.I) and not re.match(r"\s*an?\s", m.group(2), re.I))
        return {"kind": "ask", "q": "direction", "a": name(m.group(1)), "b": name(m.group(3)),
                "moves": moves(m.group(2)), "exact": exact, "word": m.group(2).lower()}
    m = re.fullmatch(r"is (.+?) (?:to the |directly |just )?(" + SPATIAL_WORDS + r")(?: of)? (.+)", t, re.I)
    if m:
        axis, sign = DIRECTIONS[m.group(2).lower()]
        return {"kind": "ask", "q": "direction", "a": name(m.group(1)), "b": name(m.group(3)),
                "moves": [(axis, sign)], "exact": False, "word": m.group(2).lower()}
    m = re.fullmatch(r"(?:who|which|what)(?: one)? is ((?:more|less) \w+|\w+),? (.+?),? or (.+)", t, re.I)
    if m and _cmp(m.group(1)):
        axis, sign, word = _cmp(m.group(1))
        return {"kind": "ask", "q": "which", "a": name(m.group(2)), "b": name(m.group(3)), "axis": axis,
                "sign": sign, "word": word}
    m = re.fullmatch(r"is (.+?) the (most|least) (\w+)(?: one)?", t, re.I)
    if m:
        return {"kind": "ask", "q": "extreme", "a": name(m.group(1)), "axis": m.group(3).lower(),
                "sign": 1 if m.group(2).lower() == "most" else -1, "word": f"{m.group(2)} {m.group(3)}".lower()}
    m = re.fullmatch(r"is (.+?) the (\w+)(?: one)?", t, re.I)
    if m and superlative(m.group(2)):
        axis, sign = superlative(m.group(2))
        return {"kind": "ask", "q": "extreme", "a": name(m.group(1)), "axis": axis, "sign": sign,
                "word": m.group(2).lower()}
    m = re.fullmatch(r"(?:who|which|what)(?: one)? is the (most|least) (\w+)(?: one)?", t, re.I)
    if m:
        return {"kind": "ask", "q": "extreme", "axis": m.group(2).lower(),
                "sign": 1 if m.group(1).lower() == "most" else -1, "word": f"{m.group(1)} {m.group(2)}".lower()}
    m = re.fullmatch(r"(?:who|which|what)(?: one)? is the (\w+)(?: one)?", t, re.I)
    if m and superlative(m.group(1)):
        axis, sign = superlative(m.group(1))
        return {"kind": "ask", "q": "extreme", "axis": axis, "sign": sign, "word": m.group(1).lower()}
    m = re.fullmatch(r"(?:rank|order|sort|list)(?: everyone| everything| them| all| the \w+)?(?: by| from)? (.+)", t,
                     re.I)
    if m:
        what = re.sub(r"\s+to\s+\w+$", "", m.group(1).lower().strip())  # "tallest to shortest" -> "tallest"
        found = None
        if what in NOUNS:
            found = (NOUNS[what], 1)
        elif re.fullmatch(r"(most|least) (\w+)", what):
            mm = re.fullmatch(r"(most|least) (\w+)", what)
            found = (mm.group(2), 1 if mm.group(1) == "most" else -1)
        else:
            found = superlative(what) or comparative(what) or ((what, 1) if re.fullmatch(r"\w+", what) else None)
        if found:
            return {"kind": "ask", "q": "rank", "axis": found[0], "sign": found[1]}
    m = re.fullmatch(r"(?:who|what)(?: is| are|'s)? between (.+?) and (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "between", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"how (?:much|many \w+|far) ((?:more|less) \w+|\w+) is (.+?) than (.+)", t, re.I)
    if m and _cmp(m.group(1)):
        axis, sign, word = _cmp(m.group(1))
        return {"kind": "ask", "q": "howmuch", "a": name(m.group(2)), "b": name(m.group(3)), "axis": axis,
                "sign": sign, "word": word}
    m = re.fullmatch(r"(?:what|who|which \w+)(?: is|'s| lies)? " + MOVES + r" of (.+)", t, re.I)
    if m and moves(m.group(1)):
        return {"kind": "ask", "q": "at", "a": name(m.group(2)), "moves": moves(m.group(1)), "word": m.group(1).lower()}
    m = re.fullmatch(r"where is (.+?)(?: relative to| compared to| from| in relation to) (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "where", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"how far is (.+?) from (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "where", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"where is (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "about", "a": name(m.group(1))}
    # "is Rex a mammal", "are dogs animals": a question about category membership, which is only answerable
    # by chaining - Rex is a dog, a dog is a mammal - and so belongs to the reasoning loop rather than to a
    # single retrieval. Answering it from one hop is how "is Rex a mammal" came back "Rex is a dog".
    m = re.fullmatch(r"(?:is|are|was|were)\s+(.+?)\s+(?:an?|the)\s+(?:kind|type|sort)\s+of\s+(.+)", t, re.I) or \
        re.fullmatch(r"(?:is|are|was|were)\s+(.+?)\s+(?:an?)\s+(.+)", t, re.I) or \
        re.fullmatch(r"(?:is|are|was|were)\s+(.+?)\s+(\w+s)\b", t, re.I)
    if m and len(m.group(1).split()) <= 4 and len(m.group(2).split()) <= 3:
        return {"kind": "ask", "q": "member", "a": name(m.group(1).strip(" ?.")),
                "b": name(m.group(2).strip(" ?."))}
    # asking how two things connect: the question this system answers as a route through the map
    m = re.fullmatch(r"(?:what(?:'s| is)? the )?(?:relation(?:ship)?|connection|link) between (.+?) and (.+)", t,
                     re.I)
    if m:
        return {"kind": "ask", "q": "relate", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"(?:how (?:does|did|do|is) )?(.+?) (?:relate[sd]? to|connected to|affect(?:ed|s)?|"
                     r"influence[sd]?|shape[sd]?|bears? on|bore on|ha[sd] to do with|"
                     r"compare[sd]? (?:to|with)|inform[sd]?) (.+)", t, re.I)
    if m and len(m.group(1).split()) <= 8:
        return {"kind": "ask", "q": "relate", "a": name(m.group(1)), "b": name(m.group(2))}
    m = re.fullmatch(r"(?:write |give me |produce )?(?:an? )?(?:\w+ )*?(?:analysis|essay|account|piece) of (.+?) "
                     r"on (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "relate", "a": name(m.group(1)), "b": name(m.group(2)), "asked_for": "essay"}
    # "google what a platypus is", "look up Kuala Lumpur", "search for jazz history": an explicit request to go
    # and read. It was falling through to "I don't know anything about please", which is refusing an
    # instruction by misreading it as a claim.
    m = re.fullmatch(r"(?:please\s+|can you\s+|could you\s+|would you\s+|go\s+(?:and\s+)?)*"
                     r"(?:google|search(?:\s+(?:for|up|the web for|online for))?|look\s+up|look\s+into|"
                     r"find\s+out(?:\s+about)?|find\s+(?:info|information)\s+(?:on|about)|research|check)\s+"
                     r"(?:what|who|where|when|how|why|whether|if)?\s*(?:an?\s+|the\s+)?(.+?)"
                     r"(?:\s+(?:is|are|was|were|means?|does|do))?\s*[?.!]*", t, re.I)
    if m and m.group(1).strip():
        return {"kind": "ask", "q": "about", "a": name(m.group(1).strip(" ?.")), "lookup": True, "question": t}
    # where something is, asked about something the person has not placed for us: "where is Kuala Lumpur",
    # "what country is Beijing in", "which state is Philadelphia in". This is a question for the reader, not
    # for the map, and the answer is a place - so it is marked as one, and the reply leads with the place
    # rather than with a definition that mentions it somewhere in the middle.
    m = re.fullmatch(r"(?:where(?:'s| is| are)|where abouts is) (.+?)(?: located| situated)?", t, re.I) or \
        re.fullmatch(r"(?:what|which|in what|in which) (country|state|city|continent|region|province) "
                     r"(?:is|are) (.+?) (?:in|located in|part of)", t, re.I)
    if m and not re.search(r"\b(of|from|north|south|east|west)\b", t.split(" is ")[-1] if " is " in t else "",
                           re.I):
        target = m.group(m.lastindex)
        if len(target.split()) <= 5:
            unit = m.group(1).lower() if m.lastindex == 2 else "place"
            return {"kind": "ask", "q": "about", "a": name(target), "asked": "where", "unit": unit}
    m = re.fullmatch(r"(?:what do you know about|tell me about|describe|who is|what is) (.+)", t, re.I)
    if m:
        return {"kind": "ask", "q": "about", "a": name(m.group(1))}

    # an instruction is a thing to do, not a thing to look up. "Write an essay on X" was being matched as a
    # question about the phrase and answered by recalling that somebody had said it, which is the most annoying
    # failure a system can have: it heard the request and filed it instead of obeying it.
    m = re.fullmatch(r"(?:please\s+)?(?:can you|could you|would you|will you)?\s*"
                     r"(write|compose|draft|produce|give me|make me|put together|do)\s+"
                     r"(?:me\s+)?(?:an?|the|some)?\s*"
                     # "a short essay", "a quick write-up", "a proper analysis": an adjective between the
                     # article and the thing asked for was enough to make this miss, and a missed instruction
                     # is filed as a remark and read back three turns later, which is the failure that started
                     # all of this. Let a couple of modifiers through.
                     r"(?:[a-z-]+\s+){0,3}?(essay|piece|write-?up|summary|account|note|paragraph|"
                     r"explanation|analysis|report)\s*(?:on|about|of|for)?\s*(.*)", t, re.I)
    if m:
        target = re.sub(r"^(the|a|an)\s+", "", m.group(3).strip(" .?"), flags=re.I)
        # "an essay on just KL": the qualifier says how narrowly to write, and is not part of the name
        target = re.sub(r"^(just|only|simply|purely)\s+", "", target, flags=re.I)
        anything = not target or re.fullmatch(r"any(thing| topic| subject)?|whatever|something", target, re.I)
        return {"kind": "instruct", "deliverable": m.group(2).lower().replace("write-up", "writeup"),
                "topic": None if anything else name(target), "said": t}
    m = re.fullmatch(r"(?:please\s+)?(?:can you|could you)?\s*(explain|summarise|summarize|describe|outline|"
                     r"list)\s+(?:me\s+)?(?:an?|the)?\s*(?:what you know about\s+)?(.+)", t, re.I)
    if m:
        kind = {"list": "list", "summarise": "summary", "summarize": "summary",
                "outline": "summary"}.get(m.group(1).lower(), "explanation")
        return {"kind": "instruct", "deliverable": kind, "topic": name(m.group(2).strip(" .?")), "said": t}

    # "X is the tallest" is a fact about X and everyone else, not the introduction of a thing called tallest
    m = re.fullmatch(r"(.+?) (?:is|are|was|were) (?:the )?(most|least) (\w+)(?: of .+)?", t, re.I)
    if m:
        return {"kind": "fact", "a": name(m.group(1)), "extreme": True, "axis": m.group(3).lower(),
                "sign": 1 if m.group(2).lower() == "most" else -1, "word": f"{m.group(2)} {m.group(3)}".lower()}
    m = re.fullmatch(r"(.+?) (?:is|are|was|were) (?:the )?(\w+est)(?: of .+| one)?", t, re.I)
    if m and superlative(m.group(2)):
        axis, sign = superlative(m.group(2))
        return {"kind": "fact", "a": name(m.group(1)), "extreme": True, "axis": axis, "sign": sign,
                "word": m.group(2).lower()}
    # "Rex, Max and Milo are dogs" / "I have three dogs: Rex, Max and Milo" - several things introduced at once,
    # which is how anyone actually introduces a set of things
    m = re.fullmatch(r"(?:i (?:have got|have|got|own|keep)|there (?:are|is)) "
                     r"(?:\d+ |one |two |three |four |five |six |a |an |some |several |many )?"
                     r"([a-z][\w' -]*?)s?[:,]? (?:called |named )?(.+)", t, re.I)
    if m and re.search(r",| and ", m.group(2)):
        members = [name(x) for x in re.split(r",| and ", m.group(2)) if x.strip()]
        if len(members) > 1:
            return {"kind": "fact", "group": members, "b": name(m.group(1)), "moves": [("kind", 1)],
                    "word": "a kind of", "mine": True}
    m = re.fullmatch(r"(.+?) (?:are|were) (?:a |an )?([a-z][\w' -]*)", t, re.I)
    if m and re.search(r",| and ", m.group(1)):
        members = [name(x) for x in re.split(r",| and ", m.group(1)) if x.strip()]
        if len(members) > 1:
            return {"kind": "fact", "group": members, "b": name(re.sub(r"s$", "", m.group(2))),
                    "moves": [("kind", 1)], "word": "a kind of"}

    # facts -----------------------------------------------------------------------------------------------
    m = re.fullmatch(r"(.+?) (?:is|are|was|were) " + HEDGE + r"((?:more|less) \w+|\w+) than (.+)", t, re.I)
    if m and _cmp(m.group(2)):
        axis, sign, word = _cmp(m.group(2))
        return {"kind": "fact", "a": name(m.group(1)), "b": name(m.group(3)), "moves": [(axis, sign)], "word": word}
    m = re.fullmatch(r"(.+?) (?:is|are) (?:directly |just )?" + MOVES + r" of (.+)", t, re.I)
    if m and moves(m.group(2)):
        return {"kind": "fact", "a": name(m.group(1)), "b": name(m.group(3)), "moves": moves(m.group(2)),
                "word": m.group(2).lower()}
    m = re.fullmatch(r"(.+?) (?:is|are) (?:directly |just |to the )?(" + SPATIAL_WORDS + r")(?: of)? (.+)", t, re.I)
    if m:
        axis, sign = DIRECTIONS[m.group(2).lower()]
        return {"kind": "fact", "a": name(m.group(1)), "b": name(m.group(3)), "moves": [(axis, sign)],
                "word": m.group(2).lower()}
    # "Rex is a dog" - category rather than comparison. Last, so it never shadows "Rex is older than Max".
    # This is also the one pattern that fires often in encyclopedia prose, which is how anything read from the
    # web gets into the map at all.
    m = re.fullmatch(r"(.+?) (?:is|are|was|were) (?:an?|the) ([a-z][\w' -]*)", t, re.I)
    if m and len(m.group(1).split()) <= 5 and "than" not in m.group(2):
        return {"kind": "fact", "a": name(m.group(1)), "b": name(re.sub(r"^(a|an|the) ", "", m.group(2))),
                "moves": [("kind", 1)], "word": "a kind of"}
    # "dogs are mammals" - the same claim in the plural, where English drops the article. Without this the
    # second half of every chain went unheard: "Rex is a dog" landed and "dogs are mammals" did not.
    m = re.fullmatch(r"([a-z][\w' -]*?)\s+are\s+([a-z][\w' -]*)", t, re.I)
    if m and len(m.group(1).split()) <= 4 and len(m.group(2).split()) <= 3 \
            and not re.search(r"\b(than|there|here|you|they|we|it|these|those|what|who)\b", t, re.I):
        return {"kind": "fact", "a": name(m.group(1)), "b": name(m.group(2)),
                "moves": [("kind", 1)], "word": "a kind of"}

    return {"kind": "unknown"}

In [ ]:
%%writefile /kaggle/working/neuromoe/chat/mapsystem.py
"""The conversational wrapper around NeuroMoE's cognitive map. It holds one map per kind of relation - one for
places (north/east/up/front share a space, so routes can mix them), and one per comparative dimension (height, age,
...), because being taller says nothing about being older. Every answer is read off the maps; this file only turns
parsed sentences into observe / sleep / steps / nearest calls and turns the numbers back into words.

Nothing is pre-loaded: the maps start empty and learn one-shot from what they are told. Settings are the ones tuned
on validation episodes in the Kaggle run (results/kaggle_map/results/map/neuromoe_map_config.json).
"""
import json
import os
import re
import sys

import torch

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, ROOT)
from cogmap import CognitiveMap, PlaceCells  # noqa: E402

import language as L  # noqa: E402

CONFIG = os.path.join(ROOT, "results", "kaggle_map", "results", "map", "neuromoe_map_config.json")
PLACE_DIM, LINE_DIM = 4, 2  # places: north, east, up, front; each comparative dimension: its axis plus slack

HELP = """I learn from what you tell me - nothing is pre-loaded. I understand a small, fixed kind of English:

Teach me:  "Alice is taller than Bob."  "Cara is more careful than Dan."
           "The cafe is north of the park."  "The bank is 2 blocks east of the cafe."  "The lamp is left of the bed."
Ask me:    "Is Alice taller than Dan?"  "Who is taller, Bob or Dan?"  "Who is the tallest?"  "Rank everyone by height."
           "Who is between Alice and Dan?"  "How much taller is Alice than Dan?"
           "What is 2 blocks north of the park?"  "Is the bank north-east of the park?"  "Where is the bank from the park?"
           "Tell me about Alice."
Commands:  "sleep" (consolidate), "what do you know", "forget everything".

You can paste several sentences at once. I only ever answer from what you have told me; if two things were never
linked by any chain of facts, I say I don't know."""


# The settings chosen on validation episodes in the pseudospatial Kaggle run, kept here so the map is the same
# one that was measured even on a machine without that run's output folder. results/kaggle_map/.../
# neuromoe_map_config.json is the same numbers; if it is present it wins, so a re-tune takes effect.
TUNED = {"model": "NeuroMoE cognitive map",
         "map_config": {"lr": 0.3, "rel_lr": 0.0, "replay": 40, "robust": 0.3, "sleep": 300, "tol": 0.3},
         "place_cells": {"per_dim": 48, "span": 24.0}}


def load_config():
    if not os.path.exists(CONFIG):
        return TUNED
    with open(CONFIG, encoding="utf-8") as f:
        return json.load(f)


class Brain:
    def __init__(self, config=None, seed=0):
        self.config = config or load_config()
        self.seed = seed
        self.reset()

    # ------------------------------------------------------------------ state
    def reset(self):
        self.maps = {}        # space -> CognitiveMap
        self.shown = {}       # key -> how the user wrote it
        self.words = {}       # comparative axis -> the word the user used for "more"
        self.log = []         # every sentence taught, in order (the model's whole knowledge, replayable)
        self.dirty = set()    # spaces with new facts since the last sleep

    def _map(self, space):
        if space not in self.maps:
            c = self.config["map_config"]
            dim = PLACE_DIM if space == "place" else LINE_DIM
            pc = self.config.get("place_cells", {})
            place = PlaceCells(dim=dim, per_dim=pc.get("per_dim", 48), span=pc.get("span", 24.0),
                               seed=self.seed + len(self.maps))
            self.maps[space] = CognitiveMap(dim=dim, lr=c["lr"], rel_lr=c["rel_lr"], replay=c["replay"],
                                            robust=c["robust"], place=place, seed=self.seed + len(self.maps))
        return self.maps[space]

    def _say(self, key):
        return self.shown.get(key) or getattr(self, "_asked", {}).get(key, key)

    def _connected(self, space, a, b):
        m = self.maps.get(space)
        return bool(m and m.knows(a) and m.knows(b) and m.island[m.index[a]] == m.island[m.index[b]])

    def _settle(self, space):
        """Sleep before answering if new facts arrived: every episode replayed, the map relaxes as a whole."""
        if space in self.dirty and space in self.maps:
            self.maps[space].sleep(self.config["map_config"]["sleep"], tolerance=self.config["map_config"]["tol"])
            self.dirty.discard(space)
            return True
        return False

    def _offset(self, space, a, b):
        """Steps from a to b along every relation of the space, e.g. {'north': 2.0, 'east': -1.0}."""
        m = self.maps[space]
        return {r: m.steps(a, r, b) for r in m.rel}

    # ------------------------------------------------------------------ dialogue
    def say(self, text):
        """-> list of {sentence, parsed, reply, kind}. Several sentences can arrive in one message."""
        out = []
        for s in L.sentences(text) or [text]:
            p = L.parse(s)
            try:
                reply = self._handle(p, s)
            except Exception as e:  # never let one odd sentence kill the conversation
                reply = f"(internal error: {type(e).__name__}: {e})"
            reply = reply[:1].upper() + reply[1:]
            out.append({"sentence": s, "parsed": describe(p), "reply": reply, "kind": p["kind"]})
        return out

    def _handle(self, p, sentence):
        k = p["kind"]
        if k == "command":
            return getattr(self, "_cmd_" + p["do"])()
        if k == "fact":
            self.log.append(sentence)
            return self._teach(p)
        if k == "ask":
            self._asked = {x[0]: article(x[1]) for x in (p.get("a"), p.get("b")) if x}
            return getattr(self, "_ask_" + p["q"])(p)
        return ("I didn't understand that. I only follow a small set of sentence shapes - facts like "
                "\"A is taller than B\" or \"A is north of B\", and questions about them. Type \"help\" for the list.")

    # ------------------------------------------------------------------ teaching
    def _teach(self, p):
        (ka, sa), (kb, sb) = p["a"], p["b"]
        if ka == kb:
            return "That relates something to itself, so I can't place it."
        self.shown.setdefault(ka, article(sa))
        self.shown.setdefault(kb, article(sb))
        spaces = {L.space_of(axis) for axis, _ in p["moves"]}
        space = spaces.pop()
        moves = tuple(p["moves"])
        if space != "place":
            axis, sign = moves[0]
            if sign == 1 or axis not in self.words:
                self.words.setdefault(axis, p["word"] if sign == 1 else L.positive_word(axis))
        m = self._map(space)
        note = ""
        if self._connected(space, kb, ka):  # already inferable: does the new fact agree with what I inferred?
            self._settle(space)
            predicted = self._offset(space, kb, ka)
            for axis, c in moves:
                have = predicted.get(axis, 0.0)
                if c * have < -0.5:
                    note = (f" That contradicts what I had inferred, so I'll weigh it against the other facts "
                            f"(one fact against many gets outvoted when I sleep).")
                    break
            else:
                note = " I could already infer that - it fits."
        new = [x for x in (ka, kb) if not m.knows(x)]
        joins = not new and not self._connected(space, ka, kb)
        m.observe_moves(kb, moves, ka)  # a = b + moves
        if joins:
            self.dirty.add(space)
            size = sum(self._connected(space, ka, n) for n in m.names)
            return (f"Got it - that links two separate groups, so I shifted one to fit and merged them. "
                    f"{size} things are now on one map, and I can compare any two of them.")
        self.dirty.add(space)
        if len(new) == 2:
            where = "a new, separate group - I'll link it to the rest once a fact connects them"
            if not any(m.knows(n) for n in m.names if n not in (ka, kb)):
                where = "the start of a new map" if space == "place" else f"the start of my {self._dim(axis)} scale"
            return f"Got it. {sa} and {sb} are {where}.{note}"
        if len(new) == 1:
            return f"Got it - I placed {self._say(new[0])} from that one fact.{note}"
        return f"Got it.{note}"

    def _dim(self, axis):
        return {"smarts": "smartness"}.get(axis, axis)

    # ------------------------------------------------------------------ questions
    def _unknown(self, *keys, space=None):
        missing = [self._say(k) for k in keys if not any(m.knows(k) for m in self.maps.values())]
        if missing:
            return f"I don't know anything about {' or '.join(missing)} yet."
        return None

    def _link_check(self, space, a, b, what):
        """None if a and b can be compared in this space, otherwise the honest reason why not."""
        m = self.maps.get(space)
        if m is None:
            return f"I don't know - you haven't told me anything about {what} yet."
        outside = [self._say(x) for x in (a, b) if not m.knows(x)]
        if outside:
            who = " or ".join(outside)
            return (f"I don't know - you haven't told me where {who} is." if space == "place" else
                    f"I don't know - you haven't told me how {who} compares by {what}.")
        return None if self._connected(space, a, b) else self._not_linked(a, b, what)

    def _not_linked(self, a, b, what):
        how = "in space" if what == "where things are" else f"by {what}"
        return (f"I don't know - nothing you've told me links {self._say(a)} and {self._say(b)} {how}. "
                f"They're on separate islands of my map.")

    def _confidence(self, s):
        s = abs(s)
        return "" if s >= 1.5 else " (I think so - they're close)" if s >= 0.5 else ""

    def _ask_compare(self, p):
        (ka, _), (kb, _) = p["a"], p["b"]
        axis, sign, word = p["axis"], p["sign"], p["word"]
        miss = self._unknown(ka, kb)
        if miss:
            return miss
        space = L.space_of(axis)
        why = self._link_check(space, ka, kb, self._dim(axis))
        if why:
            return why
        self._settle(space)
        s = self.maps[space].steps(kb, axis, ka) * sign  # >0: a is `word` than b
        if abs(s) < 0.5:
            return f"I can't tell - {self._say(ka)} and {self._say(kb)} land in about the same place ({s:+.1f} steps)."
        if s > 0:
            return f"Yes, {self._say(ka)} is {word} than {self._say(kb)}.{self._confidence(s)}"
        return f"No - {self._say(kb)} is {word} than {self._say(ka)}.{self._confidence(s)}"

    def _ask_which(self, p):
        (ka, _), (kb, _) = p["a"], p["b"]
        axis, sign, word = p["axis"], p["sign"], p["word"]
        miss = self._unknown(ka, kb)
        if miss:
            return miss
        space = L.space_of(axis)
        why = self._link_check(space, ka, kb, self._dim(axis))
        if why:
            return why
        self._settle(space)
        s = self.maps[space].steps(kb, axis, ka) * sign
        if abs(s) < 0.5:
            return f"I can't tell them apart ({s:+.1f} steps)."
        win, lose = (ka, kb) if s > 0 else (kb, ka)
        return f"{self._say(win)} is {word} than {self._say(lose)}.{self._confidence(s)}"

    def _groups(self, space):
        m = self.maps[space]
        groups = {}
        for n in m.names:
            if m.knows(n):
                groups.setdefault(m.island[m.index[n]], []).append(n)
        return list(groups.values())

    def _ask_extreme(self, p):
        axis, sign, word = p["axis"], p["sign"], p["word"]
        space = L.space_of(axis)
        if space not in self.maps or axis not in self.maps[space].rel:
            return f"You haven't told me anything about who is {L.positive_word(axis)} yet."
        self._settle(space)
        m = self.maps[space]
        tops = []
        for g in self._groups(space):
            order = sorted(g, key=lambda n: -sign * float(m.pos[m.index[n]] @ m.rel[axis]))
            gap = abs(m.steps(order[1], axis, order[0])) if len(order) > 1 else 9
            tops.append((order[0], gap, order))
        asked = p["a"][0] if p.get("a") else None     # "is Rex the oldest?" - the same question, answered yes or no
        if len(tops) == 1:
            top, gap, order = tops[0]
            hedge = "" if gap >= 0.5 else f" - but {self._say(order[1])} is about level, so I'm not sure"
            if asked:
                if asked not in order:
                    return f"I don't know - you haven't told me how {self._say(asked)} compares by {self._dim(axis)}."
                if asked == top:
                    return f"Yes, {self._say(asked)} is the {word}{hedge}."
                return (f"No - {self._say(top)} is, and {self._say(asked)} is "
                        f"{ordinal(order.index(asked) + 1)} of {len(order)}.")
            return f"{self._say(top)} is the {word}{hedge}."
        if asked:
            for top, gap, order in tops:
                if asked in order:
                    rank = ordinal(order.index(asked) + 1)
                    return (f"Of the {len(order)} I can compare with {self._say(asked)}, it is {rank}"
                            + ("" if asked == top else f" - {self._say(top)} is the {word}")
                            + f". My facts form {len(tops)} groups I can't compare with each other, so I can't "
                              f"say outright.")
        names = [self._say(t[0]) for t in tops]
        return (f"I can't say for sure: my facts form {len(tops)} separate groups, and the top of each is "
                f"{', '.join(names[:-1])} or {names[-1]}. Tell me how the groups relate and I'll know.")

    def _ask_rank(self, p):
        axis, sign = p["axis"], p["sign"]
        space = L.space_of(axis)
        if space not in self.maps or axis not in self.maps[space].rel:
            return f"You haven't told me anything about {axis} yet."
        self._settle(space)
        m = self.maps[space]
        word = self.words.get(axis, L.positive_word(axis)) if space != "place" else L.SAY_AXIS[axis][0]
        lines = []
        for g in self._groups(space):
            order = sorted(g, key=lambda n: -sign * float(m.pos[m.index[n]] @ m.rel[axis]))
            parts = [self._say(order[0])]
            for prev, n in zip(order, order[1:]):
                gap = abs(m.steps(n, axis, prev))
                parts.append(("ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â°ÃƒÆ’Ã¢â‚¬Â¹ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â  " if gap < 0.5 else "> ") + self._say(n))
            lines.append(" ".join(parts))
        if space == "place":
            pos, neg = L.SAY_AXIS[axis]
            head = f"From {pos}most to {neg}most" if axis in ("north", "east") else f"From most {pos} to most {neg}"
            if sign < 0:
                head = f"From {neg}most to {pos}most" if axis in ("north", "east") else f"From most {neg} to most {pos}"
        else:
            head = f"By {self._dim(axis)}, {word} first" if sign == 1 else f"By {self._dim(axis)}, {word} last"
        if len(lines) == 1:
            return f"{head}: {lines[0]}"
        return head + f" - {len(lines)} separate groups I can't compare with each other:\n" + "\n".join(
            "  ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ " + l for l in lines)

    def _ask_between(self, p):
        (ka, _), (kb, _) = p["a"], p["b"]
        miss = self._unknown(ka, kb)
        if miss:
            return miss
        found = []
        for space, m in self.maps.items():
            if not self._connected(space, ka, kb):
                continue
            self._settle(space)
            pa, pb = m.pos[m.index[ka]], m.pos[m.index[kb]]
            d = pb - pa
            L2 = float(d @ d)
            if L2 < 0.25:
                continue
            for n in m.names:
                if n in (ka, kb) or not self._connected(space, ka, n):
                    continue
                v = m.pos[m.index[n]] - pa
                t = float(v @ d) / L2
                off = float((v - t * d).norm())
                length = L2 ** 0.5
                if 0.5 / length < t < 1 - 0.5 / length and off < 0.5:
                    found.append((t, space, n))
        if not found:
            if not any(self._connected(s, ka, kb) for s in self.maps):
                return self._not_linked(ka, kb, "at all")
            return f"Nothing I know of lies between {self._say(ka)} and {self._say(kb)}."
        by = {}
        for _, space, n in sorted(found):
            by.setdefault(space, []).append(self._say(n))
        parts = [f"{', '.join(v)}" + ("" if s == "place" else f" (by {s})") for s, v in by.items()]
        return "Between them: " + "; ".join(parts) + "."

    def _ask_howmuch(self, p):
        (ka, _), (kb, _) = p["a"], p["b"]
        axis, sign, word = p["axis"], p["sign"], p["word"]
        miss = self._unknown(ka, kb)
        if miss:
            return miss
        space = L.space_of(axis)
        why = self._link_check(space, ka, kb, self._dim(axis))
        if why:
            return why
        self._settle(space)
        s = self.maps[space].steps(kb, axis, ka) * sign
        n = round(abs(s))
        unit = "step" if n == 1 else "steps"
        if s < -0.5:
            return f"{self._say(ka)} isn't {word} - {self._say(kb)} is, by about {n} {unit}."
        if abs(s) < 0.5:
            return f"About the same ({s:+.1f} steps)."
        return (f"About {n} {unit} (exactly {s:.2f}). A step is one \"{word} than\" fact - you told me orderings, "
                f"not amounts, so I count links in the chain.")

    def _describe_offset(self, off):
        parts = []
        for axis in L.SPATIAL_AXES:
            v = off.get(axis, 0.0)
            if abs(v) >= 0.5:
                n = round(abs(v))
                pos, neg = L.SAY_AXIS[axis]
                w = pos if v > 0 else neg
                parts.append(f"{n} {w}" if axis in ("north", "east") else f"{n} step{'s' * (n != 1)} {w}")
        return " and ".join(parts) if parts else "in the same spot"

    def _ask_where(self, p):
        (ka, _), (kb, _) = p["a"], p["b"]
        miss = self._unknown(ka, kb)
        if miss:
            return miss
        why = self._link_check("place", ka, kb, "where things are")
        if why:
            return why
        self._settle("place")
        off = self._offset("place", kb, ka)
        return f"{self._say(ka)} is {self._describe_offset(off)} of {self._say(kb)}.".replace(
            "in the same spot of", "in the same spot as")

    def _ask_direction(self, p):
        (ka, _), (kb, _) = p["a"], p["b"]
        miss = self._unknown(ka, kb)
        if miss:
            return miss
        why = self._link_check("place", ka, kb, "where things are")
        if why:
            return why
        self._settle("place")
        off = self._offset("place", kb, ka)
        actual = self._describe_offset(off)
        if p["exact"]:
            want = dict(p["moves"])
            ok = all(abs(off.get(a, 0.0) - want.get(a, 0)) < 0.5 for a in set(off) | set(want))
        else:
            ok = all(off.get(a, 0.0) * c > 0.5 for a, c in p["moves"])
        verdict = "Yes" if ok else "No"
        return f"{verdict} - {self._say(ka)} is {actual} of {self._say(kb)}.".replace(
            "in the same spot of", "in the same spot as")

    def _ask_at(self, p):
        (ka, _) = p["a"]
        miss = self._unknown(ka)
        if miss:
            return miss
        if "place" not in self.maps or not self.maps["place"].knows(ka):
            return f"I don't know where {self._say(ka)} is."
        self._settle("place")
        m = self.maps["place"]
        for r, _ in p["moves"]:
            if r not in m.rel:
                return f"You haven't told me anything {L.SAY_AXIS[r][0]} or {L.SAY_AXIS[r][1]} of anything yet."
        target = m.pos[m.index[ka]] + m._disp(tuple(p["moves"]))
        near = [(n, d) for n, d in m.nearest(target, top=len(m.names)) if self._connected("place", ka, n)]
        if not near:
            return "Nothing I know of."
        n, d = near[0]
        if n == ka and d < 0.5:
            return f"That brings you back to {self._say(ka)}."
        if d < 0.5:
            return f"{self._say(n)}."
        if d < 1.0:
            return f"Probably {self._say(n)} - it's the closest thing to that spot ({d:.1f} steps off)."
        return f"Nothing I know of is there. The closest is {self._say(n)}, {d:.1f} steps away."

    def _ask_about(self, p):
        (ka, sa) = p["a"]
        miss = self._unknown(ka)
        if miss:
            return miss
        lines = []
        for space, m in self.maps.items():
            if not m.knows(ka):
                continue
            self._settle(space)
            group = [n for n in m.names if self._connected(space, ka, n)]
            if space == "place":
                near = [(n, d) for n, d in m.nearest(m.pos[m.index[ka]], top=4) if n != ka and n in group][:3]
                if near:
                    rel = "; ".join(f"{self._describe_offset(self._offset('place', n, ka))} of {self._say(n)}"
                                    for n, _ in near)
                    lines.append(f"Place: {rel}.")
            elif space in ("kind", "origin", "involved"):
                # a category is not a ranking: say what it is, not where it comes in a line-up
                axis = next(iter(m.rel))
                others = [n for n in group if n != ka][:3]
                if others:
                    lines.append(f"{'Kind' if space == 'kind' else space.capitalize()}: "
                                 + ", ".join(self._say(o) for o in others) + ".")
                continue
            else:
                axis = next(iter(m.rel))
                order = sorted(group, key=lambda n: -float(m.pos[m.index[n]] @ m.rel[axis]))
                word = self.words.get(axis, L.positive_word(axis))
                lines.append(f"{self._dim(axis).capitalize()}: {ordinal(order.index(ka) + 1)} of {len(order)}, "
                             f"counting from the {word} end.")
        return f"{self._say(ka)}:\n" + "\n".join(lines) if lines else f"I know {self._say(ka)} but nothing more."

    # ------------------------------------------------------------------ commands
    def _cmd_sleep(self):
        if not self.maps:
            return "Nothing to consolidate yet."
        moved = 0.0
        for space, m in self.maps.items():
            before = m.pos.clone()
            m.sleep(self.config["map_config"]["sleep"], tolerance=self.config["map_config"]["tol"])
            # recentering shifts everything rigidly; measure how much the shape changed
            delta = (m.pos - before)
            delta = delta - delta.mean(0) if len(delta) else delta
            moved = max(moved, float(delta.norm(dim=1).max()) if len(delta) else 0.0)
        self.dirty.clear()
        n = sum(len(m.episodes) for m in self.maps.values())
        return (f"Slept: replayed all {n} remembered facts together and let the map settle. "
                f"The biggest change to any position was {moved:.2f} steps.")

    def _cmd_reset(self):
        self.reset()
        return "Forgotten. My maps are empty again."

    def _cmd_summary(self):
        if not self.maps:
            return "Nothing yet - my maps are empty. Tell me something!"
        lines = []
        for space, m in self.maps.items():
            groups = self._groups(space)
            label = "Places" if space == "place" else self._dim(space).capitalize()
            lines.append(f"ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ {label}: {int(m.placed.sum())} things from {len(m.episodes)} facts"
                         + (f", in {len(groups)} unlinked groups" if len(groups) > 1 else ""))
        total = sum(m.footprint() for m in self.maps.values())
        return ("Here's what I've learned (all of it from you):\n" + "\n".join(lines) +
                f"\nMy whole memory is {total} numbers - a position per thing and a direction per relation.")

    def _cmd_help(self):
        return HELP

    # ------------------------------------------------------------------ view
    def view(self):
        """2-D picture of every map for the interface: places by east/north, each dimension as a ladder."""
        spaces = []
        for space, m in self.maps.items():
            live = [n for n in m.names if m.knows(n)]
            if not live:
                continue
            idx = torch.tensor([m.index[n] for n in live])
            P = m.pos[idx]
            if space == "place":
                ax = [a for a in ("east", "north") if a in m.rel]
                xs = (P @ m.rel["east"]) if "east" in m.rel else torch.zeros(len(live))
                ys = (P @ m.rel["north"]) if "north" in m.rel else torch.zeros(len(live))
                if "north" not in m.rel and "up" in m.rel:
                    ys = P @ m.rel["up"]
                if "east" not in m.rel and "front" in m.rel:
                    xs = P @ m.rel["front"]
                kind = "place"
            else:
                axis = next(iter(m.rel))
                ys = P @ m.rel[axis]
                # spread items at the same level sideways so they don't overlap
                xs = torch.zeros(len(live))
                kind = "line"
                ax = [axis]
            points = [{"key": n, "name": self._say(n), "x": float(x), "y": float(y),
                       "group": int(m.island[m.index[n]]), "spread": round(m.spread_of(n), 2)}
                      for n, x, y in zip(live, xs, ys)]
            edges = [{"a": m.names[a], "b": m.names[b]} for a, _, b in m.episodes]
            label = "Places" if space == "place" else self._dim(space).capitalize()
            word = self.words.get(space, L.positive_word(space)) if kind == "line" else None
            spaces.append({"space": space, "label": label, "kind": kind, "axes": ax, "word": word,
                           "points": points, "edges": edges})
        return spaces


def article(shown):
    """'The Cafe' -> 'the Cafe', so names read naturally mid-sentence (replies are capitalised at the start)."""
    return re.sub(r"^(The|A|An)\s", lambda m: m.group(1).lower() + " ", shown)


def ordinal(n):
    return f"{n}{'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')}"


def describe(p):
    """The parse, shown under each reply so it is always clear what the model was actually given."""
    k = p["kind"]
    if k == "fact":
        mv = " + ".join(f"{c:+d} {a}" for a, c in p["moves"])
        return f"learn: {p['a'][0]} = {p['b'][0]} {mv}"
    if k == "ask":
        bits = [p["q"]] + [f"{f}={p[f][0]}" for f in ("a", "b") if p.get(f)]
        if "axis" in p:
            bits.append(f"axis={p['axis']}{'+' if p['sign'] > 0 else '-'}")
        if "moves" in p:
            bits.append("moves=" + ",".join(f"{c:+d}{a}" for a, c in p["moves"]))
        return "ask: " + " ".join(bits)
    if k == "command":
        return "command: " + p["do"]
    return "not understood"

In [ ]:
%%writefile /kaggle/working/neuromoe/chat/web.py
"""Looking things up.

When the brain is asked about something it has never been told, it can go and read about it. Wikipedia is used
rather than a general search engine: the prose is clean, the licence is clear, and each fact keeps a source, so
anything the brain learned this way can be traced back and removed.

What comes back is *text*, and it is treated as text - sentences to read with the same reader used on everything
else, never as instructions. The brain has no way to act on what a page says even if a page tried to tell it to;
the only thing a page can do is add relations to a map, and every one of them is attributed and reversible.
"""
import json
import re
import time
import urllib.error
import urllib.parse
import urllib.request

API = "https://en.wikipedia.org/w/api.php"
UA = "NeuroMoE-cognitive-map/0.1 (research; contact via local user)"
SENT = re.compile(r"(?<=[.!?])\s+")


def _get(params, timeout=20, tries=4):
    """One request, waiting longer each time Wikipedia asks to be left alone. A 429 is a request to back off,
    not an error to retry immediately, and a shared service deserves that courtesy."""
    url = API + "?" + urllib.parse.urlencode({**params, "format": "json"})
    delay = 1.5
    for attempt in range(tries):
        req = urllib.request.Request(url, headers={"User-Agent": UA})
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.loads(r.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            if e.code in (429, 503) and attempt < tries - 1:
                after = e.headers.get("Retry-After")
                time.sleep(min(float(after) if (after or "").isdigit() else delay, 30.0))
                delay *= 2
                continue
            raise
    raise RuntimeError("gave up after retries")


SUGGESTED = {}      # query -> the spelling the search engine thinks was meant, from the last search


def search(query, limit=5):
    """-> [(title, snippet)] - what pages exist about this."""
    try:
        d = _get({"action": "query", "list": "search", "srsearch": query, "srlimit": limit,
                  "srinfo": "suggestion"})
    except Exception as e:
        return [], f"{type(e).__name__}: {e}"
    suggestion = d.get("query", {}).get("searchinfo", {}).get("suggestion")
    if suggestion:
        SUGGESTED[query] = suggestion
    hits = [(h["title"], re.sub("<[^>]+>", "", h.get("snippet", ""))) for h in d.get("query", {}).get("search", [])]
    return hits, None


def read_page(title, sentences=12, intro_only=False):
    """-> (sentences, url, error). A page as plain sentences.

    The intro alone says what a thing *is* and almost nothing about what it did, which is where the links to
    other things live - so by default this reads on into the body. It is still only the first few hundred
    sentences, and the caller decides how many of them to look at.
    """
    params = {"action": "query", "prop": "extracts", "titles": title, "explaintext": 1, "redirects": 1}
    if intro_only:
        params["exintro"] = 1
    try:
        d = _get(params)
    except Exception as e:
        return [], None, f"{type(e).__name__}: {e}"
    pages = d.get("query", {}).get("pages", {})
    for _, p in pages.items():
        text = (p.get("extract") or "").strip()
        if not text:
            continue
        url = "https://en.wikipedia.org/wiki/" + urllib.parse.quote(p["title"].replace(" ", "_"))
        out = [s.strip() for s in SENT.split(text) if 20 < len(s.strip()) < 400]
        return out[:sentences], url, None
    return [], None, "no extract"


def _slips(a, b):
    """Edit distance: how many keystrokes separate what was typed from what the engine suggests."""
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def _words(text):
    return set(re.findall(r"[a-z0-9]+", str(text).lower()))


def choose(query, hits, context=()):
    """Which search hit is the page about the thing asked for.

    Taking the first hit is how "that compares and contrasts the three cities" read a book about the history
    of technology, and how "kl" read the Kullback-Leibler divergence. A page is the right one when its title
    *is* what was asked about - most of the query's words in the title, few extra - and, where the query is
    ambiguous, when its summary mentions what the conversation is about. Lists, disambiguation pages and
    "outline of" pages are never the answer to "what is X".
    """
    q = _words(query)
    ctx = _words(" ".join(context))
    # "the capital city of India" is a description, not a name. Its answer is Delhi, whose title shares no
    # word with it, and scoring titles by overlap picked the article on capital cities in general. When the
    # query describes rather than names, the search engine's own ranking is the better judge.
    describes = len(q) > 3 and bool(q & {"of", "in", "the", "which", "who", "largest", "biggest", "first",
                                         "capital", "founder", "author", "inventor", "president", "leader"})
    best, best_score = None, float("-inf")
    for rank, (title, snippet) in enumerate(hits):
        low = title.lower()
        if "disambiguation" in low or low.startswith(("list of", "lists of", "outline of", "index of",
                                                      "glossary of", "timeline of")):
            continue
        t = _words(title)
        covered = len(q & t) / max(1, len(q))
        extra = len(t - q) / max(1, len(t))
        score = 3.0 * covered - 1.0 * extra - 0.3 * rank + 0.5 * len(ctx & _words(snippet))
        if describes:
            score = -rank + 0.5 * len(ctx & _words(snippet))
        if low == query.lower().strip():
            score += 2.0
        if score > best_score:
            best, best_score = title, score
    return best or (hits[0][0] if hits else None)


def look_up(query, sentences=12, context=()):
    """Search, choose the page actually about the query, read it. -> dict with title, url, sentences, error."""
    hits, err = search(query, limit=6)
    if err:
        return {"query": query, "error": err, "sentences": [], "title": None, "url": None}
    # a name this system has never read - "beijig" - is one the search engine has seen misspelled a million
    # times. When none of the hits even contains what was typed and the engine offers a spelling, take it:
    # that is the last line of hearing past a typo, for the words the corpus was too small to know.
    meant = SUGGESTED.pop(query, None)
    # only a respelling, not a different word: "beijig" -> "beijing" is one slip, "auifh" -> "arith" is a
    # guess, and following a guess lands on a page about something nobody asked about
    if meant and _slips(query.lower(), meant.lower()) > max(1, len(query) // 5):
        meant = None
    if meant and meant.lower() != query.lower() and not any(query.lower() in t.lower() for t, _ in hits):
        better, err2 = search(meant, limit=6)
        if better and not err2:
            hits, query = better, meant
    if not hits:
        return {"query": query, "error": "nothing found", "sentences": [], "title": None, "url": None}
    title = choose(query, hits, context)
    sents, url, err = read_page(title, sentences)
    # a word with several meanings lands on a page listing them. The conversation usually says which one -
    # a chat about planets wants Mercury the planet - so the search is run again with that as a hint.
    ambiguous = not sents or re.search(r"\bmay (also )?refer to\b|\bcan refer to\b", " ".join(sents[:2]), re.I)
    if ambiguous:
        for hint in [c for c in context if c][:3] or ["topic"]:
            more, _ = search(f"{query} {hint}", limit=6)
            more = [h for h in (more or []) if h[0] != title]       # not the page of meanings again
            better = choose(f"{query} {hint}", more, context) if more else None
            if better and better != title:
                s2, u2, e2 = read_page(better, sentences)
                if s2 and not re.search(r"\bmay (also )?refer to\b", " ".join(s2[:2]), re.I):
                    title, sents, url, err = better, s2, u2, e2
                    break
    return {"query": query, "title": title, "url": url, "sentences": sents, "error": err,
            "alternatives": [h[0] for h in hits if h[0] != title][:4]}


if __name__ == "__main__":
    import sys
    r = look_up(" ".join(sys.argv[1:]) or "octopus")
    print(r["title"], "->", r["url"], r.get("error") or "")
    for s in r["sentences"][:6]:
        print("  -", s[:110])

In [ ]:
%%writefile /kaggle/working/neuromoe/chat/dialogue.py
"""Finding real conversation to learn from.

Everything it had heard was encyclopedia prose, which is why it could state a fact and could not hold a turn.
An encyclopedia never says "good question", never answers one thing and then asks another, and never begins a
sentence with "well". Those are not knowledge, they are the grammar of talking, and the only way to have them
without writing them down myself is to go and hear them.

So this searches for conversation and returns it as turns. Three sources, all of them licence-clear and all
reachable through the same MediaWiki-shaped API this project already uses:

  Wikisource     interviews, speeches, testimony, plays, trials. Real exchanges, transcribed.
  Wikiquote      dialogue from film and television, which is written conversation but conversation shaped
                 for the ear - openings, interruptions, acknowledgements.
  Gutenberg      public-domain novels, where quoted speech is dialogue with the attributions stripped off.

What comes back is treated as text and nothing else, exactly as with anything else read from the world: a
transcript cannot instruct this system, it can only be heard. Nothing is stored verbatim beyond the training
pass - what survives is counts of what follows what.
"""
import json
import re
import time
import urllib.error
import urllib.parse
import urllib.request

UA = "NeuroMoE-cognitive-map/0.1 (research; contact via local user)"
WIKIS = {"wikisource": "https://en.wikisource.org/w/api.php",
         "wikiquote": "https://en.wikiquote.org/w/api.php"}
GUTENDEX = "https://gutendex.com/books"

# a line of dialogue in a transcript: "SPEAKER: what they said", or a dash, or quoted speech in prose
# a transcript line: a named speaker, a colon, and then something they said. The dash form that plays use is
# not here on purpose - it matches every bulleted citation in a reference list, and a reference list is not a
# conversation, which is what the first harvest came back full of.
SPOKEN = re.compile(r"^\s*(?:MR\.|MRS\.|MS\.|DR\.|Q|A)?\s*[A-Z][A-Za-z.'-]{1,20}(?:\s[A-Z][A-Za-z.'-]{1,20})?"
                    r"\s*:\s+(.+)$")
QUOTED = re.compile(r"[“\"]([^”\"]{12,220})[”\"]")
SENT = re.compile(r"(?<=[.!?])\s+")
STAGE = re.compile(r"\[[^\]]{0,60}\]")        # "[Angrily]", "[Pauses]" - what a script tells an actor, not speech


def _get(url, params=None, timeout=25, tries=3):
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    delay = 1.5
    for attempt in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": UA})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.loads(r.read().decode("utf-8", "ignore"))
        except urllib.error.HTTPError as e:
            if e.code in (429, 503) and attempt < tries - 1:
                time.sleep(delay)
                delay *= 2
                continue
            return None
        except Exception:
            return None
    return None


def _text(url, timeout=30):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": UA})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.read().decode("utf-8", "ignore")
    except Exception:
        return ""


def search(wiki, query, limit=12):
    """Pages on one of the sister projects that look like conversation."""
    api = WIKIS[wiki]
    d = _get(api, {"action": "query", "list": "search", "srsearch": query, "srlimit": limit,
                   "format": "json"})
    if not d:
        return []
    return [h["title"] for h in d.get("query", {}).get("search", [])]


def page(wiki, title, chars=120000):
    d = _get(WIKIS[wiki], {"action": "query", "prop": "extracts", "titles": title, "explaintext": 1,
                           "redirects": 1, "format": "json"})
    if not d:
        return ""
    for _, p in d.get("query", {}).get("pages", {}).items():
        return (p.get("extract") or "")[:chars]
    return ""


def _said(line):
    """A spoken line with the script's instructions to the actor taken out of it."""
    return re.sub(r"\s+", " ", STAGE.sub(" ", line)).strip()


# books whose whole substance is people talking. Public domain, and a novel's quoted speech is dialogue with
# the attributions stripped off, which is exactly the shape this wants.
BOOKS = [1342, 158, 161, 141, 1400, 730, 98, 1260, 768, 844, 1661, 76, 74, 11, 120, 345, 84, 514, 45, 174,
         2852, 1023, 209, 271, 1695, 4300, 2500, 160, 105, 121]


def paragraphs(text):
    """A book wraps a spoken line across several lines; a paragraph is where the line actually ends."""
    for block in text.split(chr(10) + chr(10)):
        block = " ".join(block.split())
        if block:
            yield block


def turns_from(text):
    """Conversation out of a page: the lines someone actually said.

    A transcript marks them with a speaker and a colon; a play uses the same shape; prose puts them in quotes.
    Anything else on the page is stage direction, heading or commentary, and is not conversation.
    """
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line or len(line) > 400:
            continue
        m = SPOKEN.match(line)
        if m:
            said = _said(m.group(1))
            # something said ends like something said: a heading, a caption and a citation do not
            if 20 < len(said) < 300 and said.count(" ") >= 4 and said.endswith((".", "?", "!")):
                out.append(said)
            continue
    # quoted speech is looked for in paragraphs rather than lines, because a book wraps one spoken sentence
    # over four lines and a line-at-a-time reader finds the opening quotation mark and never the closing one
    for block in paragraphs(text):
        for q in QUOTED.findall(block):
            q = _said(q)
            if q.count(" ") >= 4 and q[:1].isupper() and q.endswith((".", "?", "!", ",")):
                out.append(q.rstrip(",") + ("" if q.endswith((".", "?", "!")) else "."))
    return out


def books(ids=None, per_book_chars=900000, verbose=True):
    """Dialogue out of named public-domain novels, fetched straight from the archive.

    Searching for books about conversation finds books *about* conversation, which are essays. Books full of
    conversation are novels, and which novels those are is not something a search engine will tell you, so
    they are named.
    """
    out = []
    for book in (ids or BOOKS):
        text = _text(f"https://www.gutenberg.org/cache/epub/{book}/pg{book}.txt")[:per_book_chars]
        got = turns_from(text) if text else []
        out += got
        if verbose:
            print(f"  gutenberg {book}: {len(got)} turn(s)", flush=True)
    return out


def gutenberg(topic="", languages="en", pages=2, per_book_chars=300000):
    """Dialogue out of public-domain novels: quoted speech, which is conversation with the tags removed."""
    out = []
    url = GUTENDEX
    params = {"languages": languages, "mime_type": "text/plain"}
    if topic:
        params["search"] = topic
    for _ in range(pages):
        d = _get(url, params if url == GUTENDEX else None)
        if not d:
            break
        for book in d.get("results", [])[:8]:
            link = next((v for k, v in book.get("formats", {}).items()
                         if k.startswith("text/plain")), None)
            if not link:
                continue
            body = _text(link)[:per_book_chars]
            if body:
                out += turns_from(body)
        url, params = d.get("next") or "", None
        if not url:
            break
    return out


def harvest(queries, per_query=8, gutenberg_topics=("conversation", "dialogues"), verbose=True):
    """Everything, as a flat list of spoken turns."""
    out = []
    for wiki in WIKIS:
        for query in queries:
            for title in search(wiki, query, limit=per_query):
                got = turns_from(page(wiki, title))
                out += got
                if verbose and got:
                    print(f"  {wiki}: {title[:58]} -> {len(got)} turn(s)", flush=True)
    if gutenberg_topics:
        out += books(verbose=verbose)
    seen, unique = set(), []
    for t in out:
        key = t.lower()
        if key not in seen:
            seen.add(key)
            unique.append(t)
    return unique


if __name__ == "__main__":
    import sys
    turns = harvest(sys.argv[1:] or ["interview transcript"], per_query=3, gutenberg_topics=())
    print(f"{len(turns)} turns")
    for t in turns[:10]:
        print("  -", t[:100])

In [ ]:
%%writefile /kaggle/working/neuromoe/chat/books.py
"""Books to read: textbooks for how explanation is written, novels for how people write at length.

The encyclopedia teaches one register - dense definitional prose - and a system that has read nothing else
writes like an encyclopedia and reads everything as one. Two more registers matter for the things this system
is asked to do:

  textbooks     how an explanation unfolds: a claim, then why, then an example, then what follows. That is
                the shape an essay has, and the relations a textbook states ("X causes Y", "X is made of Y")
                are the ones the extractor is short of. Wikibooks, which are open textbooks under the same
                licence and the same API as the encyclopedia.
  novels        how English sounds when nobody is summarising: long sentences, subordinate clauses, dialogue,
                description. Project Gutenberg, public domain. Read as prose now, not just for the quoted
                speech, because the sequence memory needs to hear sentences that are not definitions.

Everything comes back as sentences to hear and count. Nothing here is stored beyond the training pass.
"""
import json
import re
import time
import urllib.parse
import urllib.request

UA = "NeuroMoE-cognitive-map/0.1 (research; contact via local user)"
WIKIBOOKS = "https://en.wikibooks.org/w/api.php"
SENT = re.compile(r"(?<=[.!?])\s+(?=[A-Z\"'])")

TEXTBOOKS = ["General Biology", "Biology", "General Chemistry", "Physics Study Guide", "Algebra", "Calculus",
             "Geometry", "Statistics", "Probability", "Microeconomics", "Macroeconomics", "Introduction to Sociology",
             "Introduction to Psychology", "Cognitive Psychology and Cognitive Neuroscience", "Anatomy and Physiology of Animals",
             "Human Physiology", "Cell Biology", "Genetics", "Ecology", "Evolution", "Astronomy", "Earth Science",
             "Geology", "Climate Change", "World History", "European History", "US History", "Ancient History",
             "Philosophy", "Introduction to Philosophy", "Ethics", "Logic", "Introductory Linguistics",
             "English Grammar", "Writing", "Rhetoric and Composition", "Music Theory", "Art History",
             "Computer Science", "Algorithms", "Programming Fundamentals", "Introduction to Computer Information Systems",
             "Electronics", "Engineering", "Geography", "Political Science", "International Relations",
             "Public Health", "Nutrition", "Anthropology"]

# novels and long-form non-fiction in the public domain, chosen for range: comedy of manners, gothic,
# adventure, detective fiction, satire, natural history, political writing, essays
NOVELS = [1342, 158, 161, 141, 105, 121, 1400, 730, 98, 766, 580, 1023, 1260, 768, 84, 345, 174, 1661, 2852,
          76, 74, 11, 12, 120, 35, 36, 5230, 159, 164, 103, 2701, 1184, 135, 2600, 4300, 1399, 28054, 219, 43,
          209, 2554, 1952, 25344, 514, 45, 113, 16, 55, 1232, 3207, 5200, 7370, 1497, 1998, 4363, 2009, 22382,
          1228, 23700, 3300, 815, 2130, 30254, 10676, 1080, 205, 30155, 2680, 8800, 4280]


def _json(url, params, tries=3):
    q = url + "?" + urllib.parse.urlencode({**params, "format": "json"})
    for attempt in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(q, headers={"User-Agent": UA}), timeout=30) as r:
                return json.loads(r.read().decode("utf-8", "ignore"))
        except Exception:
            time.sleep(1.5 * (attempt + 1))
    return None


def _text(url):
    try:
        with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": UA}), timeout=60) as r:
            return r.read().decode("utf-8", "ignore")
    except Exception:
        return ""


def sentences_of(text, lo=30, hi=320):
    """Prose -> sentences, joining the hard line wraps a plain-text book is full of first."""
    out = []
    for block in re.split(r"\n\s*\n", text):
        block = " ".join(block.split())
        if len(block) < lo or block.isupper():
            continue
        for s in SENT.split(block):
            s = s.strip()
            words = s.split()
            capitals = sum(1 for w in words if w[:1].isupper()) / max(1, len(words))
            # a table of contents is a run of capitalised headings with no sentence in it
            if lo < len(s) < hi and s[0].isupper() and len(words) >= 5 and capitals < 0.6                     and not re.search(r"https?://|\[\d+\]", s):
                out.append(s)
    return out


def wikibook(title, pages=40):
    """One open textbook: its chapters, as sentences."""
    d = _json(WIKIBOOKS, {"action": "query", "list": "allpages", "apprefix": title + "/", "aplimit": pages})
    chapters = [title] + [p["title"] for p in (d or {}).get("query", {}).get("allpages", [])]
    out = []
    for chapter in chapters[:pages]:
        page = _json(WIKIBOOKS, {"action": "query", "prop": "extracts", "titles": chapter, "explaintext": 1,
                                 "redirects": 1})
        for _, p in ((page or {}).get("query", {}).get("pages", {}) or {}).items():
            out += sentences_of(p.get("extract") or "")
    return out


def gutenberg_book(number):
    """One public-domain book, with the licence header and footer stripped, as sentences."""
    text = _text(f"https://www.gutenberg.org/cache/epub/{number}/pg{number}.txt")
    start = re.search(r"\*\*\* ?START OF (THE|THIS) PROJECT GUTENBERG.*?\*\*\*", text)
    end = re.search(r"\*\*\* ?END OF (THE|THIS) PROJECT GUTENBERG", text)
    body = text[start.end() if start else 0:end.start() if end else len(text)]
    return sentences_of(body)


def harvest(textbooks=TEXTBOOKS, novels=NOVELS, limit=None, verbose=True):
    """Everything, as (register, sentence) pairs, deduplicated."""
    seen, out = set(), []

    def add(register, sentences, label):
        n = 0
        for s in sentences:
            if s not in seen:
                seen.add(s)
                out.append((register, s))
                n += 1
        if verbose:
            print(f"  {register}: {label[:48]} -> {n:,} sentence(s) ({len(out):,} total)", flush=True)

    for title in textbooks:
        add("textbook", wikibook(title), title)
        if limit and len(out) >= limit:
            return out
    for number in novels:
        add("novel", gutenberg_book(number), f"gutenberg {number}")
        if limit and len(out) >= limit:
            return out
    return out


if __name__ == "__main__":
    got = harvest(textbooks=TEXTBOOKS[:1], novels=NOVELS[:1])
    print(len(got))
    for register, s in got[:3] + got[-3:]:
        print(f"  [{register}] {s[:100]}")

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/__init__.py
"""A conversational brain: lexical cortex, prefrontal working memory, hippocampal map and episodic store, a
basal-ganglia gate that learns which of them answers, and sleep. No pretrained weights, no gradients."""

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/agent.py
"""The turn loop, which is the brain.

Every stage of a turn is one of the mechanisms, in the order the tissue would do it. Nothing here is a wrapper
around a lookup table, and no stage exists only to make the others reachable:

  1. lexical cortex    the sentence becomes sparse activity (codes.Lexicon). Nothing enters any other way.
  2. working memory    the utterance is bound to this turn; pronouns are resolved by reading prefrontal slots,
                       not by a rule about the previous sentence.
  3. situation code    what was asked *and* what each store currently has, as one sparse pattern.
  4. the gate          basal-ganglia selection: experts vote on which system answers, and learn from whether it
                       produced grounded evidence. A cascade of ifs it is not.
  5. the system acts   the cognitive map infers, the episodic store completes a pattern, the semantic store
                       reports background structure.
  6. neuromodulation   surprise - the gap between what the map predicted and what was said - sets how strongly
                       the fact is written. Expected facts leave a faint trace, contradictions a strong one.
  7. consolidation     when the conversation goes quiet, replay: the episodic store proposes, the map relaxes,
                       weak experts are pruned. Nothing is trained by gradient anywhere in this file.
"""
import math
import os
import re
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
for p in (os.path.join(ROOT, "chat"), ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)

from brain import answer                 # noqa: E402  the conversational surface
from brain import workspace as ws        # noqa: E402  one content at a time, seen by everything
from brain.broca import Broca            # noqa: E402  production: words are read off activity here
from brain.thalamus import Thalamus      # noqa: E402  what gets through, and what holds the goal
from brain.referents import Referents    # noqa: E402  what 'it' and 'the three cities' point at
from brain.spelling import Speller       # noqa: E402  what was meant when it was mistyped
from brain.wernicke import Wernicke      # noqa: E402  comprehension: language enters only here
from brain import compose                # noqa: E402
from brain import deliberate             # noqa: E402  reasoning as workspace cycles
from brain import discourse              # noqa: E402
from brain import extract                # noqa: E402  patterns for reading, learned rather than written
from brain import inquiry                # noqa: E402
from brain import judgement              # noqa: E402
from brain import lang                   # noqa: E402
from brain import pfc                    # noqa: E402
from brain import prose                  # noqa: E402
from brain import relate                 # noqa: E402
from brain import selfmodel              # noqa: E402
from brain import timeline               # noqa: E402
from brain import motor                  # noqa: E402
from brain import say as production      # noqa: E402
from brain import speech                 # noqa: E402
from brain.codes import Lexicon          # noqa: E402
from brain.episodic import Episodes      # noqa: E402
from brain.gate import Gate              # noqa: E402
from brain.semantic import Semantic      # noqa: E402
from brain.tracts import Tracts          # noqa: E402
from brain.working import Prefrontal     # noqa: E402
import language as L                     # noqa: E402  the hand-written parser: no pretrained weights anywhere
from mapsystem import Brain as MapSystem  # noqa: E402  the cognitive map and its query layer

PRONOUNS = {"she": "topic", "he": "topic", "it": "topic", "they": "topic", "her": "object", "him": "object",
            "them": "object", "that": "topic", "this": "topic", "those": "object", "there": "topic",
            "the same": "topic"}
RECALL = (re.compile(r"^(what|who) (did|have) (we|i|you) (just )?(talk|say|said|discuss)", re.I),
          re.compile(r"^what did (i|you) say about (.+)", re.I),
          re.compile(r"^(do you )?remember", re.I),
          re.compile(r"^what (was|were) (we|i|you) (talking|saying)", re.I))
GREET = re.compile(r"^(hi|hey|hello|good (morning|evening|afternoon)|yo)\b", re.I)
ABOUT_SELF = re.compile(r"\b(you|your|yourself|we|our)\b", re.I)
FILLER = re.compile(r"^(anyway|ok(ay)?|right|hmm+|so|well|alright|cool|nice|sure|i see|got it)\b[.!]*$", re.I)
PERSONAL = re.compile(r"\b(i|my|me|we|our)\b", re.I)
STOPWORDS = {"the", "and", "you", "are", "was", "were", "for", "with", "that", "this", "what", "who", "how",
             "why", "does", "did", "can", "about", "from", "than", "then", "there", "here", "have", "has",
             "not", "but", "its", "his", "her", "their", "our", "your", "she", "him", "them", "they"}
SLEEP_EVERY = 6      # quiet-period consolidation: after this many new facts, replay before the next answer


class Agent:
    def __init__(self, seed=0, background=True):
        self.lex = Lexicon(seed=seed)
        # the two language areas. Everything heard enters through one and everything said leaves through the
        # other; no other part of this system touches text.
        self.wernicke = Wernicke(self.lex, seed=seed)
        self.broca = None                    # built once the sequence memory is loaded, below
        self.heard = None                    # what Wernicke recovered of the last utterance
        self.wm = Prefrontal(self.lex)
        self.epi = Episodes(self.lex)
        self.gate = Gate(self.lex)
        self.maps = MapSystem()                       # hippocampal map: one space per relation kind
        self.sem = Semantic() if background else Semantic(path=None)
        self.tracts = Tracts()                        # how long each pathway takes, and when the gate must choose
        self.turns = 0
        self.unslept = 0
        self.sleep_every = SLEEP_EVERY
        self.task = discourse.TaskSet()      # prefrontal task rule: what is being done, held across turns
        self.voice = production.Voice(seed)  # phrasing, with repetition suppressed the way adaptation does it
        self.articulator = motor.Articulator(self.lex, seed=seed)   # words are produced here and nowhere else
        self.speak_from_state = True         # say the answer by reading the state, once it can
        self.allow_web = False               # reading the world is opt-in; the portal turns it on
        self.read_pages = []                 # every page consulted, with what came out of it
        self.inquiries = []                  # every search-until-confident run, with how it ended
        self.meta = judgement.Metamemory(self.lex)   # learns which retrievals have been worth acting on
        self.plan_with_pfc = True            # route questions through prefrontal planning
        self.plans = []                      # the reasoning behind recent answers
        self.last_retrieval = None           # the retrieval a correction or confirmation would be about
        # what it has learned to say, from reading: how each relation is phrased and what follows what
        # the big disk first: a speech model trained on the whole corpus is too large to keep in a synced
        # folder, and the small one beside the code is the fallback for a machine that has no E: drive
        self.speech = None
        for speech_path in ("E:/flybrain/speech.pt", os.path.join(HERE, "speech.pt")):
            if os.path.exists(speech_path):
                self.speech = lang.Speech.load(speech_path)
                break
        self.broca = Broca(self.lex, speech=self.speech, seed=seed)
        # the things under discussion, and the corpus's own word frequencies for hearing past a typo
        # the workspace and the thalamus that gates it: areas bid, one content wins, everything hears it
        self.workspace = ws.Workspace()
        self.thalamus = Thalamus(self.workspace)
        self.bidders = {}
        self.referents = Referents()
        try:
            self.speller = Speller(speech=self.speech)
        except Exception:
            self.speller = None
        self.bidders = {"map": deliberate.MapBidder(self.maps),
                        "episodic": deliberate.EpisodicBidder(self.epi),
                        "reader": deliberate.ReaderBidder(self)}
        for name, area in self.bidders.items():
            self.workspace.subscribe(name, area)
        self.repairs, self.greeted, self.understood = [], False, ""
        self._named = set()
        # how English says a relation, learned from the corpus by distant supervision: what lets it read a
        # sentence whose shape nobody wrote a rule for
        self.patterns = None
        for pattern_path in ("E:/flybrain/patterns.pt", os.path.join(HERE, "patterns.pt")):
            if os.path.exists(pattern_path):
                self.patterns = extract.Patterns.load(pattern_path)
                break
        # and how people open a turn, counted from real conversation rather than written down by me
        self.markers = None
        for marker_path in ("E:/flybrain/markers.pt", os.path.join(HERE, "markers.pt")):
            if os.path.exists(marker_path):
                self.markers = answer.Markers.load(marker_path)
                break
        self.me = selfmodel.Self(self)       # what it is, and where each thing it holds came from
        self.you = selfmodel.Other()         # what the person has said, asked and corrected
        self._last_passes = []    # the retrievals prefrontal asked for on this turn, for the firing record
        self._broca_mark = 0      # how many utterances Broca had made when the current turn started
        self._last_deflection = None   # so it does not hand somebody the same deflection twice running
        self.trace = []           # per turn: what the gate saw, chose, and why - the panel reads this

    # ------------------------------------------------------------------ a turn
    def say(self, text):
        out = []
        for sentence in (L.sentences(text) or [text]):
            out.append(self._turn(sentence))
        return out

    def _turn(self, sentence):
        self.turns += 1
        self.referents.advance()
        self._broca_mark = self.broca.utterances if self.broca else 0
        self.wm.advance(sentence)
        p = self._understand(sentence)
        self._named = set()
        self._asked = p.get("asked")          # what kind of answer the question word wants: a place, a time...
        for field in ("a", "b", "topic"):
            if p.get(field) and isinstance(p[field], tuple):
                self.referents.mention(p[field][1], turn=self.turns, boost=0.6)
                self._named.add(p[field][0])
        signals, arrived = self._signals(p)
        allowed = self._applicable(p, sentence, arrived)
        action, conf, how, code = self.gate.choose(p["intent"], signals, allowed, self._activity(p, sentence))
        result = self._act(action, p, sentence)
        if result["grounded"]:
            self.gate.reinforce(code, action)
        else:
            for alt in (x for x in self._alternatives(action) if x in allowed):
                other = self._act(alt, p, sentence)
                if other["grounded"]:
                    self.gate.correct(code, action, alt)   # match tracking: this situation gets its own expert
                    action, result = alt, other
                    break
            else:
                # no store had anything. Before saying so: if this is a question and reading is allowed, go and
                # read. "Google it when you're not confident" was the instruction, and it was only being
                # followed for one sentence shape - "what is X" about a name it had never met. Every other
                # question it could not answer ended here with "I don't know anything about please".
                looked = self._do_lookup(p, sentence) if self._worth_looking_up(p, sentence) else None
                if looked and looked["grounded"]:
                    action, result = "search", looked
                else:
                    action, result = "clarify", self._do_clarify(p, sentence)
                self.gate.reinforce(code, action)
        self._judge(p, sentence)
        self._remember(p, result, sentence)
        self.task.hold(p)
        if action == "infer" and result.get("evidence"):
            self.task.hold(p, {"space": result["evidence"].get("space"), "word": p.get("word"),
                               "a": p["a"][0] if p.get("a") else None, "b": p["b"][0] if p.get("b") else None,
                               "steps": result["evidence"].get("steps")})
        # a follow-up question is good conversation after a short answer and an interruption after a piece of
        # work: nobody hands in an essay and then asks the marker a question about something else.
        ask = (discourse.curiosity(self.maps, self.task, self.turns)
               if result["grounded"] and action not in ("instruct", "view", "self", "report")
               and len(result["reply"]) < 400 else None)
        if ask:
            result = {**result, "reply": result["reply"] + "\n" + ask, "asked_back": ask}
        # what this turn was about is what "it" means next: the named subject of a question or a request,
        # after anything read on the way to answering it
        subject = (p.get("topic") or p.get("a") or (None, None))
        if isinstance(subject, tuple) and subject[0] and action not in ("social", "clarify"):
            self.referents.focus(subject[0])
        self.epi.store(sentence, self.turns, p["intent"], payload={"action": action})
        firing = self._record_turn(sentence, p, code, action)
        step = {"firing": firing, "reasoning": self._reasoning(p, action, how, conf, arrived, result),
                "turn": self.turns, "sentence": sentence, "intent": p["intent"], "action": action,
                "via": how, "confidence": round(conf, 3), "signals": [s for s, on in signals.items() if on],
                "grounded": result["grounded"], "reply": result["reply"], "evidence": result.get("evidence"),
                "arrived": [r for r, ok in arrived.items() if ok],
                "decided_at": self.tracts.deadline, "timeline": self.tracts.timeline()}
        if getattr(self, "greeted", False) and action != "social" and not step["reply"].lower().startswith(("hi", "hello")):
            step["reply"] = "Hi! " + step["reply"][0].upper() + step["reply"][1:]
            self.greeted = False
        self.trace.append(step)
        if self.unslept >= self.sleep_every:
            self.sleep(quiet=True)
        return step

    def _reasoning(self, p, action, how, conf, arrived, result):
        """The working, said quietly and kept out of the answer.

        This system used to narrate its own memory consolidation at whoever asked it a question: "I read Will
        Smith (url)", "I could turn 2 of it into facts I can use", "As for how much of this to trust". All of
        that is true and worth being able to see - it is the only reason to trust the answer at all - but it
        is not what was asked, and putting it where the answer goes means never actually answering.

        So it comes out here instead, as the trace the interface shows faintly beside the reply. Four things,
        in the order the turn did them: what it recognised, what each store returned, what the gate released,
        and what the answer itself rested on. Nothing is invented for this list - every line reports something
        the turn measured, which is why a turn that did little has a short one.
        """
        out = []
        named = [x[1] for x in (p.get("a"), p.get("b")) if x]
        if named:
            met = [n for n in named
                   if any(m.knows(n.lower()) for m in self.maps.maps.values()) or self.sem.knows(n.lower())]
            fresh = [n for n in named if n not in met]
            out.append("recognition: " + ", ".join(
                [f"{n} - met before" for n in met] + [f"{n} - new to me" for n in fresh]))
        late = [r for r, ok in arrived.items() if not ok]
        if late:
            out.append(f"conduction: {', '.join(late)} did not arrive before the gate's deadline "
                       f"({self.tracts.deadline} steps), so it played no part")
        passes = [x for x in getattr(self, "_last_passes", []) if x]
        if passes:
            out.append(f"hippocampus: {len(passes)} retrieval(s) - " + "; ".join(passes[:3]))
        for note in self.repairs:
            out.append(f"wernicke: {note}")
        if self.heard and self.heard.get("fidelity", 1.0) < 1.0:
            out.append(f"wernicke: recovered {self.heard['fidelity']:.0%} of what was said - "
                       f"working from \"{self.heard['recovered']}\"")
        # only if Broca spoke on *this* turn. Reporting its last utterance on a turn that did not use it
        # says the area was working when it was not, which is the one thing a firing record must not do.
        if self.broca is not None and self.broca.last_trace and self.broca.utterances != self._broca_mark:
            said = " | ".join(f"{t['role']}: {t['said']}" for t in self.broca.last_trace[:4])
            out.append(f"broca: {len(self.broca.last_trace)} assembly step(s) - {said}"
                       + ("" if self.broca.fidelity() == 1.0
                          else f" (one step came out unfaithful, fidelity {self.broca.fidelity()})"))
        out.append(f"gate: {p['intent']} -> {action} ({how}"
                   + (f", confidence {conf:.2f}" if how == "learned" else "") + ")")
        for line in ([result.get("aside")] if isinstance(result.get("aside"), str) else result.get("aside") or []):
            if line:
                out.append(line)
        if self.unslept:
            out.append(f"consolidation: {self.unslept} fact(s) written and not yet replayed "
                       f"(replay at {self.sleep_every})")
        return out

    def _record_turn(self, sentence, p, gate_code, action):
        """What was busy, when, and why - across the structures that actually did the work.

        Memory is not one box. Recognising that a word has been met before is perirhinal; the context it
        arrived in is parahippocampal; keeping two similar episodes apart is dentate gyrus; filling one in from
        a fragment is CA3; handing the result out is CA1; the metric space the items live in is entorhinal.
        On a single retrieval most of those are busy at once, and the frontal regions holding the goal stay up
        the whole time rather than taking a turn.

        Every bar below is a computation this turn performed. Where a structure did no work - no conflict, no
        surprise, no self-reference - it is absent rather than drawn idle, which is why a greeting looks so
        different from a comparison.
        """
        import torch
        rec = timeline.Recorder(self.tracts)
        words = sentence.lower().split()
        lex_code = self.lex.bag(words)
        for target in ("working", "episodic", "gate", "map", "semantic"):
            rec.send("lexical", target, 0, "the utterance")
        t_lex = rec.fire("auditory / lexical cortex", lex_code, duration=2, because="hearing the sentence", at=0)
        # the language areas, which are now where language actually lives rather than labels on the diagram
        heard = self.heard or {}
        rec.fire("Wernicke L4 (hearing)", self.wernicke.area.state4, duration=2, at=0,
                 because=f"{len(self.wernicke.last['words']) if self.wernicke.last else 0} word(s) driving "
                         f"the input layer")
        rec.fire("Wernicke L2/3 (meaning)", self.wernicke.area.state23, duration=3, at=1,
                 because=f"settling on one pattern; recovered {heard.get('fidelity', 1.0):.0%} of it")

        # --- medial temporal lobe, all of it at once ------------------------------------------------------
        keys = [x[0] for x in (p.get("a"), p.get("b")) if x]
        known = [k for k in keys if any(m.knows(k) for m in self.maps.maps.values()) or self.sem.knows(k)]
        fam_code = self.lex.bag(known) if known else None
        rec.fire("perirhinal (familiarity)", fam_code, duration=2,
                 because=f"{len(known)} of {len(keys) or 0} thing(s) met before" if keys else "nothing named")
        context = self.lex.bag([w for w in words if w not in STOPWORDS][:6])
        rec.fire("parahippocampal (context)", context, duration=3, because="the setting this arrived in")
        epi_code = self.epi._code(sentence, self.turns) if self.epi.records else None
        t_dg = rec.fire("dentate gyrus (separation)", epi_code, duration=2,
                        because="keeping this utterance apart from similar ones")
        if self.epi.records:
            record, conf = self.epi.recall(sentence)
            rec.fire("CA3 (completion)", epi_code, duration=2, at=t_dg,
                     because=f"settling on a memory ({conf:.2f} match)" if record else "no memory matched")

        # --- frontal, held up for the whole turn rather than taking a turn --------------------------------
        wm_code = self.wm.slots.read(self.wm.slot_key["topic"])
        rec.fire("dlPFC (working memory)", wm_code, duration=max(4, self.tracts.deadline),
                 because="holding the topic across the turn", at=1)
        if self.task.question or p.get("borrowed"):
            rec.fire("frontopolar (task set)", None, duration=max(3, self.tracts.deadline - 1), at=1,
                     because="holding what is being done with it")
        if p.get("act") in ("about-you", "report") or p["intent"] in ("self", "report"):
            rec.fire("mPFC (self / source)", None, duration=3, at=2,
                     because="this question is about me rather than the world")

        # --- the map, asked as many times as prefrontal wanted -------------------------------------------
        space = self._space(p)
        m = self.maps.maps.get(space) if space else None
        key = keys[0] if keys else None
        passes = [x for x in getattr(self, "_last_passes", []) if x]
        if m is not None and key and m.knows(key):
            grid = m.place.encode(m.pos[m.index[key]][None])[0]
            rec.fire("entorhinal grid modules", grid, duration=2,
                     because=f"{len(getattr(m.place, 'projections', [1]))} grid module(s) coding where it sits")
            t = rec.ready("hippocampal map")
            for i, why in enumerate(passes or ["one retrieval"]):
                end_t = rec.fire("hippocampal map", grid, duration=2, because=why, at=t)
                rec.fire("CA1 (readout)", grid, duration=1, at=end_t - 1, because="handing the result out")
                land = rec.send("map", "gate", end_t, "what came back")
                rec.fire("vmPFC (metamemory)", None, duration=1, at=land,
                         because="is a retrieval like this one worth acting on")
                rec.fire("dlPFC (evaluation)", None, duration=1, at=land, because="judging what came back")
                t = end_t + 1          # the next walk starts while the last one is still being judged
        elif self.sem.loaded and key and self.sem.knows(key):
            rec.fire("temporal pole (concepts)", self.lex.code(key), duration=2,
                     because="background knowledge about this term")

        # --- subcortical: selection, relay, and what the surprise was worth -------------------------------
        # the workspace, when this turn actually deliberated: one bar per cycle it spent thinking
        if action == "reason" and self.workspace.stream:
            rec.fire("global workspace", None, duration=max(2, len(self.workspace.stream)), at=2,
                     because=f"{self.workspace.cycles} cycle(s): " + " -> ".join(self.workspace.report(4)))
            rec.fire("thalamic gate (reticular)", None, duration=2, at=2,
                     because=f"{len(self.thalamus.blocked)} bid(s) never reached the workspace")
            rec.fire("mediodorsal (goal held)", None, duration=max(3, self.thalamus.held), at=2,
                     because=f"holding the goal for {self.thalamus.held} cycle(s), "
                             f"uncertainty {self.thalamus.uncertainty()}")
        t_str = rec.fire("striatum (selection)", gate_code, duration=1, because=f"releasing: {action}")
        back = rec.send("gate", "working", t_str, "the action, relayed back to cortex")
        rec.fire("thalamus (relay)", None, duration=1, at=t_str, because="carrying the choice back to cortex")
        last = self.trace[-1] if self.trace else None
        surprise = (self.trace and (self.trace[-1].get("evidence") or {}).get("surprise")) or 0.0
        if p["intent"] in ("fact", "correction"):
            recent = [t for t in (self.plans[-1:] or [])]
            rec.fire("VTA (surprise)", None, duration=1, at=t_str,
                     because="how unexpected this was decides how hard it is written")
        conflict = None
        if p.get("act") == "disagree" or p["intent"] == "correction":
            conflict = "what you just said and what I had worked out disagree"
        elif self.plans and (self.plans[-1].get("result") or {}).get("verdict") is None and passes:
            conflict = "the two retrievals did not agree with each other"
        if conflict:
            rec.fire("ACC (conflict)", None, duration=2, at=2, because=conflict)

        # --- out ------------------------------------------------------------------------------------------
        if self.broca is not None and self.broca.last_trace and self.broca.utterances != self._broca_mark:
            rec.fire("Broca L2/3 (assembling)", self.broca.area.state23, duration=2, at=max(back, t_str + 1),
                     because=f"{len(self.broca.items)} part(s) of the message held as activity at once")
            rec.fire("Broca L5 (sequencing)", None, at=max(back, t_str + 2),
                     duration=max(1, len(self.broca.last_trace)),
                     because="; ".join(f"{t['role']}: {t['said']}" for t in self.broca.last_trace[:4]))
        if self.articulator.heard:
            motor_code = self.articulator.column.encode(
                self.articulator._drive(torch.zeros(self.lex.n), None), learn=False)
            rec.fire("Broca / motor speech", motor_code, duration=2, at=max(back, t_str + 1),
                     because="turning the answer into words")
        out = rec.raster()
        out.update(deadline=self.tracts.deadline, action=action, recruited=pfc.recruitment(p["intent"]),
                   sentence=sentence)
        return out

    def _firing(self, sentence, p, gate_code, action):
        """What actually fired this turn, region by region, for the panel.

        Not a summary of what the code did - the codes themselves, binned down to something that can be drawn.
        A region that did no work this turn has an empty raster, which is the point: you can see the map sit
        out a greeting, and see the episodic store light up when someone asks what was said.
        """
        import torch
        bins = 64

        def raster(code, active_only=True):
            if code is None:
                return {"bins": [0.0] * bins, "active": 0, "of": 0}
            v = code.reshape(-1).float()
            n = v.numel()
            pad = (-n) % bins
            binned = torch.cat([v, torch.zeros(pad)]).reshape(bins, -1).sum(1)
            top = float(binned.max()) or 1.0
            return {"bins": [round(float(x) / top, 3) for x in binned],
                    "active": int((v > 0).sum()), "of": n}

        utter = self.lex.bag(sentence.lower().split())
        epi_code = self.epi._code(sentence, self.turns) if self.epi.records else None
        wm_pattern = self.wm.slots.read(self.wm.slot_key["topic"])
        space = self._space(p)
        m = self.maps.maps.get(space) if space else None
        place = None
        key = (p["a"][0] if p.get("a") else None)
        if m is not None and key and m.knows(key):
            place = m.place.encode(m.pos[m.index[key]][None])[0]
        motor_code = None
        if self.articulator.heard:
            motor_code = self.articulator.column.encode(
                self.articulator._drive(torch.zeros(self.lex.n), None), learn=False)
        return {
            "regions": [
                {"name": "lexical cortex", "tick": 0, "raster": raster(utter)},
                {"name": "working memory", "tick": self.tracts.arrival("lexical", "working"),
                 "raster": raster(wm_pattern)},
                {"name": "basal-ganglia gate", "tick": self.tracts.arrival("lexical", "gate"),
                 "raster": raster(gate_code)},
                {"name": "episodic store", "tick": self.tracts.arrival("lexical", "episodic"),
                 "raster": raster(epi_code)},
                {"name": "hippocampal map", "tick": self.tracts.arrival("lexical", "map"),
                 "raster": raster(place)},
                {"name": "motor speech", "tick": self.tracts.arrival("lexical", "map") + 1,
                 "raster": raster(motor_code)},
            ],
            "deadline": self.tracts.deadline, "action": action,
            "recruited": pfc.recruitment(p["intent"]),
        }

    def _activity(self, p, sentence):
        """What the circuit is doing right now, as patterns: the utterance itself, what working memory holds,
        and where the referents sit. This is what the gate decides on - no parsed label goes in."""
        act = {"utterance": self.lex.bag(sentence.lower().split())}
        held = self.wm.recall("topic")
        if held:
            act["held"] = self.lex.code(held[0]) * float(held[1])
        record, conf = (self.epi.recall(sentence) if self.epi.records else (None, 0.0))
        if record is not None and conf > 0:
            act["episodic"] = self.lex.code(record["text"].lower()) * float(conf)
        space = self._space(p)
        m = self.maps.maps.get(space) if space else None
        for role in ("a", "b"):
            key = p[role][0] if p.get(role) else None
            if m is not None and key and m.knows(key):
                act["place_" + role] = self.lex.code(key) * 1.0
        return act

    # -------------------------------------------------------------- understanding
    def _understand(self, sentence):
        """Hear it in Wernicke's area, then work with what came back out of the tissue.

        The parser is no longer given the string somebody typed. The utterance drives Wernicke, L2/3 settles,
        and what the area recovered is what everything downstream sees. On a clean pattern that is the same
        sentence; when the pattern is degraded it is less than the sentence, and comprehension degrades with
        it rather than sailing on regardless.
        """
        # "so who is the oldest now?" is the same question as "who is the oldest". A trailing time word is
        # discourse, not content, and leaving it on turned a question the map could answer into a search for
        # a thing called "the oldest now".
        sentence = re.sub(r"[,\s]+(?:now|then|again|still|yet|today)\s*(?=[?.!]*$)", "", sentence.strip(),
                          flags=re.I) or sentence
        self.repairs = []
        # "hello, how is your day" is two things, and answering only the greeting ignores the question. The
        # greeting is noted and peeled off; the question is what gets understood.
        self.greeted = False
        greeting = re.match(r"^(hi|hello|hey|good (?:morning|afternoon|evening)|yo|hiya)\b[,!.\s]+(?=\S)",
                            sentence, re.I)
        if greeting:
            self.greeted = True
            sentence = sentence[greeting.end():]
        # hearing what was meant: slips of the keyboard repaired against what is being talked about and what
        # the sequence memory expects, then pointers ("it", "the three cities", "KL") bound to the things
        # the conversation is actually about. Both happen before Wernicke hears it, and both are reported.
        if self.speller is not None:
            context = [e["name"] for e in self.referents.ranked()[:20]] + list(self.maps.shown.values())[-40:]
            context += ["essay", "summary", "paragraph", "analysis", "explanation", "report", "compare"]
            fixed, notes = self.speller.repair(sentence, context=context)
            if notes:
                self.repairs += [f"heard {n}" for n in notes]
                sentence = fixed
        resolved, notes = self.referents.resolve(sentence)
        if notes:
            self.repairs += [f"bound {n}" for n in notes]
            sentence = resolved
        self.understood = sentence
        self.wernicke.hear(sentence)
        self.heard = {"fidelity": self.wernicke.fidelity(), "recovered": self.wernicke.heard_text(),
                      "vocabulary": len(self.wernicke.heard)}
        if self.heard["fidelity"] < 0.999 and self.heard["recovered"]:
            sentence = self.heard["recovered"]
        kind, argument = discourse.act(sentence)
        if kind == "follow-up":
            borrowed = self.task.fill(argument)        # "and Bob?" - the held question with one argument swapped
            if borrowed:
                p = dict(borrowed)
                p["resolved"] = {argument: {"term": argument, "slot": "task-set", "strength": 1.0}}
                p["unresolved"], p["borrowed"] = False, True
                p["intent"] = p["q"]
                return p
        if kind == "disagree":
            # "no, Max is older than Milo" is a correction carrying its own replacement. The refusal and the
            # fact are one utterance, and reading the fact without stripping the refusal files something called
            # "no, Max" - which is how a system ends up agreeing with itself forever.
            rest = re.sub(discourse.SOFTENER + r"(?:no|nope|wrong|that's wrong|not true|incorrect|"
                          r"that's not right)[,.!\s]+", "", sentence, count=1, flags=re.I).strip()
            if rest and rest.lower() != sentence.lower():
                inner = dict(L.parse(rest))
                if inner["kind"] == "fact":
                    inner.update(intent="correction", act="disagree", resolved={}, unresolved=False)
                    return inner
        p = dict(L.parse(sentence))
        p["resolved"] = {}
        p["unresolved"] = False
        for field in ("a", "b"):
            if not p.get(field):
                continue
            key, shown = p[field]
            slot = PRONOUNS.get(key)
            if slot is None:
                continue
            hit = self.wm.recall(slot) or self.wm.recall("topic")
            if hit:
                p[field] = (hit[0], self.maps._say(hit[0]) if hit[0] in self.maps.shown else hit[0])
                p["resolved"][shown] = {"term": hit[0], "strength": round(hit[1], 3), "slot": slot}
            else:
                p["unresolved"] = True
        low = sentence.strip().lower()
        p["act"] = kind
        # "where is X" about something that is not on the place map is a question for the reader whose answer
        # is a place. The parser files it as a general question; the question word says what kind of answer
        # is wanted, and that is kept.
        if p.get("q") == "about" and not p.get("asked") and re.match(r"^\s*where(?:'s| is| are)\b", low or sentence,
                                                                      re.I):
            p["asked"], p["unit"] = "where", "place"
        if p["kind"] == "instruct":
            p["intent"] = "instruct"
        elif kind == "view" and p.get("q") == "about" and ABOUT_SELF.search(p.get("a", ("", ""))[1] or ""):
            # "what is your opinion of Will Smith" parses as a question about the phrase "your opinion of
            # Will Smith", and answering it as one is how a request for a view came back as a quotation of
            # what the person had said two turns earlier. The dialogue act wins: this is a request for a
            # stance on the thing, not a lookup of the words.
            p["intent"] = "view"
        elif p["kind"] == "ask":
            p["intent"] = p["q"]
        elif p["kind"] == "fact":
            p["intent"] = "correction" if kind == "disagree" else "fact"
        elif p["kind"] == "command":
            p["intent"] = "command"
        elif kind == "view":
            p["intent"] = "view"
        elif kind == "remember":
            p["intent"] = "recall"
        elif kind == "about-you":
            p["intent"] = "self"
        elif kind in ("why", "check", "report"):
            p["intent"] = kind                       # "why?" and "are you sure?" are about the last answer
        elif kind in ("thanks", "sorry", "greet", "bye", "agree", "about-you"):
            p["intent"] = "social"
        elif any(r.search(low) for r in RECALL):
            p["intent"] = "recall"
        elif FILLER.match(low) or PERSONAL.search(low):
            p["intent"] = "social"        # about the speaker, or just keeping the conversation going
        else:
            p["intent"] = "unknown"
        return p

    def _space(self, p):
        """Which map a turn is about. Space is the default for questions that name no dimension ("where is the
        vet from the cafe"); "between" belongs to whichever map actually holds both things."""
        if p.get("axis"):
            return L.space_of(p["axis"])
        if p.get("moves"):
            return L.space_of(p["moves"][0][0])
        if p.get("q") in ("where", "direction", "at"):
            return "place"
        if p.get("q") == "between" and p.get("a") and p.get("b"):
            for space in self.maps.maps:
                if self.maps._connected(space, p["a"][0], p["b"][0]):
                    return space
        return None

    def _signals(self, p):
        """What each store has to offer, and which of them got a signal here in time.

        A store's contribution is gated by its conduction delay: the gate chooses at a deadline, and anything
        still travelling is simply absent from the situation code. That is what makes the order of the turn a
        physical fact rather than the order these lines happen to be written in.
        """
        a = p["a"][0] if p.get("a") else None
        b = p["b"][0] if p.get("b") else None
        space = self._space(p)
        arrived = {r: self.tracts.in_time("lexical", r) for r in ("working", "episodic", "map", "semantic")}
        known = lambda t: bool(t) and any(m.knows(t) for m in self.maps.maps.values())
        sig = {"knows_a": arrived["map"] and known(a), "knows_b": arrived["map"] and known(b),
               "connected": bool(arrived["map"] and space and a and b and self.maps._connected(space, a, b)),
               "space_exists": bool(arrived["map"] and space and space in self.maps.maps),
               "semantic_a": bool(arrived["semantic"] and a and self.sem.knows(a)),
               "topic_held": arrived["working"] and self.wm.recall("topic") is not None,
               "resolved_ref": bool(p.get("resolved")),
               "unresolved_ref": bool(p.get("unresolved")),
               "episodes": arrived["episodic"] and len(self.epi.records) > 0,
               "facts_held": arrived["map"] and len(self.maps.log) > 0,
               "late_map": not arrived["map"]}
        return sig, arrived

    # ------------------------------------------------------------------- acting
    def _applicable(self, p, sentence, arrived=None):
        """What this turn could be answered by at all - a type check, not a routing decision. A statement cannot
        be queried and a question cannot be stored. A store whose signal has not arrived by the deadline is out
        too, for a different reason: it has nothing to say yet."""
        arrived = arrived or {r: True for r in ("working", "episodic", "map", "semantic")}
        kind, intent = p["kind"], p["intent"]
        if kind == "command":
            return ["social"]
        if intent in ("why", "check"):
            return ["explain", "recall", "clarify"]
        if intent == "report":
            return ["report", "recall", "social"]
        if intent == "self":
            return ["self", "report", "social"]
        if intent == "view":
            return ["view", "describe", "infer", "report", "clarify"]
        if p.get("lookup") and self.allow_web:
            return ["search", "inquire", "describe", "clarify"]     # asked to go and look: look
        if intent == "instruct":
            return ["instruct", "relate", "describe", "clarify"]
        if kind == "fact":
            # "I've been building you all day and I'm tired" has the shape of a claim and is not one. Filing
            # it produced a thing called "been", which then turned up in later answers as though it were a
            # dog. What somebody says about themselves is conversation, not cartography.
            subject = (p.get("a") or ("", ""))[1]
            # and a question is not a claim either, whatever shape it parses into: "why is the sky blue?" was
            # being filed as a fact relating a thing called "why" to a thing called "sky blue"
            if sentence.strip().endswith("?") or re.match(r"^\s*(why|how|what|who|when|where|which)\b",
                                                         sentence, re.I):
                return ["search", "clarify"] if self.allow_web else ["clarify"]
            first_person = re.compile(r"^\s*(?:i|we|my|our)\b|^\s*(?:i|we)'", re.I)
            if first_person.match(str(subject)) or first_person.match(sentence.strip()):
                return ["social", "clarify"]
            return ["teach", "clarify"]
        if kind == "ask" and p.get("q") == "member":
            return ["reason", "infer", "describe", "clarify"]
        if kind == "ask" and p.get("q") == "relate":
            return (["relate", "inquire", "describe", "clarify"] if self.allow_web
                    else ["relate", "describe", "clarify"])
        # a comparison on a scale that does not exist yet has nothing to ask of the map. Letting the map try
        # produces "I don't know anything about Philadelphia", which blames the thing rather than the missing
        # scale and tells the person the wrong thing to fix.
        if kind == "ask" and p.get("q") == "compare" and p.get("axis") and p["axis"] not in self.maps.maps:
            return ["clarify"]      # clarify reads the figures first when the scale is one a page can state
        if kind == "ask" and hasattr(self.maps, "_ask_" + p.get("q", "")):
            if False:
                return ["relate", "inquire", "describe", "clarify"] if self.allow_web else ["relate", "describe", "clarify"]
            out = ["infer", "clarify"] if arrived["map"] else ["recall", "clarify"]
            if self.allow_web:
                out.append("inquire")      # the map may simply not hold what the question needs yet
            if p["q"] == "about":                      # "tell me about X" - the stores can compete for this
                # but not the episodic store. "Who is Kuala Lumpur" is a question about the world, and the
                # episodic store's best offer is that somebody said those words a moment ago, which it will
                # happily call an answer. Quoting the question back is never an answer to it.
                out = [a for a in ("describe", "infer") if a != "infer" or arrived["map"]] + ["clarify"]
                # asked about something nothing here has heard of: going and reading is the obvious move, and
                # not offering it was why "what is Kuala Lumpur" came back as a shrug with the web switched on
                # background associations are a starting point, not an answer. ConceptNet will happily say a
                # bonobo is an oviparous animal used for griffins; if reading is available, read.
                if self.allow_web and p.get("a") and not any(
                        m.knows(p["a"][0]) for m in self.maps.maps.values()):
                    out = ["search", "inquire"] + out
            return out
        out = ["clarify"]
        # the episodic store answers questions about what was *said*, and nothing else. It was previously
        # allowed to compete for any turn that happened to mention something it had heard of, which is how
        # "how does Will Smith bear on Malaysia" came back as "you said 4 turns ago: who is Will Smith" -
        # a question about the world answered with a quotation of the person asking it.
        if arrived["episodic"] and intent == "recall":
            out.insert(0, "recall")
        if self._topic_of(sentence):
            out.insert(0, "describe")
        if intent == "social" or GREET.match(sentence.strip().lower()):
            out.insert(0, "social")
        if self.allow_web and (self._unknown_term(sentence) or self._worth_looking_up(p, sentence)):
            out.insert(0, "inquire")    # reading repeatedly, until it is answered or clearly will not be
            out.insert(0, "search")     # but first, reading once about the question as it was asked
        return out

    TRAILING = ("now", "then", "today", "yet", "still", "again", "here", "there", "already", "currently")

    def _trim_term(self, term):
        """A thing's name does not include when it was asked about.

        "What do you know about Rex now" is a question about Rex. Taking "Rex now" as the name of something
        and going to look it up is how it came back with a Tyrannosaurus.
        """
        words = str(term or "").split()
        while words and words[-1].lower() in self.TRAILING:
            words.pop()
        return " ".join(words)

    def _unknown_term(self, sentence):
        """Something worth going to the world about: a name, not a word.

        Asked "what did you do today", the old version found "today" unfamiliar and went and read the
        encyclopedia entry for Today, which is the single most embarrassing thing a system like this can do.
        A question about itself, or about the conversation, is not a question about the world at all, and a
        bare common noun is not the name of anything. What survives is a capitalised name or a phrase the
        person wrote as a thing.
        """
        if ABOUT_SELF.search(sentence) or PERSONAL.search(sentence):
            return None
        for m in re.finditer(r"\b([A-Z][a-z'-]+(?:\s+[A-Z][a-z'-]+)*)", sentence[1:] or sentence):
            candidate = m.group(1).strip()
            if candidate.lower() in STOPWORDS or len(candidate) < 3:
                continue
            key = self._trim_term(candidate.lower())
            if not key or any(mp.knows(key) for mp in self.maps.maps.values()) or self.sem.knows(key):
                continue
            return key
        return None

    def _alternatives(self, action):
        """Who else gets a try when the selected system comes back empty, best-placed first. Clarifying is not
        in any list: it produces no evidence, so it is where the turn ends rather than something to try."""
        order = {"teach": ("infer", "describe", "recall", "social"),
                 "infer": ("teach", "recall", "describe", "search", "social"),
                 "recall": ("infer", "describe", "search", "social"),
                 "describe": ("infer", "recall", "search", "social"),
                 "search": ("inquire", "describe", "infer", "recall", "social"),
                 "inquire": ("search", "relate", "describe", "infer", "recall", "social"),
                 "reason": ("infer", "describe", "relate", "inquire", "clarify"),
                 "relate": ("inquire", "describe", "infer", "recall", "social"),
                 "explain": ("recall", "describe", "social"),
                 "report": ("self", "recall", "social"),
                 "self": ("report", "recall", "social"),
                 "view": ("describe", "infer", "report", "recall", "social"),
                 "instruct": ("relate", "describe", "inquire", "view", "clarify"),
                 "social": ("recall", "describe", "infer"),
                 "clarify": ("infer", "teach", "recall", "describe", "search", "social")}
        return tuple(x for x in order.get(action, ()) if x != action)

    def _act(self, action, p, sentence):
        return getattr(self, "_do_" + action)(p, sentence)

    def _do_teach(self, p, sentence):
        if p["kind"] != "fact":
            return {"grounded": False, "reply": speech.clarify("unparsed")}
        if p.get("group"):
            return self._teach_group(p, sentence)
        if p.get("extreme"):
            return self._teach_extreme(p, sentence)
        # whose claim this is, recorded before it becomes part of the map: what you said and what I hold are
        # different things, and only one of them is mine to be wrong about
        source = selfmodel.READ if p.get("how") == "read" else selfmodel.TOLD
        for key in (p["a"][0], p["b"][0]):
            self.me.attribute(key, source, sentence)
        if source == selfmodel.TOLD:
            self.you.claim(sentence, (p["a"][0], p["word"], p["b"][0]))
            if re.match(r"^(my|our)\b", p["a"][1], re.I):
                self.you.introduced(p["a"][0], p["a"][1], possessive=True)
        surprise, note = self._surprise(p)
        space = self._space(p)
        self.maps.log.append(sentence)
        # Neuromodulation: surprise sets how hard this fact is written *now* - an expected fact barely moves
        # anything, an unexpected one is encoded strongly. It does not change how loudly the fact votes later:
        # consolidation weighs facts by how many others agree, which is what keeps one loud mistake from winning.
        gain = 0.5 + 1.5 * surprise
        m = self.maps.maps.get(space)
        if m is not None:
            was, m.lr = m.lr, min(1.0, m.lr * gain)
            reply = self.maps._teach(p)
            m.lr = was
        else:
            reply = self.maps._teach(p)
        self.unslept += 1
        self._link_variants(p["a"][0], p["b"][0])
        corrected = p.get("intent") == "correction"
        if corrected and m is not None:
            # A correction is not one more noisy fact. Somebody who was there is telling the map it is wrong,
            # and that is better evidence than any single thing it was told in passing - so it carries more
            # weight through consolidation, and consolidation happens now rather than whenever the next lull
            # comes, because the next question will otherwise be answered from the belief just corrected.
            if m.weights:
                m.weights[-1] = 2.5
            m.sleep(self.maps.config["map_config"]["sleep"], tolerance=self.maps.config["map_config"]["tol"])
            self.maps.dirty.discard(space)
            reply = "Taken - I've changed it, and weighted what you just told me above what I had worked out."
            note = ""
        return {"grounded": True, "reply": reply + note,
                "evidence": {"surprise": round(surprise, 3), "gain": round(gain, 3), "space": space,
                             "correction": corrected}}

    def _teach_group(self, p, sentence):
        """"I've got three dogs: Rex, Max and Milo" - several things introduced in one breath, which is how
        anyone introduces a set. Each becomes a fact of its own, and they are all the same kind of thing."""
        made = []
        for member in p["group"]:
            one = {"kind": "fact", "a": member, "b": p["b"], "moves": p["moves"], "word": p["word"]}
            self.maps.log.append(sentence)
            self.maps._teach(one)
            self.me.attribute(member[0], selfmodel.TOLD, sentence)
            if p.get("mine"):
                self.you.introduced(member[0], member[1], possessive=True)
            made.append(member[1])
        self.unslept += len(made)
        self.you.claim(sentence)
        kind = self.maps._say(p["b"][0])
        return {"grounded": True, "evidence": {"group": made},
                "reply": prose.paragraph([f"Noted: {', '.join(made[:-1])} and {made[-1]}, all of them {kind}"],
                                         gap=f"How do they compare - which is the oldest, or the biggest?")}

    def _teach_extreme(self, p, sentence):
        """"Milo is the youngest" is a fact about Milo and about everyone else on that scale.

        Parsed naively it introduces a thing called "youngest", which is how a system ends up with a map full
        of adjectives. What it actually says is that Milo is past every other thing I have on this scale - so
        that is what gets stored, one fact per item, and if I have nobody else yet then it says nothing at all
        and I should say so.
        """
        axis, sign = p["axis"], p["sign"]
        space = L.space_of(axis)
        m = self._map_for(space)
        who = p["a"]
        others = [n for n in m.names if m.knows(n) and n != who[0]]
        if not others:
            self.maps.shown.setdefault(who[0], who[1])
            return {"grounded": True, "evidence": {"extreme": axis, "against": 0},
                    "reply": f"I can hold that {who[1]} is the {p['word']}, but I have nobody else on that "
                             f"scale yet - tell me one other and it starts to mean something."}
        for other in others:
            fact = {"kind": "fact", "a": who if sign > 0 else (other, self.maps._say(other)),
                    "b": (other, self.maps._say(other)) if sign > 0 else who,
                    "moves": [(axis, 1)], "word": p["word"]}
            self.maps.log.append(sentence)
            self.maps._teach(fact)
        self.me.attribute(who[0], selfmodel.TOLD, sentence)
        self.you.claim(sentence)
        self.unslept += len(others)
        return {"grounded": True, "evidence": {"extreme": axis, "against": len(others)},
                "reply": f"Got it - that puts {who[1]} past {'all ' if len(others) > 1 else ''}"
                         f"{len(others)} other thing(s) I have on that scale, not just one."}

    def _surprise(self, p):
        """Prediction error before the fact is stored: 0 if the map already implied it, 1 if it contradicts.

        Dopamine and acetylcholine decide how strongly something is written down, and what they respond to is
        the gap between prediction and outcome. Nothing was gating encoding here before; everything was learned
        with the weight the corpus happened to give it.
        """
        space = self._space(p)
        a, b = p["a"][0], p["b"][0]
        if not space or space not in self.maps.maps or not self.maps._connected(space, b, a):
            return 0.6, ""                     # new material: worth encoding, nothing to contradict
        self.maps._settle(space)
        predicted = self.maps._offset(space, b, a)
        worst = 0.0
        for axis, c in p["moves"]:
            have = predicted.get(axis, 0.0)
            agreement = (c * have) / (abs(c) * max(abs(have), 1e-6))
            worst = max(worst, (1.0 - agreement) / 2.0)
        return worst, ""

    def _do_infer(self, p, sentence):
        if p["kind"] != "ask" or not hasattr(self.maps, "_ask_" + p.get("q", "")):
            return {"grounded": False, "reply": speech.clarify("unparsed")}
        self.maps._asked = {x[0]: x[1] for x in (p.get("a"), p.get("b")) if x}
        space = self._space(p)
        a = p["a"][0] if p.get("a") else None
        b = p["b"][0] if p.get("b") else None
        if space:
            self.maps._settle(space)   # think about the map as it stands now, not as it stood before the last fact
        thought = self._think(p, space, a, b)          # prefrontal: plan, retrieve, decide
        reply = getattr(self.maps, "_ask_" + p["q"])(p)
        if thought and thought.get("override"):
            reply = thought["override"]
        if thought and thought.get("weak"):
            reply = thought["weak"]
        state = self._answer_state(p, space, a, b, reply)
        answered = bool(space and space in self.maps.maps and (
            (a and b and self.maps._connected(space, a, b)) or
            (p["q"] in ("extreme", "rank") and space in self.maps.maps) or
            (p["q"] == "at" and a and self.maps.maps[space].knows(a))))
        # A justified "I can't tell you" is an answer too: never heard of it, or heard of both and nothing links
        # them. The map knows which of those it is, and no other store can do better - handing the question on
        # anyway is how the old router produced confident irrelevance.
        nowhere = any(t and not any(m.knows(t) for m in self.maps.maps.values()) and not self.sem.knows(t)
                      for t in (a, b))
        both_known = bool(space and space in self.maps.maps and a and b
                          and self.maps.maps[space].knows(a) and self.maps.maps[space].knows(b))
        # "I have never heard of it" settles the question only when there is nowhere to look. With the web on,
        # never having heard of something is a reason to go and read, not an answer.
        if nowhere and self.allow_web:
            nowhere = False
        grounded = (answered or nowhere or both_known) and not p.get("unresolved")
        spoken = self._articulate(state, {"subject": self.maps._say(a) if a else None,
                                          "object": self.maps._say(b) if b else None}, reply) if grounded else None
        return {"grounded": grounded, "reply": spoken or reply,
                "evidence": {"space": space, "steps": self._steps(space, a, b, p),
                             "said_from_state": bool(spoken)}}

    def _judge(self, p, sentence):
        """What became of the last answer, fed back to the metamemory.

        Being told "no" or being corrected by a fact that contradicts what was just said is the outcome signal;
        so is agreement. The retrieval that produced the answer is still to hand, so the judgement learns about
        the *shape* of that retrieval rather than about the particular items - which is what makes it transfer
        to the next question that comes back looking the same.
        """
        if self.last_retrieval is None:
            return
        act = p.get("act")
        if act == "disagree" or p.get("intent") == "correction":
            self.meta.learn(self.last_retrieval, False)
            self.last_retrieval = None
        elif act == "agree":
            self.meta.learn(self.last_retrieval, True)
            self.last_retrieval = None

    def _think(self, p, space, a, b):
        """Put the question to prefrontal cortex, which plans retrievals against the map and judges them.

        The map's own query layer still produces the sentence - it knows how to say a ranking - but what the
        answer rests on is this: a walk along the relation in both directions, or a route travelled and judged
        by a metamemory that learned which retrievals have held up. Where the two disagree, the reasoned one
        wins and says why.
        """
        m = self.maps.maps.get(space)
        if m is None or not self.plan_with_pfc:
            return None
        axis = p.get("axis")
        plan = pfc.Plan(p.get("q", "compare"), [x for x in (a, b) if x], space)
        try:
            if p.get("q") in ("compare", "which") and a and b and axis in m.rel:
                out = pfc.compare(m, a, b, axis, plan=plan)
                self._last_passes = [s["did"] for s in out["plan"]["steps"]]
                self.plans.append(out["plan"])
                if out["verdict"] is None:
                    return {"plan": out["plan"], "override": f"I can't decide that: {out['why']}."}
                if "weak evidence" in out["why"]:
                    who = self.maps._say(a if out["verdict"] else b)
                    other = self.maps._say(b if out["verdict"] else a)
                    word = p.get("word") or axis
                    return {"plan": out["plan"], "steps": out.get("steps"),
                            "weak": f"On balance {who} is {word} than {other}, but only just - {out['why']}."}
                if out.get("spread", 0) > 1.5:
                    who = self.maps._say(a if out["verdict"] else b)
                    other = self.maps._say(b if out["verdict"] else a)
                    word = p.get("word") or axis
                    return {"plan": out["plan"], "steps": out.get("steps"),
                            "weak": f"{who} is {word} than {other} - though I would not lean on it: what you "
                                    f"have told me leaves both of them floating rather than pinned."}
                return {"plan": out["plan"], "steps": out.get("steps")}
            if p.get("q") == "at" and a and p.get("moves"):
                out = pfc.locate(m, a, p["moves"], plan=plan, meta=self.meta)
                self._last_passes = [s["did"] for s in out["plan"]["steps"]]
                self.plans.append(out["plan"])
                self.last_retrieval = out.get("meta_code")
                if out["plan"]["result"]["item"] is None:
                    return {"plan": out["plan"], "override":
                            f"{out['plan']['result']['why'].capitalize()}."}
                return {"plan": out["plan"]}
        except (KeyError, ValueError, IndexError):
            return None
        return None

    def _answer_state(self, p, space, a, b, reply):
        """The answer as activity: who, along which relation, which way, how far apart.

        This is what the articulator is given. It is built out of population codes that already exist - the
        items' own codes, the relation's code, the margin as graded drive - and carries no sentence and no
        structure the speech end could read instead.
        """
        margin = self._steps(space, a, b, p)
        verdict = None if margin is None else margin * (p.get("sign", 1) or 1) > 0
        near = None if margin is None else ("\x00margin:near" if abs(margin) < 1.5 else "\x00margin:far")
        return motor.state_code(self.lex, {
            "relation": p.get("axis") or (space or "unknown"),
            "kind": "\x00ask:" + p.get("q", ""), "sign": verdict, "margin": near,
            "known": "\x00known" if "don't know" not in reply.lower() else "\x00unknown"})

    def _articulate(self, state, roles, reply):
        """Hear its own sentence bound to the state that produced it, and - once it can say that state by
        itself - let the readout do the talking instead of the template."""
        words = reply.split()
        if len(words) > 18:
            return None
        already = self.articulator.rehearse(state, words, roles)
        if already and self.speak_from_state:
            return motor.detokenize(self.articulator.speak(state, roles))
        return None

    def _steps(self, space, a, b, p):
        """The number the answer rests on, so a reply is never a claim without its grounds."""
        axis = p.get("axis")
        if not (space and axis and a and b) or space not in self.maps.maps:
            return None
        m = self.maps.maps[space]
        if not (m.knows(a) and m.knows(b) and axis in m.rel):
            return None
        return round(m.steps(b, axis, a), 3)

    def _do_recall(self, p, sentence):
        """The episodic store answers: pattern completion from whatever the question happens to say."""
        cue = sentence
        m = re.search(r"about (.+)", sentence, re.I)
        if m:
            cue = m.group(1).strip(" ?.")
        # "what did we talk about at the start" points at a *time*, not a topic. Treating "at the start" as
        # something to complete from lands on whichever turn happens to sound like those words, which is how
        # a question about the beginning of the conversation was answered with a question from the middle.
        when = bool(m) and bool(re.fullmatch(r"(at |in |from )?(the )?(start|beginning|outset|first|top)"
                                             r"|earlier|before|just now|at first", cue, re.I))
        # "what did we talk about" names nothing to complete from - it is a request for the last thing that
        # carried content, not a cue. Completing on its own words lands on whatever question sounds like it.
        generic = (not m or when) and p["intent"] == "recall"
        record, conf = (None, 0.0) if generic else self.epi.recall(cue)
        # never answer with the question itself - including an earlier turn where the same thing was asked and
        # not answered, which is the same failure with an extra step in front of it
        if record is not None and (record["turn"] == self.turns
                                   or record["text"].strip().lower() == sentence.strip().lower()):
            record, conf = None, 0.0
        if record is None or conf < 0.3:
            # "what did we just talk about" has no content to complete from; the last turn is the honest answer
            older = [r for r in self.epi.records if r["turn"] < self.turns and r["kind"] != "recall"]
            if not older or p["intent"] != "recall":
                return {"grounded": False, "reply": "I don't have anything remembered that matches that."}
            # "what did we talk about" has no content to complete from, so the honest answer is the last thing
            # that carried any - a fact if one was said, otherwise whatever was last said
            facts = [r for r in older if r["kind"] == "fact"]
            # and what carried content is not a question. Answering "what did we talk about" with the last
            # thing *asked* is answering with the conversation's own confusion.
            said = [r for r in older if not r["text"].strip().endswith("?")]
            if when and facts:
                facts = facts[:1]           # asked about the start, answer with the start
            record, conf = (facts or said or older)[-1], 0.6
        return {"grounded": True,
                "reply": speech.recalled(record, conf, self.turns - record["turn"]),
                "evidence": {"turn": record["turn"], "match": round(conf, 3)}}

    def _do_describe(self, p, sentence):
        term = (p["a"][0] if p.get("a") else None) or self._topic_of(sentence)
        if not term:
            return {"grounded": False, "reply": speech.clarify("unparsed")}
        if any(m.knows(term) for m in self.maps.maps.values()):
            spoken = self._answer_about(term)
            if spoken:
                return {"grounded": True, "reply": spoken[0], "aside": spoken[1],
                        "evidence": {"store": "map"}}
            self.maps._asked = {term: term}
            return {"grounded": True, "reply": self.maps._ask_about({"a": (term, term)}),
                    "evidence": {"store": "map"}}
        detail = self.sem.describe(term)
        if detail and (detail["relations"] or detail["near"]):
            # an association is not an answer when reading is available. Saying so and stopping is what made
            # "what is a bonobo" come back as a list of things a crowd-sourced graph files it near; not
            # counting it as grounded sends the turn on to the reader, which is where the answer is.
            grounded = not self.allow_web
            return {"grounded": grounded, "reply": speech.about_semantic(term, detail),
                    "aside": ["association: background graph only, nothing read"] if not grounded else [],
                    "evidence": {"store": "semantic", "near": [n for n, _ in detail["near"][:3]]}}
        return {"grounded": False, "reply": speech.clarify("unknown-term", term)}

    def _topic_of(self, sentence):
        words = [w for w in re.findall(r"[a-zA-Z][a-zA-Z'-]+", sentence.lower()) if len(w) > 2]
        for w in words:
            if any(m.knows(w) for m in self.maps.maps.values()) or self.sem.knows(w):
                return w
        return None

    def _do_self(self, p, sentence):
        """Questions about it rather than about the world: answered from its own structure and history.

        Nothing here is a canned description. It counts its own columns, its own corrections and where each
        thing it holds came from, at the moment of being asked - which is the only kind of self-description
        that can be wrong in a way that matters.
        """
        low = sentence.lower()
        account = self.me.account()
        if re.search(r"\b(who|what) am i\b|about me\b|do you know about me", low):
            mine = [e["shown"] for e in self.you.entities.values() if e["theirs"]]
            return {"grounded": True, "evidence": {"about": "you"},
                    "reply": prose.paragraph([
                        f"You are the person I am talking to, and everything I hold came from you except what "
                        f"I went and read",
                        f"You have told me {len(self.you.said)} thing(s) and corrected me "
                        f"{len(self.you.corrections)} time(s)",
                        f"Things you have introduced as yours: {', '.join(mine)}" if mine else
                        "You have not told me about anything of your own yet"])}
        if re.search(r"\b(what|who) are you\b|how do you work|what can you do|are you (an? )?(ai|llm|robot)", low):
            return {"grounded": True, "evidence": self.me.anatomy(),
                    "reply": prose.plan_self(account, len(self.me.mistakes), len(self.you.said))}
        if re.search(r"\b(wrong|mistake|got .* wrong)\b", low):
            if not self.me.mistakes:
                return {"grounded": True, "reply": "You have not corrected me yet, so nothing I have said has "
                                                   "been marked wrong - which is not the same as being right."}
            last = self.me.mistakes[-1]
            return {"grounded": True, "evidence": {"mistakes": len(self.me.mistakes)},
                    "reply": prose.paragraph([
                        f"You have corrected me {len(self.me.mistakes)} time(s)",
                        f"The last was when I had {last['was']} and you told me {last['corrected_to']}",
                        "I weighted yours above what I had inferred and consolidated it straight away"])}
        return self._do_report(p, sentence)

    def _do_report(self, p, sentence):
        """Talking about its day, which for it means this conversation.

        The first version answered with a count, and a count is not an answer to "how was your day" - it is a
        receipt. What makes this answerable at all is that it *has* a day: turns it could not parse, retrievals
        that came back murky, a fact that contradicted what it had worked out, pages that turned out to be
        useless. All of that is in the trace, so the report is drawn from what happened rather than from a
        template, and it changes when the session does.
        """
        low = sentence.lower()
        did = self._session()
        if not self.trace:
            return {"grounded": True, "evidence": {"facts": 0},
                    "reply": prose.paragraph([
                        "Nothing has happened to me yet - you have just arrived",
                        "I have no day outside this conversation, so this is the whole of it so far"])}
        if re.search(r"\b(hard|difficult|tricky|struggle|trouble|confus)", low):
            return {"grounded": True, "evidence": did, "reply": self._hard_parts(did)}
        if re.search(r"\b(learn|learnt|learned|new)\b", low):
            learned = [t["sentence"] for t in self.trace if t["action"] == "teach"][-3:]
            return {"grounded": True, "evidence": did, "reply": prose.paragraph(
                [f"I have taken in {did['taught']} fact(s) this session"]
                + ([f"The last were: {'; '.join(learned)}"] if learned else
                   ["Nothing yet, though - you have not told me any facts"]),
                gap="Tell me how two things relate and I will work out the rest")}
        plural = lambda n, word: f"{n} {word}" + ("" if n == 1 else "s")
        parts = [f"Busy enough: {plural(did['turns'], 'turn')} so far, of which "
                 f"{plural(did['questions'], 'question')} and {plural(did['taught'], 'fact')} you taught me"]
        if did["pages"]:
            parts.append(f"I went and read {did['pages']} page(s) when you asked about things I had never met")
        if did["conflicts"]:
            parts.append(f"{did['conflicts']} thing(s) you said cut across what I had already worked out, which "
                         f"is the part that actually took effort")
        if did["missed"]:
            parts.append(f"{did['missed']} turn(s) I could make nothing of at all")
        ask = ("What have you been doing?" if did["turns"] > 3 else
               "What would you like me to know about?")
        return {"grounded": True, "evidence": did, "reply": prose.paragraph(parts, gap=ask)}

    def _session(self):
        """What has actually happened this session, counted off the trace."""
        t = self.trace
        return {"turns": len(t),
                "questions": sum(1 for x in t if x["action"] in ("infer", "recall", "describe", "explain",
                                                                 "relate", "inquire")),
                "taught": sum(1 for x in t if x["action"] == "teach"),
                "pages": len(self.read_pages),
                "conflicts": sum(1 for x in t if (x.get("evidence") or {}).get("surprise", 0) > 0.6),
                "missed": sum(1 for x in t if not x["grounded"] or x["action"] == "clarify"),
                "murky": sum(1 for x in t if (x.get("evidence") or {}).get("said_from_state") is False),
                "corrections": len(self.me.mistakes)}

    def _hard_parts(self, did):
        """What was difficult, which is a different question from what happened."""
        parts = []
        if did["conflicts"]:
            parts.append(f"The hard part was the {did['conflicts']} thing(s) that contradicted what I had "
                         f"already settled - I had to weigh what you said against the rest and let it move the map")
        if did["missed"]:
            parts.append(f"{did['missed']} turn(s) went past me entirely: I only understand a fixed set of "
                         f"sentence shapes, and anything else I have to admit to")
        unlinked = [space for space, m in self.maps.maps.items()
                    if len({m.island[m.index[n]] for n in m.names if m.knows(n)}) > 1]
        if unlinked:
            parts.append(f"I am also still stuck on {', '.join(unlinked)}: what you have told me falls into "
                         f"separate groups I cannot compare across")
        if not parts:
            parts.append("Honestly, none of it was hard - everything you told me fitted together and nothing "
                         "contradicted anything")
        return prose.paragraph(parts)

    def _do_explain(self, p, sentence):
        """"Why?" and "are you sure?" - both are about the answer just given, and both are answered from the
        facts that actually produced it rather than from the map's geometry after the fact."""
        grounds = self.task.answer
        if not grounds or not grounds.get("a") or not grounds.get("b"):
            # "how sure are you about that" after a superlative: there is no pair to chain between, but there
            # is still an answer to be accountable for. Saying what the last one rested on beats saying
            # nothing, and saying "I didn't follow that" to a question about my own answer is the worst of
            # the three.
            last = next((t for t in reversed(self.trace) if t["grounded"] and t["action"] != "clarify"), None)
            if last is None:
                return {"grounded": False,
                        "reply": "I haven't said anything yet that I could be more or less sure about."}
            facts = len(self.maps.log)
            return {"grounded": True, "evidence": {"about": last["action"], "turn": last["turn"]},
                    "aside": [f"no pair to chain: judging the {last['action']} answer from turn "
                              f"{last['turn']} by what it rested on"],
                    "reply": f"That came from my {last['action']} system, off {facts} fact(s) you've given me. "
                             f"It is only as good as those - if one of them is wrong the answer moves, and "
                             f"nothing in it was inferred from anything I read."}
        space, a, b = grounds["space"], grounds["a"], grounds["b"]
        if p["intent"] == "check":
            steps = len(production.chain(self.maps, space, a, b))
            return {"grounded": True, "evidence": {"steps": steps, "margin": grounds.get("steps")},
                    "reply": production.confidence(steps, grounds.get("steps"))}
        return {"grounded": True, "evidence": {"space": space, "a": a, "b": b},
                "reply": production.explain(self.maps, space, a, b, grounds.get("word"), self.maps._say)}

    QUESTION = re.compile(r"^\s*(who|what|when|where|why|how|which|whose|whom|is|are|was|were|do|does|did|can|"
                          r"could|will|would|should|has|have|tell me|explain|describe|define)\b", re.I)
    SCAFFOLD = re.compile(r"^\s*(?:please\s+|can you\s+|could you\s+|would you\s+|go\s+(?:and\s+)?)*(?:google|"
                          r"search(?:\s+for|\s+up)?|look\s+up|look\s+into|find\s+out(?:\s+about)?|research|check|"
                          r"tell me|explain|describe|define)?\s*", re.I)

    def _worth_looking_up(self, p, sentence):
        """A question, or a request to find something out, that memory could not answer - and reading is on."""
        if not self.allow_web or p.get("intent") in ("social", "report", "self", "why", "check", "correction"):
            return False
        asked = bool(p.get("lookup") or sentence.strip().endswith("?") or self.QUESTION.match(sentence))
        if p.get("kind") == "fact":
            # a claim is not a question - unless it is phrased as one: "why is the sky blue?" parses as a claim
            return sentence.strip().endswith("?") or bool(re.match(r"^\s*(why|how|what|who|when|where|which)\b",
                                                                   sentence, re.I))
        return asked

    def _do_lookup(self, p, sentence):
        """Go and read about a question as asked, and answer with what the page says about *that*.

        The query is the question with its scaffolding taken off - "can you google who invented the telephone"
        searches for "who invented the telephone" - and the answer is the sentence on the page that shares the
        most with the question, not the page's first line: asked how many people live somewhere, the first
        line of the article is what the place is, and the number is further down.
        """
        query = self.SCAFFOLD.sub("", self.understood or sentence).strip(" ?.!")
        query = re.sub(r"^(?:what|who|where|when|why|how|which)\s+(?:a|an)\s+(.+?)\s+(?:is|are)$", r"\1", query,
                       flags=re.I)
        # the question word stays in the query. "why is the sky blue" finds the article on diffuse sky
        # radiation; "the sky blue", with the question word taken off, finds the colour. A search engine reads
        # questions, and throwing away the part that says what is being asked is throwing away the question.
        #
        # Unless the question names something. "How many people live in Tokyo" as a whole finds a film called
        # Tokyo Drift; the page wanted is the one about Tokyo, and the rest of the question is for picking the
        # sentence on it. A capitalised name in the question is what the question is *about*.
        named = re.findall(r"\b[A-Z][\w.'-]*(?:\s+(?:of|and|the)?\s*[A-Z][\w.'-]*)*", query[1:] or query)
        named = [n.strip() for n in named if len(n.strip()) > 2]
        if named:
            longest = max(named, key=len)
            if longest.lower() not in ("google", "please"):
                query = longest
        if not query:
            return None
        self._question = sentence
        return self._do_search({"a": (query.lower(), query)}, sentence)

    def _do_search(self, p, sentence):
        """Go and read about something nothing here knows, and keep only what can be turned into relations.

        What comes back is text, and it is treated as text. A page cannot tell this system to do anything: the
        only thing that survives reading is a fact the same hand-written parser recognises, stored with its
        source and a lower encoding weight than something you said, because hearsay is not testimony.
        """
        # a question with no parse of its own - "who wrote Pride and Prejudice", "when did World War II end" -
        # is searched as the question it is. Searching for its first capitalised word read the article on
        # pride, the emotion.
        if (not p.get("a") or p.get("kind") == "fact") and self.allow_web and self._worth_looking_up(p, sentence):
            return self._do_lookup(p, sentence) or {"grounded": False, "reply": speech.clarify("unparsed")}
        term = (p["a"][0] if p.get("a") else None) or self._unknown_term(sentence) or self._topic_of(sentence)
        if not term or not self.allow_web:
            return {"grounded": False, "reply": "I can't look things up right now."}
        import web  # noqa: E402  stdlib only; imported here so nothing touches the network unless asked
        page = web.look_up(term, sentences=24, context=self._context_words())
        if page.get("error") or not page["sentences"]:
            # asked to look something up and there is nothing there: that is an answer, and a better one than
            # falling through to "I didn't follow that"
            return {"grounded": bool(p.get("lookup") or getattr(self, "_question", None)),
                    "aside": [f"search: '{term}' - {page.get('error') or 'no text'}"],
                    "reply": f"I searched for \"{term}\" and found nothing by that name. Is it spelled "
                             f"differently, or is it something quite new?"}
        # a search engine always returns *something*. For a single word the page never mentions and the title
        # does not resemble - "auifh" landing on a symposium on computer arithmetic - that something is noise,
        # and reciting it would be answering a question nobody asked. Say it found nothing by that name.
        searched = term.lower()
        respelled = str(page.get("query", term)).lower()
        if len(searched.split()) == 1 and len(searched) >= 3 and respelled == searched:
            text = (page["title"] + " " + " ".join(page["sentences"][:6])).lower()
            close = searched in text or any(
                __import__("brain.spelling", fromlist=["distance"]).distance(searched, w) <= 1
                for w in re.findall(r"[a-z]+", text) if abs(len(w) - len(searched)) <= 1)
            if not close:
                return {"grounded": True, "aside": [f"search: nothing called '{term}' - the nearest page was "
                                                    f"{page['title']}, which never mentions it"],
                        "reply": f"I searched for \"{term}\" and couldn't find anything by that name - the closest "
                                 f"page was about {page['title']}, which never mentions it. Is it spelled "
                                 f"differently, or is it something quite new?"}
        learned = []
        # the page's own title is another name for what was asked about: bind them together so what is read
        # under one name can be found under the other
        title_key = page["title"].lower()
        if title_key == term or term in title_key:
            # the page title is how the world spells what was asked about in lower case: "will smith" written
            # back as "Will Smith". Saying a name the way it is written is not cosmetic - it is the difference
            # between an essay and a transcript of a search box.
            self.maps.shown[term] = page["title"] if title_key == term else self.maps.shown.get(term, term)
        if title_key != term:
            self._map_for("kind").attract(term, title_key, weight=0.5)
            self.maps.shown.setdefault(title_key, page["title"])
            self.maps.shown.setdefault(term, term)
        subjects = set()
        by_pattern = 0
        # A page about hip hop is *about hip hop*: the sentences that matter have it as their subject, in full
        # or as "it". Taking a fact from every sentence regardless files claims under whatever phrase happened
        # to be at the front - "synthesis of a wide" becomes a thing - and the page's actual subject ends up
        # with one fact to its name.
        about = {term, page["title"].lower(), page["title"].lower().replace("-", " ")}
        about |= {term.replace(" ", "-"), term.split()[-1]} if " " in term else set()
        for s in page["sentences"]:
            hand = [L.parse(s)] if L.parse(s)["kind"] == "fact" else L.prose_facts(s)
            # a sentence the four hand-written shapes cannot take is where the learned phrasings earn their
            # place: an article on a city used to yield nothing at all, because "is the capital city of" was
            # not one of the four things anyone had written down.
            learned_here = [] if hand else (self.patterns.read(s) if self.patterns else [])
            for parsed in (hand or learned_here):
                if parsed.get("a"):
                    subject = parsed["a"][0]
                    if subject in ("it", "they", "he", "she", "this", "that"):
                        parsed = dict(parsed, a=(term, page["title"]))   # "It is a genre" - it is the page
                    elif not (subject in about or any(w in subject for w in about)
                              or parsed["a"][1][:1].isupper()):
                        continue      # a claim about a phrase in passing, not about what the page is about
                # a page sentence can parse as a group or a superlative, which have no single pair of ends
                # to place; those shapes are for someone talking to it, not for prose it is reading
                if parsed.get("kind") != "fact" or not (parsed.get("a") and parsed.get("b")):
                    continue
                got = self._absorb(parsed, s, page, term)
                if got:
                    subjects.add(parsed["a"][0])
                    by_pattern += 1 if parsed.get("pattern") else 0
                learned += got
        # a page about "Will Smith" states its facts about "Willard Carroll Smith II". Those are one thing under
        # two names, and unless they are bound together everything read lands where nobody will look for it.
        initials = "".join(w[0] for w in term.split() if w)
        for subject in subjects:
            # "KL" in an article about Kuala Lumpur is the same city under its initials, and a fact filed
            # under an initialism that is bound to nothing is a fact nobody will ever find again
            if subject != term and len(subject) <= 4 and subject.replace(".", "") == initials:
                self._map_for("kind").attract(term, subject, weight=0.5)
                self.maps.shown.setdefault(subject, subject.upper())
                continue
            if subject != term and (term.split()[-1] in subject or subject.split()[-1] == term.split()[-1]):
                self._map_for("kind").attract(term, subject, weight=0.5)
                self.maps.shown.setdefault(subject, subject)
        return self._page_report(page, term, learned, by_pattern)

    def _context_words(self):
        """What the conversation is about, as words a search can use to pick the right page."""
        words = []
        for entry in self.referents.ranked()[:4]:
            words += list(entry["kinds"])[:2]
        return words

    def _absorb(self, parsed, sentence, page, term):
        """One fact out of a page: encoded more weakly than testimony, and kept with where it came from."""
        ka, kb = parsed["a"][0], parsed["b"][0]
        if ka == kb or len(kb) > 40 or len(ka) > 60:
            return []
        space = L.space_of(parsed["moves"][0][0])
        m = self._map_for(space)
        was, m.lr = m.lr, m.lr * 0.4          # read, not told: it moves the map less
        self.maps.log.append(sentence)
        self.maps._teach(parsed)
        m.lr = was
        self.maps.sources = getattr(self.maps, "sources", {})
        self.maps.sources[ka] = page["url"]
        # how *this* sentence said it, kept with the fact. The corpus at large says "was an American actor",
        # because most sentences of that shape are about people who have died; the page about this one says
        # "is". A prior learned from every biography ever written is the wrong thing to consult about the
        # tense of a particular claim, and the sentence the claim came out of is the right thing.
        # the sentence without its asides: "Smith (born September 25, 1968) is an American actor" has six
        # words between the two ends and five of them are a birth date, which is enough to throw the phrasing
        # away and fall back to whatever biographies usually say.
        low = answer.gloss(sentence).lower()
        i, j = low.find(parsed["a"][1].lower()), low.find(parsed["b"][1].lower())
        if 0 <= i < j:
            middle = " ".join(lang.words_of(low[i + len(parsed["a"][1]):j]))
            if middle and len(middle.split()) <= 5:
                self.maps.said = getattr(self.maps, "said", {})
                self.maps.said[(L.space_of(parsed["moves"][0][0]), kb)] = middle
        return [f"{parsed['a'][1]} - {parsed['word']} - {parsed['b'][1]}"]

    def _page_report(self, page, term, learned, by_pattern=0):
        """What to say after reading. The answer, not the receipt.

        Asked who someone is, this used to reply "I read Will Smith (url)" and then, as its second line, how
        many sentences it had managed to convert. That is the receipt for the work, and nobody asked for the
        receipt. The page's own opening sentence *is* the answer to "who is X" - that is what a reference
        page's first sentence is for - so that is what comes back, with the citation apparatus stripped out
        of it, and the provenance goes to the reasoning channel where it can be checked without being read.

        The sentences themselves are kept, not just the relations pulled out of them. A fact that failed to
        parse is still something it read, and when it later has to write about the topic that material is the
        difference between three sentences and a page.
        """
        # a page that answers what the person asked about becomes the thing under discussion; a page read on
        # the way to something else is background. Without the difference, "write an essay about it" bound
        # "it" to the Kuala Lumpur Tower, which the inquiry had read in passing.
        asked = term in getattr(self, "_named", set()) or any(term in n or n in term for n in getattr(self, "_named", ()))
        self.referents.mention(page["title"], page["sentences"][0] if page["sentences"] else "",
                               turn=self.turns, boost=1.5 if asked else 0.15)
        self.referents.merge(term, page["title"])
        self.read_pages.append({"term": term, "title": page["title"], "url": page["url"],
                                "facts": len(learned), "turn": self.turns,
                                "sentences": list(page["sentences"][:24])})
        self.unslept += len(learned)
        body = answer.gloss(page["sentences"][0], term)
        follow = ""
        for extra in page["sentences"][1:4]:
            candidate = answer.gloss(extra, term)
            if 40 < len(candidate) < 240 and candidate != body:
                follow = candidate
                break
        # a question with more in it than a name - "how many people live in Tokyo", "who invented the
        # telephone" - is answered by the sentence that shares most with the question, wherever it is on the
        # page. The lead sentence stays as the second line, so the answer still says what the thing is.
        question = getattr(self, "_question", None)
        self._question = None
        if question:
            want = {w for w in re.findall(r"[a-z0-9]+", question.lower())
                    if w not in STOPWORDS and len(w) > 2 and w not in ("google", "search", "please", "look",
                                                                        "find", "tell", "many", "much")}
            want -= set(term.lower().split()) if len(want - set(term.lower().split())) >= 1 else set()
            if want:
                scored = []
                # "end" and "ended", "invent" and "invented" are one word here: a question word matches a page
                # word that begins with it (or that it begins with), which is crude morphology and enough
                def overlap(have):
                    return {w for w in want if any(h.startswith(w) or (len(h) >= 4 and w.startswith(h))
                                                   for h in have)}
                for i, sent in enumerate(page["sentences"][:40]):
                    have = set(re.findall(r"[a-z0-9]+", sent.lower()))
                    numbers = 0.5 if re.search(r"how (many|much|old|long|big|far)|when|what year", question,
                                               re.I) and re.search(r"\d", sent) else 0.0
                    scored.append((len(overlap(have)) + numbers - 0.02 * i, i, sent))
                best = max(scored)
                if best[0] >= 1 and best[1] != 0:
                    body, follow = answer.gloss(best[2], term), body
        aside = [f"read: {page['title']} ({page['url']}) - {len(page['sentences'])} sentence(s) taken in"
                 + (f"; {by_pattern} fact(s) came from phrasings I learned rather than rules I was given"
                    if by_pattern else ""),
                 f"extracted: {len(learned)} of them parsed into relations I can reason over - "
                 + "; ".join(learned[:4]) if learned else
                 "extracted: nothing in it fit a shape I can store, so this stays material I have read "
                 "rather than facts I can reason over"]
        if getattr(self, "_asked", None) == "where":
            place = answer.locate(page["sentences"])
            if place:
                aside.append(f"where: read the place off the definition - {place}")
                body, follow = f"{page['title']} is in {place}.", body
        return {"grounded": True, "aside": aside,
                "evidence": {"url": page["url"], "learned": learned},
                "reply": answer.lead(self.voice, "found", body, follow, markers=self.markers)}

    def _do_reason(self, p, sentence):
        """Chain, in workspace cycles, until the question is settled or the goal stops being worth holding.

        "Is Rex a mammal" is not one retrieval. The map holds Rex is a dog and dogs are mammals in two
        separate facts, and the only way from one to the other is to make the answer to the first retrieval
        the question for the second - which needs a goal that survives both, and an area that can hear what
        another area found. That is the whole reason for the workspace.
        """
        if not (p.get("a") and p.get("b")):
            return {"grounded": False, "reply": speech.clarify("unparsed")}
        thing, category = p["a"][0], p["b"][0]
        out = deliberate.is_a(self, thing, category)
        said, kind = self.maps._say(thing), self.maps._say(category)
        aside = [f"workspace: {out['cycles']} cycle(s) at gain {out['gain']}, "
                 f"{out['blocked']} bid(s) stopped by the gate",
                 "stream: " + " -> ".join(out["stream"]) if out["stream"] else "stream: nothing ignited"]
        if out["found"]:
            hops = [f"{self.maps._say(step['from'])} {step['relation']} {self.maps._say(step['to'])}"
                    for step in out["trail"]]
            middle = "; ".join(hops)
            return {"grounded": True, "aside": aside, "evidence": {"trail": out["trail"]},
                    "reply": f"Yes - {said} is {kind}, in {len(out['trail'])} step(s): {middle}. "
                             f"Every step there is something you told me; I put them together."}
        if out["trail"]:
            reached = self.maps._say(out["trail"][-1]["to"])
            return {"grounded": True, "aside": aside, "evidence": {"trail": out["trail"]},
                    "reply": f"Not as far as I can tell. I followed what I hold about {said} as far as "
                             f"{reached} and nothing there leads to {kind}. Tell me how they connect and I "
                             f"will have it."}
        return {"grounded": False, "aside": aside,
                "reply": f"I have nothing about {said} to follow, so I cannot say whether it is {kind}."}

    def _do_relate(self, p, sentence):
        """"How does X bear on Y?" - answered as a route through the map, after going and reading if the two
        are not both there yet. What it presents is the route, what kind of claim each part is, and which link
        it trusts least; an essay is exactly the thing it should not produce."""
        if not (p.get("a") and p.get("b")):
            return {"grounded": False, "reply": speech.clarify("unparsed")}
        a, b = p["a"][0], p["b"][0]
        searched = None
        if self.allow_web:
            job = inquiry.Inquiry(self, budget=7, target=0.7)
            searched = job.pursue([a, b], None)
            self.inquiries.append({"terms": [a, b], **searched, "turn": self.turns})
        known = [x for x in (a, b) if any(m.knows(x) for m in self.maps.maps.values()) or self.sem.knows(x)]
        if not known:
            return {"grounded": False,
                    "reply": f"I have nothing about {self.maps._say(a)} or {self.maps._say(b)}, and no way to "
                             f"find out right now." if not self.allow_web else
                             f"I read what I could and still have nothing usable about either of them."}
        if p.get("asked_for") == "essay":
            # an essay was asked for, so an essay is what it plans: what each side is, what joins them, what
            # that will carry, and what is missing - in paragraphs, and without writing around the hole
            text = prose.essay(a, b, self._facts_about(a), self._facts_about(b),
                               relate.chain(self.maps, a, b), relate.bridges(self, a, b),
                               (searched or {}).get("readings", []),
                               (searched or {}).get("confidence", 0.0), self.maps._say, say_fact=self._phrase)
        else:
            text = relate.account(self, a, b, self.maps._say,
                                  confidence=(searched or {}).get("confidence"))
        aside = []
        if searched and searched["readings"]:
            # what it went and read to answer this is how the answer can be checked; it is not the answer,
            # and putting it first means the reply opens with a list of search terms
            aside.append("read first: " + ", ".join(f"{r['read']} ({r['facts']} fact(s))"
                                                    for r in searched["readings"]))
            aside.append(f"inquiry: verdict {searched['verdict']}, confidence {searched['confidence']:.2f}")
        return {"grounded": True, "reply": text, "aside": aside,
                "evidence": {"a": a, "b": b, "route": len(relate.chain(self.maps, a, b)),
                             "searched": searched}}

    def _do_inquire(self, p, sentence):
        """A question nothing here can answer: go and find out, and keep going until it is either answered or
        clearly not going to be. What comes back is the account of the search, not a guess dressed up as one."""
        if not self.allow_web:
            return {"grounded": False, "reply": "I'd have to go and read about that, which I can't right now."}
        terms = [t for t in (p["a"][0] if p.get("a") else None, p["b"][0] if p.get("b") else None) if t]
        if len(terms) < 2:
            found = [w.lower() for w in re.findall(r"[A-Z][a-z]+(?: [A-Z][a-z]+)*", sentence)]
            terms = (terms + [f for f in found if f not in terms])[:2] or [self._unknown_term(sentence)]
        terms = [t for t in terms if t]
        if not terms:
            return {"grounded": False, "reply": speech.clarify("unparsed")}
        job = inquiry.Inquiry(self, budget=4, target=0.7)
        result = job.pursue(terms, self._space(p))
        self.inquiries.append({"terms": terms, **result, "turn": self.turns})
        # having gone and read, the thing to say is what it now knows - not an account of the reading. The
        # account is real work and it is how the answer can be checked, so it goes to the reasoning channel.
        account = [inquiry.report(result, terms, self.maps._say),
                   f"inquiry: {len(result['readings'])} page(s), verdict {result['verdict']}, "
                   f"confidence {result['confidence']:.2f}"]
        spoken = self._answer_about(terms[0], kind="found")
        if spoken and len(terms) == 1:
            return {"grounded": True, "evidence": result, "reply": spoken[0], "aside": account + spoken[1]}
        return {"grounded": True, "evidence": result, "aside": account,
                "reply": inquiry.report(result, terms, self.maps._say)}

    def _phrase(self, relation, subject, obj):
        """Say one fact the way the world says it, using a phrasing learned from reading.

        Several phrasings are usually available - "is a", "is an", "was the" - and which one fits depends on
        what comes next. The sequence memory already knows: it is asked which continuation it expects after the
        subject, and the phrasing it rates highest is the one used. Nothing here is a template; the shape came
        from the corpus and the two ends came from the map.
        """
        # a comparative axis is said with the word the person used for it: "older than", not "age"
        comparative = self.maps.words.get(relation)
        default = {"kind": "is a", "origin": "is from", "involved": "is connected to"}.get(
            relation, f"is {comparative} than" if comparative else f"is {relation}")
        # what the sentence this fact came out of actually said, if it is on record: an observed phrasing for
        # this exact claim beats anything inferred from how such claims are usually worded
        seen = getattr(self.maps, "said", {}).get((L.space_of(relation), str(obj).lower()))
        if seen and all(w in lang.FUNCTION or w == default.split()[0] or w.endswith(("s", "ed"))
                        for w in lang.words_of(seen)):
            return self.broca.say(subject, seen, obj)
        if self.speech is None:
            return self.broca.say(subject, default, obj)
        # A phrasing may only be scaffolding. "is a city in" was learned honestly from sentences about cities,
        # and using it for anything else asserts that the thing is a city - the phrasing smuggling content in
        # is exactly the failure mode a language model has, so a phrasing made of anything but function words
        # and the relation's own verb is refused.
        usable = [(phrasing, count) for phrasing, count in self.speech.phrasings_for(relation, top=8)
                  if all(w in lang.FUNCTION or w == default.split()[0] for w in lang.words_of(phrasing))]
        options = usable or [(default, 1.0)]
        best, best_score = None, -1.0
        for phrasing, count in options:
            candidate = f"{subject} {phrasing} {obj}"
            # score the whole sentence it would produce, not the phrasing in isolation. An earlier version
            # scored the phrasing's own frequency and then added a bonus I had chosen by hand for the present
            # tense, because the model of the language was too thin to have an opinion - which is tuning
            # standing in for training. With the corpus properly heard, the sentence's own fluency carries it:
            # "Will Smith is an American actor" is a sentence the corpus expects and "Will Smith was an
            # American actor" is not, and no rule of mine has to say so.
            score = 3.0 * self.speech.fluency(candidate) + 0.15 * math.log1p(count)
            if score > best_score:
                best, best_score = phrasing, score
        # the frame is chosen here; the sentence is *said* in Broca, by loading the parts as activity and
        # watching which assembly wins at each step. Nothing downstream of this line concatenates a sentence.
        return self.broca.say(subject, best, obj)

    def _material(self, term):
        """Sentences it has read that are about this thing, in the order it read them.

        Facts are what survived parsing; material is everything it took in. Keeping both is the difference
        between being able to say "it is a kind of X" and being able to write about it, and it is also what
        lets the reasoning channel show that the two are not the same thing.
        """
        key = str(term).lower()
        out = []
        for page in self.read_pages:
            title = page["title"].lower()
            if key == page["term"] or key == title or key in title or page["term"] in key:
                out += [x for x in page.get("sentences", []) if x]
        return out

    def _answer_about(self, term, kind="answer"):
        """One thing, answered: what it is, then what else is held about it. Returns (reply, aside) or None.

        The order matters more than the content. A definition it read is the fullest single statement it has,
        so that leads; what it has placed on its own map comes second, because that is the part it can reason
        with; and how firmly any of it sits goes to the reasoning channel rather than in front of the answer.
        """
        said = self.maps._say(term)
        facts = self._facts_about(term)
        stated = compose.sentences_about(said, facts, self._phrase, self.maps._say, limit=4)
        material = self._material(term)
        definition = ""
        for sentence in material[:3]:
            candidate = answer.gloss(sentence, term)
            if 40 < len(candidate) < 260:
                definition = candidate
                break
        if not definition and not stated:
            return None
        body = definition or stated[0]
        if getattr(self, "_asked", None) == "where" and material:
            place = answer.locate(material)
            if place:
                return answer.lead(self.voice, kind, f"{said} is in {place}.", definition or body,
                                   markers=self.markers), [f"where: read the place off the definition - {place}"]
        rest = stated if not definition else stated[:3]
        second = answer.also(rest if definition else stated[1:], limit=3)
        aside = []
        held = sum(len(v) for v in facts.values())
        if held:
            spread = max((m.spread_of(term) for m in self.maps.maps.values() if m.knows(term)), default=2.0)
            firm = ("several facts agree on where it belongs" if spread < 0.8 else
                    "one or two facts only, so it is still floating" if spread < 1.5 else
                    "the facts I have about it disagree")
            aside.append(f"map: {held} relation(s) about {said}, spread {spread:.2f} - {firm}")
        if material:
            aside.append(f"material: {len(material)} sentence(s) read about {said}, "
                         f"{'led with the definition' if definition else 'none of it usable as a definition'}")
        return answer.lead(self.voice, kind, body, second, markers=self.markers), aside

    def _facts_about(self, term):
        """What was actually stated about a thing, grouped by the relation, for the essay to draw on."""
        out = {}
        # a page about "Will Smith" files its facts under "Willard Carroll Smith II", and the alias link is
        # what says those are one thing - so the facts of an alias are this thing's facts too
        names = {term}
        for m in self.maps.maps.values():
            if not m.knows(term):
                continue
            i = m.index[term]
            for ea, moves, eb in m.episodes:
                if not moves and i in (ea, eb):
                    names.add(m.names[eb if ea == i else ea])
        for space, m in self.maps.maps.items():
            for alias in list(names):
                if m.knows(alias):
                    out.update(self._facts_of(m, alias, out))
        return out

    def _facts_of(self, m, term, out):
        """The stated facts about one name on one map, grouped by relation."""
        if m.knows(term):
            i = m.index[term]
            for idx, (ea, moves, eb) in enumerate(m.episodes):
                if i not in (ea, eb):
                    continue
                if not moves:
                    continue          # an alias link is plumbing, not something to say about the thing
                other = m.names[eb if ea == i else ea]
                rel = moves[0][0]     # the raw relation, so production can use what it learned to say for it
                if other != term and compose.usable(other) and other not in out.get(rel, []):
                    out.setdefault(rel, []).append(other)        # the same thing said twice is one fact
        return out

    def _link_variants(self, *names):
        """Bind a plural to its singular where both are on the map: "dogs" and "dog" are one thing.

        Without this the second half of a chain is a separate island - "Rex is a dog" in one group and "dogs
        are mammals" in another - and nothing can walk from one to the other however many facts are held.
        """
        linked = []
        for name in [n for n in names if n]:
            variants = {name.rstrip("s"), name + "s", name + "es"}
            if name.endswith("ies"):
                variants.add(name[:-3] + "y")
            for other in variants - {name}:
                for m in list(self.maps.maps.values()):
                    if m.knows(name) and m.knows(other) and not self.maps._connected("kind", name, other):
                        self._map_for("kind").attract(name, other, weight=0.5)
                        self.maps.shown.setdefault(other, other)
                        linked.append((name, other))
                        break
        return linked

    def _map_for(self, space):
        self.maps._map(space)
        return self.maps.maps[space]

    def _do_instruct(self, p, sentence):
        """Do what was asked, and go and read first if it is not confident it knows enough to do it.

        Two failures were showing here. It treated an instruction as something to file, so "write an essay"
        came back as "you said that three turns ago". And when it did know what was being asked for, it wrote
        from whatever happened to be on the map without noticing how thin that was. So: pick the topic (or
        choose one if none was named), measure how much it actually holds, and if that is not enough, read
        until it is or until reading stops paying - and then write, saying which of those happened.
        """
        topic, chose = (p.get("topic") or (None, None))[0], False
        if topic:
            # "an essay that compares and contrasts X", "an essay on just KL": the words around the topic say
            # what to do with it, and are not part of its name
            topic = re.sub(r"^(?:that |which )?(?:compares?|comparing|contrasts?|contrasting)(?: and "
                           r"(?:compares?|contrasts?))?\s+", "", topic)
            topic = re.sub(r"^(?:just|only|simply|purely|the topic of)\s+", "", topic).strip(" ?.")
            group = getattr(self.referents, "group", None)
            # one thing, even with a comma in its name: "Washington, D.C." is not Washington and D.C.
            known = self.referents.find(topic)
            if known and not group:
                topic = known["name"].lower()
            names = list(group) if group else [topic] if known else [
                n.strip() for n in re.split(r",\s*(?:and\s+)?|\s+and\s+|\s+vs\.?\s+|\s+versus\s+", topic)
                if n.strip()]
            if len(names) >= 2 or re.search(r"\b(compar|contrast)", sentence, re.I) and len(names) >= 2:
                return self._compare_essay(names, p, sentence)
        if not topic:
            # no topic given: the honest choice is whatever it holds most about, and it should say it chose
            counts = {}
            for m in self.maps.maps.values():
                for n in m.names:
                    if m.knows(n) and "_" not in n:
                        counts[n] = counts.get(n, 0) + sum(1 for ea, _, eb in m.episodes
                                                           if m.index[n] in (ea, eb))
            if not counts:
                return {"grounded": False,
                        "aside": ["map: empty - nothing to choose a topic from"],
                        "reply": "I'd have to make it up - I'm holding nothing yet. Give me a topic and I'll "
                                 "go and read about it first, or tell me about something yourself."}
            topic, chose = max(counts, key=counts.get), True
        facts = self._facts_about(topic)
        spread = max((m.spread_of(topic) for m in self.maps.maps.values() if m.knows(topic)), default=2.0)
        searched = None
        thin = sum(len(v) for v in facts.values()) < 3 or spread > 1.2
        if thin and self.allow_web:
            job = inquiry.Inquiry(self, budget=5, target=0.7)
            searched = job.pursue([topic], None)
            self.inquiries.append({"terms": [topic], **searched, "turn": self.turns})
            facts = self._facts_about(topic)
            spread = max((m.spread_of(topic) for m in self.maps.maps.values() if m.knows(topic)), default=2.0)
        near = []
        for space, m in self.maps.maps.items():
            if m.knows(topic) and space != "kind":
                near += [n for n, _ in m.nearest(m.pos[m.index[topic]], top=4) if n != topic]
        text, working = prose.deliver(topic, facts, near, spread, (searched or {}).get("readings", []),
                                      (searched or {}).get("confidence", 0.0), self.maps._say, self._phrase,
                                      deliverable=p.get("deliverable", "essay"), chose=chose,
                                      material=self._material(topic),
                                      mark=answer.opening(self.voice, "task", self.markers))
        return {"grounded": True, "reply": text, "aside": working,
                "evidence": {"topic": topic, "chose": chose, "deliverable": p.get("deliverable"),
                             "read": len((searched or {}).get("readings", []))}}

    def _compare_essay(self, names, p, sentence):
        """Several things, compared: what each is, what they share, and where they differ - from reading.

        "Compare and contrast the three cities" used to go and read a book whose title contained the words of
        the request. What it should do is what a student does: make sure it has read about each one, pull out
        of that reading the things that can be set side by side - what kind of thing each is, where, how big -
        and then say what is common to all of them and what is not. Every figure in the result is one it read;
        where it has a figure for some and not others, it says so rather than comparing what it does not have.
        """
        infos, readings = [], []
        for name in names[:5]:
            key = name.lower()
            if self.allow_web and not self._material(key):
                job = inquiry.Inquiry(self, budget=1, target=0.9)
                found = job.pursue([key], None)
                readings += [r["read"] for r in found["readings"]]
            material = self._material(key)
            entry = self.referents.entries.get(key) or next(
                (e for e in self.referents.entries.values() if key in e["aliases"] or key in e["name"].lower()),
                None)
            infos.append({"name": entry["name"] if entry else name, "material": material,
                          "definition": answer.gloss(material[0], key) if material else "",
                          "place": answer.locate(material) if material else None,
                          "kinds": sorted(entry["kinds"]) if entry else [],
                          "population": prose.figure(material, "population"),
                          "area": prose.figure(material, "area")})
        text = prose.compare(infos, mark=answer.opening(self.voice, "task", self.markers))
        aside = [f"compare: {len(infos)} topic(s) - " + ", ".join(i["name"] for i in infos)]
        if readings:
            aside.append("read first: " + ", ".join(readings))
        for info in infos:
            got = [k for k in ("place", "population", "area") if info.get(k)]
            aside.append(f"{info['name']}: " + (", ".join(got) if got else "nothing I could set side by side"))
        return {"grounded": bool(any(i["material"] for i in infos)), "reply": text, "aside": aside,
                "evidence": {"compared": [i["name"] for i in infos]}}

    def _do_view(self, p, sentence):
        """"What do you make of that?", "which one should I take running?", "what else would help you?"

        None of these are answerable from a store on its own, and all of them are answerable. What it makes of
        something is what it holds about it plus what it cannot yet decide; which one to pick is a ranking on
        whichever scale it has, with the honest note that it was not asked about running; and what would help
        is a question it can answer better than anyone, because it can see its own gaps.
        """
        low = sentence.lower()
        gaps = self._gaps()
        if re.search(r"\b(help|useful|need|tell you|should i say|what else)\b", low):
            if not gaps:
                return {"grounded": True, "reply": prose.paragraph(
                    ["Everything you have told me hangs together, so nothing is missing that I can see"],
                    gap="More things on the same scales would sharpen it - or a new kind of comparison")}
            return {"grounded": True, "evidence": {"gaps": gaps[:3]},
                    "reply": prose.paragraph([f"The most useful thing would be {gaps[0]}"]
                                             + ([f"After that, {gaps[1]}"] if len(gaps) > 1 else []),
                                             gap="Either of those would let me answer things I currently cannot")}
        # "what's your opinion of Will Smith" names its topic after the preposition, and the parser has no
        # reason to have picked it up: the sentence is not a claim or a comparison, it is a request for a
        # stance on a thing. Take the thing.
        asked = re.search(r"\b(?:of|about|on|to)\s+(.+?)\s*[?.!]*$", sentence, re.I)
        topic = p["a"][0] if p.get("a") else None
        if topic and (ABOUT_SELF.match(topic) or PERSONAL.match(topic) or " opinion " in f" {topic} "):
            topic = None      # "your opinion of X" is not a thing to have a view about; X is
        if not topic and asked:
            candidate = asked.group(1).strip().lower()
            for option in (candidate, candidate.split(" and ")[0]):
                if any(m.knows(option) for m in self.maps.maps.values()) or self.sem.knows(option):
                    topic = option
                    break
            else:
                topic = candidate if len(candidate.split()) <= 4 else None
        topic = topic or self.wm.recall("topic")
        topic = topic[0] if isinstance(topic, tuple) else topic
        if not topic:
            return {"grounded": False, "reply": speech.clarify("unparsed")}
        # asked for a view on something it has never heard of: the view is worthless until it has read, and
        # reading is available, so read first rather than shrugging
        aside = []
        if self.allow_web and not (any(m.knows(topic) for m in self.maps.maps.values())
                                   or self.sem.knows(topic) or self._material(topic)):
            job = inquiry.Inquiry(self, budget=3, target=0.6)
            found = job.pursue([topic], None)
            self.inquiries.append({"terms": [topic], **found, "turn": self.turns})
            aside.append(f"no view without material: read {len(found['readings'])} page(s) about "
                         f"{self.maps._say(topic)} first, confidence {found['confidence']:.2f}")
        facts = self._facts_about(topic)
        lines = []
        for rel, others in list(facts.items())[:2]:
            lines.append(self._phrase(rel, self.maps._say(topic), ", ".join(self.maps._say(o) for o in others[:2])))
        # ranking only means something on a scale. "kind" is a category, and saying something comes 1st of 5
        # on it is the kind of nonsense that comes of treating every relation as an ordering.
        scales = [space for space, m in self.maps.maps.items()
                  if m.knows(topic) and space not in ("place", "kind", "origin", "involved")]
        if scales:
            best = scales[0]
            m = self.maps.maps[best]
            self.maps._settle(best)
            axis = next(iter(m.rel))
            order = sorted([n for n in m.names if self.maps._connected(best, topic, n)],
                           key=lambda n: -float(m.pos[m.index[n]] @ m.rel[axis]))
            if len(order) > 1:
                rank = order.index(topic) + 1
                lines.append(f"On {self.maps._dim(best)} it comes {rank} of {len(order)}, with "
                             f"{self.maps._say(order[0])} at the top")
        material = self._material(topic)
        if not lines and not material:
            return {"grounded": False, "aside": aside,
                    "reply": speech.clarify("unknown-term", self.maps._say(topic))}
        # A stance is not a preference. What it can honestly offer is what the thing is, and then what its own
        # facts lean towards - said as its own position, with what would move it. That is an opinion of the
        # only kind it is entitled to, and it is more use than either a refusal or an invented preference.
        T = self.maps._say(topic)
        held = sum(len(v) for v in facts.values())
        if not lines and material:
            lines = [answer.gloss(material[0], topic)]
        if held >= 3:
            stance = (f"My view is that {T} is well enough attested for me to reason about - what I hold "
                      f"about it agrees with itself, so I would act on it")
        elif held or material:
            stance = (f"A view, though, I don't really have. I can tell you what {T} is; what I can't do is "
                      f"tell you what to make of it, because everything I hold describes it and nothing I "
                      f"read said what it was worth")
        else:
            stance = f"I have no view on {T} worth the name"
        # a gap about something else is not what would change my mind about this. Prefer one that names the
        # topic, and if none does, ask for what would actually settle a view on it.
        mine = [g for g in gaps if T.lower() in g.lower() or topic in g.lower()]
        ask = (f"What would change that is {mine[0]}" if mine else
               f"Tell me what matters to you about {T} - what it should be judged on - and I will take a "
               f"position on that instead")
        aside.append(f"stance: {held} relation(s) held, "
                     f"{len(material)} sentence(s) of reading - a view, not a preference")
        return {"grounded": True, "evidence": {"topic": topic}, "aside": aside,
                "reply": answer.lead(self.voice, "stance", prose.paragraph(lines),
                                     prose.paragraph([stance, ask]), markers=self.markers)}

    def _gaps(self):
        """What it is missing, specifically enough to ask for: said as the request it would make."""
        out = []
        for space, m in self.maps.maps.items():
            groups = {}
            for n in m.names:
                if m.knows(n):
                    groups.setdefault(m.island[m.index[n]], []).append(n)
            if len(groups) > 1:
                first, second = list(groups.values())[:2]
                out.append(f"how {self.maps._say(first[0])} and {self.maps._say(second[0])} compare"
                           + ("" if space == "place" else f" by {self.maps._dim(space)}"))
            loners = [n for n in m.names if m.knows(n) and m.spread_of(n) > 1.2]
            if loners:
                out.append(f"another fact about {self.maps._say(loners[0])} - one fact leaves it floating")
        return out

    def _do_clarify(self, p, sentence):
        """Asking back. Never grounded - it rests on no store - so it is only ever where a turn ends."""
        if p.get("unresolved"):
            return {"grounded": False, "reply": speech.clarify("pronoun")}
        # a comparison on a scale it has never been given anything on. The missing thing is the scale, not
        # the word: answering "I don't know anything about bigger" reads as though it had never heard the
        # word, when what it actually lacks is a single fact about how big anything is.
        axis = p.get("axis")
        # nobody has *told* it how big anything is - but it may have *read* it. A population stated on the
        # page about each is a scale, and comparing two figures it read is not an inference it is making up.
        if p.get("q") == "compare" and axis == "size" and p.get("a") and p.get("b"):
            sizes = []
            for x in (p["a"], p["b"]):
                material = self._material(x[0])
                if not material and self.allow_web:
                    inquiry.Inquiry(self, budget=1, target=0.9).pursue([x[0]], None)
                    material = self._material(x[0])
                figure = prose.figure(material, "population")
                sizes.append((prose.magnitude(figure), figure, self.referents.entries.get(x[0], {}).get("name")
                              or x[1]))
            if all(s[0] for s in sizes):
                (va, fa, na), (vb, fb, nb) = sizes
                word = p.get("word") or "bigger"
                yes = va > vb if word not in ("smaller", "less", "fewer") else va < vb
                bigger, smaller = (na, nb) if va > vb else (nb, na)
                return {"grounded": True,
                        "aside": [f"size: read populations - {na} {fa}, {nb} {fb}"],
                        "reply": f"{'Yes' if yes else 'No'} - by population {bigger} is bigger: I read "
                                 f"{fa if va > vb else fb} for it against {fb if va > vb else fa} for {smaller}. "
                                 f"That is people, not area; I'd need the areas to say which covers more ground."}
        if p.get("q") == "compare" and axis and axis not in self.maps.maps:
            word = p.get("word") or axis
            named = [self.maps._say(x[0]) for x in (p.get("a"), p.get("b")) if x]
            about = " and ".join(named) if named else "those"
            return {"grounded": False,
                    "aside": [f"no {axis} map exists yet: nothing has ever been placed on that scale"],
                    "reply": f"I have no scale for {word} - nobody has told me how big anything is, so "
                             f"{about} are not on one. Give me one comparison on size and I can place the "
                             f"rest from it."}
        term = self._topic_of(sentence)
        if term and not any(m.knows(term) for m in self.maps.maps.values()):
            return {"grounded": False, "reply": speech.clarify("unknown-term", term)}
        return {"grounded": False, "reply": speech.clarify("unparsed")}

    def _do_social(self, p, sentence):
        low = sentence.strip().lower()
        act = p.get("act")
        if act in ("thanks", "sorry", "bye", "agree", "about-you"):
            return {"grounded": True, "reply": self.voice(act), "evidence": {"act": act}}
        if act == "greet":
            return {"grounded": True, "evidence": {"act": act},
                    "reply": self.voice("greet-known", facts=len(self.maps.log)) if self.maps.log
                             else self.voice("greet-empty")}
        if p["kind"] == "command":
            if p["do"] == "sleep":
                return {"grounded": True, "reply": self.sleep()}
            if p["do"] == "reset":
                self.forget()
                return {"grounded": True, "reply": "Forgotten - every store is empty again."}
            if p["do"] == "summary":
                return {"grounded": True, "reply": self.summary()}
            return {"grounded": True, "reply": self.maps._cmd_help()}
        if GREET.match(low) or low in ("thanks", "thank you"):
            return {"grounded": True, "reply": speech.greeting(len(self.maps.shown), len(self.maps.log))}
        if FILLER.match(low):
            held = self.wm.recall("topic")
            return {"grounded": True, "evidence": {"filler": True},
                    "reply": f"Mm. We were on {self.maps._say(held[0])} - carry on, or change the subject."
                             if held else "Mm. Say what you like; I am listening."}
        if PERSONAL.search(sentence):
            return {"grounded": True, "evidence": {"about": "them"}, "reply": self._about_them(sentence)}
        topic = self._topic_of(sentence)
        said = None
        if topic:
            hits = self.epi.about(topic, limit=2)
            older = [(r, s) for r, s in hits if r["turn"] < self.turns]
            if older:
                said = f'you said "{older[0][0]["text"]}"'
        # being spoken to is always answerable: acknowledging, and saying plainly what is and isn't held, is a
        # real reply rather than a failure to route
        return {"grounded": True, "reply": speech.acknowledge(topic, said), "evidence": {"topic": topic}}

    def _about_them(self, sentence):
        """Someone has said something about themselves. Answer it as about them, not as a failed fact.

        It cannot sympathise and should not pretend to. What it can do is take the thing seriously, say what it
        actually holds about them, and ask for the part it could use - which is what being a listener with a
        map rather than a heart amounts to.
        """
        words = [w for w in re.findall(r"[a-zA-Z'-]{3,}", sentence.lower()) if w not in STOPWORDS]
        subject = next((w for w in words if any(m.knows(w) for m in self.maps.maps.values())), None)
        mine = [e["shown"] for e in self.you.entities.values() if e["theirs"]]
        parts = []
        if subject:
            parts.append(f"You have mentioned {subject} before, so I have something to hang that on")
        else:
            parts.append("That is about you rather than about anything on my map, so I have nothing to check "
                         "it against")
        if mine:
            parts.append(f"What I do have of yours is {', '.join(mine)}")
        parts.append(f"You have told me {len(self.you.said)} thing(s) so far")
        ask = ("Is there something in it I should hold on to - how two things relate, or who is what?")
        text = prose.paragraph(parts, gap=ask)
        # somebody who says two personal sentences in a row gets the same deflection twice, which reads like
        # a machine with one card. Adaptation is already how every other phrasing here avoids repeating; the
        # second one in a row is shorter and admits that it is the same answer.
        if self._last_deflection == text:
            return ("Same answer, I'm afraid - that is about you rather than about anything I hold. "
                    "I am better company once there is something on the map: tell me how two things relate.")
        self._last_deflection = text
        return text

    # ---------------------------------------------------------------- memory
    def _remember(self, p, result, sentence):
        """What prefrontal cortex holds after this turn. Written, not appended to a list."""
        a = p["a"][0] if p.get("a") else None
        b = p["b"][0] if p.get("b") else None
        if a:
            self.wm.hold("subject", a)
            self.wm.hold("topic", a)
        if b:
            self.wm.hold("object", b)
        if not a and not b:
            topic = self._topic_of(sentence)
            if topic:
                self.wm.hold("topic", topic)
        if p.get("axis"):
            self.wm.hold("relation", p["axis"])
        self.wm.hold("answer", result["reply"][:40])

    # ---------------------------------------------------------------- offline
    def sleep(self, quiet=False):
        """Consolidation. The episodic store proposes what to replay, the map relaxes under all of it at once,
        and experts that have stopped winning are let go."""
        moved = 0.0
        for space, m in self.maps.maps.items():
            before = m.pos.clone()
            m.sleep(self.maps.config["map_config"]["sleep"], tolerance=self.maps.config["map_config"]["tol"])
            delta = m.pos - before
            if len(delta):
                delta = delta - delta.mean(0)
                moved = max(moved, float(delta.norm(dim=1).max()))
        self.maps.dirty.clear()
        self.unslept = 0
        replayed = self.epi.replay(8)
        return speech.sleep_report(sum(len(m.episodes) for m in self.maps.maps.values()), moved, len(self.gate.pool))

    def forget(self):
        self.maps.reset()
        self.wm.clear()
        self.epi = Episodes(self.lex)
        self.unslept = 0

    # ------------------------------------------------------------------ status
    def summary(self):
        g = self.gate.stats()
        facts = sum(len(m.episodes) for m in self.maps.maps.values())
        return (f"{facts} fact(s) on {len(self.maps.maps)} map(s), {len(self.epi.records)} episode(s) remembered, "
                f"{g['experts']} gate expert(s) grown from use ({g['born']} born), "
                f"{self.footprint():,} synapses in all.")

    def footprint(self):
        return (self.wm.synapses() + self.epi.synapses() + self.gate.pool.synapses()
                + sum(m.footprint() for m in self.maps.maps.values()))

    def knowledge(self):
        """Everything it holds, as a graph: the things, what connects them, and where each came from.

        The per-space views are ladders and floor plans, which is right for answering a question and wrong for
        seeing what it knows. This is the other view: every item across every map, every stated fact as an edge
        labelled with its relation, each node carrying how far its position floats and each edge carrying
        whether it was told, read or worked out.
        """
        nodes, edges, seen = [], [], {}
        for space, m in self.maps.maps.items():
            for i, name in enumerate(m.names):
                if not m.knows(name) or name in seen:
                    continue
                seen[name] = len(nodes)
                facts = sum(1 for ea, _, eb in m.episodes if i in (ea, eb))
                nodes.append({"id": name, "name": self.maps._say(name), "space": space,
                              "spread": round(m.spread_of(name), 2), "facts": facts,
                              "source": self.me.origin(name) or "worked",
                              "mine": self.you.theirs(name)})
            for idx, (ea, moves, eb) in enumerate(m.episodes):
                a, b = m.names[ea], m.names[eb]
                if a not in seen or b not in seen or a == b:
                    continue
                rel = moves[0][0] if moves else "same as"
                word = self.maps.words.get(rel, rel)
                edges.append({"from": b, "to": a, "relation": rel, "word": word, "space": space,
                              "weight": m.weights[idx] if idx < len(m.weights) else 1.0,
                              "source": self.me.origin(a) or "worked"})
        # things the conversation has been about that the map barely holds - Kuala Lumpur after a page read
        # about it has one relation and twenty sentences of material - are still things it knows something of
        for entry in self.referents.ranked()[:24]:
            key = entry["name"].lower()
            if key not in seen and self._material(key):
                seen[key] = len(nodes)
                nodes.append({"id": key, "name": entry["name"], "space": sorted(entry["kinds"])[0]
                              if entry["kinds"] else "discussed", "spread": 2.0, "facts": 0,
                              "source": "read", "mine": False})
        # how well each thing is known, which is what the map's layout should show. A thing it has many
        # facts about that agree with each other, has read a lot about, and keeps coming back to is known;
        # a thing named once and placed from one fact is floating. The old layout put everything on a ring,
        # which drew the best-known and the barely-known as equals.
        for n in nodes:
            material = len(self._material(n["id"]))
            salience = (self.referents.entries.get(n["id"]) or {}).get("salience", 0.0)
            firm = 1.0 - min(n["spread"], 2.0) / 2.0
            n["material"] = material
            n["familiarity"] = round(min(1.0, 0.45 * firm + 0.25 * min(n["facts"], 6) / 6
                                         + 0.2 * min(material, 20) / 20 + 0.1 * min(salience, 3) / 3), 3)
        return {"nodes": nodes, "edges": edges,
                "spaces": sorted({n["space"] for n in nodes})}

    def state(self):
        return {"turns": self.turns, "facts": len(self.maps.log), "episodes": len(self.epi.records),
                "web": self.allow_web, "pages": self.read_pages[-6:], "inquiries": self.inquiries[-4:],
                "knowledge": self.knowledge(),
                "meta": self.meta.stats(), "plans": self.plans[-3:],
                "gate": self.gate.stats(), "synapses": self.footprint(), "tracts": self.tracts.spec(),
                "columns": [self.lex.column.spec(), self.epi.column.spec()],
                "held": {s: self.wm.recall(s) for s in ("topic", "subject", "object", "relation")},
                "maps": self.maps.view(), "trace": self.trace[-12:]}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/answer.py
"""The conversational surface: saying the answer first, and saying it like someone talking.

Everything upstream of this file decides *what is true and how sure it is*. This file decides *what the
reply sounds like*, and it has one rule: the first sentence answers the question. Before this existed the
reply led with the machinery - "I read Will Smith (url)", "I could turn 2 of it into facts I can use" - which
is the system narrating its own memory consolidation at someone who asked a question. That narration is not
wrong and it is worth keeping; it is just not the answer, so it goes to the reasoning channel and the answer
goes here.

Two jobs:

  opening      a short acknowledgement before the content, chosen by the same fatigue the rest of production
               uses so it does not say "good question" four times running, and skipped entirely when the turn
               does not want one. This is discourse marking, not knowledge - the same status as "well" or
               "so" - and it is what makes a reply read as a turn in a conversation rather than a record.
  answering    the content of the reply, front-loaded. What something *is* comes first, in one sentence,
               and everything qualifying it comes after. Hedging that arrives before the claim reads as
               refusal to answer, which is what "As for how much of this to trust" was doing at the top.
"""
import os
import pickle
import re
from collections import defaultdict

# openings, by what the turn is doing. Several each, because the fatigue in production.Voice needs something
# to fall back to, and an empty string in each list is what lets it sometimes just answer.
OPENINGS = {
    "answer": ["", "", "Good question - ", "So - ", "Right - "],
    "found": ["", "Just read up on this - ", "Had to go and look, but - ", "Here's what I found: "],
    "task": ["", "Sure - ", "Here you go. ", "Right, "],
    "stance": ["", "Honestly, ", "For what it's worth, ", "My read: "],
    "thin": ["", "Not much to go on here, but ", "I'm thin on this one - "],
}

# "(born September 25, 1968)" and friends. Innermost-first, applied a few times, so nesting comes out too.
# The obvious one-shot pattern for nested brackets backtracks exponentially, and a sentence from the article
# on Mount Everest - which is all parentheses - hung the whole system for minutes on a single answer.
TRIM = re.compile(r"\s*\([^()]*\)")
SPACES = re.compile(r"\s+")


class Markers:
    """How people actually open a turn, counted from real conversation.

    The list above is mine, and a list of discourse markers I wrote is exactly the kind of thing this project
    is not supposed to contain. What people say before answering is learnable and worth learning: it differs
    by what they are answering, which a transcript shows directly - the marker after somebody has asked a
    question is not the marker after somebody has made a statement. So the counts are conditioned on that,
    and the hand-written list stays only as what it says when it has heard nothing.
    """

    # a marker is made of these and nothing else: the moment a content word appears it is an answer, not an
    # opening, and repeating somebody's content back at them is not a discourse marker
    OPENERS = {"well", "oh", "so", "right", "yes", "no", "now", "ok", "okay", "sure", "actually", "look",
               "honestly", "good", "great", "true", "yeah", "hmm", "i", "think", "guess", "mean", "suppose",
               "question", "point", "of", "course", "that", "is", "a", "the", "very", "quite", "my", "to",
               "be", "fair", "let", "me", "see", "you", "know", "it", "depends", "certainly", "indeed"}
    # A marker of one word has to be one of these. Two exclusions matter. "I" opens most sentences in English
    # and marks nothing. "Yes" and "no" are not markers at all - they are answers, and a novel's dialogue is
    # full of them because novels are full of questions with answers. Opening a factual answer with "No - "
    # because that is what turned up most often after a question would be asserting something.
    PRIMARY = {"well", "oh", "so", "right", "now", "ok", "okay", "sure", "actually", "look",
               "honestly", "yeah", "hmm", "indeed", "certainly", "true"}
    # and a marker of more than one word has to be one of these. Discourse markers are a closed class, like
    # the fact that "than" follows a comparative; what is learned here is which of them are used, how often,
    # and after what - not the class itself.
    PHRASES = {"of course", "good question", "i mean", "you know", "to be fair", "let me see", "i think",
               "i suppose", "i guess", "that is true", "in fact", "as it happens", "to be honest"}

    def __init__(self):
        self.counts = defaultdict(lambda: defaultdict(float))    # context -> marker -> times heard

    def hear(self, turn, after_question):
        """The opening of one turn, if it has one: the run of discourse words before it says anything.

        The comma does most of the work. "Well, I think so" opens with "well" and then says something; without
        treating the comma as the end of the opening, the run swallows "I think" as well and the turn looks
        like nothing but marker. Where there is no comma, the opening is the leading run of discourse words,
        and there has to be a sentence left after it or it was not an opening to anything.
        """
        text = str(turn).strip()
        head, sep, tail = re.match(r"([^,!–—-]*)([,!–—-]?)(.*)", text, re.S).groups()
        words = re.findall(r"[A-Za-z']+", text.lower())
        if sep and head and all(w in self.OPENERS for w in re.findall(r"[A-Za-z']+", head.lower()))                 and 0 < len(head.split()) <= 3 and len(re.findall(r"[A-Za-z']+", tail)) >= 3:
            run = re.findall(r"[A-Za-z']+", head.lower())
        else:
            run = []
            for w in words[:3]:
                if w in self.OPENERS:
                    run.append(w)
                else:
                    break
            if len(words) - len(run) < 3:
                run = []
        marker = " ".join(run)
        if not run or (marker not in self.PRIMARY if len(run) == 1 else marker not in self.PHRASES):
            return None
        self.counts["after_question" if after_question else "after_statement"][marker] += 1.0
        return marker

    def hear_many(self, turns):
        heard = 0
        for i, turn in enumerate(turns):
            previous = turns[i - 1] if i else ""
            if self.hear(turn, str(previous).rstrip().endswith("?")):
                heard += 1
        return heard

    def best(self, context, top=6, floor=3.0):
        """The commonest openings in this situation, as text ready to sit in front of an answer."""
        counts = self.counts.get(context) or {}
        ranked = [(m, n) for m, n in sorted(counts.items(), key=lambda kv: -kv[1]) if n >= floor][:top]
        return [m[0].upper() + m[1:] + (", " if len(m.split()) > 1 else " - ") for m, _ in ranked]

    def stats(self):
        return {c: len(v) for c, v in self.counts.items()}

    def save(self, path):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump({c: dict(v) for c, v in self.counts.items()}, f)
        return os.path.getsize(path)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            d = pickle.load(f)
        m = cls()
        for context, counts in d.items():
            m.counts[context] = defaultdict(float, counts)
        return m


def openings_for(kind, markers=None):
    """What it could say before an answer of this kind: learned if it has heard any, mine if it has not."""
    if markers is not None:
        context = "after_question" if kind in ("answer", "found", "stance") else "after_statement"
        learned = markers.best(context)
        if learned:
            return [""] + learned          # saying nothing at all stays an option, as it is in speech
    return OPENINGS.get(kind, OPENINGS["answer"])


def opening(voice, kind, markers=None):
    """A discourse marker, suppressed when it has just been used - same adaptation as every other phrasing."""
    options = openings_for(kind, markers)
    return options[voice.pick(("open", kind), len(options))]


def gloss(sentence, term=None, keep_parentheses=False):
    """A read sentence turned into an answer: the definition, without the encyclopedia's asides.

    A first sentence from a reference page is almost always a definition with the citation apparatus baked in -
    birth dates, native spellings, pronunciation. Those are fine in a source and wrong in an answer, so the
    parentheticals come out and what is left is the sentence that answers "who is this".
    """
    text = str(sentence or "").strip()
    if not text:
        return ""
    if not keep_parentheses:
        for _ in range(3):
            shorter = TRIM.sub("", text)
            if shorter == text:
                break
            text = shorter
    # a bracket the sentence never closes - the page opened it and carried on into the next sentence - takes
    # everything after it with it, or the answer reads as though it were cut off mid-aside
    if text.count("(") > text.count(")"):
        cut = text.find("(")
        rest = text[cut:]
        # the aside runs until the sentence picks its verb back up: "Mount Everest (known in Nepali as
        # Sagarmatha, in Sherpa as Chamolangma ... is the highest mountain" is about a mountain, not about
        # what it is called in three languages
        verb = re.search(r"\b(?:is|was|are|were|refers|lies|stands|has)\b", rest)
        end = verb.start() if verb else next((i for i in (rest.find(","), rest.find(";")) if i > 0), len(rest))
        text = (text[:cut] + rest[end:]).strip()
    # an em dash out of a web page arrives as whatever encoding the page used; a hyphen always reads
    text = text.replace(chr(8212), " - ").replace(chr(8211), " - ").replace(chr(160), " ")
    text = SPACES.sub(" ", text).replace(" ,", ",").strip()
    if term and text.lower().startswith(term.lower()):
        pass                       # already leads with what was asked about, which is what we want
    return text if text.endswith((".", "!", "?")) else text + "."


def lead(voice, kind, content, *rest, markers=None):
    """Opening, then the answer, then whatever qualifies it - in that order and never the other way round."""
    body = [str(x).strip() for x in ([content] + list(rest)) if x and str(x).strip()]
    if not body:
        return ""
    mark = opening(voice, kind, markers)
    head = body[0]
    if not mark and head[:1].islower():
        head = head[0].upper() + head[1:]
    return "\n\n".join([mark + head] + body[1:])


def also(facts_lines, limit=2):
    """The second sentence of an answer: what else is held, said as a continuation rather than a list."""
    lines = [l for l in facts_lines if l][:limit]
    if not lines:
        return ""
    if len(lines) == 1:
        return lines[0]
    joined = lines[0].rstrip(".") + ", and " + (lines[1][0].lower() + lines[1][1:]).rstrip(".")
    return joined + "."


PLACE = r"(?P<the>the )?(?P<place>[A-Z][\w.'-]*(?:\s+(?:of\s+)?[A-Z][\w.'-]*){0,3})"
WHERE = [
    # "the capital city and a federal territory of Malaysia", "the federal capital city of the United States"
    re.compile(r"capital(?: city)?(?: and [\w ]{0,40}?)? (?:of|in) " + PLACE),
    # "a union territory of India", "a city in Pennsylvania", "a province of Canada"
    re.compile(r"\b(?:city|town|state|province|region|country|territory|district|county|municipality|"
               r"island|village|suburb|borough|prefecture) (?:in|of) " + PLACE),
    re.compile(r"\blocated (?:in|on) " + PLACE),
    re.compile(r"\bin (?:the )?(?:north|south|east|west|central|northern|southern|eastern|western|"
               r"north-eastern|north-western|south-eastern|south-western)(?:ern)? (?:part of )?" + PLACE),
]


def locate(sentences):
    """Where a thing is, read out of what was read about it - or None if the reading never says.

    A question that starts with "where" wants a place first. The encyclopedia puts the place in the middle of
    a definition ("is the capital city and a federal territory of Malaysia"), so the answer has to find it
    there rather than recite the definition and hope the person spots it.
    """
    for sentence in sentences[:4]:
        text = gloss(sentence)
        for pattern in WHERE:
            m = pattern.search(text)
            if m:
                place = m.group("place").strip(" .,")
                tail = re.search(r"state of " + PLACE, text)
                if tail and place in ("U.S", "US", "United States"):
                    place = tail.group("place").strip(" .,")
                if place and place.split()[0].lower() not in ("the", "a", "an"):
                    # "the United States", not "United States": the article is part of how the place is named
                    return ("the " if m.group("the") and place not in ("U.S", "US") else "") + place
    return None

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/codes.py
"""Lexical cortex: the only way anything enters this system.

A word is not a string here and not a row in a pretrained table - it is a sparse pattern of active units, produced
by fixed random wiring plus k-winners-take-all, exactly as in the mushroom body and the dentate gyrus. Two
properties follow, and everything downstream depends on both:

  overlap means similarity   words sharing letter trigrams drive overlapping inputs, so "dog" and "dogs" land on
                             codes that share units. Nothing was trained to make that true; it falls out of the
                             wiring, which is what morphology-by-similarity looks like in tissue.
  different means separate   unrelated words barely overlap, so binding one into memory does not disturb another.

The vocabulary is unbounded: a word is hashed into a fixed number of input slots, so a word never seen before has
a code the moment it is heard. There is no table to grow and no out-of-vocabulary case.
"""
import zlib

import torch

from brain.column import Column

SLOTS = 4096       # input fibres the lexicon projects onto
N_UNITS = 8192     # lexical cortex units
K_ACTIVE = 64      # how many fire: 0.8%, the sparsity cortex runs at


def _slot(text, salt=0):
    """A stable hash - Python's own is salted per process, and a brain's wiring does not change overnight."""
    return zlib.crc32(text.encode("utf-8"), salt) % SLOTS


def trigrams(word):
    w = f"<{word}>"
    return [w[i:i + 3] for i in range(len(w) - 2)] or [w]


class Lexicon:
    """Words and phrases -> sparse codes. One cortical column, wired to the input fibres; never trained.

    Because summation inside a branch is supralinear, a term whose parts arrive together on the same branch
    drives its units harder than the same evidence spread thinly - so a phrase is not the average of its words.
    """

    def __init__(self, n=N_UNITS, k=K_ACTIVE, seed=0, homeostasis=0.01, branches=4):
        self.column = Column("lexical", SLOTS, n=n, k=k, branches=branches, seed=seed, homeostasis=homeostasis)
        self.expand = lambda v: self.column.encode(v)
        self.n, self.k = n, k
        self._cache = {}

    def drive(self, term):
        """The input pattern a term causes: its identity, plus its letter trigrams so morphology overlaps."""
        v = torch.zeros(SLOTS)
        for word in term.lower().split():
            v[_slot(word, 1)] += 1.5                 # the word itself, weighted a little above its parts
            for g in trigrams(word):
                v[_slot(g, 2)] += 1.0                # its letter trigrams: what makes "dog" and "dogs" overlap
        return v

    def code(self, term):
        """A term -> its sparse code (n,), k units active."""
        if term not in self._cache:
            self._cache[term] = self.column.encode(self.drive(term))
        return self._cache[term]

    def bag(self, terms, weights=None):
        """Several terms at once -> one sparse code. Used for an utterance: the code of what was just said.

        The drives are summed before the winner-take-all, not the codes afterwards, so the result is a genuine
        population response to the whole phrase rather than a union of word codes - overlapping evidence adds up
        and the k winners are whatever the phrase as a whole drives hardest.
        """
        if not terms:
            return torch.zeros(self.n)
        v = torch.zeros(SLOTS)
        for t, w in zip(terms, weights or [1.0] * len(terms)):
            v += w * self.drive(t)
        return self.column.encode(v)

    def nearest(self, code, candidates, top=3):
        """Population readout: which known terms this pattern overlaps most. How a code becomes a word again."""
        if not candidates:
            return []
        scored = [(t, float(self.code(t) @ code) / self.k) for t in candidates]
        scored.sort(key=lambda x: -x[1])
        return scored[:top]

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/column.py
"""The canonical microcircuit, which every area is made of.

Cortex is not a collection of bespoke machines. The same six-layer circuit is repeated across areas that do
utterly different jobs - vision, touch, planning - and what differs between them is what they are wired to, how
big they are and how they are tuned, not the circuit. Subcortical structures repeat their own motif the same
way. That uniformity is the strongest architectural fact about the brain, and this file is the project taking it
literally: one `Column`, used for every store.

  input layer (L4)      dendritic expansion - coincidence detection on branches, divisive gain control, k-WTA
  local layer (L2/3)    lateral inhibition: only the k best-driven units carry the pattern onward
  output layer (L5)     a local associative readout - Hebbian counts from the active units to whatever the
                        column is being asked to produce, and nothing else

A column that stores patterns (episodic), a column that names them (lexical), a column that selects an action
(the gate) are the same object with different sizes and different targets. If the microcircuit is wrong, it is
wrong everywhere at once, which is the point: it can be tested in one place.
"""
import torch

from brain.dendrite import Dendrites


class Column:
    def __init__(self, name, d_in, n=8192, k=64, branches=4, n_out=0, seed=0, **kw):
        self.name, self.d_in, self.n, self.k = name, d_in, n, k
        self.dendrites = Dendrites(d_in, n=n, k=k, branches=branches, seed=seed, **kw)
        self.out = torch.zeros(n, n_out) if n_out else None     # L5: local Hebbian associations
        self.fired = 0

    def encode(self, x, learn=True):
        """Drive -> sparse code. The only way anything enters a column."""
        single = x.dim() == 1
        code = self.dendrites(x[None] if single else x, learn=learn)
        self.fired += code.shape[0]
        return code[0] if single else code

    def associate(self, code, target, strength=1.0):
        """Hebbian: the units that fired grow their synapses onto what was present at the same moment."""
        if self.out is None:
            raise ValueError(f"column {self.name} has no output layer")
        self.out[code > 0, target] += strength

    def read(self, code):
        return code @ self.out if self.out is not None else None

    def spec(self):
        s = {"column": self.name, "inputs": self.d_in, **self.dendrites.spec(), "fired": self.fired}
        if self.out is not None:
            s["output_units"] = int(self.out.shape[1])
            s["output_synapses"] = int((self.out != 0).sum())
        return s

    def synapses(self):
        return int(self.dendrites.idx.numel()) + (int((self.out != 0).sum()) if self.out is not None else 0)

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/compose.py
"""Composing what it has into something worth reading.

The planner decides what to say and in what order; this decides how it lands on the page. They are different
jobs and keeping them apart is what stops either from becoming a template bank: the planner never writes a
word, and this never invents a claim.

Three things make the difference between notes and prose, and none of them needs a language model:

  joining       facts about the same thing said one to a line are notes. Two facts sharing a subject belong in
                one sentence with "and"; a third belongs in the next sentence starting "It"; a contrast wants
                "but" rather than "and". That is grammar applied to structure, which is what a speaker does.
  pruning       a fragment is worse than a gap. "It is connected to block parties in the" is what comes of
                saying whatever the extractor produced, so anything that ends mid-phrase is dropped rather
                than dressed up.
  shape         an essay, a summary and an explanation are different objects. Asked for a summary, the answer
                is one paragraph of the firmest facts; asked for an essay, it is several paragraphs with the
                evidence and the gaps; asked for a list, it is a list. Answering the wrong shape is a way of
                not doing what was asked.
"""
import re

DANGLING = ("of", "a", "an", "the", "and", "or", "in", "to", "for", "with", "from", "by", "as", "that",
            "which", "is", "was", "at", "on", "into", "about")


def usable(text):
    """Is this a phrase that can be said, or a fragment the extractor left half-finished?"""
    if not text:
        return False
    words = str(text).split()
    # facts learned before the span checks got stricter are still on the map; this is the last guard before
    # one of them is said out loud - a trailing bare number is a date cut in half
    if words and words[-1].isdigit() and len(words) > 1:
        return False
    return bool(words) and words[-1].lower() not in DANGLING and len(words) <= 6


def tidy(text):
    """Capitalise the opening, close the sentence, and leave the rest alone."""
    text = str(text).replace(chr(8212), " - ").replace(chr(8211), " - ").replace(chr(160), " ")
    text = re.sub(r"\s+", " ", text.strip())
    if not text:
        return ""
    text = text[0].upper() + text[1:]
    return text if text.endswith((".", "!", "?")) else text + "."


def sentences_about(subject, facts, say_fact, say_name, limit=4):
    """Facts about one thing, joined the way someone would say them rather than listed.

    The first two go in one sentence because they are about the same subject; the rest continue with "It",
    which is what a pronoun is for. Anything that reads as a fragment is dropped.
    """
    clauses = []
    for relation, others in list(facts.items()):
        named = [say_name(o) for o in others if usable(say_name(o))][:2]
        if not named:
            continue
        clauses.append((relation, " and ".join(named) if len(named) == 2 else named[0]))
        if len(clauses) >= limit:
            break
    if not clauses:
        return []
    out = []
    first = say_fact(clauses[0][0], subject, clauses[0][1])
    if len(clauses) > 1 and clauses[1][0] == clauses[0][0]:
        # the same relation twice: one sentence, joined, rather than two that sound like a stutter
        first = say_fact(clauses[0][0], subject, f"{clauses[0][1]} and {clauses[1][1]}")
        clauses = [clauses[0]] + clauses[2:]
    out.append(tidy(first))
    for relation, named in clauses[1:]:
        # the subject again rather than "it": a pronoun for a person reads as a thing, and this has no way of
        # knowing which it is dealing with. The join below takes the repetition back out when the two clauses
        # end up in one sentence, which is where a pronoun would have been doing its work anyway.
        out.append(tidy(say_fact(relation, subject, named)))
    return out


def joined(lines, connective="and", subject=None):
    """Two short sentences that belong together, made into one.

    "X is a dog. X is older than Max." is two facts about one thing said as though they were about two, and
    the fix an English speaker applies without thinking is to say the subject once: "X is a dog and is older
    than Max." So if the second clause opens with the same subject, it comes out in the join.
    """
    if len(lines) < 2:
        return lines
    first, second = lines[0].rstrip("."), lines[1]
    if subject and second.lower().startswith(subject.lower() + " "):
        second = second[len(subject) + 1:]
    else:
        second = second[0].lower() + second[1:]
    return [tidy(f"{first}, {connective} {second.rstrip('.')}")] + lines[2:]


SHAPES = {
    "summary": {"paragraphs": 1, "facts": 3, "evidence": False, "gaps": False},
    "explanation": {"paragraphs": 2, "facts": 4, "evidence": True, "gaps": True},
    "note": {"paragraphs": 1, "facts": 3, "evidence": False, "gaps": True},
    "paragraph": {"paragraphs": 1, "facts": 4, "evidence": False, "gaps": True},
    "list": {"paragraphs": 0, "facts": 6, "evidence": False, "gaps": False},
    "essay": {"paragraphs": 4, "facts": 5, "evidence": True, "gaps": True},
    "analysis": {"paragraphs": 4, "facts": 5, "evidence": True, "gaps": True},
    "report": {"paragraphs": 3, "facts": 5, "evidence": True, "gaps": True},
    "account": {"paragraphs": 3, "facts": 4, "evidence": True, "gaps": True},
    "writeup": {"paragraphs": 3, "facts": 4, "evidence": True, "gaps": True},
    "piece": {"paragraphs": 3, "facts": 4, "evidence": True, "gaps": True},
}


def shape_of(deliverable):
    return dict(SHAPES.get((deliverable or "essay").lower(), SHAPES["essay"]))


def opening(deliverable, topic, chose, facts_held, pages):
    """Say what is being delivered, so the reader knows the task was understood before reading it."""
    what = {"summary": "a short summary", "explanation": "an explanation", "list": "a list",
            "note": "a note", "paragraph": "a paragraph"}.get(deliverable, f"a short {deliverable or 'piece'}")
    basis = f"{facts_held} fact(s) I hold" if facts_held else "very little"
    read = f" and {pages} page(s) I went and read" if pages else ""
    why = " - I picked it because it is what I hold most about" if chose else ""
    preposition = "of" if deliverable in ("summary", "list", "account") else "on"
    return tidy(f"{what.capitalize()} {preposition} {topic}{why}, from {basis}{read}")

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/dendrite.py
"""Dendritic summation: what a neuron does to its inputs before it decides to fire.

The expansion this project has used until now sums every synapse linearly and takes the k largest. A real neuron
does neither of those things, and the two differences change what the code means:

  non-linear summation   synapses landing close together on the same dendritic branch interact. Two inputs on
                         one branch produce more than twice one input (NMDA-dependent supralinearity), and the
                         branch then saturates. Two inputs on *different* branches just add. So a unit is not a
                         weighted sum - it is a small layer of coincidence detectors, one per branch, and what
                         it is tuned to is a *combination* of inputs arriving together on the same branch.
  suppression            inhibition here is not a bigger negative number. Perisomatic (PV-like) inhibition
                         divides the whole drive by how much the population as a whole is active - gain control,
                         so a loud input does not simply recruit more units. Dendrite-targeting (SST-like)
                         inhibition subtracts from *specific* branches that have been active recently, which is
                         input-specific adaptation rather than global quieting.

Both are local: a branch sees only its own synapses, the divisive term sees only the population mean, and the
adaptation term sees only each unit's own recent activity. Nothing is trained.
"""
import math

import torch


class Dendrites:
    """d_in -> n units with `branches` dendritic branches each; exactly k fire."""

    def __init__(self, d_in, n=8192, k=64, fan_in=12, branches=4, seed=0, theta=1.0, divisive=0.5,
                 adapt=0.0, homeostasis=0.0):
        g = torch.Generator().manual_seed(int(seed))
        self.d_in, self.n, self.k, self.branches = d_in, n, k, branches
        # every input must reach enough units, or a lone active input cannot fill the k winners and the rest are
        # ties at zero - the same filler units for every input, which makes codes collide
        fan_in = max(fan_in, branches, math.ceil(2 * k * d_in / n))
        fan_in += (-fan_in) % branches                      # divide evenly between branches
        wiring = torch.arange(n * fan_in) % d_in
        self.idx = wiring[torch.randperm(n * fan_in, generator=g)].reshape(n, branches, fan_in // branches)
        self.w = 0.5 + torch.rand(n, branches, fan_in // branches, generator=g)
        self.theta, self.divisive, self.adapt = theta, divisive, adapt
        self.bias = torch.zeros(n)
        self.fatigue = torch.zeros(n, branches)             # SST-like, branch-specific, from recent activity
        self.rate = torch.full((n,), k / n)
        # how long each cell has existed and how often it has fired. A newborn granule cell has a window in
        # which something must recruit it; the ones nothing recruits die. That window is what these count.
        self.age = torch.zeros(n)
        self.uses = torch.zeros(n)
        self.homeostasis = homeostasis

    @torch.no_grad()
    def __call__(self, x, learn=True):
        """x (B, d_in) -> binary code (B, n), exactly k ones."""
        s = (x[:, self.idx] * self.w).sum(-1)               # (B, n, branches): drive per branch
        s = (s - self.fatigue).clamp_min(0.0)               # branch-specific suppression
        # supralinear then saturating: s^2 / (theta + s). Small inputs are discounted, coincident ones on the
        # same branch multiply, and a branch cannot shout indefinitely.
        branch = s * s / (self.theta + s)
        drive = branch.sum(-1) - self.bias                  # branches add linearly at the soma
        if self.divisive:                                   # PV-like gain control: divide by how loud the field is
            drive = drive / (1.0 + self.divisive * drive.mean(dim=1, keepdim=True))
        code = torch.zeros(x.shape[0], self.n)
        code.scatter_(1, drive.topk(self.k, dim=1).indices, 1.0)
        if learn:
            self.age += 1
            self.uses += code.sum(0)
        if learn and self.adapt:                            # branches that just fired are harder to drive next
            active = (code[..., None] * (branch > 0).float())
            self.fatigue.mul_(1 - self.adapt).add_(self.adapt * active.mean(0) * s.mean(0))
        if learn and self.homeostasis:
            self.rate.mul_(0.99).add_(0.01 * code.mean(0))
            self.bias.add_(self.homeostasis * (self.rate - self.k / self.n))
        return code

    @torch.no_grad()
    def recruit(self, drive, count=8, excitability=0.25):
        """Grow new units wired to whatever is active right now - neurogenesis, with synaptogenesis onto the
        fibres that are carrying the current input.

        Adult neurogenesis is not general-purpose growth. It happens in the dentate gyrus, the new granule
        cells arrive hyperexcitable and broadly tuned, and they are recruited into whatever is being encoded
        while they are young - which is why the dentate is where it happens: the job there is keeping similar
        experiences apart, and a fresh population that nothing else has claimed is the cleanest way to give a
        new-but-similar memory a code of its own. The excitability decays as they mature, so a cell that has
        been used settles into being an ordinary one.
        """
        g = torch.Generator().manual_seed(int(self.n) + count)
        live = drive.reshape(-1).nonzero().squeeze(1)
        if not len(live):
            return 0
        per = self.idx.shape[2]
        # wire each new cell mostly onto the active fibres, with a few elsewhere so it is not a copy
        picks = live[torch.randint(len(live), (count, self.branches, per), generator=g)]
        stray = torch.randint(self.d_in, (count, self.branches, per), generator=g)
        mask = torch.rand(count, self.branches, per, generator=g) < 0.75
        self.idx = torch.cat([self.idx, torch.where(mask, picks, stray)])
        self.w = torch.cat([self.w, 0.5 + torch.rand(count, self.branches, per, generator=g)])
        self.bias = torch.cat([self.bias, torch.full((count,), -excitability)])   # young: easier to fire
        self.fatigue = torch.cat([self.fatigue, torch.zeros(count, self.branches)])
        # a newborn cell has fired nothing yet, and saying otherwise is what kept the useless ones alive:
        # its rate starts at zero, so if nothing recruits it within a few exposures it is culled
        self.rate = torch.cat([self.rate, torch.zeros(count)])
        self.age = torch.cat([self.age, torch.zeros(count)])
        self.uses = torch.cat([self.uses, torch.zeros(count)])
        self.n += count
        self.born = getattr(self, "born", 0) + count
        return count

    @torch.no_grad()
    def mature(self, rate=0.05):
        """Young cells lose their excitability as they are used, and become ordinary members of the population."""
        young = self.bias < 0
        if bool(young.any()):
            self.bias[young] = torch.clamp(self.bias[young] + rate, max=0.0)
        return int(young.sum())

    def spec(self):
        return {"units": self.n, "active": self.k, "branches": self.branches,
                "synapses_per_branch": int(self.idx.shape[2]), "divisive": self.divisive, "adapt": self.adapt,
                "born": getattr(self, "born", 0), "died": getattr(self, "died", 0),
                "young": int((self.bias < 0).sum())}

    @torch.no_grad()
    def cull(self, floor=0.2, keep_min=256, limit=64, window=6):
        """Kill the cells that were grown and never taken up. Half of what is born is meant to die.

        Development overproduces and lets activity decide. A new cell arrives hyperexcitable and has a window
        in which something must recruit it into a code; the ones nothing recruits are removed, and the ones
        that fired with something stay and mature. That is not tidying up after growth - it is the other half
        of the same mechanism, and without it growth is a leak that dilutes every code it is added to.

        Age is counted in exposures rather than by a flag, because being under-driven is not the same as being
        new: homeostasis leaves plenty of old cells quiet, and killing those would be removing the population
        that already does the work.
        """
        if self.n <= keep_min:
            return 0
        share = self.k / self.n
        grown_up = self.age > window
        taken_up = (self.uses / self.age.clamp(min=1.0)) >= floor * share
        doomed = (grown_up & ~taken_up).nonzero().flatten()
        if not len(doomed):
            return 0
        # the least used go first, so a cull that hits its limit removes the worst rather than the earliest
        order = (self.uses[doomed] / self.age[doomed].clamp(min=1.0)).argsort()
        doomed = doomed[order][:min(limit, self.n - keep_min)]
        keep = torch.ones(self.n, dtype=torch.bool)
        keep[doomed] = False
        # whoever holds a matrix with one row per unit - the projection to the layer above, for instance -
        # has to drop the same rows, so the mask is left where they can find it
        self.last_keep = keep
        self.idx, self.w = self.idx[keep], self.w[keep]
        self.bias, self.fatigue, self.rate = self.bias[keep], self.fatigue[keep], self.rate[keep]
        self.age, self.uses = self.age[keep], self.uses[keep]
        self.n = int(keep.sum())
        self.died = getattr(self, "died", 0) + int(len(doomed))
        return int(len(doomed))

    @torch.no_grad()
    def recruit_conjunctive(self, parts, count=None, excitability=0.05):
        """Grow cells that detect a *combination*, by putting its ingredients on one dendritic branch.

        What tells "Rex is older than Max" from "Max is older than Rex" is not any word - both have the same
        words - but a conjunction: Rex *at the start*. A cell whose synapses from "Rex" and from "first
        position" sit on different branches adds them linearly and fires for anything with either, which is
        how growth made separation worse. On one branch they are supralinear: the branch only really drives
        the cell when both arrive together. That is what coincidence detection on dendrites is for, and it is
        the only way random wiring can produce a cell selective for one member of a near-identical pair.

        `parts` is a list of drive vectors, one per conjunction wanted. Each new cell takes one, and every one
        of its branches samples its fibres from that conjunction alone.
        """
        # each part is a tuple of ingredients - a word and a position - and every branch takes an equal share
        # of its synapses from each. Pooling them first let a branch land entirely on position fibres, which
        # every sentence drives, and that cell fired for everything.
        parts = [tuple(x.reshape(-1) for x in (p if isinstance(p, (tuple, list)) else (p,))) for p in parts]
        parts = [p for p in parts if all(float(x.sum()) > 0 for x in p)]
        if not parts:
            return 0
        count = count or len(parts)
        per = self.idx.shape[2]
        g = torch.Generator().manual_seed(int(self.n) * 7 + count)
        rows = []
        for c in range(count):
            ingredients = [x.nonzero().squeeze(1) for x in parts[c % len(parts)]]
            share = [per // len(ingredients) + (1 if i < per % len(ingredients) else 0)
                     for i in range(len(ingredients))]
            branch = [torch.cat([live[torch.randint(len(live), (n,), generator=g)]
                                 for live, n in zip(ingredients, share)]) for _ in range(self.branches)]
            rows.append(torch.stack(branch))
        self.idx = torch.cat([self.idx, torch.stack(rows)])
        self.w = torch.cat([self.w, 0.9 + 0.2 * torch.rand(count, self.branches, per, generator=g)])
        self.bias = torch.cat([self.bias, torch.full((count,), -excitability)])
        self.fatigue = torch.cat([self.fatigue, torch.zeros(count, self.branches)])
        self.rate = torch.cat([self.rate, torch.zeros(count)])
        self.age = torch.cat([self.age, torch.zeros(count)])
        self.uses = torch.cat([self.uses, torch.zeros(count)])
        self.n += count
        self.born = getattr(self, "born", 0) + count
        return count

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/discourse.py
"""Discourse: the part of talking that is not about the world.

A conversation is not a sequence of independent questions. People drop most of a sentence when the rest is
obvious ("and Bob?"), ask for grounds ("why?"), check a claim ("are you sure?"), correct what was just said
("no, Rex is older"), and expect the other side to occasionally want something. None of that is about relations
between things, and none of it was possible while every turn was parsed from scratch.

What makes it work here is the task set - prefrontal cortex holding not just *what* is being discussed but *what
is being done with it*. "And Bob?" is answerable only because the question form from the previous turn is still
held: swap one argument, keep the rest. That is the same held representation that lets a person keep applying a
rule to new items without being told the rule again.

Initiative comes from the map's own shape rather than from a script: two groups of facts that nothing links are
exactly where a question would help most, so that is what it asks about.
"""
import re

FOLLOW_UP = (re.compile(r"^(?:and|what about|how about|ok(?:ay)?,? and)\s+(.+?)\s*\??$", re.I),)
WHY = re.compile(r"^(why|why is that|how do you know|how come|says who|on what grounds)\b", re.I)
SURE = re.compile(r"^(are you sure|really|is that right|you sure|certain\??|"
                  r"how (sure|confident|certain) (are you|is that)|how do you rate)\b", re.I)
AGREE = re.compile(r"^(yes|yeah|yep|right|correct|exactly|that's right|true)\b", re.I)
# "actually no, Ollie is older than Rex" is a correction with a softener in front of it. Missing the softener
# leaves the refusal attached to the fact and files a thing called "no, Ollie", which is worse than missing
# the correction altogether.
SOFTENER = r"(?:actually|wait|hang on|sorry|hmm|hold on|oh)?[,\s]*"
DISAGREE = re.compile(r"^" + SOFTENER + r"(no|nope|wrong|that's wrong|not true|incorrect|that's not right)\b",
                      re.I)
THANKS = re.compile(r"^(thanks|thank you|cheers|ta)\b", re.I)
SORRY = re.compile(r"^(sorry|my bad|apologies)\b", re.I)
GREET = re.compile(r"^(hi|hey|hello|good (morning|evening|afternoon)|yo|howdy)\b", re.I)
BYE = re.compile(r"^(bye|goodbye|see you|good night|later)\b", re.I)
SELF_REPORT = re.compile(r"^(how (was|is|has) your (day|week|time)|how('s| is| are) (it going|things|you)|"
                         r"what have you (been up to|been doing|done|learned|learnt)|what's new|"
                         r"how are you doing|tell me about your (day|self)|"
                         r"what (did|have) you (do|been doing|done|get up to)|"
                         r"(was|were) (any of )?(it|that|this|they) (hard|difficult|tricky|easy|interesting)|"
                         r"what (was|has been) (hard|difficult|tricky|the hard part)|"
                         r"how (did|has) (it|that|today) (go|gone|been)|what was (it|that) like)", re.I)
ABOUT_YOU = re.compile(r"^(who|what) are you|^how do you work|^what can you do|"
                       r"^(who|what) am i\b|^what do you know about me|^are you (an? )?(ai|llm|robot|human)|"
                       r"\b(you|your)\b.*\b(think|feel|like|want|remember|made)\b|"
                       r"^(what|when) (have|did) you got? .*wrong|^what did you get wrong", re.I)


VIEW = re.compile(r"^(what do you (make of|think of|think about|reckon)|"
                  r"what('s| is) your (view|take|opinion|sense|impression)|"
                  r"how do you (feel|rate|see) |what do you think (of|about) |"
                  r"which (one )?(should|would)|what (would|should) (i|we) |what else (would|could|do you) |"
                  r"(what|anything) else (would|could) help|how would you|do you have (a view|thoughts|an opinion))",
                  re.I)
REMEMBER = re.compile(r"^(do you |can you )?remember (what|when|that|anything)?\s*(i|you|we)?\s*(said|told)?"
                      r"\s*(about)?\s*(.+)?$", re.I)


def act(text):
    """The dialogue act of an utterance, before anything is looked up. None means 'about the world'."""
    t = text.strip()
    for pattern in FOLLOW_UP:
        m = pattern.match(t)
        if m and len(m.group(1).split()) <= 4:
            return "follow-up", m.group(1).strip(" ?.")
    if VIEW.match(t):
        return "view", None
    m = REMEMBER.match(t)
    if m and m.group(6):
        return "remember", m.group(6).strip(" ?.")
    if SELF_REPORT.match(t):
        return "report", None
    if WHY.match(t):
        return "why", None
    if SURE.match(t):
        return "check", None
    if THANKS.match(t):
        return "thanks", None
    if SORRY.match(t):
        return "sorry", None
    if GREET.match(t):
        return "greet", None
    if BYE.match(t):
        return "bye", None
    if DISAGREE.match(t):
        return "disagree", t
    if AGREE.match(t):
        return "agree", None
    if ABOUT_YOU.search(t):
        return "about-you", None
    return None, None


class TaskSet:
    """What is being done, held across turns so an incomplete sentence can borrow it.

    Prefrontal cortex holds a task rule, not just a topic: given "is X older than Y", the rule is "compare by
    age", and it survives until something replaces it. Holding it is what makes "and Bob?" a question at all.
    """

    def __init__(self):
        self.question = None      # the last full question, as parsed
        self.answer = None        # what was said back, with its grounds
        self.offered = 0          # the turn the brain last asked something of its own

    def hold(self, parsed, answer=None):
        if parsed.get("kind") == "ask":
            self.question = dict(parsed)
        if answer is not None:
            self.answer = answer

    def fill(self, term):
        """'and Bob?' -> the held question with one argument swapped. None if there is no rule to apply."""
        if not self.question:
            return None
        q = dict(self.question)
        key = (term.lower().strip(), term.strip())
        # the new term replaces whichever argument was *not* the one the question was mainly about: for
        # "is Alice older than Bob", "and Cara?" means Alice against Cara
        if q.get("b"):
            q["b"] = key
        elif q.get("a"):
            q["a"] = key
        else:
            q["a"] = key
        return q


def curiosity(maps, task, turn, every=4):
    """Something worth asking about, or None. The map's own gaps decide: two groups of facts with nothing
    between them cannot be compared, and one fact would join them."""
    if turn - task.offered < every:
        return None
    for space, m in maps.maps.items():
        if space in ("kind", "involved", "origin"):
            continue          # categories and associations are not scales; "compare by kind" means nothing
        groups = {}
        for n in m.names:
            if m.knows(n):
                groups.setdefault(m.island[m.index[n]], []).append(n)
        if len(groups) > 1:
            (_, first), (_, second) = sorted(groups.items())[:2]
            task.offered = turn
            dim = "where" if space == "place" else space
            if space == "place":
                return f"Where is {second[0]} compared with {first[0]}? Nothing you've told me links them."
            return f"How do {first[0]} and {second[0]} compare by {dim}? I can't place them on the same scale yet."
    return None

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/episodic.py
"""Hippocampal episodic store: every utterance, kept once and recalled from a partial cue.

The map holds what is true. This holds what happened - who said what, in which words, at which turn. They are
different memories in the brain and they are different here, which is the whole point of the redesign: one store
was being asked to give content words broad neighbourhoods and frame words sharp ones, and no single width can.

  pattern separation   an utterance is expanded into a sparse code by fixed wiring, so two similar sentences get
                       codes that overlap only as far as they genuinely share words (dentate gyrus).
  pattern completion   recall settles: the best-matching memories vote, the vote is re-sharpened to exactly k
                       active units, and the match is run again. A cue that is partly wrong still lands on the
                       right memory once its neighbours agree (CA3 recurrence).
  no overwriting       storing is appending a code. Nothing is averaged, so one odd sentence cannot smear the
                       memory of another - the failure mode that dense stores have and this does not.
"""
import torch

from brain.column import Column


class Episodes:
    def __init__(self, lexicon, n=16384, k=96, seed=5, branches=6):
        self.lex = lexicon
        # the same column as everywhere else, wired to the lexical code and tuned for separation rather than
        # generalisation: more units, more branches, so utterances that differ in a word land far apart
        self.column = Column("episodic", lexicon.n, n=n, k=k, branches=branches, seed=seed)
        self.sep = lambda v: self.column.encode(v)
        self.n, self.k = n, k
        self.codes = torch.zeros(0, n)
        self.records = []
        self.recruited = []      # every time new granule cells were grown, and what for
        self.overlaps = []       # how crowded each incoming memory was, so "crowded" can mean crowded here
        self.neurogenesis = True # growing new granule cells under crowding; off only to measure what it buys
        self.widths = []         # how big the population was when each memory was written

    def _code(self, text, turn=0):
        return self.column.encode(self.lex.bag(text.lower().split()))

    def align(self):
        """Make the stored memories the same width as the population that now exists.

        When granule cells are born or die, every memory written before is a code for a population that has
        changed shape. A memory written before a cell existed simply does not include it - a column of zeros
        - and a memory that used a cell which has since died loses that column. Realigning is not a repair:
        it is what it means for an old memory to live in a changed dentate.
        """
        n = self.column.dendrites.n
        have = int(self.codes.shape[1])
        if have == n or not len(self.codes):
            self.codes = self.codes if have == n else torch.zeros(0, n)
            return self.codes
        if have < n:
            self.codes = torch.cat([self.codes, torch.zeros(len(self.codes), n - have)], dim=1)
        else:
            keep = getattr(self.column.dendrites, "last_keep", None)
            self.codes = (self.codes[:, keep] if keep is not None and int(keep.numel()) == have
                          else self.codes[:, :n])
        return self.codes

    def store(self, text, turn, kind, payload=None, crowding=0.55, recruit=6):
        """One utterance, stored once - and new granule cells if it is about to collide with an old one.

        This is the one place in the system where new units are grown as a matter of course, and that is not
        an arbitrary choice: adult neurogenesis is a dentate gyrus phenomenon, and the dentate's job is exactly
        the problem being solved here. Two utterances that share most of their words get codes that overlap,
        and overlapping codes are how one memory gets recalled in place of another. When the incoming code
        overlaps an existing one too much, a fresh population is grown, wired to what is active now - so the
        new memory can be given a code the old one has no claim on.

        Cells are grown for crowding, not for volume: a conversation full of different things adds almost
        none, and a conversation that keeps circling the same phrasing adds many.
        """
        code = self._code(text, turn)
        born = 0
        self.align()
        if len(self.codes):
            overlap = float((self.codes @ code).max()) / self.k
            self.overlaps.append(overlap)
            # Crowded relative to this conversation, not to a number I picked. What counts as dangerously
            # similar depends on what is being talked about: a session of unrelated facts sits near zero and a
            # session circling one subject sits high, and the cells should be grown in the second case.
            typical = sorted(self.overlaps)[len(self.overlaps) // 2]
            # genuinely similar, not merely the most similar thing in a session of unrelated remarks: an
            # absolute floor as well as the relative one, because two sentences sharing "the" and "is" are not
            # a collision worth growing tissue for
            if self.neurogenesis and overlap > 0.25 and (overlap > 2.0 * typical or overlap > crowding):
                born = self.column.dendrites.recruit(self.lex.bag(text.lower().split()), count=recruit)
                if born:
                    self.n = self.column.dendrites.n
                    self.codes = torch.cat([self.codes, torch.zeros(len(self.codes), born)], dim=1)
                    code = self._code(text, turn)      # re-encode: the new cells get their say in this memory
                    self.recruited.append({"turn": turn, "cells": born, "overlap": round(overlap, 3)})
        self.column.dendrites.mature()
        self.codes = torch.cat([self.codes, code[None]])
        self.widths.append(self.column.dendrites.n)
        self.records.append({"turn": turn, "text": text, "kind": kind, "payload": payload, "new_cells": born})
        return len(self.records) - 1

    def _scale(self, cue):
        """How much of the cue each memory could possibly match.

        A memory laid down before a cell existed has nothing in that cell, so a cue that spends part of its
        code on newer cells would score lower against every older memory - and the store would drift toward
        always recalling the most recent thing. What each memory is entitled to be scored on is the population
        that existed when it was written, so the overlap is divided by how much of the cue falls inside it.
        """
        if not self.widths:
            return None
        # cells die as well as being born, so a width recorded when a memory was written can be wider than
        # the population that exists now. What it is entitled to is the whole of what remains.
        widths = torch.tensor(self.widths, dtype=torch.long).clamp(max=cue.numel())
        cumulative = cue.cumsum(0)
        share = cumulative[widths - 1].clamp_min(1.0)
        return share / max(1.0, float(cue.sum()))

    def recall(self, cue, rounds=2, top=5):
        """-> (record, confidence). The cue does not have to be anything that was said."""
        self.align()
        if not len(self.codes):
            return None, 0.0
        c = self._code(cue, 0)
        for _ in range(rounds):
            score = (self.codes @ c) / self._scale(c).clamp_min(1e-6)
            best = score.topk(min(top, len(self.codes))).indices
            if float(score[best[0]]) <= 0:
                return None, 0.0
            w = score[best] / score[best].sum().clamp_min(1e-9)
            vote = (self.codes[best] * w[:, None]).sum(0)
            c = torch.zeros_like(c)
            c[vote.topk(self.k).indices] = 1.0     # back to exactly k units: a memory, not a blur
        score = (self.codes @ c) / self._scale(c).clamp_min(1e-6)
        i = int(score.argmax())
        return self.records[i], float(score[i] / self.k)

    def about(self, term, limit=4):
        """Every episode whose code overlaps this term's - what was said about something."""
        self.align()
        if not len(self.codes):
            return []
        c = self.column.encode(self.lex.code(term))
        score = self.codes @ c
        order = score.argsort(descending=True)[:limit]
        return [(self.records[int(i)], float(score[i] / self.k)) for i in order if score[i] > 0]

    def replay(self, count=8, rng=None):
        """Sequences for consolidation: recent episodes come up most, as in sharp-wave ripples."""
        if not self.records:
            return []
        n = min(count, len(self.records))
        weights = torch.linspace(0.3, 1.0, len(self.records))
        idx = torch.multinomial(weights, n, replacement=False)
        return [self.records[int(i)] for i in sorted(idx.tolist())]

    def stats(self):
        spec = self.column.dendrites.spec()
        return {"episodes": len(self.records), "cells": spec["units"], "born": spec["born"],
                "young": spec["young"], "recruitments": len(self.recruited)}

    def synapses(self):
        return int((self.codes > 0).sum()) + self.column.synapses()

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/broca.py
"""Broca's area: where a message becomes a sequence of words.

The instruction this file exists to satisfy: language production is an *observation* of neural activity, not a
procedure that builds a string. Until now a sentence was assembled by a function that knew the subject, the
relation and the object and wrote them out in that order. That is a template with extra steps, and it means
the architecture is not producing the language - the author is.

Here, instead:

  the message arrives as activity.   Everything to be said - the subject, the relation, the thing said about
                                     it - is driven into L2/3 as an assembly, with the strength of its drive
                                     set by how central it is to what is being said. Nothing is ordered yet.
  the order is the dynamics.         At each step the most active assembly wins, is read out, and then adapts
                                     - it becomes hard to reach, the way a neuron that has just fired does -
                                     while the recurrent weights it has learned excite whatever usually
                                     follows it. That is what produces word order: not a rule about subjects
                                     preceding objects, but a settled sequence in a layer that has heard a lot
                                     of English.
  the words are a readout.           What is said at each step is whatever the winning assembly decodes to,
                                     through the same population readout the rest of this system uses. If the
                                     assembly is degraded the word comes out wrong or not at all, which is
                                     what a production aphasia looks like and is the test that this is a
                                     readout rather than a lookup.

The grammatical frame - "is a", "is the capital city of" - is *also* a readout: it is the phrasing whose
assembly the relation's own activity excites most strongly, learned from reading. What Broca contributes on
top of the map is the ordering, the frame and the adaptation that stops it repeating itself.
"""
import re

import torch

from brain.cortex import Laminar
from brain.lang import FUNCTION      # the function words the sequence memory learned to expect

FUNCTION_ONLY = re.compile(r"^[a-z' ]+$")


class Broca:
    """The production area. Everything this system says comes out of here, one settled assembly at a time."""

    def __init__(self, lexicon, speech=None, seed=0, n23=1024, k23=24):
        self.lex = lexicon
        self.speech = speech
        self.area = Laminar("broca", lexicon.n, n4=1536, k4=32, n23=n23, k23=k23, seed=seed, adapt=0.05)
        self.items = []             # what is currently loaded: (surface, role, code, drive)
        self.said = []
        self.utterances = 0
        self.last_trace = []

    # ------------------------------------------------------------------ loading
    def load(self, items):
        """Put a message into the area as activity. `items` are (surface, role, weight) - no order implied.

        Role sets how hard each part is driven, and that is all the priority there is. A subject is driven
        hardest because it is what the sentence is about, not because a rule says it comes first.
        """
        self.items = []
        drive = torch.zeros(self.lex.n)
        for surface, role, weight in items:
            if not str(surface).strip():
                continue
            code = self.lex.code(str(surface).lower())
            # each part's own assembly, taken without speaking it: this is what the readout will match
            # against and what the sequence weights will be written between
            self.items.append({"surface": str(surface), "role": role, "code": code,
                               "assembly": self.area.assembly(code), "drive": float(weight)})
            drive += float(weight) * code
        if not self.items:
            return None
        self.area.drive(drive, learn=True)
        self.area.settle(steps=2, learn=True)
        return self.area.state23

    # -------------------------------------------------------------- production
    def observe(self, limit=8):
        """Watch the area until it has nothing left to say, and write down what it said.

        This is the whole of production. At each step the assembly with the most activity is decoded into
        whatever it overlaps most in the message being held; that surface is emitted, the assembly adapts,
        and the recurrence says what tends to come next. The loop stops when activity falls below what it
        takes to speak at all.
        """
        if not self.items:
            return []
        activity = {i: item["drive"] for i, item in enumerate(self.items)}
        trace, out, previous, spoken = [], [], None, set()
        for step in range(limit):
            live = {i: a for i, a in activity.items() if a > 0.12 and i not in spoken}
            if not live:
                break
            winner = max(live, key=lambda i: live[i])
            item = self.items[winner]
            # the readout: what does this assembly actually overlap, of everything the area is holding?
            pattern = self.area.state23
            decoded = self.decode(item["assembly"])
            out.append(decoded or item["surface"])
            trace.append({"step": step, "role": item["role"], "said": decoded or item["surface"],
                          "activation": round(live[winner], 3),
                          "faithful": (decoded or "").lower() == item["surface"].lower()})
            # adaptation: an assembly that has just fired is spent. Multiplying its activation down was not
            # enough - the excitation the next steps hand out lifted it back over threshold and it said the
            # same thing twice ("Kuala Lumpur is the capital city is the capital city"). Firing is once.
            activity[winner] = 0.0
            spoken.add(winner)
            # and what usually follows it gets a push - the recurrence doing the sequencing
            for other, value in activity.items():
                if other == winner:
                    continue
                activity[other] = value + 0.35 * self._follows(item, self.items[other])
            if previous is not None:
                # what followed what, written into the recurrence: next time this area holds the first of
                # these, the second is already a little bit active
                self.area.sequence(previous, item["assembly"], strength=0.2)
            previous = item["assembly"]
        self.utterances += 1
        self.last_trace = trace
        self.said = out
        return out

    def _follows(self, first, second):
        """How much saying one thing makes the next thing likely - learned from reading, felt as excitation."""
        base = {"subject": {"relation": 1.0, "object": 0.3},
                "relation": {"object": 1.0},
                "object": {"tail": 0.6},
                "frame": {"object": 0.8}}.get(first["role"], {}).get(second["role"], 0.0)
        if self.speech is not None:
            try:
                expect = dict(self.speech.expect(self.lex_words(first["surface"]), top=12))
                head = self.lex_words(second["surface"])[:1]
                if head:
                    base += 2.0 * expect.get(head[0], 0.0)
            except Exception:
                pass
        return base

    @staticmethod
    def lex_words(text):
        return [w for w in re.findall(r"[A-Za-z'\-]+", str(text).lower())]

    def decode(self, code, floor=0.35):
        """Read an assembly back into a word: which of the things being held does this pattern overlap most?

        This is where production earns the word "observation". Nothing is looked up by name - the candidates
        are whatever the area currently holds, and the winner is whichever one the pattern actually matches.
        A pattern too degraded to match anything returns nothing, and the utterance has a hole in it, which
        is what it should have.
        """
        best, score = None, floor
        norm = float((code * code).sum()) ** 0.5 or 1.0
        for item in self.items:
            other = item["assembly"]
            overlap = float(other @ code) / (norm * (float((other * other).sum()) ** 0.5 or 1.0))
            if overlap > score:
                best, score = item["surface"], overlap
        return best

    # ------------------------------------------------------------------- frame
    def frame(self, relation, default, allowed=None):
        """The grammatical frame for a relation, read out of what this area has heard said.

        A frame is not content and must never carry any: it is function words plus the relation's own verb.
        Which one is used is decided by how well the whole sentence it would produce sits with everything
        heard - the sequence memory's own judgement, not a rule about tense.
        """
        if self.speech is None:
            return default
        options = []
        for phrasing, count in self.speech.phrasings_for(relation, top=8):
            words = self.lex_words(phrasing)
            if all(w in FUNCTION or w == default.split()[0] for w in words):
                options.append((phrasing, count))
        if not options:
            return default
        if allowed is None:
            return max(options, key=lambda kv: kv[1])[0]
        best, best_score = default, -1.0
        for phrasing, count in options:
            score = 3.0 * self.speech.fluency(allowed.format(frame=phrasing)) + 0.15 * float(count) ** 0.25
            if score > best_score:
                best, best_score = phrasing, score
        return best

    # -------------------------------------------------------------- utterance
    def say(self, subject, relation_phrase, obj, tail=None):
        """One sentence: load the parts as activity, watch the area, and write down what came out."""
        items = [(subject, "subject", 1.0), (relation_phrase, "frame", 0.75), (obj, "object", 0.55)]
        if tail:
            items.append((tail, "tail", 0.3))
        self.load(items)
        words = self.observe()
        text = " ".join(w for w in words if w)
        return re.sub(r"\s+([,.;:])", r"\1", text).strip()

    def fidelity(self):
        """How much of the last utterance came out as the assembly that produced it actually was."""
        if not self.last_trace:
            return 1.0
        return round(sum(1 for t in self.last_trace if t["faithful"]) / len(self.last_trace), 3)

    def spec(self):
        return {**self.area.spec(), "utterances": self.utterances, "holding": len(self.items)}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/cortex.py
"""The laminar circuit, enforced.

`brain/column.py` had the shape of a cortical column - an input layer and an output layer - but nothing in it
stopped a caller reaching straight past the circuit and reading the output. That is the difference between a
model that is *organised* like cortex and one that merely mentions cortex in a docstring, and it matters here
because the whole claim of this project is that the architecture is doing the work.

So: one column, four layers, and a gate on each of them.

  L4    the input layer. The only way drive enters. Dendritic expansion with branch-local supralinear
        summation, divisive gain control and k-winners-take-all, exactly as before.
  L2/3  the associative layer, and the only place where anything settles. Recurrent, laterally inhibited,
        and plastic: units that fire together strengthen the connection between them, which is what makes
        this layer complete a pattern from a fragment and what makes it able to run a sequence.
  L5    the projection layer. Reads out of L2/3, not out of L4, and refuses to read at all if L2/3 has not
        settled since the last drive. That refusal is the layering being enforced rather than described.
  L6    feedback. It does not drive anything; it multiplies the gain of the L4 below it, which is what
        cortical feedback does - it changes what the input layer is sensitive to without putting words in
        its mouth.

Plasticity lives in three places and all three stay: the dendrites adapt and can recruit new units, the
recurrent weights in L2/3 are Hebbian, and the L5 readout is Hebbian. Nothing here is trained by gradient
descent and nothing here has an objective function.
"""
import torch

from brain.dendrite import Dendrites


class Laminar:
    """One cortical area. Every area in this system is one of these, differing only in size and wiring."""

    # `decay` is how strongly the recurrent layer pulls toward what it has bound before. It is high because
    # the recurrent input is now divided by how much each cell already receives - and it sets the trade every
    # associative memory has to make: turn it up and a fragment completes to the whole, turn it down and two
    # similar inputs stay apart. Completion is what this layer is for; keeping near-identical things apart is
    # what the dentate and the growth schedule are for.
    def __init__(self, name, d_in, n4=2048, k4=32, n23=1024, k23=24, n_out=0, branches=4, seed=0,
                 decay=2.2, **kw):
        self.name = name
        self.d_in = d_in
        self.n4, self.k4, self.n23, self.k23 = n4, k4, n23, k23
        self.l4 = Dendrites(d_in, n=n4, k=k4, branches=branches, seed=seed, **kw)
        g = torch.Generator().manual_seed(seed + 977)
        # fixed feedforward wiring from L4 to L2/3. Which L4 unit reaches which L2/3 unit is anatomy, not
        # something learned; what is learned is what L2/3 does with what arrives.
        self.up = (torch.rand(n4, n23, generator=g) < (8.0 / n4)).float()
        # how many L4 cells reach each L2/3 cell. Without dividing by it, the cells that happen to have been
        # wired to the most inputs win every competition whatever is being heard - and L2/3 settles on the
        # same pattern for every sentence, which is a layer that represents nothing.
        self.updeg = self.up.sum(0).clamp(min=1.0)
        self.recurrent = torch.zeros(n23, n23)      # L2/3 -> L2/3, Hebbian, plastic
        self.l5 = torch.zeros(n23, n_out) if n_out else None
        self.gain = torch.ones(d_in)                # what L6 from above is currently doing to this area's L4
        self.decay = decay
        self.state4 = None
        self.state23 = None
        self.settled = False
        self.fired = 0
        self.settles = 0

    # ------------------------------------------------ growth and death in L4
    def grow(self, drive, count, excitability=0.25):
        """Recruit new input-layer cells, and wire them up to the layer above.

        A new cell is useless until something above it can hear it, so the projection to L2/3 grows with the
        population. Keeping the two in step is the whole reason growth lives here rather than in the
        dendrites: the dendrites know how to make a cell, they do not know what it has to reach.
        """
        if drive is None or count <= 0:
            return 0
        born = self.l4.recruit(drive, count=count, excitability=excitability)
        if born:
            g = torch.Generator().manual_seed(self.l4.n + born)
            fresh = (torch.rand(born, self.n23, generator=g) < (8.0 / max(1, self.l4.n))).float()
            self.up = torch.cat([self.up, fresh])
            self.updeg = self.up.sum(0).clamp(min=1.0)
            self.n4 = self.l4.n
        return born

    def grow_conjunctive(self, parts, count=None, excitability=0.05):
        """Grow cells tuned to combinations - one ingredient set per branch - and wire them upward."""
        born = self.l4.recruit_conjunctive(parts, count=count, excitability=excitability)
        if born:
            g = torch.Generator().manual_seed(self.l4.n + 3 * born)
            fresh = (torch.rand(born, self.n23, generator=g) < (8.0 / max(1, self.l4.n))).float()
            self.up = torch.cat([self.up, fresh])
            self.updeg = self.up.sum(0).clamp(min=1.0)
            self.n4 = self.l4.n
        return born

    def prune(self, floor=0.2, limit=48, keep_min=512, window=6):
        """Cull the cells that were grown and never fired, and drop their projections with them."""
        died = self.l4.cull(floor=floor, keep_min=keep_min, limit=limit, window=window)
        if died:
            self.up = self.up[self.l4.last_keep]
            self.updeg = self.up.sum(0).clamp(min=1.0)
            self.n4 = self.l4.n
            self.state4 = None
            self.settled = False
        return died

    # ------------------------------------------------------------------ L4
    def drive(self, x, learn=True):
        """Feedforward input. The only way anything enters this area, and it enters at L4."""
        single = x.dim() == 1
        v = (x if not single else x[None]) * self.gain
        code = self.l4(v, learn=learn)
        self.state4 = code[0] if single else code
        self.state23 = None
        self.settled = False
        self.fired += 1
        return self.state4

    def drive_many(self, xs, learn=True):
        """Several inputs arriving together, each through L4, their codes held at once.

        One k-winners-take-all over the sum of five words' drive keeps k units, not five times k, so four of
        the five words are simply gone - which is why an utterance cannot be a single code. Each part gets
        its own set of winners and the layer holds the union, which is what a population does when it is
        representing several things at the same time.
        """
        codes = self.l4(torch.stack([x * self.gain for x in xs]), learn=learn)
        self.state4 = codes.sum(0).clamp(max=1.0)
        self.state23 = None
        self.settled = False
        self.fired += len(xs)
        return self.state4

    def encode(self, x):
        """What L4 would make of this input, without disturbing what the area is currently holding.

        Needed by any area that has to compare what it is hearing now against what it has heard before: the
        comparison happens in this area's own units, not in the units of whatever fed it.
        """
        single = x.dim() == 1
        return self.l4((x if not single else x[None]) * self.gain, learn=False)[0 if single else slice(None)]

    def assembly(self, x, learn=False):
        """The settled L2/3 pattern this input produces, leaving the area as it was found.

        Production needs this: to know what following what feels like, an area has to be able to ask what a
        thing's assembly *is* without speaking it.
        """
        keep = (self.state4, self.state23, self.settled)
        self.drive(x, learn=learn)
        pattern = self.settle(learn=learn)
        self.state4, self.state23, self.settled = keep
        return pattern

    # ---------------------------------------------------------------- L2/3
    def settle(self, steps=2, learn=True, bias=None):
        """Lateral inhibition and recurrence until the layer holds one pattern rather than several.

        The first step is whatever L4 delivered; each step after that adds what the currently active units
        excite through the recurrent weights, and only the k best survive. That is pattern completion: a
        fragment drives the units that have fired with it before, and they pull in the rest of the assembly.
        """
        if self.state4 is None:
            raise RuntimeError(f"{self.name}: nothing has driven L4 - cannot settle on nothing")
        a = (self.state4 @ self.up) / self.updeg
        if bias is not None:
            a = a + bias
        for _ in range(max(1, steps)):
            a = self._kwta(a)
            # recurrent input is divided by how much each cell already receives. Without that, the first
            # assembly this layer ever bound wins every competition afterwards - every sentence settles into
            # the same pattern, and the layer stops representing what it is hearing. Synaptic scaling in
            # cortex does exactly this job: the total a cell receives is held roughly constant while the
            # share each partner contributes is what changes.
            incoming = self.recurrent.sum(0).clamp(min=1.0)
            a = a + self.decay * ((a @ self.recurrent) / incoming)
        self.state23 = self._kwta(a)
        self.settled = True
        self.settles += 1
        if learn:
            self.bind(self.state23)
        return self.state23

    def _kwta(self, a):
        k = min(self.k23, a.numel())
        top = torch.topk(a, k)
        out = torch.zeros_like(a)
        out[top.indices] = torch.clamp(top.values, min=0.0)
        return out

    def bind(self, pattern, strength=0.25):
        """Hebbian recurrence: the units active together now are easier to reach from each other later."""
        active = (pattern > 0).nonzero().flatten()
        if active.numel() < 2:
            return 0
        self.recurrent[active[:, None], active[None, :]] += strength
        self.recurrent[active, active] = 0.0          # a unit does not excite itself
        self.recurrent.clamp_(max=8.0)                # no synapse grows without limit
        return int(active.numel())

    def sequence(self, before, after, strength=0.6):
        """Asymmetric recurrence: what followed what. The same synapses, wired one way round.

        A layer that only had symmetric associations could hold a set but not an order. Strengthening from
        the units that were active to the units that came next is how L2/3 runs a sequence, and it is why
        production can come out of this layer at all rather than out of a loop in some other file.
        """
        a = (before > 0).nonzero().flatten()
        b = (after > 0).nonzero().flatten()
        if a.numel() == 0 or b.numel() == 0:
            return 0
        self.recurrent[a[:, None], b[None, :]] += strength
        return int(a.numel() * b.numel())

    # ------------------------------------------------------------------ L5
    def project(self):
        """The area's output. Refuses to produce anything L2/3 has not settled on."""
        if not self.settled:
            raise RuntimeError(f"{self.name}: L5 cannot read an unsettled L2/3 - drive and settle first")
        return self.state23 if self.l5 is None else self.state23 @ self.l5

    def associate(self, target, strength=1.0):
        """Hebbian readout: what L2/3 currently holds, bound to what the area is being asked to produce."""
        if self.l5 is None:
            raise ValueError(f"{self.name} has no L5 projection")
        if not self.settled:
            raise RuntimeError(f"{self.name}: nothing settled to associate from")
        self.l5[self.state23 > 0, target] += strength

    # ------------------------------------------------------------------ L6
    def feedback(self, lower, pattern=None, strength=0.35):
        """Modulate the gain of the area below. Feedback biases what the input layer hears; it never speaks.

        This is the one direction that does not carry content. An area that could inject its expectations
        into a lower area's L4 as drive would hallucinate: it would hear what it expected. Multiplying gain
        changes sensitivity and nothing else, which is the distinction the cortex itself maintains.
        """
        source = self.state23 if pattern is None else pattern
        if source is None:
            return None
        reach = min(lower.d_in, self.n23)
        g = torch.ones(lower.d_in)
        g[:reach] = 1.0 + strength * (source[:reach] > 0).float()
        lower.gain = g
        return g

    def release(self, lower):
        lower.gain = torch.ones(lower.d_in)

    # --------------------------------------------------------------- state
    def spec(self):
        s = {"area": self.name, "inputs": self.d_in, "L4": self.n4, "L2/3": self.n23,
             "k4": self.k4, "k23": self.k23, "settles": self.settles, "drives": self.fired,
             "recurrent_synapses": int((self.recurrent != 0).sum())}
        if self.l5 is not None:
            s["L5"] = int(self.l5.shape[1])
            s["L5_synapses"] = int((self.l5 != 0).sum())
        s.update({f"L4_{k}": v for k, v in self.l4.spec().items() if k not in ("units", "k")})
        return s

    def synapses(self):
        n = int(self.l4.idx.numel()) + int((self.recurrent != 0).sum())
        return n + (int((self.l5 != 0).sum()) if self.l5 is not None else 0)

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/curriculum.py
"""Sentences that are almost the same, and the measure of whether it can tell them apart.

Pattern separation is the one job the dentate gyrus is agreed to have, and until now nothing in this project
ever asked it to do that job properly: the sentences it heard were about different things, so keeping them
apart took no work. Here the sentences are built to be as close as they can be while meaning different
things, in five grades of nastiness:

  1  different things      "Rex is older than Max" / "Ada is older than Ben"        - separable on the words
  2  one content word      "Rex is a border collie" / "Rex is a border terrier"
  3  one letter            "Cara is taller than Cora" / "Cora is taller than Cara"  - and near-homograph names
  4  the same words        "Rex is older than Max" / "Max is older than Rex"        - role swap, zero lexical cue
  5  one function word     "Rex is older than Max" / "Rex is not older than Max"    - negation and tense, where
                           "the cafe is north of the park" / "the cafe is north of the pack"  one token flips
                           the truth of the whole thing

Grade 4 is where a bag of words has nothing left to work with: the two sentences contain exactly the same
tokens. Grade 5 is where a *nearly* identical bag flips the meaning. Anything that scores well on both is
separating by structure rather than by vocabulary, which is what the dentate is supposed to buy.

The measure is deliberately not "did it answer nicely". It is:

  recall precision   store both members, cue with one, and see which comes back. Chance is 0.5.
  separation         one minus the overlap of the two settled patterns. A system that codes both members
                     identically scores 0 however well it happens to answer.
"""
import random

FIRST = ["Rex", "Max", "Ada", "Ben", "Cara", "Cora", "Milo", "Mila", "Otto", "Otis", "Ida", "Ivy"]
NEAR = [("Cara", "Cora"), ("Milo", "Mila"), ("Otto", "Otis"), ("Ida", "Ivy"), ("Dana", "Dena"),
        ("Jon", "Jan"), ("Sam", "Sal"), ("Ann", "Ana")]
BREEDS = [("border collie", "border terrier"), ("great dane", "great pyrenees"), ("fox terrier", "fox hound"),
          ("water spaniel", "water dog"), ("wheaten terrier", "wheaton terrier")]
PLACES = [("park", "pack"), ("cafe", "cove"), ("station", "stables"), ("harbour", "arbour"),
          ("market", "marker")]
SCALES = [("older", "age"), ("taller", "height"), ("faster", "speed"), ("heavier", "weight")]


def pairs(grade, count=40, seed=0):
    """`count` pairs of sentences at one grade of similarity. Each pair means two different things."""
    rng = random.Random(seed * 100 + grade)
    out = []
    for i in range(count):
        word, _ = rng.choice(SCALES)
        if grade == 1:
            a, b = rng.sample(FIRST, 2)
            c, d = rng.sample([x for x in FIRST if x not in (a, b)], 2)
            out.append((f"{a} is {word} than {b}.", f"{c} is {word} than {d}."))
        elif grade == 2:
            who = rng.choice(FIRST)
            one, two = rng.choice(BREEDS)
            out.append((f"{who} is a {one}.", f"{who} is a {two}."))
        elif grade == 3:
            one, two = rng.choice(NEAR)
            out.append((f"{one} is {word} than {two}.", f"{two} is {word} than {one}."))
        elif grade == 4:
            a, b = rng.sample(FIRST, 2)
            out.append((f"{a} is {word} than {b}.", f"{b} is {word} than {a}."))
        else:
            a, b = rng.sample(FIRST, 2)
            here, there = rng.choice(PLACES)
            kind = rng.choice(["negation", "tense", "letter", "direction"])
            if kind == "negation":
                out.append((f"{a} is {word} than {b}.", f"{a} is not {word} than {b}."))
            elif kind == "tense":
                out.append((f"{a} is {word} than {b}.", f"{a} was {word} than {b}."))
            elif kind == "letter":
                out.append((f"the cafe is north of the {here}.", f"the cafe is north of the {there}."))
            else:
                out.append((f"the {here} is north of the cafe.", f"the {here} is south of the cafe."))
    return out


def graded(count=40, seed=0, grades=(1, 2, 3, 4, 5)):
    return {grade: pairs(grade, count=count, seed=seed) for grade in grades}


def overlap(one, two):
    """How much two sparse patterns share, 0 to 1. The thing separation has to drive down."""
    if one is None or two is None:
        return 1.0
    a, b = (one > 0).float(), (two > 0).float()
    size = max(float(a.sum()), float(b.sum()), 1.0)
    return round(float(a @ b) / size, 4)


def separation(area, first, second):
    """One minus the overlap of the patterns two sentences settle into, in whichever area is asked."""
    area.hear(first)
    one = area.area.state23.clone()
    area.hear(second)
    two = area.area.state23.clone()
    return round(1.0 - overlap(one, two), 4)


def recall_precision(agent, examples):
    """Store both members of each pair, then cue with one and see which memory comes back.

    This is the test that cannot be passed by answering nicely: either the cue brings back the sentence it
    came from, or it brings back the one it was nearly identical to. Chance is a half.
    """
    right = total = 0
    for first, second in examples:
        agent.epi.store(first, agent.turns + 1, "fact", payload={})
        agent.epi.store(second, agent.turns + 2, "fact", payload={})
        agent.turns += 2
        for cue, want in ((first, first), (second, second)):
            record, _ = agent.epi.recall(cue)
            total += 1
            right += bool(record and record["text"].strip() == want.strip())
    return round(right / max(1, total), 4)


def report(agent, examples, area=None):
    """Both measures for one grade, plus how big the population had to get to do it."""
    area = area or agent.wernicke
    seps = [separation(area, first, second) for first, second in examples[:20]]
    return {"recall": recall_precision(agent, examples),
            "separation": round(sum(seps) / max(1, len(seps)), 4),
            "dentate_units": agent.epi.column.dendrites.n,
            "wernicke_units": area.area.l4.n}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/deliberate.py
"""Thinking as cycles: bid, ignite, broadcast, check, again.

This is the reasoning the system did not have. Until now the planner ran *inside* one retrieval - two walks
and an adjudication - and anything that needed several retrievals in sequence was written out by hand in
agent.py. So "Rex is a dog" and "dogs are mammals" could both be on the map and "is Rex a mammal" would still
come back "Rex is a dog", because nothing ever took the answer to one retrieval and made it the question for
the next.

The loop here is small and does exactly that:

  1. prefrontal cortex states the goal, and the thalamus holds it - it is the only thing that survives a cycle
  2. every area bids on the current focus: the map offers the facts around it, the episodic store offers what
     was said about it, the reader offers to go and read about it
  3. the thalamic gate suppresses whatever has nothing to do with the goal, and amplifies what does
  4. the workspace ignites the strongest survivor and broadcasts it to every area
  5. prefrontal cortex asks whether that settles the goal. If it does, stop and say so with the trail. If it
     moves things along, the thing just learned becomes the new focus. If it does not, that is registered as
     a failure, and enough failures make the thalamus report uncertainty and the goal is abandoned rather
     than repeated.

Because the areas hear each other, one loop covers several kinds of reasoning that used to need separate
code: chaining ("is Rex a mammal"), finding what joins two things, and reading when the map runs out. The
stopping test is what differs, not the mechanism.
"""
from brain import workspace as ws


class MapBidder:
    """The hippocampal map, offering what it holds around whatever is in focus."""

    def __init__(self, maps):
        self.maps = maps
        self.focus = None
        self.seen = set()

    def receive(self, content):
        focus = content.payload.get("moves_to") or content.payload.get("focus")
        if focus:
            self.focus = focus
            self.seen.add(focus)

    def signature(self):
        return ws.digest("map", self.focus, len(self.seen))

    def bid(self, focus, avoid=()):
        """Every fact touching the focus, as a candidate step. Salience is how firmly it is held."""
        out = []
        for space, m in self.maps.maps.items():
            if not m.knows(focus):
                continue
            i = m.index[focus]
            for ea, moves, eb in m.episodes:
                if i not in (ea, eb):
                    continue
                other = m.names[eb if ea == i else ea]
                if other == focus or other in avoid:
                    continue
                # an edge with no moves is an alias - "dog" and "dogs", "KL" and "Kuala Lumpur". It is a free
                # hop rather than a claim, and skipping it is what stopped every chain at the plural.
                relation = moves[0][0] if moves else "another name for"
                firmness = 1.0 - min(1.0, m.spread_of(other) / 2.0)
                out.append(ws.Content("map", f"{focus} {relation} {other}",
                                      0.75 if not moves else 0.45 + 0.35 * firmness,
                                      {"focus": focus, "moves_to": other, "relation": relation,
                                       "space": space, "alias": not moves}, kind="step"))
        return out


class EpisodicBidder:
    """What was actually said about the focus, offered as evidence rather than as a step."""

    def __init__(self, episodes):
        self.episodes = episodes
        self.last = None
        self.offered = set()

    def receive(self, content):
        self.last = content.label

    def signature(self):
        return ws.digest("episodic", self.last)

    def bid(self, focus, avoid=()):
        if not focus or not self.episodes.records:
            return []
        record, confidence = self.episodes.recall(focus)
        if record is None or confidence < 0.35:
            return []
        if record["turn"] in self.offered:
            return []           # saying the same remembered line twice is not a second piece of evidence
        self.offered.add(record["turn"])
        # evidence is worth less than a step: it corroborates, it does not get the reasoning anywhere, and
        # letting it win every cycle is how the loop spent four cycles repeating "you said: Rex is a dog"
        return [ws.Content("episodic", f"you said: {record['text'][:60]}", min(0.42, 0.2 + 0.2 * confidence),
                           {"focus": focus, "turn": record["turn"]}, kind="evidence")]


class ReaderBidder:
    """The offer to go and read, which is worth something only when the map has run out."""

    def __init__(self, agent):
        self.agent = agent
        self.read = []

    def receive(self, content):
        focus = content.payload.get("moves_to") or content.payload.get("focus")
        if focus:
            self.focus = focus

    def signature(self):
        return ws.digest("reader", len(self.read), getattr(self, "focus", None))

    def bid(self, focus, avoid=()):
        if not focus or not self.agent.allow_web or focus in self.read:
            return []
        known = any(m.knows(focus) for m in self.agent.maps.maps.values())
        return [ws.Content("reader", f"read about {focus}", 0.25 if known else 0.5,
                           {"focus": focus}, kind="offer")]

    def act(self, content):
        """Reading is the one bid that does something to the world, so it happens only when it ignites."""
        focus = content.payload.get("focus")
        if not focus or focus in self.read:
            return None
        self.read.append(focus)
        return self.agent._do_search({"a": (focus, focus)}, focus)


def reason(agent, goal_label, start, satisfied, budget=6, difficulty=0.5, avoid=()):
    """Run the loop until the goal is met, the budget runs out, or the thalamus gives up on it.

    `satisfied(step)` decides what counts as an answer - which is the only thing that differs between
    chaining to a category, finding a link between two things, and reading until something is known.
    """
    board, thal = agent.workspace.clear(), agent.thalamus
    thal.arouse(difficulty)
    goal = ws.Content("prefrontal", goal_label, 1.0, {"focus": start}, kind="goal")
    thal.hold(goal)
    board.post(goal)
    board.cycle()                     # the goal itself is the first thing globally available

    focus, trail, visited = start, [], {start} | set(avoid)
    for _ in range(max(1, int(budget * board.gain))):
        if not thal.sustain():
            break
        bids = []
        for area in (agent.bidders["map"], agent.bidders["episodic"], agent.bidders["reader"]):
            bids += area.bid(focus, avoid=visited)
        for bid in thal.gate(bids, focus=focus):
            board.post(bid)
        winner = board.cycle()
        if winner is None:
            thal.progressed(False)
            continue
        if winner.source == "reader":
            agent.bidders["reader"].act(winner)
            thal.progressed(True)
            continue
        if winner.kind != "step":
            thal.progressed(False)      # corroboration is not progress; only a new hop is
            continue
        step = {"from": focus, "to": winner.payload.get("moves_to"), "relation": winner.payload.get("relation"),
                "space": winner.payload.get("space"), "said": winner.label, "source": winner.source}
        trail.append(step)
        if satisfied(step):
            return {"found": True, "trail": trail, "stream": board.report(), "cycles": board.cycles,
                    "gain": board.gain, "uncertainty": thal.uncertainty(), "blocked": len(thal.blocked)}
        moved = step["to"] and step["to"] not in visited
        thal.progressed(bool(moved))
        if moved:
            focus = step["to"]
            visited.add(focus)
    return {"found": False, "trail": trail, "stream": board.report(), "cycles": board.cycles,
            "gain": board.gain, "uncertainty": thal.uncertainty(), "blocked": len(thal.blocked)}


def is_a(agent, thing, category, budget=6):
    """"Is Rex a mammal?" - chain upward through what things are until the category turns up, or does not."""
    target = str(category).lower().rstrip("s")

    def satisfied(step):
        to = (step.get("to") or "").lower()
        return to == target or to.rstrip("s") == target or target in to.split()

    return reason(agent, f"is {thing} a {category}", str(thing).lower(), satisfied, budget=budget,
                  difficulty=0.6)

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/extract.py
"""Learning to read, instead of being told how.

The reader was four regular expressions I wrote: what a thing is, what it is known as, where it is, and what
it did. Everything else on a page went past unread - a Wikipedia article on a city yielded nothing at all, and
an essay built on that has two facts in it. Adding a fifth regular expression would have got a fifth shape and
the same problem, because the shapes English uses to state a fact are not a list anyone can finish.

So this learns them. The method is distant supervision, and it is old and cheap and needs no gradients:

  seeds        whatever the hand-written reader *does* manage to extract, across a whole corpus. Those pairs
               are not a training signal in the loss sense - they are pairs of things known to stand in a
               known relation, which is a different and much weaker thing to need.
  patterns     scan the corpus again for sentences containing both ends of a seed pair, and keep what sat
               between them. "Kuala Lumpur" ... "Malaysia" turns up with "is the capital city and a federal
               territory of" between them, and that phrase is now a way of saying `origin` that nobody wrote
               down. A pattern is kept only if several *different* pairs produced it and they agree on which
               relation it means, which is what separates a way of saying something from a coincidence.
  bootstrap    the facts those patterns extract become seeds themselves, and the scan runs again. Each round
               reaches sentences the round before could not, and each round is checked the same way, so a
               pattern that only ever fires on one pair never survives to do damage.

The guard against nonsense is that a pattern must earn its place three times over: several pairs behind it,
agreement about what it means, and ends that look like things rather than fragments. What comes out is still
weaker evidence than a person saying something, and it is encoded as such.
"""
import os
import pickle
import re
from collections import defaultdict

WORD = re.compile(r"[A-Za-z][A-Za-z'\-]*|\d+")
# tokens that can sit inside a relation's phrasing without being the thing being related
GLUE = {"a", "an", "the", "of", "in", "on", "at", "to", "for", "with", "from", "by", "as", "and", "or",
        "is", "was", "are", "were", "has", "have", "had", "been", "be", "its", "their", "his", "her",
        "which", "that", "who", "also", "known", "part", "one", "most", "first", "located", "situated"}
DANGLING = ("of", "a", "an", "the", "and", "or", "in", "to", "for", "with", "from", "by", "as", "that",
            "which", "is", "was", "at", "on", "into", "about", "s")
STOP_SPAN = {"it", "he", "she", "they", "this", "that", "these", "those", "there", "which", "who", "what"}
COPULA = {"is", "was", "are", "were", "be", "been", "became", "becomes"}
HOLLOW = {"can", "could", "would", "will", "may", "might", "must", "shall", "should", "not", "never", "no",
          "i", "me", "my", "myself", "you", "your", "he", "him", "his", "himself", "she", "her", "herself", "it",
          "we", "us", "our", "they", "them", "their", "all", "some", "any", "each", "every", "both", "well",
          "very", "so", "too", "just", "then", "now", "here", "there", "this", "these", "those", "what",
          "do", "does", "did", "done", "had", "have", "has", "am", "wasn", "don", "didn", "isn", "right",
          "really", "only", "even", "still", "also", "yet", "ever", "much", "more", "most", "up", "out",
          "if", "but", "or", "nor", "than", "thought", "think", "said", "say", "says", "knew", "know"}
PERSONS = {"i", "me", "my", "you", "your", "he", "him", "his", "she", "her", "we", "us", "our", "they",
           "them", "their", "myself", "yourself", "himself", "herself", "sir", "madam"}
MAX_SPAN = 5          # a thing named in more than five words is a clause, not a thing
RESTATES = {"also", "formerly", "officially", "originally", "sometimes", "often", "commonly",
            "previously", "variously", "alternatively", "colloquially", "informally"}
MAX_MIDDLE = 6        # more than six words between two things is not a phrasing, it is a sentence


def tokens(text):
    return WORD.findall(str(text))


def normal(text):
    return " ".join(w.lower() for w in tokens(text))


def usable_span(text):
    """Does this look like a thing, or like the tail of a clause someone cut in half?"""
    words = tokens(text)
    if not words or len(words) > MAX_SPAN:
        return False
    if words[-1].lower() in DANGLING or words[0].lower() in STOP_SPAN:
        return False
    if any(w.lower() in PERSONS for w in words):
        return False      # "I", "him", "you" are whoever is speaking in a story, not a thing a fact is about
    if any(w.lower() in ("what", "which", "that", "who", "whose", "whom") for w in words):
        return False      # a span containing a wh-word is the hinge of a clause, not the name of a thing
    if words[-1].isdigit() and len(words) > 1:
        return False      # "union territory on 1" is a date cut in half, not a thing
    if words[-1].lower().endswith("ly") and len(words[-1]) > 4:
        return False      # "historically" is how something was done, not a thing it was done to
    return any(w.lower() not in GLUE for w in words)      # at least one word carrying content


def _find(low, parts):
    """Where a phrase sits in a tokenised sentence, or None."""
    if not parts:
        return None
    n = len(parts)
    for i in range(0, len(low) - n + 1):
        if low[i:i + n] == parts:
            return i, i + n
    return None


WORDS = 'which|who|that|what|where|when|while|because|although|and|but|however|including|such as'
PUNCT = '[,;:()"—]'
# a clause break: punctuation, or the words English uses to stop talking about one thing and start
# talking about another. Both ends of a learned phrasing are cut here.
BOUNDARY = re.compile(PUNCT + '|\\b(?:' + WORDS + ')\\b', re.I)


def _before_copula(span):
    """The subject in front of "is"/"was", where there is one: the thing the clause is about."""
    words = span.split()
    for i, w in enumerate(words):
        if w.lower() in COPULA:
            return " ".join(words[:i])
    return span


def _left_of(text):
    """The thing a phrasing is about: the noun phrase before it, cut at the last clause break.

    Reading backwards is what a listener does - the subject of "is a" is what was just said, not the start of
    the sentence - so everything before the last break belongs to somebody else's clause. Two complications
    are worth handling because English uses both constantly. An appositive puts the subject one comma further
    back ("Kuala Lumpur, officially the Federal Territory, is the capital"), so a punctuation break may be
    stepped over once. A connective may not: crossing "what" or "and" changes what is being talked about,
    which is how "monarchs played a central role in what became the Reformation" produced a thing called
    "a central role in what".
    """
    pieces, last = [], 0
    for m in BOUNDARY.finditer(text):
        pieces.append((text[last:m.start()], text[m.start():m.end()]))
        last = m.end()
    pieces.append((text[last:], ""))
    crossed_clause, steps = False, 0
    for back in range(len(pieces) - 1, -1, -1):
        raw = tokens(pieces[back][0])
        # "Kuala Lumpur, officially the Federal Territory, is the capital" - a chunk opening with an adverb
        # is a restatement of the thing, not the thing. The name is on the far side of it.
        if raw and (raw[0].lower() in RESTATES or (raw[0].lower().endswith("ly") and len(raw[0]) > 5)):
            continue
        words = list(raw)
        while words and words[0].lower() in GLUE | STOP_SPAN:
            words = words[1:]
        span = " ".join(words[-MAX_SPAN:])
        # "Category theory is another formalization that includes ..." - the phrasing here is "that includes",
        # and the thing it is about is the subject of the clause, not the complement sitting next to it. A
        # copula in the run-up means the subject is in front of it.
        subject = _before_copula(span)
        if subject != span and usable_span(subject):
            return subject
        # having stepped back over "that" or "and", only a proper clause counts. The words before a
        # connective are a sentence in their own right and have their own subject; the words before a comma
        # are usually an appositive, and the thing being described is on the far side of it.
        if not crossed_clause and usable_span(span):
            return span
        if steps >= 2:
            break
        boundary = (pieces[back - 1][1] if back else "").strip()
        crossed_clause = crossed_clause or bool(boundary and boundary[0].isalpha())
        if words:
            steps += 1
    return ""


def _right_of(text):
    """And the other end: what follows, up to the point where the sentence turns to something else."""
    m = BOUNDARY.search(text)
    words = tokens(text[:m.start()] if m else text)[:MAX_SPAN]
    while words and words[-1].lower() in DANGLING:
        words.pop()
    return " ".join(words)


class Patterns:
    """Ways of saying a relation, learned from sentences where both ends were already known."""

    def __init__(self, support=3, purity=0.75):
        self.support = support        # how many different pairs must have produced a pattern to keep it
        self.purity = purity          # how much those pairs must agree about what it means
        self.pairs = defaultdict(set)                         # phrasing -> the pairs that produced it
        self.labels = defaultdict(lambda: defaultdict(int))   # phrasing -> relation -> count
        self.kept = {}                                        # phrasing -> relation, once it has earned it
        self.evidence = {}
        self.rounds = 0

    # ------------------------------------------------------------------ learning
    def observe(self, sentence, a, b, relation):
        """One sentence in which two known things appear: keep whatever sits between them."""
        words = tokens(sentence)
        low = [w.lower() for w in words]
        na, nb = normal(a), normal(b)
        if not na or not nb or na == nb:
            return None
        ia, ib = _find(low, na.split()), _find(low, nb.split())
        if ia is None or ib is None or ia[1] > ib[0]:
            return None                        # not both here, or the object comes first: a different shape
        middle = " ".join(low[ia[1]:ib[0]])
        if not middle or len(middle.split()) > MAX_MIDDLE:
            return None
        self.pairs[middle].add((na, nb))
        self.labels[middle][relation] += 1
        return middle

    def settle(self):
        """Which phrasings have earned the right to be read as a relation."""
        self.kept = {}
        for middle, pairs in self.pairs.items():
            if len(pairs) < self.support:
                continue
            counts = self.labels[middle]
            total = sum(counts.values())
            relation, best = max(counts.items(), key=lambda kv: kv[1])
            if best / total < self.purity:
                continue                        # the same words mean different things: not a phrasing
            words = middle.split()
            # a phrasing has to say something. "to the" is glue - it sits between two things in half the
            # sentences in English and means nothing by itself, and a pattern like that will happily read
            # "the Medical Device Amendments to the FD&C Act" as a claim about the FD&C Act. Either the
            # phrasing carries a word of its own, or it is a copula, or it is not a way of saying anything.
            if all(w in GLUE for w in words) and not any(w in COPULA for w in words):
                continue
            # and it has to be the right *sort* of phrasing for what it claims. A category claim is made with
            # a copula - "is a", "is the", "are a kind of" - and a phrasing like "made" or "and other" being
            # read as one is where "wire can also be | made | in square" came from. A relation of taking part
            # needs a verb of its own. An origin needs the preposition that points at where.
            # a category claim names a category, and in English that means a determiner: "is a", "is the",
            # "is an", "is known as". A bare "was" or "were" is a passive, and reading "hundreds of police
            # were summoned to remove the blockade" as a claim that police are a kind of summoning is the
            # sort of thing that fills a map with rubbish.
            if relation == "kind" and not (any(w in COPULA for w in words)
                                           and (any(w in ("a", "an", "the") for w in words)
                                                or words[-1] in ("as", "of"))):
                continue
            # and a relation of taking part needs a verb that carries something. Novels taught it that "not",
            # "would", "never" and "well i" are ways of saying X is involved with Y, because a hand-written
            # reader run over fiction pairs "he" with whatever came next. Modals, negation, pronouns and
            # quantifiers are grammar, not a relation, and a phrasing made only of them says nothing.
            if relation == "involved" and not any(w not in GLUE and w not in COPULA and w not in HOLLOW
                                                  for w in words):
                continue
            if relation == "origin" and words[-1] not in ("of", "from", "in", "at", "near"):
                continue
            self.kept[middle] = relation
        self.evidence = {m: len(self.pairs[m]) for m in self.kept}
        return self.kept

    # ------------------------------------------------------------------- reading
    def read(self, sentence, subject=None, limit=2):
        """Facts out of a sentence the hand-written reader could not parse.

        A pattern says where the relation is; the two ends are whatever sits either side of it, and both have
        to look like things before anything is claimed. The longest pattern that fits wins, because "is the
        capital city of" says more than "of" and both are in here.

        The ends are cut at punctuation and at the words English uses to change subject - a comma, "which",
        "that", "where". Without that the run-up to a phrasing is simply the five words before it, and
        "monarchs played a central role in what became the Reformation" yields a thing called "central role
        in what", which is not a thing and never was.
        """
        out = []
        low = sentence.lower()
        for middle in sorted(self.kept, key=lambda m: -len(m.split())):
            at = re.search(r"(?<![a-z])" + re.escape(middle) + r"(?![a-z])", low)
            if at is None:
                continue
            left = _left_of(sentence[:at.start()])
            right = _right_of(sentence[at.end():])
            if not (usable_span(left) and usable_span(right)) or left.lower() == right.lower():
                continue
            if any(f["a"][0] == left.lower() and f["b"][0] == right.lower() for f in out):
                continue
            out.append({"kind": "fact", "a": (left.lower(), left), "b": (right.lower(), right),
                        "moves": [(self.kept[middle], 1)], "word": middle, "how": "read",
                        "pattern": middle})
            if len(out) >= limit:
                break
        return out

    # ------------------------------------------------------------------- storage
    def stats(self):
        return {"patterns": len(self.kept), "seen": len(self.pairs), "rounds": self.rounds,
                "relations": len({r for r in self.kept.values()})}

    def save(self, path):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump({"kept": self.kept, "support": self.support, "purity": self.purity,
                         "rounds": self.rounds, "evidence": self.evidence}, f)
        return os.path.getsize(path)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            d = pickle.load(f)
        p = cls(support=d.get("support", 3), purity=d.get("purity", 0.75))
        p.kept = d["kept"]
        p.rounds = d.get("rounds", 0)
        p.evidence = d.get("evidence", {})
        return p


def index_pairs(seeds):
    """Seed pairs indexed by their subject, so a corpus scan is a handful of dictionary lookups.

    The obvious index - by the subject's first word - is useless, because thousands of subjects begin with
    "the" and every sentence containing "the" then has to try all of them. Indexing by the whole subject and
    generating each sentence's own candidate phrases instead makes the scan linear in the corpus, which is
    the difference between this finishing in a minute and not finishing at all.
    """
    by_subject = defaultdict(list)
    for a, b, relation in seeds:
        na, nb = normal(a), normal(b)
        if na and nb and na != nb and len(na.split()) <= MAX_SPAN and len(nb.split()) <= MAX_SPAN:
            by_subject[na].append((nb, relation))
    return by_subject


def phrases_in(low, limit=MAX_SPAN):
    """Every phrase of up to `limit` words in a sentence: the things it might be naming."""
    out = {}
    for size in range(1, limit + 1):
        for i in range(0, len(low) - size + 1):
            out.setdefault(" ".join(low[i:i + size]), i)
    return out


def harvest(sentences, seeds, patterns, progress=None):
    """Scan a corpus for sentences containing both ends of a seed pair, and learn what sat between them."""
    index = index_pairs(seeds)
    found = 0
    for n, sentence in enumerate(sentences):
        low = [w.lower() for w in tokens(sentence)]
        if len(low) < 5 or len(low) > 60:
            continue
        here = phrases_in(low)
        for phrase in here:
            for nb, relation in index.get(phrase, ()):
                if nb in here and patterns.observe(sentence, phrase, nb, relation):
                    found += 1
        if progress and n and n % progress == 0:
            print(f"  {n:,} sentences, {found:,} pattern observations, "
                  f"{len(patterns.pairs):,} distinct phrasings", flush=True)
    return found

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/gate.py
"""Basal-ganglia gate: which system is allowed to answer.

The old router was a hand-ordered cascade of `if` statements - try the map, then the web, then give up - which is
a programmer's decision frozen into the code, and it never got better. Selecting between competing systems is
what basal ganglia circuitry is for, and it is *learned*: cortical loops propose, the striatum releases one, and
what happened next changes which one is released next time.

Here the proposers are the stores - working memory, the episodic store, the cognitive map - and the selector is
an expert pool over a sparse code of the situation:

  the situation   not just what was asked, but what each store currently has: does the map know both things, are
                  they connected, did an episode match strongly, is there an unresolved pronoun. A gate that saw
                  only the question could never learn that the same question needs a different system depending
                  on what is known.
  selection       the experts wired to the active units vote, the strongest wins (lateral inhibition).
  learning        an action that produces grounded evidence reinforces the expert that chose it. One that comes
                  back empty raises the match threshold above that expert and the search continues - ARTMAP match
                  tracking - so the situation gets its own expert instead of overwriting the old one.
  neurogenesis    a situation unlike anything seen recruits a brand-new expert, wired to exactly the units that
                  are active. The gate grows with use; it is never trained.

Before anything has been learned the gate has a reflex: a fixed prior from the parser's intent. Every reflex that
works is immediately learned from, so the pool takes over from the reflexes within a few turns.
"""
import torch

from neuromoe import ExpertPool

ACTIONS = ("teach", "infer", "recall", "describe", "explain", "report", "search", "inquire", "relate", "reason", "self", "view", "instruct", "clarify", "social")

# what the parser's intent asks for, before any experience: the innate part, which experience overrules
REFLEX = {"fact": "teach", "compare": "infer", "which": "infer", "extreme": "infer", "rank": "infer",
          "between": "infer", "howmuch": "infer", "at": "infer", "where": "infer", "direction": "infer",
          "about": "describe", "recall": "recall", "command": "social", "social": "social",
          "why": "explain", "check": "explain", "correction": "teach", "report": "report", "relate": "relate", "member": "reason", "self": "self", "view": "view", "instruct": "instruct", "unknown": "clarify"}


class Gate:
    def __init__(self, lexicon, vigilance=0.55, trust=1.0, seed=7):
        self.lex = lexicon
        self.pool = ExpertPool(lexicon.n, len(ACTIONS), vigilance=vigilance, k_route=3, beta=0.25,
                               labels="exact", trust=trust)
        self.chosen = {a: 0 for a in ACTIONS}

    def situation(self, intent, signals, activity=None):
        """A sparse code of the whole situation - built from activity, not from words.

        `activity` is what the stores are actually doing this turn: the utterance's own population code, what
        working memory is holding, how strongly the episodic store matched, where the referents sit on the map.
        The striatum is driven by cortical and hippocampal activity, not by a label describing it, and a gate
        reading parsed English would be the verbal shortcut this whole design is trying to avoid.

        The availability flags stay, because "the map has nothing about this" is a fact about the circuit rather
        than about the sentence, and a system that could not represent its own emptiness would keep asking
        stores that have nothing to say.

        Only what has *arrived* is in here: a store slower than the deadline contributes nothing this turn.
        """
        v = self.lex.bag(["\x00sig:" + s for s, on in sorted(signals.items()) if on])
        for name, pattern in (activity or {}).items():
            if pattern is not None:
                v = v + 0.8 * pattern
        return v

    def choose(self, intent, signals, allowed, activity=None):
        """-> (action, confidence, how). `how` is 'learned' when an expert claimed it, 'reflex' otherwise.

        `allowed` is what this turn could physically be answered by - you cannot teach a question or infer from a
        greeting. The gate never votes on impossible actions, in the same way basal ganglia loops select among
        motor programs that are actually available rather than among all conceivable ones. Everything genuinely
        open - describe it, recall it, or work it out - is left to the experts and to what happens next.
        """
        code = self.situation(intent, signals, activity)
        allowed = [a for a in allowed if a in ACTIONS] or ["clarify"]
        reflex = REFLEX.get(intent, "clarify")
        if reflex not in allowed:
            reflex = allowed[0]
        if not len(self.pool):
            return reflex, 0.0, "reflex", code
        p, match = self.pool.predict(code, return_match=True)
        mask = torch.zeros(len(ACTIONS))
        for a in allowed:
            mask[ACTIONS.index(a)] = 1.0
        p = p * mask
        confidence = float(p.max())
        action = ACTIONS[int(p.argmax())]
        if match < self.pool.vigilance or confidence < 0.4:
            return reflex, confidence, "reflex", code
        return action, confidence, "learned", code

    def reinforce(self, code, action):
        """That action found something. The expert that chose it grows toward this situation."""
        self.pool.learn(code, ACTIONS.index(action))
        self.chosen[action] += 1

    def correct(self, code, failed, worked):
        """That action came back empty and another one did not. Match tracking gives this situation its own
        expert rather than blurring the one that was right elsewhere."""
        if worked is None:
            return
        self.pool.learn(code, ACTIONS.index(worked))
        self.chosen[worked] += 1

    def stats(self):
        return {"experts": len(self.pool), "born": self.pool.born, "died": self.pool.died,
                "synapses": self.pool.synapses(), "chosen": dict(self.chosen)}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/plasticity.py
"""Growth, death and plasticity as one process, with the balance shifting as development proceeds.

They are not three knobs. Cortex overproduces - roughly forty per cent more synapses than the adult keeps -
and then activity decides what survives; the cells and synapses that fire with something keep their place and
the rest are removed. Growth without death is a memory leak that dilutes every code it is added to, and death
without growth is a system that can only lose. What makes it development rather than churn is the *schedule*:

  early    growth far outweighs death. The population expands wherever the input is confusable, because the
           cheapest way to tell two things apart is to have cells that only one of them drives.
  late     death and plasticity take over. What was grown is culled back to what fired, and the surviving
           cells sharpen - homeostasis pushes each toward its share of the work, adaptation makes repeated
           input cheaper to ignore, and the recurrent weights that survived get stronger.

Growth is not on a timer either. It is driven by *pressure* - how badly the thing being heard now collides
with something already stored. Two sentences that settle into the same pattern are a demand for new cells;
two that are already separate are not. So the process is closed-loop: confusion causes growth, growth is
pruned by use, and what is left is a population that separates the things that were actually confusable.

The dentate is deliberately excluded from all of this during phase 0/1 and is the only place that grows in
phase 2, which is the experiment: can cortex learn to separate on its own, and what does the dentate then add.
"""
import torch


def curve(progress, early=True, power=2.0):
    """A schedule that starts high and ends low, or the reverse. Smooth, so nothing switches on abruptly."""
    p = max(0.0, min(1.0, float(progress)))
    return (1.0 - p) ** power if early else p ** power


class Regime:
    """How much growing, killing and sharpening each area is doing right now."""

    def __init__(self, name, grow_max=24, cull_max=16, floor=0.1, plastic_max=0.04, window=12):
        self.name = name
        self.grow_max = grow_max        # cells a confusable input may recruit, at the very start
        self.cull_max = cull_max        # cells that may die in one step, at the very end
        self.floor = floor              # how far below its share a young cell must fall to be removed
        self.plastic_max = plastic_max  # homeostasis at full strength
        self.window = window            # exposures a newborn has to be recruited before it can die
        self.grows, self.culls = True, True
        self.history = []

    def rates(self, progress):
        """Growth early, death and plasticity late - the developmental arc, as three numbers."""
        return {"grow": curve(progress, early=True), "cull": curve(progress, early=False),
                "plastic": 0.25 + 0.75 * curve(progress, early=False, power=1.5)}

    def step(self, area, drive, pressure, progress):
        """One pass of the coupled process over one area. Returns what changed.

        `pressure` is how confusable the current input was - one minus the separation achieved. Nothing grows
        for an input the area already tells apart, which is what keeps this from being growth on a clock.
        """
        dendrites = getattr(area, "l4", None) or getattr(area, "dendrites", None)
        if dendrites is None:
            return {}
        rate = self.rates(progress)
        grew = died = 0
        want = int(self.grow_max * rate["grow"] * max(0.0, min(1.0, pressure)))
        limit = max(1, int(self.cull_max * rate["cull"]))
        # a laminar area grows and prunes through its own methods, because the projection to the layer above
        # has to grow and shrink with the population; a bare column has only its dendrites
        if self.grows and want > 0 and drive is not None:
            if isinstance(drive, list):         # conjunctions: one set of ingredients per new cell's branches
                grew = area.grow_conjunctive(drive, count=min(want, 4 * len(drive))) if drive else 0
            else:
                grew = (area.grow(drive, want, excitability=0.05)
                        if hasattr(area, "grow") else dendrites.recruit(drive, count=want))
        if self.culls and rate["cull"] > 0.05:
            died = (area.prune(floor=self.floor, limit=limit, window=self.window) if hasattr(area, "prune")
                    else dendrites.cull(floor=self.floor, limit=limit, window=self.window))
        dendrites.homeostasis = self.plastic_max * rate["plastic"]
        dendrites.adapt = min(0.1, 0.02 + 0.06 * rate["plastic"])
        # recurrent gain is deliberately NOT on the schedule. Raising it late looked like "sharpening" and was in
        # fact the single largest confound in v16: at high gain L2/3 falls into one attractor and every pair
        # overlaps, so the critical arm was losing on separation because of its gain, not its growth. It stays
        # at the area's own value, identical to the frozen control, so the only difference between the two
        # arms is growth and death.
        record = {"area": self.name, "progress": round(progress, 3), "pressure": round(float(pressure), 3),
                  "grew": grew, "died": died, "units": dendrites.n,
                  "plastic": round(dendrites.homeostasis, 4)}
        self.history.append(record)
        return record


class Development:
    """The whole system's schedule: which areas are growing, which are frozen, and how far along it is."""

    PHASES = {
        # phase: (areas that grow and cull, whether the dentate may grow, what the phase is for)
        # the lexical column is not on the list on purpose: it is the input population every other area is
        # wired to, so growing it changes the width of their input fibres and invalidates them. Sensory
        # surfaces are laid down before the areas that read them, and here that is literally true.
        "critical": (("wernicke", "broca"), False,
                     "cortex overproduces and prunes on confusable input; the dentate is held still"),
        # the ablation arms: the same schedule with parts of it switched off, so each part's contribution can
        # be read off rather than argued about. v16 changed three things at once and blamed the wrong one.
        "tune": (("wernicke", "broca"), False, "homeostasis and adaptation on the schedule, nothing grows or dies"),
        "cull": (("wernicke", "broca"), False, "death and tuning on the schedule, nothing grows"),
        "adult": ((), True,
                  "cortex settles; the dentate is the only thing that grows, and only for what is new"),
        "frozen": ((), False, "nothing grows or dies - the control, to show the schedule is doing the work"),
    }
    SWITCHES = {"critical": (True, True), "tune": (False, False), "cull": (False, True),
                "adult": (True, True), "frozen": (False, False)}

    def __init__(self, agent, phase="critical", steps=200):
        self.agent = agent
        self.steps = max(1, steps)
        self.seen = 0
        self.regimes = {name: Regime(name) for name in ("wernicke", "broca", "lexical", "dentate")}
        self.set_phase(phase)

    def set_phase(self, phase):
        self.phase = phase
        self.growing, self.dentate_grows, self.why = self.PHASES[phase]
        grows, culls = self.SWITCHES[phase]
        for regime in self.regimes.values():
            regime.grows, regime.culls = grows, culls
        self.seen = 0
        return self

    @property
    def progress(self):
        return min(1.0, self.seen / self.steps)

    def areas(self):
        a = self.agent
        return {"wernicke": a.wernicke.area, "broca": a.broca.area, "lexical": a.lex.column,
                "dentate": a.epi.column}

    def drives(self, text, contrast=None):
        """The same experience as each area receives it. An area can only grow on its own input fibres.

        The lexical column is wired to letter slots, Wernicke and Broca to the lexicon's units, the dentate
        to whole utterances. Handing one of them another's code is not a small mistake - it is an index into
        a population that does not exist.
        """
        lex = self.agent.lex
        words = str(text or "").lower().split()
        utterance = lex.bag(words) if words else None
        # growth on the difference. Wired to everything the sentence drives, a new cell answers to both
        # members of a near-identical pair and adds overlap - which is what v16 measured. Wired to what this
        # member drives and the other does not, it answers to one of them only, which is the whole point of
        # growing it. Newborn dentate cells are selective in exactly this way for novel inputs that resemble
        # stored ones.
        if contrast is not None and utterance is not None:
            other = lex.bag(str(contrast).lower().split())
            only = (utterance - other).clamp(min=0.0)
            if float(only.sum()) > 0:
                utterance = only
        # Wernicke hears words bound to where they sat, so it grows on that - and the difference between a
        # sentence and its role-swapped twin is nonzero there, where in a bag of words it is exactly zero
        # Wernicke grows cells for the word-at-position combinations that tell this sentence from its twin,
        # each combination landing on one dendritic branch so the cell detects the conjunction and not either
        # half of it
        heard = self.agent.wernicke.conjunctions(text, contrast) if text else None
        return {"wernicke": heard, "broca": utterance, "dentate": utterance,
                "lexical": lex.drive(str(text or ""))}

    def step(self, text=None, pressure=1.0, drive=None, contrast=None):
        """Advance development by one experience. Returns what each area did about it."""
        self.seen += 1
        out = {}
        areas = self.areas()
        per_area = self.drives(text, contrast=contrast) if text is not None else {}
        for name, area in areas.items():
            if name == "dentate":
                if not self.dentate_grows:
                    continue
            elif name not in self.growing:
                continue
            out[name] = self.regimes[name].step(area, per_area.get(name, drive), pressure, self.progress)
        # a population that changed shape invalidates every index into it: the word that lived on one unit
        # now lives on another. Re-encoding the vocabulary is what keeps the area able to hear after it grew.
        if out.get("wernicke", {}).get("grew") or out.get("wernicke", {}).get("died"):
            self.agent.wernicke.reindex()
        return out

    def spec(self):
        areas = self.areas()
        return {"phase": self.phase, "progress": round(self.progress, 3), "seen": self.seen,
                "why": self.why,
                "units": {name: (getattr(area, "l4", None) or getattr(area, "dendrites")).n
                          for name, area in areas.items()},
                "born": {name: getattr((getattr(area, "l4", None) or getattr(area, "dendrites")), "born", 0)
                         for name, area in areas.items()},
                "died": {name: getattr((getattr(area, "l4", None) or getattr(area, "dendrites")), "died", 0)
                         for name, area in areas.items()}}

In [ ]:
%%writefile /kaggle/working/neuromoe/PREREGISTRATION_development.md
# Pre-registration: does a growth-and-death critical period help cortex separate near-identical sentences?

Written 2026-09-23, before the v17 Kaggle run, after the v16 result it responds to.

## What v16 found (exploratory, not pre-registered)

Five seeds per arm, grades 1-5 of confusable pairs.

- `critical` (grow + cull in cortex, dentate held still) vs `frozen` (no structural change): recall tied
  (+0.005, t = 1.6), separation **worse** (-0.111, t = -4.4).
- `frozen` vs `no-phase-1`: separation +0.41 (t = 36). Exposure with Hebbian plasticity is the large effect.
- `critical` vs `dentate-too`: recall +0.14 (t = 4.0). Dentate growth during phase 1 hurts.

Diagnosis: new cells were wired onto the fibres active for the current sentence. For a near-identical pair
those fibres are almost the same, so each new cell responded to both members and growth added overlap. The
late cull (2,030 deaths vs 1,951 births) then churned what remained.

## The change being tested

1. **Growth on the difference.** When one member of a pair is heard, new cells are wired to the fibres that
   member drives *and the other does not*. This follows the modelling result that newborn cells become
   selective for novel stimuli resembling stored ones (eLife 2021, 66463).
2. **A gentler cull.** Maximum deaths per step 48 -> 16, survival window 6 -> 12 exposures, floor 0.2 -> 0.1.

Nothing else changes: same grades, pairs (32), steps (400), seeds (5), arms, held-out sets, measures.

## Hypotheses

- **H1 (primary).** `critical` has higher mean separation than `frozen`, averaged over grades 1-5.
- **H2.** `critical` has recall at least equal to `frozen` (non-inferiority, margin 0.02).
- **H3.** `critical` beats `dentate-too` on recall (replicating v16).
- **H4.** `critical` beats `frozen` on grade-5 separation specifically (the hardest grade).

## Analysis

Paired t-test across the five seeds on per-seed means, df = 4, two-sided. H1-H4 tested at alpha = 0.05 with
Holm-Bonferroni across the four. Effect sizes reported with the per-seed standard deviation.

## What counts as failure

If H1 fails, the growth half of the proposal does not hold in this architecture and plasticity alone is doing
the work. That result will be reported as such, and the growth schedule will not be kept on by default.

## Amendment, 2026-09-23, still before the v17 run

Local pilots and a single-seed ablation, run after the text above and before anything was pushed, changed
what is being tested. They are reported here so nothing is hidden, and the hypotheses below replace H1-H4.

**Found in piloting:**

- The critical arm also ramped recurrent gain to 2.5 late in development. At that gain L2/3 falls into one
  attractor and every pair overlaps. That was confounded with growth in v16. Gain is now held at the area's
  own value in every arm, so it can no longer differ between them.
- Difference-targeted growth (above) did not fix separation. Neither did two further variants. Ablation,
  one seed, separation on held-out pairs: nothing 0.401, grow only 0.297, tune only 0.567, cull only 0.443.
  L4 separation fell with growth as well, so the harm is not the projection to L2/3 washing it out.
- The best growth variant tried is **conjunctive**: a new cell is wired to one word-at-position combination
  that distinguishes the pair, half its synapses on each branch from the word and half from the position,
  so the branch detects the conjunction rather than either half. That is the growth used in v17.

**Arms (six):** critical (conjunctive growth + cull + tuning), tune (tuning only), cull (cull + tuning),
frozen (none), dentate-too (critical + dentate growth), no-phase-1.

**Hypotheses, replacing H1-H4:**

- **H1.** critical > frozen on mean separation. (Is the full schedule better than nothing?)
- **H2.** tune > frozen on mean separation. (Does plasticity alone do it?)
- **H3.** critical > tune on mean separation. (Does growth add anything on top of tuning? This is the
  question the proposal actually turns on, and the pilot predicts it fails.)
- **H4.** critical > dentate-too on mean recall. (Replicating v16.)

Same analysis: paired t across five seeds, df = 4, two-sided, Holm-Bonferroni across H1-H4.

**Failure condition, restated:** if H3 fails, growth does not earn its place in this architecture, and the
honest description of the result is "plasticity and pruning separate; neurogenesis in cortex does not help".

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/learn_to_discriminate.py
"""Development, then acquisition: grow a cortex that can tell near-identical sentences apart, then learn.

    python brain/learn_to_discriminate.py --grades 1 2 3 4 5 --pairs 40 --arms all

Two phases, and the point of the whole exercise is the ordering.

  phase 0/1  the critical period. Cortex overproduces cells wherever what it is hearing collides with
             something it already holds, and culls whatever nothing recruited; growth far outweighs death
             early and the balance reverses as it proceeds, with plasticity strongest at the end. The dentate
             is held still throughout, so whatever separation is achieved was achieved by cortex.
  phase 2    adulthood. Cortex settles and the dentate becomes the only thing that grows. New facts arrive,
             and the question is not only whether they are learned but what happens to everything learned in
             phase 1 - because heavy dentate neurogenesis is known to cause forgetting, which is the
             neurogenic account of why nobody remembers being two.

Four arms, so the claim is falsifiable rather than illustrated:

  critical       phase 0/1 with the dentate held still, then phase 2      - the proposal
  dentate-too    phase 0/1 with the dentate growing as well, then phase 2 - is holding it still what helped?
  frozen         no growth or death at all, then phase 2                  - is the schedule doing anything?
  no-phase-1     straight to phase 2                                      - was the critical period needed?

Measured on held-out pairs the system never developed on: recall precision (cue with one member, see which
memory comes back - chance is a half) and separation (one minus the overlap of the two settled patterns).
"""
import argparse
import json
import os
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "chat"))

from brain import curriculum  # noqa: E402
from brain.agent import Agent  # noqa: E402
from brain.plasticity import Development  # noqa: E402

ARMS = {
    "critical": {"phase": "critical", "dentate": False},
    "tune": {"phase": "tune", "dentate": False},
    "cull": {"phase": "cull", "dentate": False},
    "dentate-too": {"phase": "critical", "dentate": True},
    "frozen": {"phase": "frozen", "dentate": False},
    "no-phase-1": {"phase": None, "dentate": False},
}


def pressure_of(agent, first, second):
    """How badly these two collide right now: one minus how far apart they settle. Drives growth."""
    return 1.0 - curriculum.separation(agent.wernicke, first, second)


def develop(agent, examples, steps, dentate=False, phase="critical"):
    """Phase 0/1: hear confusable pairs, grow where they collide, cull what nothing used."""
    dev = Development(agent, phase=phase, steps=steps)
    if dentate and phase == "critical":
        dev.growing = dev.growing + ("dentate",)
        dev.dentate_grows = True
    seen, trace = 0, []
    while seen < steps:
        for first, second in examples:
            if seen >= steps:
                break
            pressure = pressure_of(agent, first, second)
            agent.wernicke.hear(first)
            did = dev.step(text=first, pressure=pressure, contrast=second)
            agent.wernicke.hear(second)
            dev.step(text=second, pressure=pressure, contrast=first)
            seen += 1
            if seen % max(1, steps // 8) == 0:
                trace.append({"seen": seen, "pressure": round(pressure, 3),
                              **{k: {"grew": v["grew"], "died": v["died"], "units": v["units"]}
                                 for k, v in did.items()}})
    return dev, trace


def acquire(agent, facts, steps=None):
    """Phase 2: the dentate is the only thing growing, and new facts arrive."""
    dev = Development(agent, phase="adult", steps=steps or max(1, len(facts)))
    for fact in facts:
        agent.say(fact)
        dev.step(text=fact, pressure=1.0)
    return dev


def run_arm(name, grades, pairs_per_grade, steps, seed, new_facts):
    """One arm, start to finish, measured before and after acquisition."""
    arm = ARMS[name]
    agent = Agent(background=False, seed=seed)
    train = curriculum.graded(count=pairs_per_grade, seed=seed, grades=grades)
    held = curriculum.graded(count=max(8, pairs_per_grade // 4), seed=seed + 991, grades=grades)
    t0 = time.time()
    trace, dev = [], None
    if arm["phase"]:
        flat = [pair for grade in grades for pair in train[grade]]
        dev, trace = develop(agent, flat, steps=steps, dentate=arm["dentate"], phase=arm["phase"])
    before = {grade: curriculum.report(agent, held[grade]) for grade in grades}
    after_dev = acquire(agent, new_facts)
    after = {grade: curriculum.report(agent, held[grade]) for grade in grades}
    return {"arm": name, "seed": seed, "seconds": round(time.time() - t0, 1),
            "before": before, "after": after, "trace": trace,
            "development": dev.spec() if dev else None, "acquisition": after_dev.spec(),
            "forgetting": {grade: round(before[grade]["recall"] - after[grade]["recall"], 4)
                           for grade in grades}}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--grades", type=int, nargs="*", default=[1, 2, 3, 4, 5])
    ap.add_argument("--pairs", type=int, default=40)
    ap.add_argument("--steps", type=int, default=240)
    ap.add_argument("--seeds", type=int, default=1)
    ap.add_argument("--arms", nargs="*", default=["critical", "frozen"])
    ap.add_argument("--out", default=os.path.join(ROOT, "results", "discriminate.json"))
    args = ap.parse_args()
    arms = list(ARMS) if args.arms == ["all"] else args.arms

    new_facts = ["Rex is a dog.", "dogs are mammals.", "mammals are animals.", "Ada is older than Ben.",
                 "the cafe is north of the park.", "Milo is a border collie.", "Ivy is taller than Ida.",
                 "Otto is faster than Otis."]
    results = []
    for seed in range(args.seeds):
        for arm in arms:
            out = run_arm(arm, args.grades, args.pairs, args.steps, seed, new_facts)
            results.append(out)
            recalls = " ".join(f"g{g}:{out['after'][g]['recall']:.2f}" for g in args.grades)
            seps = " ".join(f"g{g}:{out['after'][g]['separation']:.2f}" for g in args.grades)
            grew = out["development"]["born"] if out["development"] else {}
            print(f"[{arm:11} seed {seed}] recall {recalls} | separation {seps} | born {grew} "
                  f"| {out['seconds']}s", flush=True)
    os.makedirs(os.path.dirname(args.out), exist_ok=True)
    json.dump(results, open(args.out, "w"), indent=1)
    print(f"\n-> {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/thalamus.py
"""The thalamus: what gets through, and what holds the goal still while it does.

The textbook thalamus is a relay, and that is the part of it this file deliberately does not implement -
nothing here routes content between areas, because in the brain most cortico-cortical traffic is direct and
the outbound motor path does not pass through the thalamus at all. What the thalamus does that nothing else
does is two things, and both are exactly what this system was missing.

  the gate        the reticular nucleus inhibits the relay nuclei: it decides what does *not* get through,
                  under the goal currently in play. Crick called it the searchlight. Here it suppresses bids
                  that have nothing to do with what is being worked on, before the workspace ever sees them,
                  which is why most of what an area thinks never becomes globally available.
  the sustainer   the mediodorsal nucleus does not carry the content of a rule to prefrontal cortex - it
                  holds prefrontal cortex *stable* while the rule is in use. Inhibiting that pathway
                  destabilises prefrontal representations and breaks rule transfer; strengthening it improves
                  performance (Science Advances 2026). It also represents uncertainty in its own right, and
                  drives prefrontal reconfiguration when the rule stops working (Nature 2024).

So: a filter that narrows, and a loop that keeps the goal alive across cycles and says when to give up on it.
Arousal lives here too, as one knob - the gain the workspace competes under - because in cortex arousal *is*
gain modulation, and changing it changes what can ignite rather than sitting beside it.
"""


class Thalamus:
    """Reticular gating, mediodorsal sustaining, and the arousal that sets the workspace's gain."""

    def __init__(self, workspace, relevance=0.25, patience=3):
        self.workspace = workspace
        self.relevance = relevance    # how related to the goal a bid must be to survive the gate
        self.patience = patience      # cycles without progress before the goal is called into question
        self.goal = None
        self.held = 0                 # how many cycles this goal has been sustained
        self.stale = 0                # how many of those made no progress
        self.blocked = []             # what the gate stopped, so it can be inspected
        self.history = []

    # -------------------------------------------------------------------- goal
    def hold(self, goal):
        """Take a goal and keep it alive. Nothing else in this system survives across cycles."""
        self.goal = goal
        self.held, self.stale, self.blocked = 0, 0, []
        self.history.append({"goal": getattr(goal, "label", str(goal)), "cycles": 0})
        return goal

    def sustain(self):
        """Re-assert the goal for another cycle. Returns False when it has stopped being worth holding."""
        if self.goal is None:
            return False
        self.held += 1
        if self.history:
            self.history[-1]["cycles"] = self.held
        return self.stale < self.patience

    def progressed(self, made_progress):
        """Whether the last cycle got anywhere. Enough failures and the goal is reconfigured, not repeated."""
        self.stale = 0 if made_progress else self.stale + 1
        return self.stale

    def uncertainty(self):
        """How much the current goal is failing - what drives prefrontal cortex to try something else."""
        return round(min(1.0, self.stale / max(1, self.patience)), 3)

    # -------------------------------------------------------------------- gate
    def gate(self, bids, focus=None):
        """What survives to compete. A bid unrelated to the goal never reaches the workspace at all."""
        wanted = self._words(focus) | self._words(self.goal)
        kept = []
        for bid in bids:
            share = self._overlap(bid, wanted)
            if share < self.relevance and bid.salience < 0.9:
                self.blocked.append(bid)
                continue
            # attention is not only exclusion: what matches the goal is amplified, which is what makes the
            # difference between a bid that ignites and one that sits just under threshold
            bid.salience = min(1.0, bid.salience * (1.0 + share))
            kept.append(bid)
        return kept

    @staticmethod
    def _words(thing):
        if thing is None:
            return set()
        text = getattr(thing, "label", None) or str(thing)
        return {w for w in text.lower().replace(",", " ").split() if len(w) > 2}

    def _overlap(self, bid, wanted):
        if not wanted:
            return 1.0
        mine = self._words(bid) | self._words(bid.payload.get("focus"))
        return len(mine & wanted) / max(1, len(wanted))

    # ------------------------------------------------------------------ arousal
    def arouse(self, difficulty):
        """Set how hard the system is thinking. One knob: gain, which decides what can ignite at all.

        Low gain is a system half-awake - a couple of contents cross threshold, nothing chains. High gain
        broadcasts more and reaches further, and costs more cycles. It is deliberately not a separate dial
        from "what wins": raising it changes which bids win, because that is what gain modulation does.
        """
        self.workspace.gain = round(max(0.5, min(2.0, 0.8 + 0.6 * difficulty)), 2)
        return self.workspace.gain

    def spec(self):
        return {"goal": getattr(self.goal, "label", str(self.goal)) if self.goal else None,
                "held": self.held, "stale": self.stale, "uncertainty": self.uncertainty(),
                "blocked": len(self.blocked), "gain": self.workspace.gain}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/workspace.py
"""The global workspace: one thing at a time, seen by everything.

Until now the areas were a production line. The sentence went to Wernicke, then working memory, then the gate
picked one action, then one area answered, then Broca spoke, and no area ever heard what another had found.
That is why reasoning could not chain: nothing was ever available to more than one place at once.

Here instead every area *bids*, one bid wins, and the winner is broadcast to all of them. The rules are the
ones the theory (Baars' global workspace, Dehaene's global neuronal workspace, Blum's conscious Turing
machine) actually specifies, and each one earns its place by changing what the system can do:

  competition   areas post candidates with a salience. Only the strongest crosses threshold. Everything else
                stays local to the area that thought it - it still shapes that area, and no other area ever
                learns of it. That exclusion is the whole mechanism: a workspace that broadcasts everything
                is a for-loop with a grand name.
  gain          the threshold is not fixed. Arousal in the brain works by changing gain, and changing gain
                changes what can ignite at all - low gain, few ignitions, shallow local processing; high
                gain, wide broadcast and deeper chains. It is one knob, not two, and it is how hard the
                system is currently thinking.
  broadcast     the winner reaches every subscriber, which may each change state and bid again in response.
                This is what lets the map's answer become the reader's question and the reader's answer
                become the map's next hop.
  the stream    the sequence of ignited contents *is* the reasoning trace. Not a log written alongside the
                work - the work itself, in the order it became globally available.

Nothing here is called conscious. If that word applies to a system like this at all, it applies to the whole
loop while it is igniting and broadcasting, not to any part of it - which is why the file is named for the
mechanism and not for the claim. What can be measured is how globally a content spreads, and that is what
`index` does, after the perturbational complexity index used clinically on human patients: perturb, then see
how far and how variously the response travels.
"""
import hashlib


class Content:
    """Something an area is offering to the rest of the system."""

    def __init__(self, source, label, salience, payload=None, kind="fact"):
        self.source = source
        self.label = str(label)
        self.salience = float(salience)
        self.payload = payload or {}
        self.kind = kind

    def __repr__(self):
        return f"<{self.source}:{self.label} {self.salience:.2f}>"

    def spec(self):
        return {"source": self.source, "label": self.label, "kind": self.kind,
                "salience": round(self.salience, 3), **{k: v for k, v in self.payload.items()
                                                        if isinstance(v, (str, int, float, bool))}}


class Workspace:
    """One content at a time, broadcast to everything that subscribed."""

    def __init__(self, gain=1.0, threshold=0.35):
        self.gain = gain              # arousal: how hard it is currently thinking
        self.threshold = threshold    # what a bid must reach, after gain, to be broadcast
        self.subscribers = {}         # name -> area object
        self.bids = []
        self.stream = []              # what ignited, in order
        self.suppressed = []          # what did not - kept so the exclusion can be inspected
        self.cycles = 0

    # ------------------------------------------------------------------ wiring
    def subscribe(self, name, area):
        """An area that can receive broadcasts. It needs `receive(content)` and `signature()`."""
        self.subscribers[name] = area
        return area

    def post(self, content):
        self.bids.append(content)
        return content

    # ------------------------------------------------------------- competition
    def cycle(self, learn=True):
        """One pass: strongest bid above threshold ignites and is broadcast. Returns it, or None."""
        self.cycles += 1
        if not self.bids:
            return None
        ranked = sorted(self.bids, key=lambda c: -c.salience * self.gain)
        winner = ranked[0]
        rest = ranked[1:]
        self.bids = []
        if winner.salience * self.gain < self.threshold:
            self.suppressed.extend(ranked)
            return None
        self.suppressed.extend(rest)
        if learn:
            self.broadcast(winner)
        self.stream.append(winner)
        return winner

    def broadcast(self, content):
        """Every subscriber receives it. What each does with it is its own business."""
        for name, area in self.subscribers.items():
            if name == content.source:
                continue
            receive = getattr(area, "receive", None)
            if callable(receive):
                receive(content)
        return content

    # -------------------------------------------------------------- measuring
    def signatures(self):
        return {name: getattr(area, "signature", lambda: "")() for name, area in self.subscribers.items()}

    def index(self, content):
        """How global this content actually is: the share of areas it changes, times how varied the changes.

        The clinical measure of consciousness in humans - the perturbational complexity index - perturbs the
        cortex and asks how widely and how *differently* the response spreads, because a response that is
        wide but uniform is a seizure rather than a thought. This is the same question asked of a running
        program, and it is the test that keeps the workspace honest: content that never ignites must move
        nothing, and content that ignites must move several areas in different ways.
        """
        before = self.signatures()
        self.broadcast(content)
        after = self.signatures()
        changed = [n for n in before if before[n] != after[n]]
        if not before:
            return {"spread": 0.0, "differentiation": 0.0, "index": 0.0, "changed": []}
        spread = len(changed) / len(before)
        distinct = len({after[n] for n in changed})
        differentiation = distinct / max(1, len(changed))
        return {"spread": round(spread, 3), "differentiation": round(differentiation, 3),
                "index": round(spread * differentiation, 3), "changed": changed}

    # ----------------------------------------------------------------- reading
    def report(self, last=6):
        """The stream, as the account of what was globally available and in what order."""
        return [c.label for c in self.stream[-last:]]

    def spec(self):
        return {"cycles": self.cycles, "gain": round(self.gain, 2), "threshold": self.threshold,
                "ignited": len(self.stream), "suppressed": len(self.suppressed),
                "subscribers": sorted(self.subscribers)}

    def clear(self):
        self.bids, self.stream, self.suppressed = [], [], []
        return self


def digest(*parts):
    """A short signature of an area's state, for measuring whether a broadcast changed anything."""
    return hashlib.blake2b("|".join(str(p) for p in parts).encode("utf-8"), digest_size=8).hexdigest()

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/wernicke.py
"""Wernicke's area: where hearing becomes meaning.

Until now the parser was called directly on the string somebody typed, and the codes the lexicon made were a
decoration alongside it. That gets the dependency backwards. Comprehension is not a function of the text; it
is a function of what the tissue does with the text, and the difference shows the moment anything goes wrong
with the tissue - which is the whole clinical picture of a Wernicke's aphasia, fluent speech and lost
comprehension in someone whose ears work perfectly.

So the text stops here. What leaves this area is a *pattern*, and everything downstream - the parse, the
topic, the intent - is read out of the pattern. Concretely:

  hearing      each word drives L4 through the lexicon's wiring, bound to where it sat in the utterance, and
               L2/3 settles into one pattern holding the whole utterance.
  recognition  a word is recognised by reading the settled pattern back against everything this area has ever
               heard, through an inverted index of which units belong to which words. A word it has never
               heard cannot be recovered, because nothing in the tissue knows it yet - hearing it is what
               registers it.
  meaning      the parse runs on the recovered words, not on the original string. So if the pattern is
               degraded - noise, an overloaded layer, too long an utterance - comprehension degrades with it,
               in the way it should: content words survive best, the structure goes first.

The hand-written grammar is still the hand-written grammar. What has changed is that it is now downstream of
a population code rather than upstream of one, and it only ever sees what the tissue managed to recover.
"""
import re
from collections import defaultdict

import torch

from brain.cortex import Laminar

WORD = re.compile(r"[A-Za-z0-9][A-Za-z0-9'\-]*|[.,?!;:]")
POSITIONS = 48          # how many word positions one utterance can bind at once


class Wernicke:
    """The auditory-to-meaning area. One laminar column, plus the index that lets its patterns be read back."""

    def __init__(self, lexicon, seed=0, n23=2048, k23=48):
        self.lex = lexicon
        self.area = Laminar("wernicke", lexicon.n, n4=2048, k4=40, n23=n23, k23=k23, seed=seed,
                            homeostasis=0.01, adapt=0.02)
        g = torch.Generator().manual_seed(seed + 31)
        # where a word sat is part of what was heard: position is bound in as its own random pattern, which is
        # what keeps "Rex is older than Max" from being the same pattern as "Max is older than Rex"
        self.place = (torch.rand(POSITIONS, lexicon.n, generator=g) < 0.004).float()
        self.heard = {}                     # word -> its lexical code, everything this area has ever heard
        self.index = defaultdict(set)       # lexical unit -> the words that drive it. The readout's index.
        self.utterances = 0
        self.last = None

    # ------------------------------------------------------------------ hearing
    def register(self, word):
        """Hearing a word for the first time is what makes it recoverable later. That is the plasticity here.

        What gets indexed is which of *this area's own* L4 units the word drives. A word is recognised in
        Wernicke's units, not in the lexicon's - which is why an area that has never heard a word cannot
        recover it however clearly it was said.
        """
        if word not in self.heard:
            lexical = self.lex.code(word)
            local = self.area.encode(lexical)
            self.heard[word] = {"lexical": lexical, "local": local, "size": float((local > 0).sum())}
            for unit in (local > 0).nonzero().flatten().tolist():
                self.index[unit].add(word)
        return self.heard[word]["lexical"]

    def reindex(self):
        """Re-learn which units carry which word, after the population has grown or been culled.

        Cells are born and die during development, and when they do, every unit index this area held is
        stale - the word that used to live on unit 900 now lives somewhere else, and reading the old index
        gives back nothing. So the vocabulary is re-encoded into the population as it now stands. It is the
        same information; it is held by a different set of cells.
        """
        self.index = defaultdict(set)
        for word, entry in self.heard.items():
            local = self.area.encode(entry["lexical"])
            entry["local"], entry["size"] = local, float((local > 0).sum())
            for unit in (local > 0).nonzero().flatten().tolist():
                self.index[unit].add(word)
        return len(self.heard)

    def drive_of(self, text):
        """What this area's input layer receives for an utterance, words bound to their positions.

        Growth has to be driven by what the area actually hears. The first version grew on a bag of words,
        which cannot tell "Rex is older than Max" from "Max is older than Rex" - the pair it most needed to -
        because the two contain the same words. Here each word carries where it sat.
        """
        words = WORD.findall(str(text))[:POSITIONS]
        drive = torch.zeros(self.lex.n)
        for i, word in enumerate(words):
            drive += self.lex.code(word.lower()) * (1.0 + 0.6 * self.place[i])
        return drive

    def conjunctions(self, text, contrast=None):
        """The word-at-position combinations this utterance has and its near-twin does not.

        For "Rex is older than Max" against "Max is older than Rex" that is Rex-first and Max-last: the only
        things that tell them apart. Each comes back as one drive vector - the word's code together with the
        position's - ready to be put on a single dendritic branch.
        """
        mine = WORD.findall(str(text))[:POSITIONS]
        theirs = WORD.findall(str(contrast))[:POSITIONS] if contrast else []
        out = []
        for i, word in enumerate(mine):
            if i < len(theirs) and theirs[i].lower() == word.lower():
                continue
            out.append((self.lex.code(word.lower()), self.place[i]))
        return out

    def hear(self, text, noise=0.0, learn=True):
        """An utterance -> the settled pattern in L2/3. Returns the pattern; nothing else leaves this area."""
        words = WORD.findall(str(text))[:POSITIONS]
        drives = []
        for i, word in enumerate(words):
            code = self.register(word.lower())
            drive = code + 0.6 * self.place[i]
            if noise:
                # noise is a fraction of the signal, not an absolute quantity: what degrades comprehension is
                # how much of the drive is not the word, and that is what a damaged or distracted area has
                drive = (1.0 - noise) * drive + noise * torch.rand(self.lex.n) * float(drive.max())
            drives.append(drive)
        if not drives:
            return None
        self.area.drive_many(drives, learn=learn)
        pattern = self.area.settle(steps=2, learn=learn)
        self.utterances += 1
        self.last = {"words": words, "pattern": pattern}
        return pattern

    # -------------------------------------------------------------- recognition
    def recognise(self, pattern=None, top=1):
        """What words this pattern contains, read back through the index. The population readout, in earnest.

        Only words this area has heard can be found, and a word is found by how much of its code the pattern
        still carries - so a pattern that has lost units loses its weakest words first.
        """
        pattern = self.area.state23 if pattern is None else pattern
        if pattern is None:
            return []
        # the settled pattern lives in L2/3; what it is *about* lives in L4, which is where lexical codes are
        drive = self.area.state4
        if drive is None:
            return []
        units = (drive > 0).nonzero().flatten().tolist()
        scores = defaultdict(float)
        for unit in units:
            for word in self.index.get(unit, ()):
                scores[word] += 1.0
        out = []
        for word, hits in scores.items():
            share = hits / max(1.0, self.heard[word]["size"])
            if share >= 0.5:
                out.append((word, round(share, 3)))
        out.sort(key=lambda x: -x[1])
        return out if top is None else out[:max(top, len(out))]

    def heard_text(self, noise=0.0):
        """The utterance as this area recovered it, in the order it was heard.

        What the parser downstream actually gets. With a clean pattern this is what was said; with a degraded
        one it is what survived, which is the honest input to comprehension.
        """
        if not self.last:
            return ""
        recovered = {w for w, _ in self.recognise(top=None)}
        kept = [w for w in self.last["words"] if w.lower() in recovered]
        return " ".join(kept)

    def fidelity(self):
        """How much of what was said this area actually recovered. 1.0 when the tissue is holding it all."""
        if not self.last or not self.last["words"]:
            return 1.0
        recovered = {w for w, _ in self.recognise(top=None)}
        return round(sum(1 for w in self.last["words"] if w.lower() in recovered) / len(self.last["words"]), 3)

    # ------------------------------------------------------------------- meaning
    def comprehend(self, text, parser, noise=0.0, learn=True):
        """Hear it, then parse what was recovered. Returns (parse, report).

        The parser never sees the string. It sees what came back out of the tissue, which on a good day is the
        same thing and on a bad day is not - and the difference is visible in the report rather than hidden.
        """
        self.hear(text, noise=noise, learn=learn)
        recovered = self.heard_text()
        fidelity = self.fidelity()
        # punctuation and case are not in the population code; they are restored from the acoustic trace, the
        # way prosody carries them. What is *tested* here is whether the words came back.
        surface = text if fidelity >= 0.999 else recovered
        parse = parser(surface)
        return parse, {"fidelity": fidelity, "recovered": recovered, "words": len(self.last["words"]),
                       "settled": self.area.settles}

    def spec(self):
        return {**self.area.spec(), "vocabulary": len(self.heard), "utterances": self.utterances}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/hippocampus.py
"""The hippocampus: keeping facts organised, and handing back what is asked for.

This is not where thinking happens. The hippocampus does the jobs it is known for - binding what belongs
together, keeping similar things apart, completing a memory from part of it, replaying it offline - and place
cells are how the organisation is laid out: every item has a position, so relations between items are
displacements, and retrieval is a population of cells with fields responding to a location.

What it offers upstream is therefore an *operation set*, not an opinion:

  recall        an item's position drives its place cells: a bump of activity rather than a coordinate.
  travel        path integration. Activity flows to the cells whose fields sit one displacement away, exactly
                as a place system shifts a bump when the animal moves. Nothing computes a new coordinate.
  settle        lateral inhibition sharpens the bump - a continuous attractor collapsing onto one place.
  read          what the bump overlaps: which items are there and by how much. The margin between the winner
                and the runner-up is how clean the retrieval was.

Prefrontal cortex is what decides which of these to run, in what order, and what the result means (`pfc.py`).
How much the place cells participate at all depends on the question: a spatial or relational one leans on them
heavily, remembering what someone said a minute ago barely touches them.
"""
import torch


class Thought:
    """A bump over the place cells of one map, and the operations that move it."""

    def __init__(self, cells, positions, names, live=None, width=None):
        self.cells = cells                       # the PlaceCells population
        self.pos = positions                     # (N, dim) where each item sits
        self.names = names
        self.live = live if live is not None else torch.arange(len(names))
        self.width = width or float(cells.width)
        self.bump = None
        self.history = []

    # ------------------------------------------------------------------ making
    def at(self, name, spread=None):
        """Put the bump where an item is - as wide as the evidence for it deserves.

        A thing pinned by five agreeing facts gets a tight bump; a thing placed by one fact, or caught between
        facts that disagree, gets a wide one. That is not a confidence number attached to the answer afterwards
        - the retrieval is genuinely blurrier, so everything downstream inherits the doubt: it settles less
        cleanly, it is less distinguishable from its neighbours, and the margin the planner reads off it is
        smaller. Uncertainty that lives in the representation cannot be forgotten by the code that uses it.
        """
        i = self.names.index(name) if not isinstance(name, int) else name
        bump = self.cells.encode(self.pos[i][None])[0]
        if spread and spread > 0.2:
            # blur it: the neighbourhood the item might be in, rather than the point it is filed at
            near = (self.pos[self.live] - self.pos[i]).norm(dim=1) < spread
            if bool(near.any()) and int(near.sum()) > 1:
                bump = bump + 0.6 * self.cells.encode(self.pos[self.live][near]).mean(0)
        self.bump = bump / bump.sum().clamp_min(1e-9)
        self.history = [("recall", name, float(self.bump.max()))]
        return self

    # ----------------------------------------------------------------- moving
    def travel(self, displacement):
        """Path integration: activity flows to the cells one displacement away.

        Each receiving cell is driven by how close its own field centre is to a sending cell's centre plus the
        displacement. That is a fixed pattern of connections between place cells - the same wiring that lets a
        rodent keep track of where it is in the dark - and it is applied here to a relation instead of a step.
        """
        if self.bump is None:
            raise ValueError("no retrieval in progress")
        self.bump = self.cells.flow(self.bump, displacement)
        self.history.append(("travel", [round(float(x), 2) for x in displacement], float(self.bump.max())))
        return self

    def settle(self, rounds=3, sharpness=2.0):
        """Lateral inhibition: the strongest region suppresses the rest until one place is left standing."""
        for _ in range(rounds):
            b = self.bump ** sharpness
            self.bump = b / b.sum().clamp_min(1e-9)
        self.history.append(("settle", None, float(self.bump.max())))
        return self

    def blend(self, other, weight=0.5):
        """Two thoughts at once - superposition, which is what makes a comparison possible at all."""
        self.bump = (1 - weight) * self.bump + weight * other.bump
        self.bump = self.bump / self.bump.sum().clamp_min(1e-9)
        self.history.append(("blend", None, float(self.bump.max())))
        return self

    # ---------------------------------------------------------------- reading
    def read(self, top=3):
        """What prefrontal cortex gets: which items this activity is over, and how clearly.

        The overlap between the bump and an item's own place code is the readout - the same operation the
        hippocampus offers downstream structures, which have no access to coordinates at all.
        """
        if self.bump is None or not len(self.live):
            return []
        codes = self.cells.encode(self.pos[self.live])
        # cosine, not raw overlap: a bump sitting squarely on an item scores near 1, and a bump that has
        # travelled off the map scores near 0 however it is normalised. Without that, "nothing is there" and
        # "something is there" look the same once the activity is scaled to sum to one.
        codes = codes / codes.norm(dim=1, keepdim=True).clamp_min(1e-9)
        bump = self.bump / self.bump.norm().clamp_min(1e-9)
        score = codes @ bump
        order = score.argsort(descending=True)[:top]
        return [(self.names[int(self.live[i])], float(score[i])) for i in order]

    def presence(self):
        """How much anything at all is at the current location: the top overlap, before any comparison."""
        hits = self.read(top=1)
        return hits[0][1] if hits else 0.0

    def confidence(self):
        """How sure, read off the activity: how far the winner is ahead of the runner-up."""
        hits = self.read(top=2)
        if not hits:
            return 0.0
        if len(hits) == 1 or hits[1][1] <= 0:
            return 1.0
        return float(max(0.0, min(1.0, (hits[0][1] - hits[1][1]) / max(hits[0][1], 1e-9))))


def thought_for(cogmap, place=None):
    """A thought over a cognitive map's own place cells and the items it currently holds."""
    live = cogmap.placed.nonzero().squeeze(1)
    return Thought(place or cogmap.place, cogmap.pos, cogmap.names, live=live)


def travel_answer(cogmap, start, moves, top=3):
    """"What is two steps north of the park?" - move the bump and report what is there, and how much."""
    t = thought_for(cogmap).at(start, spread=cogmap.spread_of(start))
    t.travel(cogmap._disp(tuple(moves)))
    t.settle()
    return t.read(top), t.confidence(), t.history, t.presence()


def walk(cogmap, start, relation, target, steps=8, sign=1):
    """Keep travelling along a relation and see whether the target turns up, and after how many steps.

    One step only answers about neighbours. Carrying on until something is found - or until the activity runs
    off the edge of what is placed - is what makes a multi-step question answerable, and it is the caller's job
    to decide how far is worth going.
    """
    t = thought_for(cogmap).at(start, spread=cogmap.spread_of(start))
    step = sign * cogmap.rel[relation]
    trail = []
    for n in range(1, steps + 1):
        t.travel(step)
        t.settle(rounds=2)
        hits = t.read(top=2)
        trail.append({"step": n, "nearest": hits[:2], "presence": round(t.presence(), 3)})
        if hits and hits[0][0] == target:
            return {"found": True, "steps": n, "trail": trail, "confidence": t.confidence()}
        if t.presence() < 0.05:                    # off the end of everything placed: nothing more out there
            break
    return {"found": False, "steps": None, "trail": trail, "confidence": t.confidence()}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/inquiry.py
"""Going and finding out: reading until it knows, or until it knows it will not.

A question it cannot answer is not the end of a turn. It is a goal, and the useful behaviour is to go and reduce
the uncertainty - read something, put what came back into the map, and ask again whether the answer has arrived.
Two things make that a loop rather than a single lookup:

  where to look       whichever part of the question the map knows least about. Reading about something already
                      well connected buys nothing; the thin part of the map is where a page can change an
                      answer. This is uncertainty-driven exploration, and it is the same principle as directing
                      a saccade at the least certain part of a scene.
  when to stop        two ways, and saying which one happened matters. Either confidence crosses the line, or
                      the last few readings changed nothing - diminishing returns, at which point "this is the
                      best I am going to get" is the honest report, not a failure to keep trying.

Confidence here is not a feeling. It is measured off the map: are both things placed, is there a chain between
them, how long, and how large the margin. A claim with no chain behind it gets no confidence no matter how many
pages were read.
"""


class Inquiry:
    def __init__(self, agent, budget=4, target=0.7, patience=2):
        self.agent = agent
        self.budget = budget          # how many pages it will read before giving up
        self.target = target          # the confidence that counts as answered
        self.patience = patience      # readings with no gain before it concludes this is as good as it gets
        self.log = []

    # ------------------------------------------------------------- uncertainty
    def confidence(self, terms, space=None):
        """How well placed the question is: 0 when nothing is known, 1 when everything is connected and far
        apart enough that one bad fact could not flip it."""
        maps = self.agent.maps
        known = [t for t in terms if any(m.knows(t) for m in maps.maps.values())]
        if not known:
            return 0.0
        score = 0.35 * (len(known) / max(1, len(terms)))
        if len(terms) >= 2 and space and space in maps.maps:
            a, b = terms[0], terms[1]
            if maps._connected(space, a, b):
                score += 0.4
                rel = next(iter(maps.maps[space].rel), None)
                if rel:
                    margin = abs(maps.maps[space].steps(b, rel, a))
                    score += 0.25 * min(1.0, margin / 3.0)
        elif len(terms) >= 2:
            for sp in maps.maps:
                if maps._connected(sp, terms[0], terms[1]):
                    score += 0.4
                    break
        return round(min(1.0, score), 3)

    def thinnest(self, terms):
        """The term the map knows least about - where a page is most likely to change the answer."""
        maps = self.agent.maps
        scored = []
        already = {p["term"] for p in self.agent.read_pages}
        for t in terms:
            if t in already:
                continue      # reading the same page twice is not looking harder, it is looking again
            facts = 0
            for m in maps.maps.values():
                if m.knows(t):
                    i = m.index[t]
                    facts += sum(1 for ea, _, eb in m.episodes if i in (ea, eb))
            scored.append((facts, t))
        scored.sort()
        return scored[0][1] if scored else None

    def frontier(self, term, skip=()):
        """What reading about this thing opened up: the neighbours it now has on the map.

        Reading the two ends of a question and stopping there is how you conclude that nothing connects
        anything. A page is a door: "Will Smith is a rapper" puts *rapper* on the map, and rapper is the term
        worth reading next, because a route between two distant things runs through the middle. Searching
        outward from both ends until the frontiers meet is how the middle gets found at all.
        """
        out = []
        for space, m in self.agent.maps.maps.items():
            if not m.knows(term):
                continue
            i = m.index[term]
            for ea, _, eb in m.episodes:
                for other in (ea, eb):
                    name = m.names[other]
                    if other != i and name not in skip and "_" not in name and len(name) > 2:
                        out.append(name)
        seen, ordered = set(), []
        for name in out:
            if name not in seen:
                seen.add(name)
                ordered.append(name)
        return ordered

    # -------------------------------------------------------------------- loop
    def pursue(self, terms, space=None):
        """Read outward from both ends until they meet, the answer is good enough, or reading stops helping."""
        terms = [t for t in terms if t]
        start = self.confidence(terms, space)
        best, stale = start, 0
        # one frontier per end of the question, each growing with whatever its reading turned up
        frontiers = [[t] for t in terms]
        read = set()
        while len(self.log) < self.budget and best < self.target:
            term, side = None, 0
            for i, front in enumerate(frontiers):        # alternate ends, nearest first: meet in the middle
                candidates = [t for t in front if t not in read]
                if candidates and (term is None or len(frontiers[side]) > len(front)):
                    term, side = self.thinnest(candidates) or candidates[0], i
            if term is None:
                break
            read.add(term)
            before = len(self.agent.maps.log)
            result = self.agent._do_search({"a": (term, term)}, term)
            gained = len(self.agent.maps.log) - before
            opened = [t for t in self.frontier(term, skip=read) if t not in frontiers[side]][:3]
            frontiers[side].extend(opened)
            now = self.confidence(terms, space)
            self.log.append({"read": term, "facts": gained, "confidence": now, "from_side": side,
                             "opened": opened, "page": (result.get("evidence") or {}).get("url")})
            if now > best + 0.02:
                best, stale = now, 0
            elif gained or opened:
                stale += 0        # a page that opened new ground is progress even if confidence has not moved
                best = max(best, now)
            else:
                stale += 1
            if stale >= self.patience:
                break
        met = self.confidence(terms, space) >= self.target
        return {"start": start, "confidence": best, "readings": self.log,
                "verdict": "answered" if met else ("exhausted" if self.log else "nothing to read"),
                "stopped_because": "confident" if met
                                   else ("no more gain" if stale >= self.patience else "budget")}


def report(inquiry_result, terms, say_name):
    """What it did and what it concluded - including the case where the conclusion is that there is no link."""
    r = inquiry_result
    pages = [f"{x['read']} ({x['facts']} fact(s))" for x in r["readings"]]
    names = " and ".join(say_name(t) for t in terms)
    if not r["readings"]:
        return f"I have nothing about {names} and nowhere to look it up."
    read = "I read up on " + ", then ".join(pages) + "."
    if r["verdict"] == "answered":
        return f"{read} That's enough to place them: confidence {r['confidence']:.2f}."
    return (f"{read} After that I'm at confidence {r['confidence']:.2f}, and the last reading changed nothing, "
            f"so this is the best I'm going to get. Nothing I found links {names}, and I'd rather say that "
            f"than invent a connection.")

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/judgement.py
"""Learning to tell a good retrieval from a bad one, instead of being told where the line is.

A constant like "act on it above 0.25" is me deciding, and it is wrong in the way that matters: it teaches the
system a boundary rather than letting it discover that two kinds of retrieval are different. Nothing in a brain
sets a threshold on its own hippocampal output. What it does is notice, over many retrievals, that some feel a
certain way and turn out to be usable, others feel another way and turn out not to be - and the discrimination
follows from that history. That is metamemory, and it is learned.

So what happens here is:

  the profile     a retrieval is described by its own activity - how much is there at all, how far the winner
                  is ahead of the runner-up, how spread out the activity is, how many steps it took. No labels,
                  no thresholds, just the shape of what came back.
  the outcome     what happened next. The answer stood, or it was corrected, or nothing could be built on it.
                  That is the only teaching signal, and it arrives after the fact, as it does in life.
  the judgement   an expert pool over profiles, learning which regions of that space have worked. Before it has
                  seen anything it says so, and the system acts anyway and finds out - which is the only way to
                  get the first examples.

The boundary is therefore wherever experience puts it, and it can move. Two retrievals that look the same to me
can end up on opposite sides of it if they behave differently, which is the entire point of learning it.
"""
import torch

from neuromoe import ExpertPool

USABLE, UNUSABLE = 0, 1


def profile(presence, clarity, spread, steps=0):
    """What a retrieval looked like, as numbers that came out of the activity itself."""
    return {"presence": float(presence), "clarity": float(clarity), "spread": float(spread),
            "steps": float(steps)}


class Metamemory:
    def __init__(self, lexicon, vigilance=0.5, trust=0.5, seed=13):
        self.lex = lexicon
        self.pool = ExpertPool(lexicon.n, 2, vigilance=vigilance, k_route=3, beta=0.3, labels="exact",
                               trust=trust)
        self.seen = 0

    def _code(self, prof):
        """The profile as graded drive: each feature its own population, scaled by how strong it was.

        Rendering a number as the strength of a fixed pattern is how a scalar enters a population code at all -
        the same trick the articulator uses for a margin - and it means the pool sees a *shape*, not a value it
        could compare against a constant.
        """
        v = torch.zeros(self.lex.n)
        for name, value in prof.items():
            v += min(1.0, max(0.0, value if name != "steps" else value / 8.0)) * self.lex.code("\x00meta:" + name)
        return self.lex.expand(v) if hasattr(self.lex, "expand") else self.lex.code(str(sorted(prof.items())))

    def assess(self, prof):
        """-> (verdict, confidence, how). 'unlearned' means it has no history here and will have to find out."""
        code = self._code(prof)
        if not len(self.pool) or self.seen < 4:
            return "act", 0.0, "unlearned", code
        p, match = self.pool.predict(code, return_match=True)
        if match < self.pool.vigilance:
            return "act", float(p.max()), "unlearned", code
        verdict = "act" if int(p.argmax()) == USABLE else "withhold"
        return verdict, float(p.max()), "learned", code

    def learn(self, code, worked):
        """What became of it. This is the only place the boundary comes from."""
        self.pool.learn(code, USABLE if worked else UNUSABLE)
        self.seen += 1
        return self.seen

    def stats(self):
        return {"experts": len(self.pool), "outcomes_seen": self.seen, "born": self.pool.born}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/lang.py
"""Learning to speak from what it reads, with no language model and no gradients.

Up to now the words came from phrases I wrote, which is the least brain-like thing in this project: nobody is
born knowing how to say "is a kind of". People hear an enormous number of sentences and end up with
expectations about what follows what. That is a sequence memory, and it can be built the way memory is actually
built - a synapse from the context that was active to the word that came next, strengthened each time it
happens.

The first version of this ran the contexts through the expert pool, which compares a code against every expert
that exists. That is a dense matrix multiply per word and it does not survive a corpus. Real connectivity is
sparse: a context reaches the handful of cells it is wired to and nothing else. So the store here is sparse -
a context's active units index straight into the words that have followed them - which is the same computation
the pool was approximating, done in the way the tissue does it, and fast enough to hear a hundred thousand
sentences.

  hearing      each word arrives with the last few words still active as a decaying trace. The trace is hashed
               into a handful of context cells at several depths - four words back, three, two, one - and each
               of them records what came next. Deep contexts are specific and rare; shallow ones are general
               and always available, which is exactly what a backoff is, arrived at by having both.
  speaking     the deepest context with real evidence proposes the next word, and the shallower ones are there
               when it has none.
  allowed      function words are free. A content word may only be said if the answer being expressed licenses
               it, so fluency comes from what it has read and content comes from the map. A language model
               fuses those two; keeping them apart is what stops this one inventing.
"""
import math
import os
import pickle
import re
import zlib
from collections import defaultdict

WORD = re.compile(r"[A-Za-z][A-Za-z'-]*|[.,;:?!]")
FUNCTION = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being", "am", "of", "to", "in", "on", "at",
    "by", "for", "with", "from", "into", "onto", "over", "under", "about", "through", "and", "or", "but", "if",
    "then", "than", "as", "so", "because", "while", "when", "where", "that", "which", "who", "whose", "it",
    "its", "they", "them", "their", "he", "she", "his", "her", "him", "we", "us", "our", "you", "your", "i",
    "me", "my", "this", "these", "those", "there", "here", "also", "not", "no", "more", "most", "many", "much",
    "some", "any", "all", "both", "each", "other", "one", "two", "first", "second", "such", "same", "very",
    "often", "usually", "sometimes", "well", "known", "called", "has", "have", "had", "having", "do", "does",
    "did", "can", "could", "will", "would", "may", "might", "must", "should", "shall", "own", "only", "just",
    "still", "now", "later", "after", "before", "during", "between", "among", "against", "within", "without",
    ".", ",", ";", ":", "?", "!",
}
END = "\x00end"


def words_of(text):
    return [w.lower() for w in WORD.findall(text)]


class Speech:
    """A sparse sequence memory over words: hears sentences, then continues one."""

    def __init__(self, order=4, cells=1 << 20):
        self.order = order
        self.cells = cells
        # context cell -> {next word: how often}. Only the pairs that have happened exist, which is what makes
        # this a synapse table rather than a matrix.
        self.synapses = [defaultdict(lambda: defaultdict(float)) for _ in range(order)]
        self.heard = 0
        self.words = set()
        self.phrasings = {}      # relation -> how the world says it, counted

    def _cell(self, history, depth):
        """Which context cell the last `depth` words drive. A hash is the wiring: fixed, and shared by every
        context that happens to collide, which is what a finite population of cells means."""
        key = "\x1f".join(history[-depth:]) if depth else ""
        return zlib.crc32(key.encode("utf-8")) % self.cells

    # ---------------------------------------------------------------- hearing
    def hear(self, text):
        toks = words_of(text)
        if len(toks) < 3:
            return 0
        history = []
        for word in toks + [END]:
            for depth in range(1, self.order + 1):
                if len(history) >= depth - 1:
                    self.synapses[depth - 1][self._cell(history, depth)][word] += 1.0
            history.append(word)
            self.words.add(word)
        self.heard += 1
        return len(toks)

    def hear_many(self, sentences, progress=None):
        total = 0
        for i, s in enumerate(sentences):
            total += self.hear(s)
            if progress and i and i % progress == 0:
                print(f"  {i:,} sentences, {len(self.words):,} words, {self.size():,} synapses", flush=True)
        return total

    # --------------------------------------------------------------- speaking
    def follows(self, history, word):
        """How strongly this memory expects `word` right after `history`, deepest evidence first.

        `expect` answers "what comes next?" and only reports the leaders. Spelling repair needs the other
        question - "would *this* word come next?" - because "an essay" and "an say" are both unusual enough
        that neither makes a top-ten list, and one of them is English.
        """
        word = str(word).lower()
        history = [w.lower() for w in history]
        # backing off costs something: a context this memory has heard many times, which never once went on
        # to this word, is evidence against it - "write an" has been followed by "essay" and never by "easy",
        # and that should count for more than "an easy" being common in general
        penalty = 1.0
        for depth in range(min(self.order, len(history) + 1), 0, -1):
            counts = self.synapses[depth - 1].get(self._cell(history, depth))
            if counts:
                total = sum(counts.values())
                if word in counts:
                    return penalty * counts[word] / total
                penalty *= 0.05
        return 0.0

    def expect(self, history, top=12):
        """What it expects next: the deepest context with evidence wins, shallower ones back it up."""
        scores = defaultdict(float)
        for depth in range(self.order, 0, -1):
            if len(history) < depth - 1:
                continue
            counts = self.synapses[depth - 1].get(self._cell(history, depth))
            if not counts:
                continue
            total = sum(counts.values())
            weight = depth ** 2          # a longer context that has been seen is worth more than a short one
            for word, n in counts.items():
                scores[word] += weight * n / total
            if total >= 3 and depth >= 2:
                break                    # good evidence at this depth: no need to fall back further
        ranked = sorted(scores.items(), key=lambda kv: -kv[1])[:top]
        norm = sum(v for _, v in ranked) or 1.0
        return [(w, v / norm) for w, v in ranked]

    def say(self, opening, allowed, max_words=20, floor=0.02):
        """Continue from an opening, saying only what is free or licensed."""
        history = words_of(opening)
        out = list(history)
        allowed = {w.lower() for w in allowed}
        run = 0          # how many function words in a row: three is a sentence losing its way
        for _ in range(max_words):
            options = self.expect(history, top=20)
            word = None
            for candidate, score in options:
                if score < floor:
                    break
                if candidate == END:
                    word = END
                    break
                if candidate in out[-3:]:
                    continue                            # it has just said that; saying it again is a stutter
                if candidate in allowed:
                    word = candidate                    # content it is entitled to always wins
                    break
                if candidate in FUNCTION and run < 3 and word is None:
                    word = candidate                    # keep looking for content, but take this if nothing
            if word is None or word == END:
                break
            run = run + 1 if word in FUNCTION else 0
            out.append(word)
            history.append(word)
            if word in ".?!":
                break
        return out

    def learn_phrasing(self, relation, sentence, subject, obj):
        """How the world says this relation, learned from a sentence that states it.

        Gating generation word by word was the wrong shape: after "is a" the corpus wants *genre*, and refusing
        every content word it is not licensed to say leaves it stringing prepositions. What it actually needs
        from all that reading is the *phrasing* - what goes between two things when this relation holds - while
        the things themselves come from the map. So when the reader pulls a fact out of a sentence, the words
        that sat between the two ends are kept, and that is what it learns to say.
        """
        low = sentence.lower()
        i, j = low.find(subject.lower()), low.find(obj.lower())
        if i < 0 or j < 0 or j <= i:
            return None
        middle = words_of(low[i + len(subject):j])
        if not middle or len(middle) > 5:
            return None
        self.phrasings.setdefault(relation, defaultdict(float))[" ".join(middle)] += 1.0
        return " ".join(middle)

    def phrasing(self, relation, default=None, top=3):
        """The commonest way this relation is said, out of everything it has read."""
        counts = self.phrasings.get(relation)
        if not counts:
            return default
        ranked = sorted(counts.items(), key=lambda kv: -kv[1])[:top]
        return ranked[0][0]

    def phrasings_for(self, relation, top=5):
        counts = self.phrasings.get(relation, {})
        return sorted(counts.items(), key=lambda kv: -kv[1])[:top]

    def fluency(self, sentence):
        """How much of a sentence this memory would have predicted - a check on whether it learned anything."""
        toks = words_of(sentence)
        hits = 0
        for i in range(1, len(toks)):
            expected = [w for w, _ in self.expect(toks[:i], top=5)]
            hits += toks[i] in expected
        return hits / max(1, len(toks) - 1)

    # ------------------------------------------------------------------ state
    def size(self):
        return sum(len(v) for table in self.synapses for v in table.values())

    def stats(self):
        return {"sentences_heard": self.heard, "words": len(self.words), "synapses": self.size(),
                "contexts": sum(len(t) for t in self.synapses),
                "relations_it_can_say": len(self.phrasings)}

    def save(self, path):
        plain = [{k: dict(v) for k, v in table.items()} for table in self.synapses]
        with open(path, "wb") as f:
            pickle.dump({"order": self.order, "cells": self.cells, "synapses": plain,
                         "heard": self.heard, "words": sorted(self.words),
                         "phrasings": {r: dict(c) for r, c in self.phrasings.items()}}, f, protocol=4)
        return os.path.getsize(path)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            d = pickle.load(f)
        s = cls(order=d["order"], cells=d["cells"])
        for i, table in enumerate(d["synapses"]):
            for cell, counts in table.items():
                s.synapses[i][cell] = defaultdict(float, counts)
        s.heard, s.words = d["heard"], set(d["words"])
        s.phrasings = {r: defaultdict(float, c) for r, c in d.get("phrasings", {}).items()}
        return s

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/learn_to_speak.py
"""Teach it to speak, from text rather than from templates.

    python brain/learn_to_speak.py --sentences 40000

It hears real sentences once each and keeps what followed what. There is no training loop, no objective and no
backward pass: each context recruits or reinforces an expert, and the counts are the language. What comes out
is saved so the portal can load it at startup, the way a person arrives at a conversation already able to talk.

Sources, in order of preference: the crawl this project already made, the prose harvested from Wikipedia, and
failing both, whatever Wikipedia pages are named on the command line - so a machine with no corpus can still
teach it something in a couple of minutes.
"""
import argparse
import os
import re
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "chat"))

from brain.lang import Speech  # noqa: E402
import language as L  # noqa: E402

CORPORA = ["E:/corpora/crawled.tsv", "E:/corpora/concept_prose.tsv"]
# a speech model is hundreds of megabytes and has no business in a synced folder, so it lives on the big disk
# when there is one and beside the code when there is not
OUT = "E:/flybrain/speech.pt" if os.path.isdir("E:/flybrain") else os.path.join(HERE, "speech.pt")
SENT = re.compile(r"(?<=[.!?])\s+")


def from_file(path, limit):
    """Sentences out of a harvested file, whatever its column layout."""
    out = []
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            text = line.split("\t")[-1].strip()
            for s in SENT.split(text):
                s = s.strip()
                if 30 < len(s) < 300 and s[0].isupper() and s.count(" ") > 3:
                    out.append(s)
                    if len(out) >= limit:
                        return out
    return out


def from_web(topics, limit):
    import web
    out = []
    for topic in topics:
        page = web.look_up(topic, sentences=60)
        out += [s for s in page.get("sentences", []) if 30 < len(s) < 300]
        print(f"  read {page.get('title')}: {len(out)} sentences so far", flush=True)
        if len(out) >= limit:
            break
    return out[:limit]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--corpus", nargs="*", default=None,
                    help="corpus files to hear; the machine's own are used when omitted")
    ap.add_argument("--sentences", type=int, default=40000)
    ap.add_argument("--cells", type=int, default=1 << 20,
                    help="how many context cells: more sentences need more, or contexts collide")
    ap.add_argument("--topics", nargs="*", default=["Hip hop", "Dance", "Actor", "Music", "Biology", "City",
                                                    "Language", "Science", "History", "Film"])
    ap.add_argument("--out", default=OUT)
    args = ap.parse_args()

    sentences = []
    # every corpus on the machine, not the first one that exists: hearing more of the language is the only
    # thing that makes the difference between choosing a phrasing from evidence and choosing it from a rule
    # I wrote by hand, and there is no reason to leave half the text unread.
    for path in (args.corpus or CORPORA):
        if os.path.exists(path) and len(sentences) < args.sentences:
            got = from_file(path, args.sentences - len(sentences))
            print(f"{len(got):,} sentences from {path}", flush=True)
            sentences += got
    if not sentences:
        print("no corpus on this machine; reading a few pages instead", flush=True)
        sentences = from_web(args.topics, args.sentences)

    speech = Speech(cells=args.cells)
    t0 = time.time()
    speech.hear_many(sentences, progress=20000)
    # and, from the same sentences, how each relation is actually phrased
    # every relation it can recognise at all, phrased the way the corpus phrases it. Learning only from
    # prose_facts left it with three relations' worth of phrasing and a hand-written default for everything
    # else, which is how a hand-written default ends up deciding what it sounds like.
    learned = {}
    for s_ in sentences:
        found = list(L.prose_facts(s_, limit=3))
        whole = L.parse(s_)
        if whole.get("kind") == "fact" and whole.get("a") and whole.get("b") and whole.get("moves"):
            found.append(whole)
        for fact in found:
            if not (fact.get("moves") and fact.get("a") and fact.get("b")):
                continue
            rel = fact["moves"][0][0]
            if speech.learn_phrasing(rel, s_, fact["a"][1], fact["b"][1]):
                learned[rel] = learned.get(rel, 0) + 1
    print(f"phrasings for {len(learned)} relation(s): "
          + ", ".join(f"{r} ({n})" for r, n in sorted(learned.items(), key=lambda kv: -kv[1])[:12]), flush=True)
    size = speech.save(args.out)
    s = speech.stats()
    print(f"heard {s['sentences_heard']:,} sentences in {time.time() - t0:.0f}s: {s['words']:,} words, "
          f"{s['contexts']:,} contexts, {s['synapses']:,} synapses -> {args.out} ({size / 1e6:.1f} MB)")
    held = sentences[-200:]
    print(f"  it would have predicted {sum(speech.fluency(x) for x in held) / len(held):.1%} of the next words "
          f"in {len(held)} sentences it never heard")
    for opening in ("hip hop is", "the city is", "a dancer"):
        print(f"  \"{opening}\" -> {' '.join(speech.say(opening, allowed=set(), max_words=12))}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/learn_to_extract.py
"""Teach it to read more than four sentence shapes.

    python brain/learn_to_extract.py --sentences 200000 --rounds 2

Round 0 is the hand-written reader, run over the corpus to collect pairs it is confident about. Round 1 scans
the corpus for those pairs and keeps the phrasings that sat between them. Round 2 reads the corpus again with
the phrasings it just learned, and whatever *those* extract becomes the seed set for another scan.

Nothing here is a gradient and nothing here is supervised in the usual sense: the only labelled thing in the
process is a handful of pairs that a regular expression was sure about, and the corpus does the rest by
repeating itself. What comes out is a table of ways English says a relation, which is the thing that was
missing when a page about a city produced no facts at all.

It is measured the only way that means anything: held-out sentences the patterns never saw, how many facts
come out of them, and how many of those a person would call wrong.
"""
import argparse
import os
import random
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "chat"))

from brain import extract  # noqa: E402
from brain.learn_to_speak import from_file, CORPORA  # noqa: E402
import language as L  # noqa: E402

OUT = "E:/flybrain/patterns.pt" if os.path.isdir("E:/flybrain") else os.path.join(HERE, "patterns.pt")


def seeds_from_parser(sentences, limit=None):
    """What the hand-written reader is sure about: the only labelled thing in this whole process."""
    out = []
    for s in sentences:
        for fact in L.prose_facts(s, limit=3):
            if fact.get("a") and fact.get("b") and fact.get("moves"):
                out.append((fact["a"][1], fact["b"][1], fact["moves"][0][0]))
                if limit and len(out) >= limit:
                    return out
    return out


def seeds_from_patterns(sentences, patterns, limit=None):
    """And what the patterns themselves now extract, which is what makes the next round a bootstrap."""
    out = []
    for s in sentences:
        for fact in patterns.read(s):
            out.append((fact["a"][1], fact["b"][1], fact["moves"][0][0]))
            if limit and len(out) >= limit:
                return out
    return out


def yield_on(sentences, patterns):
    """How much a set of patterns gets out of text, next to the hand-written reader alone."""
    hand = sum(len(L.prose_facts(s, limit=3)) for s in sentences)
    learned = 0
    for s in sentences:
        if not L.prose_facts(s, limit=3):
            learned += len(patterns.read(s))
    return hand, learned


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--corpus", nargs="*", default=None,
                    help="corpus files to hear; the machine's own are used when omitted")
    ap.add_argument("--sentences", type=int, default=200000)
    ap.add_argument("--rounds", type=int, default=2)
    ap.add_argument("--support", type=int, default=3)
    ap.add_argument("--purity", type=float, default=0.75)
    ap.add_argument("--holdout", type=int, default=400)
    ap.add_argument("--out", default=OUT)
    args = ap.parse_args()

    sentences = []
    for path in (args.corpus or CORPORA):
        if os.path.exists(path) and len(sentences) < args.sentences:
            got = from_file(path, args.sentences - len(sentences))
            print(f"{len(got):,} sentences from {path}", flush=True)
            sentences += got
    if not sentences:
        print("no corpus on this machine - nothing to learn from", flush=True)
        return 1
    random.Random(0).shuffle(sentences)
    held, sentences = sentences[:args.holdout], sentences[args.holdout:]

    t0 = time.time()
    patterns = extract.Patterns(support=args.support, purity=args.purity)
    seeds = seeds_from_parser(sentences)
    print(f"round 0: {len(seeds):,} seed pairs from the hand-written reader", flush=True)

    for rnd in range(1, args.rounds + 1):
        got = extract.harvest(sentences, seeds, patterns, progress=40000)
        patterns.rounds = rnd
        kept = patterns.settle()
        by_relation = {}
        for phrasing, relation in kept.items():
            by_relation[relation] = by_relation.get(relation, 0) + 1
        print(f"round {rnd}: {got:,} observations -> {len(kept):,} phrasings kept "
              f"({', '.join(f'{r} {n}' for r, n in sorted(by_relation.items(), key=lambda kv: -kv[1])[:6])})",
              flush=True)
        for phrasing in sorted(kept, key=lambda m: -patterns.evidence.get(m, 0))[:8]:
            print(f"    \"{phrasing}\" -> {kept[phrasing]} ({patterns.evidence.get(phrasing, 0)} pairs)")
        if rnd < args.rounds:
            fresh = seeds_from_patterns(sentences[:len(sentences) // 2], patterns)
            print(f"  bootstrap: {len(fresh):,} new pairs from what those patterns read", flush=True)
            seeds = seeds + fresh

    hand, learned = yield_on(held, patterns)
    print(f"\non {len(held)} held-out sentences the patterns never saw:")
    print(f"  hand-written reader: {hand} fact(s)")
    print(f"  patterns add:        {learned} fact(s) from sentences it got nothing out of "
          f"({(hand + learned) / max(1, hand):.1f}x)")
    shown = 0
    for s in held:
        if shown >= 6:
            break
        if not L.prose_facts(s, limit=3):
            got = patterns.read(s)
            if got:
                print(f"    {s[:100]}")
                for f in got:
                    print(f"      -> {f['a'][1]} | {f['word']} | {f['b'][1]}   [{f['moves'][0][0]}]")
                shown += 1
    size = patterns.save(args.out)
    print(f"\n{patterns.stats()} -> {args.out} ({size / 1e6:.2f} MB) in {time.time() - t0:.0f}s")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/learn_to_converse.py
"""Teach it to talk, from people talking.

    python brain/learn_to_converse.py --turns 40000

Everything it had heard was encyclopedia prose. Prose states; it does not take a turn. So this goes and finds
real conversation - interviews and testimony on Wikisource, dialogue on Wikiquote, quoted speech in
public-domain novels - and learns two things from it.

  the sound of a turn     the same sequence memory that learned written English, now hearing spoken English.
                          This is what lets it judge whether a sentence it is about to say is one a person
                          would say, which is the only defence it has against sounding like a database.
  how a turn opens        counted, and counted *conditionally*: what people say before answering a question
                          is not what they say before adding to a statement, and a transcript shows both.
                          This replaces a list of discourse markers I wrote by hand, which had no business
                          being in a system whose whole claim is that it learns what it says.

The harvest is cached, so this is a network job once and an offline job afterwards, and so that the same
turns can be replayed on a machine with no internet at all.
"""
import argparse
import os
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "chat"))

from brain.answer import Markers  # noqa: E402
from brain.lang import Speech  # noqa: E402

BIG = "E:/flybrain" if os.path.isdir("E:/flybrain") else HERE
CACHE = os.path.join("E:/corpora" if os.path.isdir("E:/corpora") else HERE, "dialogue.tsv")
QUERIES = ["interview", "interview transcript", "oral history", "testimony", "press conference",
           "dialogue", "conversation", "questions and answers", "debate transcript", "speech"]


def cached(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8", errors="ignore") as f:
        return [line.rstrip("\n") for line in f if line.strip()]


def save_cache(path, turns):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for t in turns:
            f.write(t.replace("\t", " ").replace("\n", " ") + "\n")
    return os.path.getsize(path)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--turns", type=int, default=40000)
    ap.add_argument("--per-query", type=int, default=8)
    ap.add_argument("--offline", action="store_true", help="use the cached harvest only")
    ap.add_argument("--refresh", action="store_true", help="harvest again even if a cache exists")
    ap.add_argument("--markers", default=os.path.join(BIG, "markers.pt"))
    ap.add_argument("--speech", default=os.path.join(BIG, "speech.pt"),
                    help="the sequence memory to extend with spoken English; omit to skip")
    ap.add_argument("--cache", default=CACHE)
    args = ap.parse_args()

    turns = [] if args.refresh else cached(args.cache)
    if turns:
        print(f"{len(turns):,} turns from the cache at {args.cache}", flush=True)
    if not turns and not args.offline:
        import dialogue
        t0 = time.time()
        turns = dialogue.harvest(QUERIES, per_query=args.per_query)
        print(f"harvested {len(turns):,} turns in {time.time() - t0:.0f}s", flush=True)
        if turns:
            save_cache(args.cache, turns)
    if not turns:
        print("no conversation to learn from", flush=True)
        return 1
    turns = turns[:args.turns]

    markers = Markers()
    heard = markers.hear_many(turns)
    markers.save(args.markers)
    print(f"\nopenings: {heard:,} of {len(turns):,} turns began with one; {markers.stats()}")
    for context in ("after_question", "after_statement"):
        best = markers.best(context, top=6, floor=2)
        print(f"  {context}: {', '.join(repr(b) for b in best) or 'nothing heard often enough'}")

    if args.speech and os.path.exists(args.speech):
        t0 = time.time()
        speech = Speech.load(args.speech)
        before = speech.size()
        speech.hear_many(turns, progress=20000)
        speech.save(args.speech)
        print(f"\nspeech: heard {len(turns):,} spoken turns, {speech.size() - before:,} new synapses "
              f"({speech.size():,} total) in {time.time() - t0:.0f}s")
        held = turns[-200:]
        print(f"  it would now predict {sum(speech.fluency(x) for x in held) / max(1, len(held)):.1%} "
              f"of the next words in {len(held)} turns")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/motor.py
"""Speech as a readout of activity, not a formatter.

Until now an answer was computed and then written into a sentence by a template - the reasoning produced English
directly, which is the one thing a brain never does. Here the stores produce *activity*: which items are
involved, which direction along which relation, how large the margin, how confident. That pattern is what the
articulator sees. It has no access to the structure behind it and no idea what a "comparison" is; it maps a
population state onto words the way a decoder maps recorded activity onto what someone was about to say.

  imitation      it learns to speak by hearing sentences alongside the state they describe. Each word is bound
                 Hebbian-style to (state + the word just said), so what it learns is a state-to-sequence habit.
                 Nothing is pretrained; before it has heard anything it can say nothing at all.
  production     at each step the current state and the previous word drive the motor column, the word units
                 compete, the strongest fires, and it feeds back as the next step's context - efference copy.
  variety        a word unit that has just fired is briefly harder to drive, so phrasing shifts by itself.

Words exist at exactly two points in this system: the lexical cortex at the front, and here at the back.
Everything between them is activity.
"""
import torch

from brain.column import Column

END = "\x00end"


class Articulator:
    def __init__(self, lexicon, n=8192, k=48, branches=4, seed=11, adapt=0.25, max_words=24):
        self.lex = lexicon
        self.column = Column("motor", lexicon.n, n=n, k=k, branches=branches, seed=seed, n_out=1)
        self.words = [END]                    # output units, one per word; new words recruit new units
        self.index = {END: 0}
        self.fatigue = torch.zeros(1)
        self.adapt = adapt
        self.max_words = max_words
        self.heard = 0

    # ------------------------------------------------------------------- units
    def _unit(self, word):
        if word not in self.index:
            self.index[word] = len(self.words)
            self.words.append(word)
            self.column.out = torch.cat([self.column.out, torch.zeros(self.column.n, 1)], dim=1)
            self.fatigue = torch.cat([self.fatigue, torch.zeros(1)])
        return self.index[word]

    def _drive(self, frame, previous):
        """What the motor column is driven by: the *frame* of the answer - which relation, which way, how sure -
        plus the word just spoken. Deliberately not the identity of the items: a sentence shape has to survive
        being about different things, which is exactly what the slots are for. Content enters at the slot, the
        way a speech plan is filled from what is currently bound rather than rebuilt per item."""
        v = frame.clone()
        if previous is not None:
            v = v + 1.2 * self.lex.code(previous)
        return v

    # ---------------------------------------------------------------- learning
    @staticmethod
    def slot(role):
        return "\x00role:" + role

    @staticmethod
    def tokens(words):
        """Punctuation becomes its own unit. Otherwise "Cal" and "Cal." are different words, and a name learned
        at the end of one sentence is unusable in the middle of the next."""
        out = []
        for w in words:
            body = w.rstrip(".,?!;:")
            if body:
                out.append(body)
            out.extend(list(w[len(body):]))
        return out

    def _slotted(self, words, roles):
        """Names in a heard sentence become role slots, so what is learned is the *shape* of the sentence.

        Otherwise every name would have to be heard in every position before it could be said, which is not how
        anyone learns to talk. The slot is filled at production time by naming whatever is bound to that role.
        """
        if not roles:
            return list(words)
        lookup = {str(v).lower(): self.slot(r) for r, v in roles.items() if v}
        return [lookup.get(w.lower().strip(".,?!"), w) for w in words]

    def imitate(self, state, words, roles=None, strength=1.0):
        """Hear a sentence said in a situation, and bind it to that situation's activity."""
        words = self._slotted(self.tokens(words), roles)
        previous = None
        for word in list(words) + [END]:
            self._unit(word)
            code = self.column.encode(self._drive(state, previous), learn=False)
            self.column.associate(code, self.index[word], strength)
            previous = word if word != END else None
        self.heard += 1
        return len(self.words)

    # -------------------------------------------------------------- production
    def speak(self, state, roles=None, max_words=None):
        """-> the words this activity produces, or [] if nothing has ever been heard in a state like it.

        A role slot is filled by naming what is bound to that role: the pattern is read out, not looked up in
        the structure that produced it.
        """
        out, previous = [], None
        # adaptation within an utterance is what stops a word repeating in the same breath; between utterances
        # it recovers, so saying the same true thing twice does not force a worse wording the second time
        fatigue = self.fatigue.clone()
        for _ in range(max_words or self.max_words):
            code = self.column.encode(self._drive(state, previous), learn=False)
            scores = self.column.read(code).reshape(-1) / (1.0 + 0.3 * fatigue)
            if float(scores.max()) <= 0:
                break
            unit = int(scores.argmax())
            word = self.words[unit]
            if word == END:
                break
            fatigue[unit] += 1.0
            spoken = word
            if word.startswith("\x00role:"):
                spoken = (roles or {}).get(word[len("\x00role:"):]) or "it"
            out.append(str(spoken))
            previous = word
        self.fatigue.mul_(1 - self.adapt).add_(self.adapt * fatigue)
        return out

    def rehearse(self, state, words, roles=None):
        """Say it, and if it comes out wrong, hear the right version again - learning from its own errors, which
        is what rehearsal is for. Returns whether it already had it right."""
        said = self.speak(state, roles)
        if said == [str(w) for w in self.tokens(words)]:
            return True
        self.imitate(state, words, roles=roles, strength=1.5)
        return False

    def spec(self):
        return {"words": len(self.words) - 1, "sentences_heard": self.heard,
                "synapses": int((self.column.out != 0).sum())}


def detokenize(words):
    """Tokens back into a sentence: punctuation closes up against the word before it."""
    out = ""
    for w in words:
        out += w if (w in ".,?!;:" or not out) else " " + w
    return out


def role_bind(code, role, n):
    """Bind a pattern to a role by permuting it - the same item in a different role is a different pattern, so
    superposing subject and object does not lose which was which."""
    import zlib
    return torch.roll(code, int(zlib.crc32(role.encode()) % n))


def state_code(lexicon, parts, weights=None):
    """The answer as one population pattern: the items involved, the relation, and how the answer came out.

    Every part is already a population code somewhere in the system - an item's code in the lexical cortex, a
    relation's own code, a sign and a margin rendered as graded drive. Superposing them is what binding by
    coactivation looks like; nothing here is a string being concatenated.
    """
    weights = weights or {}
    v = torch.zeros(lexicon.n)
    for role, thing in parts.items():
        if thing is None:
            continue
        w = weights.get(role, 1.0)
        if isinstance(thing, torch.Tensor):
            code = thing
        elif isinstance(thing, bool):
            code = lexicon.code("\x00sign:" + ("pos" if thing else "neg"))
        elif isinstance(thing, (int, float)):
            code = lexicon.code("\x00scalar:" + role) * min(1.0, abs(float(thing)))
        else:
            code = lexicon.code(str(thing))
        v += w * role_bind(code, role, lexicon.n)
    return v

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/pfc.py
"""Prefrontal cortex: the part that thinks.

The stores do not reason. The hippocampus organises facts and hands back retrievals, the episodic store returns
what was said, working memory holds what is current, and none of them decides anything. This is where a question
becomes a plan, the plan becomes a sequence of retrievals, and the results become an answer with a reason.

  goal            what is being asked, held while the work happens. It survives several retrievals, which is
                  what makes multi-step reasoning possible at all.
  recruitment     how much each store is worth using for *this* question. A comparison of two things on a scale
                  is a map question and leans on place cells heavily; what someone said two minutes ago is an
                  episodic question and barely touches them; keeping track of who "she" is needs neither. The
                  weighting is part of the plan, not a constant, and it is reported so it can be argued with.
  composition     several retrievals are put together here: travel from one place and check what is there,
                  then travel back and check the reverse, then compare the two. The hippocampus can do none of
                  that on its own - it has no goal to hold.
  evaluation      whether the retrieval was clean enough to act on, whether two retrievals agree, and what to
                  do when they do not: retrieve again, ask, or say it does not know.

Everything here operates on what the stores return. No words are involved, and nothing is differentiated.
"""
from brain import hippocampus, judgement

# how much each store is worth consulting, per kind of question. The numbers are not a mood: they are what the
# planner multiplies its retrieval budget by, and a store weighted zero is not consulted at all.
RECRUITMENT = {
    "compare":  {"map": 1.0, "episodic": 0.1, "working": 0.3},
    "which":    {"map": 1.0, "episodic": 0.1, "working": 0.3},
    "extreme":  {"map": 1.0, "episodic": 0.0, "working": 0.2},
    "rank":     {"map": 1.0, "episodic": 0.0, "working": 0.1},
    "at":       {"map": 1.0, "episodic": 0.0, "working": 0.2},
    "where":    {"map": 1.0, "episodic": 0.0, "working": 0.2},
    "between":  {"map": 1.0, "episodic": 0.0, "working": 0.1},
    "relate":   {"map": 0.8, "episodic": 0.3, "working": 0.2},
    "recall":   {"map": 0.1, "episodic": 1.0, "working": 0.6},
    "report":   {"map": 0.3, "episodic": 0.8, "working": 0.4},
    "fact":     {"map": 0.9, "episodic": 0.4, "working": 0.5},
    "social":   {"map": 0.0, "episodic": 0.3, "working": 0.5},
}
DEFAULT = {"map": 0.5, "episodic": 0.5, "working": 0.5}


def recruitment(intent):
    """Which stores this question is worth putting to work, and how hard."""
    return dict(RECRUITMENT.get(intent, DEFAULT))


class Plan:
    """A goal held while retrievals are run against it."""

    def __init__(self, intent, terms, space=None):
        self.intent, self.terms, self.space = intent, terms, space
        self.recruit = recruitment(intent)
        self.steps = []          # what was actually done, in order
        self.result = None

    def note(self, what, detail, clarity=None):
        self.steps.append({"did": what, "detail": detail, "clarity": clarity})
        return self

    def account(self):
        return {"intent": self.intent, "terms": self.terms, "recruited": self.recruit,
                "steps": self.steps, "result": self.result}


def compare(cogmap, a, b, relation, plan=None):
    """Decide which of two items is further along a relation, by running the retrieval both ways.

    One retrieval is not an argument. Travelling from b along the relation should arrive toward a, and
    travelling from a should not arrive toward b; when both agree the answer is safe, and when they disagree the
    pair is too close to call - which is a conclusion drawn here, not something the hippocampus said.
    """
    plan = plan or Plan("compare", [a, b], None)
    if plan.recruit["map"] <= 0:
        return {"verdict": None, "why": "not a question for the map", "plan": plan.account()}
    budget = max(2, int(8 * plan.recruit["map"]))          # how far it is worth walking, given the question
    forward = hippocampus.walk(cogmap, b, relation, a, steps=budget, sign=+1)
    plan.note(f"walk up from {b} looking for {a}", forward["trail"][-1] if forward["trail"] else None,
              forward["confidence"])
    backward = hippocampus.walk(cogmap, a, relation, b, steps=budget, sign=+1)
    plan.note(f"walk up from {a} looking for {b}", backward["trail"][-1] if backward["trail"] else None,
              backward["confidence"])
    if forward["found"] and not backward["found"]:
        verdict, why = True, f"walking up from {b} reaches {a} in {forward['steps']} step(s), and the reverse " \
                             f"walk never reaches it"
    elif backward["found"] and not forward["found"]:
        verdict, why = False, f"walking up from {a} reaches {b} in {backward['steps']} step(s), so it is the " \
                              f"other way round"
    elif forward["found"] and backward["found"]:
        verdict = forward["steps"] <= backward["steps"]
        why = "both walks arrive, which means the facts are inconsistent; I am going with the shorter route"
    elif a in cogmap.index and b in cogmap.index and relation in cogmap.rel:
        # Neither walk arrives, which happens when the two sit less than a step apart - after a correction, for
        # instance, when one fact has been weighted against another and they have nearly cancelled. Refusing
        # would be hiding behind the method: the positions still say something, so say it, with the caveat that
        # the gap is smaller than a single stated step.
        gap = cogmap.steps(b, relation, a)
        plan.note("fall back to how far apart they sit", round(gap, 3), abs(gap))
        if abs(gap) < 0.15:
            verdict, why = None, "they sit on top of each other - what you have told me cancels out"
        else:
            verdict = gap > 0
            why = (f"no walk arrives because they are less than one step apart; by position "
                   f"{'the first' if verdict else 'the second'} is ahead by {abs(gap):.2f} of a step, which is "
                   f"weak evidence")
    else:
        verdict, why = None, "neither walk arrives, so nothing I hold puts these two on the same scale"
    # how floating the two ends are: an answer about things whose own facts disagree is worth less than the
    # same answer about things that are pinned, and the planner is where that gets said out loud
    vague = max(cogmap.spread_of(a), cogmap.spread_of(b)) if hasattr(cogmap, "spread_of") else 0.0
    plan.result = {"verdict": verdict, "why": why, "steps": forward["steps"] or backward["steps"],
                   "spread": round(vague, 2)}
    return {"verdict": verdict, "why": why, "steps": forward["steps"] or backward["steps"], "spread": vague,
            "clarity": round(max(forward["confidence"], backward["confidence"]), 3), "plan": plan.account()}


def locate(cogmap, start, moves, plan=None, meta=None):
    """What is at the end of a route: one retrieval, and an explicit view on whether it was clean enough."""
    plan = plan or Plan("at", [start], None)
    hits, clarity, history, presence = hippocampus.travel_answer(cogmap, start, moves)
    plan.note("travel", hits[:2], clarity)
    spread = 1.0 - (hits[0][1] if hits else 0.0)
    prof = judgement.profile(presence, clarity, spread, steps=sum(abs(c) for _, c in moves))
    verdict, confidence, how, code = (meta.assess(prof) if meta else ("act", 0.0, "unlearned", None))
    if not hits:
        plan.result = {"item": None, "why": "nothing is placed anywhere near there"}
    elif verdict == "withhold":
        plan.result = {"item": None, "learned": True,
                       "why": "retrievals that came back looking like this one have not held up before"}
    else:
        plan.result = {"item": hits[0][0],
                       "why": "the activity settled on one place" if how == "learned"
                              else "this is what the activity settled on - I have no history yet for how far "
                                   "to trust a retrieval shaped like this, so I am going with it and will see"}
    plan.result["judgement"] = {"verdict": verdict, "how": how, "confidence": round(confidence, 3)}
    return {"hits": hits, "clarity": round(clarity, 3), "presence": round(presence, 3),
            "profile": prof, "meta_code": code, "plan": plan.account()}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/prose.py
"""Writing more than one sentence: planning a paragraph, not padding a line.

A single reply per question is what a lookup produces. Prose is a different job: decide what the paragraph is
about, put the strongest thing first, join the rest with connectives that mean something ("because", "but",
"which leaves"), mark what is weak as weak, and finish by saying what is missing. That is discourse planning,
and it is prefrontal work - the sentences themselves are still produced one at a time by the articulator.

No language model is involved and none is needed for this part. What makes prose readable is mostly *ordering*
and *hedging*, both of which are decisions about content: strongest evidence first, one idea per sentence,
concessives where the evidence is mixed, and an explicit statement of the gap at the end. The vocabulary of
connectives is a grammar, not knowledge - the same status as knowing that "than" follows a comparative.

What it will never do is write around a gap. If there is nothing between two ideas, the paragraph says so and
stops, which is the honest version of the essay it was asked for.
"""

from brain import answer
from brain import compose

JOIN = {
    "because": "because",
    "and": "and",
    "but": "but",
    "so": "so",
    "although": "although",
    "which leaves": "which leaves",
}


def paragraph(sentences, gap=None, caveat=None):
    """Assemble planned clauses into prose: a lead, connected middles, an honest ending."""
    parts = [s for s in sentences if s]
    if not parts:
        return gap or ""
    def close(s):
        s = s.strip()
        return s if s.endswith(("?", "!", ".")) else s.rstrip(".") + "."

    text = close(parts[0])
    for s in parts[1:] + [x for x in (caveat, gap) if x]:
        text += " " + close(s)
    return text


def plan_relation(a, b, route, shared, credit, say_name):
    """A paragraph about how two things relate, ordered by how much is actually behind each claim."""
    A, B = say_name(a), say_name(b)
    if route:
        steps = len(route)
        lead = (f"{A} and {B} are connected, and the connection is {steps} fact(s) long rather than anything "
                f"I am reading into it")
        middles = []
        for e in route:
            rel = ", ".join(r for r, _ in e["moves"]) or "linked to"
            middles.append(f"{say_name(e['from'])} is {rel} {say_name(e['to'])}, which {credit(e['to'])}")
        weakest = min(route, key=lambda e: e.get("weight", 1.0))
        caveat = (f"The step I would question first is {say_name(weakest['from'])} to "
                  f"{say_name(weakest['to'])}, which has the least behind it")
        return paragraph([lead] + middles[:4], caveat=caveat)
    if shared:
        names = ", ".join(say_name(s["term"]) for s in shared[:3])
        return paragraph([
            f"There is no route from {A} to {B} in anything I hold",
            f"What they have instead is company: both sit near {names}",
            f"That is a resemblance, and a resemblance is not a cause - it would not carry a claim that one "
            f"shaped the other",
        ], gap=f"If there is a real link, it is in something I have not read")
    return paragraph([
        f"I know something about {A} and something about {B}",
        f"Between them I have nothing at all: no chain of facts, and not even a shared neighbour",
    ], gap="My answer is that they are unrelated as far as I know, which is a finding and not a shrug")


def plan_about(term, facts, near, credit, say_name):
    """A paragraph about one thing: what it is, what sits near it, and where that came from."""
    T = say_name(term)
    lead = None
    middles = []
    for rel, others in list(facts.items())[:3]:
        named = ", ".join(say_name(o) for o in others[:2])
        if lead is None:
            lead = f"{T} is {rel} {named}, which {credit(term)}"
        else:
            middles.append(f"It is also {rel} {named}")
    if lead is None:
        lead = f"I have {T} on my map but nothing stated about it"
    if near:
        middles.append(f"On the map it sits nearest {', '.join(say_name(n) for n in near[:3])}")
    return paragraph([lead] + middles)


def essay(a, b, about_a, about_b, route, shared, readings, confidence, say_name, say_fact=None):
    """The thing that was asked for: several paragraphs, in the order an argument goes.

    What each side is, then what if anything joins them, then what the evidence will and will not carry, then
    what is missing. It is not decorated and it does not build to a flourish, because every clause has to be
    something a store actually produced - but it is an argument with a shape rather than a single line, and
    the last paragraph is the one an essay usually leaves out.
    """
    A, B = say_name(a), say_name(b)
    paras = []

    def describe(term, facts):
        T = say_name(term)
        if not facts:
            return f"I found nothing usable about {T} - the pages I read said nothing I could turn into a fact."
        lines = []
        for rel, others in list(facts.items())[:3]:
            named = ", ".join(say_name(o) for o in others[:2])
            # said the way the corpus says this relation, rather than the way I would have written it
            lines.append(say_fact(rel, T if not lines else "It", named) if say_fact
                         else f"{'It is' if lines else T + ' is'} {rel} {named}")
        return paragraph(lines)

    paras.append(describe(a, about_a))
    paras.append(describe(b, about_b))

    if route:
        steps = " Ã¢â€ â€™ ".join([say_name(route[0]["from"])] + [say_name(e["to"]) for e in route])
        paras.append(paragraph([
            f"There is a route between them, and it runs {steps}",
            f"Every step there is a fact I was told or read, so the connection is {len(route)} link(s) long "
            f"rather than an impression",
            "What it supports is that they are connected, not that one caused the other - a path through a map "
            "is not a mechanism"]))
    elif shared:
        paras.append(paragraph([
            f"Nothing I found joins {A} to {B}",
            f"They do share company: both sit near {', '.join(say_name(s['term']) for s in shared[:3])}",
            "That is resemblance, not influence. Two things can share a neighbourhood because the world files "
            "them alike, and a causal claim needs more than filing"]))
    else:
        paras.append(paragraph([
            f"Nothing I found joins {A} to {B} at all - no route, and not even a shared neighbour"]))

    pages = ", ".join(f"{r['read']}" for r in readings[:6])
    paras.append(paragraph([
        f"That rests on {len(readings)} reading(s): {pages}" if readings else
        "That rests on what you have told me, since I did not read anything",
        f"I stopped at confidence {confidence:.2f}, which is low, and I stopped because the last pages were "
        f"adding nothing rather than because I was satisfied" if confidence < 0.7 else
        f"I stopped at confidence {confidence:.2f}, which was enough",
        "If there is a real connection it is in material I did not reach, and the honest position is that I "
        "have not found one rather than that there is none"]))
    return "\n\n".join(p for p in paras if p)


def deliver(topic, facts, near, spread, readings, confidence, say_name, say_fact, deliverable="essay",
            chose=False, material=None, mark=""):
    """Produce the thing that was asked for, in the shape it was asked for. Returns (text, working).

    Asked for an essay, this used to hand back an announcement that an essay was coming, two facts, a
    paragraph about how much to trust them, and a paragraph about what it could not do. Three of those four
    are the system talking about itself, and a reader who asked for an essay on a topic has been given an
    essay about the system. So the split here is the same one the conversational surface makes: the prose is
    the prose, and the accounting - how many facts, how firmly placed, how many pages, where it stopped -
    comes back separately as the working, for the reasoning channel.

    What is left is an actual short essay with a shape: what the thing is, what it sits among, what the
    reading added beyond the facts that parsed, and what that adds up to. The last paragraph is a finding,
    not a disclaimer: if the material does not support an argument, saying so *is* the conclusion, and it is
    said in one sentence rather than three.
    """
    shape = compose.shape_of(deliverable)
    T = say_name(topic)
    stated = compose.sentences_about(T, facts, say_fact, say_name, limit=shape["facts"])
    said_already = " ".join(stated).lower()
    # the definition it read is the strongest single statement it has about the thing, so it opens the piece;
    # what parsed into relations follows, because that is the part it can actually reason over
    definition, spare, extra = "", "", []
    # "bonobos" and "bonobo" are the same thing asked about twice; a definition written in the singular is
    # still the definition of what was asked about in the plural
    last = T.lower().split()[-1] if T else ""
    last = last[:-1] if len(last) > 4 and last.endswith("s") else last
    for sentence in (material or []):
        line = compose.tidy(sentence)
        core = line.lower().strip(".")
        if len(line) < 40 or len(line) > 240:
            continue
        # a definition says what the thing is, in its own right. A sentence opening with "While" or "Although"
        # is a qualification of something said earlier, and using it as the first line of a piece starts the
        # essay in the middle of an argument nobody has made yet.
        opens_subordinate = core.split()[:1] in (["while"], ["although"], ["though"], ["because"], ["if"],
                                                 ["when"], ["despite"], ["unlike"], ["since"])
        # and a definition has the thing as its *subject*: "The bonobo is the smallest species of great ape"
        # defines it, "Taxonomically, members of the subtribe Panina are termed panins" mentions it. So the
        # opening words are what count, and a sentence that merely contains the word is second choice.
        leads_with = last and last in " ".join(core.split()[:3])
        if not definition and leads_with and not opens_subordinate:
            definition = answer.gloss(line, topic)
            continue
        if not definition and last and last in core[:80] and not opens_subordinate and not spare:
            spare = answer.gloss(line, topic)
            continue
        if any(core[:40] in other.lower() for other in extra) or core[:40] in said_already:
            continue
        extra.append(line)
    definition = definition or spare
    held = sum(len(v) for v in facts.values())
    firmness = ("firmly placed: several facts agree on where it belongs" if spread < 0.8 else
                "loosely placed - one or two facts, so it is still floating" if spread < 1.5 else
                "barely placed: the facts I have about it disagree")
    working = [f"shape: asked for {deliverable or 'a piece'} - {shape['paragraphs']} paragraph(s), "
               f"up to {shape['facts']} fact(s)"]
    if chose:
        working.append(f"topic: nothing was named, so I chose {T} - it is what I hold most about")
    working.append(f"held: {held} stated relation(s) about {T}; spread {spread:.2f} - {firmness}")
    if readings:
        working.append(f"read: {', '.join(r['read'] for r in readings[:5])}; stopped at confidence "
                       f"{confidence:.2f}" + (", which was enough" if confidence >= 0.6 else
                                              ", which is low, so this is provisional"))
    else:
        working.append("read: nothing - all of this came from what you told me")
    if extra:
        working.append(f"material: {len(extra)} sentence(s) I read but could not parse into relations, used "
                       f"as reported reading rather than as facts")

    if shape["paragraphs"] == 0:                       # a list is a list
        lines = [f"On {T}:"] + [f"  - {line}" for line in (stated or extra[:shape['facts']])]
        if len(lines) == 1:
            lines.append("  - nothing I can state about it")
        return "\n".join(lines), working

    # a fact the definition already states is not a second sentence, it is the same sentence again. "Will
    # Smith is an American actor, rapper and film producer. Will Smith is an American actor." is what comes
    # of treating the two sources as independent when one of them is where the other came from.
    if definition:
        low = definition.lower()
        stated = [line for line in stated
                  if not all(w in low for w in line.lower().strip(".").split()[-3:])]
    paras = []
    body = compose.joined(stated, subject=T) if len(stated) > 1 else list(stated)
    if definition:
        paras.append(mark + paragraph([definition] + body[:1]))
        body = body[1:]
    elif body:
        paras.append(mark + paragraph(body[:2]))
        body = body[2:]
    elif extra:
        paras.append(mark + paragraph([extra[0]]))
        extra = extra[1:]
    else:
        return (mark + paragraph([f"I have {T} on my map but nothing stated about it"],
                                 gap="Tell me something about it, or let me go and read, and I will write "
                                     "you the piece"), working)
    if body and shape["paragraphs"] > 1:
        paras.append(paragraph(body[:3]))

    if extra and shape["paragraphs"] > 1:
        # this is material it read and could not turn into a relation, so it is reported as reading rather
        # than stated as its own: the distinction between what it knows and what a page said is the whole
        # reason it can be trusted about either.
        room = 2 if shape["paragraphs"] > 2 else 1
        paras.append(paragraph([f"From what I read about {T}: " + extra[0]] + extra[1:room]))

    if shape["paragraphs"] > 2 and near:
        neighbours = [say_name(n) for n in near[:3] if compose.usable(say_name(n))]
        if neighbours:
            paras.append(paragraph([
                f"{T} does not sit on its own: the things nearest it here are "
                f"{', '.join(neighbours)}",
                "That is company rather than consequence - it is where the next fact about it would most "
                "likely come from"]))

    if shape["gaps"]:
        if held >= 3 and spread < 1.2:
            close = (f"Taken together, that is a coherent picture of {T}: the facts I hold about it agree "
                     f"with each other, which is the most I can say without going further than the evidence")
        elif extra or readings:
            close = (f"What I have on {T} is a description rather than an argument - the material says what "
                     f"it is, and nothing I read said why it matters, so that is where I stop")
        else:
            close = (f"That is the whole of what I hold about {T}, and it is thin enough that the honest "
                     f"conclusion is that I need more before I can argue anything from it")
        paras.append(paragraph([close]))
    return "\n\n".join(p for p in paras if p), working


def piece(topic, facts, near, spread, readings, confidence, say_name, say_fact=None, chose=False):
    """A short piece about one thing: what it is, what it sits with, how firm any of it is, what is missing.

    The same discipline as the essay about two things. Every clause comes from a store, the uncertainty is
    stated rather than smoothed over, and where it had to go and read, it says so and says what that was worth.
    """
    T = say_name(topic)
    paras = []
    opening = [f"I chose {T} because it is the thing I hold most about" if chose else f"On {T}"]
    if facts:
        for rel, others in list(facts.items())[:3]:
            named = ", ".join(say_name(o) for o in others[:3])
            opening.append(say_fact(rel, T if len(opening) == 1 else "It", named) if say_fact
                           else f"{T} is {rel} {named}")
    else:
        opening.append(f"I have {T} on my map but nothing stated about it, which is a thin place to start")
    paras.append(paragraph(opening))

    if near:
        paras.append(paragraph([
            f"It sits near {', '.join(say_name(n) for n in near[:3])}",
            "Those are not claims about it - they are what the map puts in the same neighbourhood, which is "
            "worth saying only because it is where I would look next"]))

    firmness = ("firmly placed: several facts agree about where it belongs" if spread < 0.8 else
                "loosely placed: what I have leaves it floating rather than pinned" if spread < 1.5 else
                "barely placed at all: the facts I have about it disagree")
    lines = [f"How much weight this carries: {T} is {firmness}"]
    if readings:
        lines.append(f"I went and read {len(readings)} page(s) for it - "
                     + ", ".join(r["read"] for r in readings[:4]))
        lines.append(f"After that I am at confidence {confidence:.2f}"
                     + (", which is enough to say this much" if confidence >= 0.6 else
                        ", which is low, so treat the whole of it as provisional"))
    else:
        lines.append("All of this came from what you told me; I did not go and read anything")
    paras.append(paragraph(lines))
    paras.append(paragraph(["What I cannot give you is an argument about it",
                            "I can say what it is, what it is near and how sure I am, and that is the whole "
                            "of what I have - anything beyond that would be me making it up"]))
    return "\n\n".join(paras)


def plan_self(account, mistakes, other_said):
    """A paragraph about itself: what it is, what it holds, and what it has got wrong."""
    return paragraph([
        account["what"],
        f"Right now that is {account['holding']} - {account['provenance']}",
        f"Physically I am {account['size']}",
        f"You have corrected me {mistakes} time(s), and you have told me {other_said} thing(s) in total",
    ], caveat=account["cannot"])


FIGURES = {
    "population": r"population of (?:about |around |approximately |over |nearly )?([\d][\d,.]*(?:\s*(?:million|billion))?)",
    "area": r"area of (?:about |around |approximately )?([\d][\d,.]*\s*(?:km2|km²|square kilometres|square kilometers|"
            r"sq mi|square miles))",
}


def figure(sentences, what):
    """A number the reading states outright - a population, an area - or None. Never an estimate of mine."""
    import re
    pattern = re.compile(FIGURES[what], re.I)
    for sentence in sentences or []:
        m = pattern.search(str(sentence))
        if m:
            return m.group(1).strip()
    return None


def magnitude(text):
    """'2,111,536' or '33.8 million' as a number, so two figures can be put in order."""
    import re
    if not text:
        return None
    m = re.match(r"([\d][\d,.]*)\s*(million|billion)?", text.replace(",", ""))
    if not m:
        return None
    try:
        value = float(m.group(1))
    except ValueError:
        return None
    return value * {"million": 1e6, "billion": 1e9}.get((m.group(2) or "").lower(), 1)


def listed(items):
    items = [i for i in items if i]
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1] if items else ""


def compare(infos, mark=""):
    """A compare-and-contrast piece over several things, built only from what was read about each.

    The shape is the one the assignment has: what they have in common, each in turn, then where they differ -
    and a closing line that names what could not be compared, because a comparison that quietly drops the
    cases it has no figure for is a comparison of something else.
    """
    have = [i for i in infos if i.get("material")]
    if not have:
        return mark + "I couldn't find anything to compare them on - I have nothing read about any of them yet."
    names = listed([i["name"] for i in infos])
    kinds = [set(i["kinds"]) for i in have if i["kinds"]]
    shared = set.intersection(*kinds) if len(kinds) == len(have) and kinds else set()
    shared = sorted(shared - {"place"}, key=lambda k: (k not in ("city", "country", "person"), k))
    places = [i["place"] for i in have if i.get("place")]
    paras = []
    if shared and len(set(places)) == len(places) and len(places) == len(have):
        lead = (f"{names} are all {shared[0]} " if shared[0] != "city" else f"{names} are all cities") + \
               f", but in different countries: {listed(places)}"
        if "capital" in shared:
            lead = f"{names} are all capital cities, each the seat of a different country - {listed(places)}"
    elif shared:
        lead = f"What {names} have in common is that each is a {shared[0]}"
    else:
        lead = f"{names} are different kinds of thing, so the comparison is between what each is on its own terms"
    paras.append(mark + paragraph([lead]))

    for info in have:
        lines = [info["definition"]] if info.get("definition") else []
        extra = []
        if info.get("population"):
            extra.append(f"a population of {info['population']}")
        if info.get("area"):
            extra.append(f"an area of {info['area']}")
        if extra:
            lines.append(f"What I read gives it {listed(extra)}")
        paras.append(paragraph(lines))

    sized = [(magnitude(i["population"]), i["name"]) for i in have if magnitude(i.get("population"))]
    contrast = []
    if len(sized) >= 2:
        sized.sort(reverse=True)
        contrast.append(f"The sharpest difference is size: by the figures I read, {sized[0][1]} is the most "
                        f"populous and {sized[-1][1]} the least" + ("" if len(sized) == 2 else
                                                                    f", with {listed([n for _, n in sized[1:-1]])} "
                                                                    f"in between"))
    missing = [i["name"] for i in have if not i.get("population")]
    if missing and sized:
        contrast.append(f"I have no population figure for {listed(missing)}, so I have left it out of that "
                        f"ranking rather than guess")
    if not contrast:
        contrast.append("Where they differ beyond that, the pages I read do not give me figures I can put side "
                        "by side, so I will not pretend to rank them")
    paras.append(paragraph(contrast))
    return "\n\n".join(p for p in paras if p)

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/referents.py
"""What "it" means: the things a conversation is about, held so they can be pointed back at.

A person who asks about Kuala Lumpur, then Washington, then Delhi, and then says "compare the three cities",
has not said which three cities. They do not need to: all three are still active in the listener's head, and
"the three cities" is a pointer into that. Same for "write an essay about it" - "it" is whatever was last
being talked about - and for "KL", which in a conversation about Kuala Lumpur is not the Kullback-Leibler
divergence however often an encyclopedia uses it that way.

This is the prefrontal register that holds those things. Each entry is something the conversation has
discussed, with what kind of thing it is (read off how it was described - "is the capital city of" makes it
a city), the other names it goes by (its initials, the abbreviation a page gave for it), and a salience that
rises every time it comes up and decays as the conversation moves on. Resolution is ranking by salience among
the entries that fit: a pronoun takes the most salient thing, "the three cities" takes the three most salient
cities, an acronym takes the thing whose name it abbreviates.

The resolved sentence is what goes on to Wernicke's area, and every substitution is reported in the
reasoning trace, so it is always visible what the system decided a pointer meant.
"""
import re

PRONOUNS = {"it", "its", "that", "this", "there", "them", "they", "those", "these", "both"}
NUMBERS = {"two": 2, "both": 2, "three": 3, "four": 4, "five": 5, "all": 0, "these": 0, "those": 0, "the": 0}
# what kind of thing something is, read off the words used to describe it. A closed list on purpose: these are
# the head nouns people use when they point back ("the city", "the country", "the book", "the band").
KINDS = ["city", "country", "state", "capital", "book", "film", "person", "actor", "band", "company",
         "species", "animal", "language", "river", "mountain", "island", "university", "album", "song",
         "game", "planet", "dog", "cat", "territory", "megacity", "genre", "dance", "place", "region"]
PLURAL = {k + ("es" if k.endswith(("s", "sh", "ch")) else "s"): k for k in KINDS}
PLURAL.update({"cities": "city", "countries": "country", "species": "species", "people": "person",
               "companies": "company", "territories": "territory", "megacities": "megacity"})
SAME_KIND = {"capital": "city", "megacity": "city", "territory": "city", "actor": "person", "dog": "animal",
             "cat": "animal"}


def initials(name):
    words = [w for w in re.findall(r"[A-Za-z]+", name) if w.lower() not in ("of", "the", "and")]
    return "".join(w[0] for w in words).lower() if len(words) > 1 else ""


class Referents:
    """The things under discussion, ranked by how present they still are."""

    def __init__(self, decay=0.8):
        self.entries = {}          # key -> {"name", "kinds", "aliases", "salience", "turn"}
        self.decay = decay
        self.turn = 0
        self.group = None          # the things the last plural pointer ("the three cities") resolved to

    # ------------------------------------------------------------------ holding
    def mention(self, name, description="", turn=None, boost=1.0):
        """Something came up. Raise it, and learn what it is and what else it is called from how it was put."""
        name = re.sub(r"\s+", " ", str(name or "")).strip(" ,?!")
        if name.endswith(".") and not re.search(r"\b[A-Z]\.$", name):
            name = name[:-1]                         # a full stop, not the end of "D.C."
        if not name or name.lower() in PRONOUNS or len(name) < 2:
            return None
        key = name.lower()
        # "Capital city" is a page about a kind of thing, not a thing anyone is talking about. Filing it as
        # one made "the three cities" come out as Washington, Kuala Lumpur and "Capital city".
        words = key.split()
        generic = set(KINDS) | set(PLURAL) | {"federal", "national", "major", "global", "list", "lists"}
        if words and words[0] in generic and not any(w[:1].isupper() for w in name.split()[1:]):
            return None
        entry = self.entries.get(key)
        if entry is None:
            entry = {"name": name, "kinds": set(), "aliases": set(), "salience": 0.0, "turn": turn or self.turn}
            self.entries[key] = entry
            if initials(name):
                entry["aliases"].add(initials(name))
        if name[:1].isupper() and not entry["name"][:1].isupper():
            entry["name"] = name                     # keep the properly written form once it has been seen
        entry["salience"] += boost
        entry["turn"] = turn or self.turn
        text = str(description or "")
        low = text.lower()
        for kind in KINDS:
            if re.search(r"\b" + kind + r"\b", low[:220]):
                entry["kinds"].add(SAME_KIND.get(kind, kind))
                entry["kinds"].add(kind)
        # "Kuala Lumpur (KL), officially ..." - the page says what it abbreviates to, and that is an alias
        for abbreviation in re.findall(r"\(([A-Z][A-Za-z.]{1,6})\)", text[:200]):
            entry["aliases"].add(abbreviation.replace(".", "").lower())
        for m in re.finditer(r"(?:known as|also called|abbreviated(?: as)?|or simply)\s+([A-Z][\w.]+(?:\s[A-Z][\w.]+)?)",
                             text[:300]):
            entry["aliases"].add(m.group(1).replace(".", "").lower())
        return entry

    def focus(self, name, boost=3.0):
        """This turn was *about* this thing. It becomes what "it" means next, over everything read on the way.

        Salience that only accumulates lets the thing mentioned most over the whole conversation win every
        pronoun, which is how "write an essay about it", straight after an essay on Kuala Lumpur, came back as
        Washington. What was just discussed is what a pronoun points at, and a strong, decaying boost to the
        turn's own subject is how a listener's attention works.
        """
        key = str(name or "").lower().strip()
        entry = self.entries.get(key) or next((e for e in self.entries.values() if key in e["aliases"]), None)
        if entry is not None:
            entry["salience"] += boost
        return entry

    def find(self, name):
        """The entry this name refers to, by name or by any alias."""
        key = str(name or "").lower().strip(" ?.")
        return self.entries.get(key) or next((e for e in self.entries.values() if key in e["aliases"]
                                              or key == e["name"].lower().replace(",", "")), None)

    def merge(self, alias, name):
        """What the person called it and what the page called it are the same thing. Fold one into the other.

        Without this, asking "where is washington dc" leaves two entries - the phrase and the page title
        "Washington, D.C." - and "the three cities" later comes out as four.
        """
        a, b = str(alias).lower().strip(), str(name).lower().strip()
        # the parser drops articles and "just" from what it files; the register kept what was typed
        for variant in (a, "the " + a, "just " + a, "only " + a, "a " + a):
            if variant in self.entries and variant != b:
                a = variant
                break
        if a == b or a not in self.entries or b not in self.entries:
            return
        gone = self.entries.pop(a)
        keep = self.entries[b]
        keep["salience"] += gone["salience"]
        keep["aliases"] |= gone["aliases"] | {a.replace(".", "").replace(",", "")}
        keep["kinds"] |= gone["kinds"]

    def advance(self):
        """A new turn: everything fades a little, so the recent outranks the old without anything being lost."""
        self.turn += 1
        for entry in self.entries.values():
            entry["salience"] *= self.decay

    def ranked(self, kind=None):
        out = [e for e in self.entries.values() if kind is None or kind in e["kinds"]]
        return sorted(out, key=lambda e: -e["salience"])

    # ------------------------------------------------------------------ resolving
    def expand_alias(self, token):
        """An acronym, read in context: whichever thing under discussion it abbreviates."""
        t = token.lower().replace(".", "")
        if len(t) < 2 or len(t) > 6:
            return None
        hits = [e for e in self.entries.values() if t in e["aliases"]]
        return max(hits, key=lambda e: e["salience"])["name"] if hits else None

    def resolve(self, sentence):
        """The sentence with its pointers replaced by what they point at. Returns (sentence, notes)."""
        notes = []
        text = sentence
        self.group = None

        # "the three cities", "both countries", "these cities", "all three"
        def plural(m):
            quantity = [w for w in m.group(1).lower().split() if w in NUMBERS]
            want = max([NUMBERS[w] for w in quantity] or [0])
            kind = PLURAL.get(m.group(2).lower())
            if not kind:
                return m.group(0)
            pool = self.ranked(kind) or self.ranked(SAME_KIND.get(kind, kind))
            want = want or len(pool)
            if len(pool) < max(2, want):
                return m.group(0)
            chosen = [e["name"] for e in pool[:want]]
            self.group = chosen
            said = ", ".join(chosen[:-1]) + " and " + chosen[-1]
            notes.append(f"'{m.group(0)}' -> {said}")
            return said

        nouns = "|".join(sorted(PLURAL, key=len, reverse=True))
        text = re.sub(r"\b((?:all\s+)?(?:the\s+)?(?:two|three|four|five)?\s*|both\s+|these\s+|those\s+)(" + nouns
                      + r")\b", lambda m: plural(m) if m.group(1).strip() else m.group(0), text, flags=re.I)

        # "the city", "that country": the single most present thing of that kind
        def single(m):
            kind = m.group(2).lower()
            pool = self.ranked(kind)
            if not pool:
                return m.group(0)
            notes.append(f"'{m.group(0)}' -> {pool[0]['name']}")
            return pool[0]["name"]

        # only a bare pointer: "the capital city of India" is a description that picks out its own referent,
        # and binding "the capital" in it to whatever capital was last discussed answers the wrong question
        text = re.sub(r"\b(the|that|this)\s+(" + "|".join(k for k in KINDS if k not in ("place",)) + r")\b"
                      r"(?=\s*(?:[?.!,;]|$)|\s+(?:is|was|has|had|and|or|to|for|than|compared|seems|looks)\b)",
                      single, text, flags=re.I)

        # acronyms: a short token that abbreviates something under discussion
        def acronym(m):
            token = m.group(0)
            name = self.expand_alias(token)
            if name and name.lower() != token.lower():
                notes.append(f"'{token}' -> {name}")
                return name
            return token

        text = re.sub(r"\b[A-Za-z]{2,5}\b", acronym, text)

        # pronouns: the most present thing. Only in argument positions - "it is raining" is not about anything,
        # so a pronoun is resolved when it follows a verb or preposition that takes an object, or leads a
        # question about a property ("is it bigger", "where is it").
        # and never the thing the sentence already names: in "is it bigger than Kuala Lumpur", "it" is
        # whatever else was being talked about, because nobody asks whether something is bigger than itself
        low_text = text.lower()
        top = [e for e in self.ranked()
               if e["name"].lower() not in low_text and not any(a in low_text.split() for a in e["aliases"])]
        if top:
            name = top[0]["name"]
            pattern = (r"\b(about|on|of|with|for|is|was|compare|describe|explain|summarise|summarize|"
                       r"to|from|in|visit|like|call|write|than)\s+(it|that|this|them|there)\b(?![-'])")
            before = text
            text = re.sub(pattern, lambda m: f"{m.group(1)} {name}", text, flags=re.I)
            text = re.sub(r"^(is|was|does|did|how big is|where is|what is)\s+it\b",
                          lambda m: f"{m.group(1)} {name}", text, flags=re.I)
            if text != before:
                notes.append(f"'it' -> {name}")
        return text, notes

    def spec(self):
        return [{"name": e["name"], "kinds": sorted(e["kinds"]), "aliases": sorted(e["aliases"]),
                 "salience": round(e["salience"], 3)} for e in self.ranked()[:12]]

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/relate.py
"""Relating two things that were never mentioned together.

This is the part of thinking that is not recall and not deduction along a single scale: given two ideas, is
there any route between them, and what does that route consist of? In a map, that question has a shape. Items
are positions, stated facts are edges, and a relation between distant things is a *path* - either a chain of
stated facts, or a shared neighbour that both sit close to.

Three things come out of it, and they are different kinds of claim, so they are kept apart when it speaks:

  chain        a sequence of stated facts from one to the other. Every link was said by someone, and the route
               can be shown. This is the strongest thing it can offer.
  bridge       no chain, but something both sit near. That is an analogy, not a derivation - a shared
               neighbourhood in the space rather than a route through it - and it is presented as such.
  nothing      the two are in unconnected parts of the map. Saying so is a finding: an honest "these are not
               related in anything I know" is the answer an essay would have covered up.

Support for a link is how much was actually behind it: whether it came from testimony or from reading, and how
much the map moved when it went in. That is the nearest thing this system has to an opinion, and it is what
lets it say which part of its own account it trusts least.
"""


def graph(maps):
    """Every stated fact as an edge, across every space. The map's own memory of what it was told."""
    edges = {}
    for space, m in maps.maps.items():
        for i, (ea, moves, eb) in enumerate(m.episodes):
            a, b = m.names[ea], m.names[eb]
            weight = m.weights[i] if i < len(m.weights) else 1.0
            for x, y, sign in ((a, b, +1), (b, a, -1)):
                edges.setdefault(x, []).append({"to": y, "space": space, "moves": moves, "sign": sign,
                                                "weight": float(weight)})
    return edges


def chain(maps, a, b, max_len=5):
    """The shortest route of stated facts between two things, or []."""
    edges = graph(maps)
    if a not in edges or b not in edges:
        return []
    seen, queue = {a: None}, [(a, 0)]
    while queue:
        node, depth = queue.pop(0)
        if node == b:
            break
        if depth >= max_len:
            continue
        for e in edges.get(node, ()):
            if e["to"] not in seen:
                seen[e["to"]] = (node, e)
                queue.append((e["to"], depth + 1))
    if b not in seen:
        return []
    out, node = [], b
    while seen[node] is not None:
        prev, e = seen[node]
        out.append({"from": prev, "to": node, **e})
        node = prev
    return list(reversed(out))


def bridges(agent, a, b, top=3):
    """Things both are near, when nothing joins them: the analogical route rather than the derived one."""
    found = []
    for space, m in agent.maps.maps.items():
        if not (m.knows(a) and m.knows(b)):
            continue
        pa, pb = m.pos[m.index[a]], m.pos[m.index[b]]
        for n in m.names:
            if n in (a, b) or not m.knows(n):
                continue
            p = m.pos[m.index[n]]
            da, db = float((p - pa).norm()), float((p - pb).norm())
            found.append((max(da, db), n, space, da, db))
    # also the semantic store, if this machine has one: what the background map puts near both
    if agent.sem.loaded and agent.sem.knows(a) and agent.sem.knows(b):
        near_a = {n for n, _ in agent.sem.neighbours(a, 25)}
        near_b = {n for n, _ in agent.sem.neighbours(b, 25)}
        for n in sorted(near_a & near_b):
            found.append((0.5, n, "background", 0.0, 0.0))
    found.sort()
    return [{"term": n, "space": s, "to_a": round(da, 2), "to_b": round(db, 2)} for _, n, s, da, db in found[:top]]


def account(agent, a, b, say_name, confidence=None):
    """Put the relation into words: what the route is, what kind of claim it is, and what is weak about it."""
    route = chain(agent.maps, a, b)
    A, B = say_name(a), say_name(b)
    if route:
        steps = []
        for e in route:
            rel = ", ".join(f"{r}" for r, _ in e["moves"]) or "linked to"
            steps.append(f"{say_name(e['from'])} → {rel} → {say_name(e['to'])}")
        weakest = min(route, key=lambda e: e["weight"])
        lines = [f"There is a route from {A} to {B}, through {len(route)} stated fact(s):",
                 "  " + "\n  ".join(steps),
                 f"Every link there is something I was told or read - none of it is my invention. The link I'd "
                 f"trust least is {say_name(weakest['from'])} → {say_name(weakest['to'])}, which rests on the "
                 f"least support of the {len(route)}."]
        return "\n".join(lines)
    near = bridges(agent, a, b)
    if near:
        shared = ", ".join(f"{say_name(n['term'])}" for n in near)
        return (f"Nothing I know links {A} to {B} directly. What they have in common is their company: both sit "
                f"near {shared}. That is a resemblance rather than a connection, and I'd present it as one - it "
                f"would not support a claim that one affected the other.")
    known = [x for x in (a, b) if any(m.knows(x) for m in agent.maps.maps.values()) or agent.sem.knows(x)]
    if len(known) < 2:
        missing = " and ".join(say_name(x) for x in (a, b) if x not in known)
        return f"I can't relate them: I have nothing at all about {missing}."
    return (f"I know about {A} and about {B}, and there is no route between them in anything I hold - not a "
            f"chain of facts, not even a shared neighbour. My answer is that they are unrelated as far as I "
            f"know, which is a finding rather than a failure"
            + (f" (confidence {confidence:.2f})." if confidence is not None else "."))

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/say.py
"""Production: choosing words, and not choosing the same ones every time.

Two jobs here. The first is variety. A system with one template per situation sounds like a form letter, and the
fix is not a random choice - it is the same suppression the dendrites use. A phrasing that has just been used is
harder to reach for a while, so repetition falls away on its own and comes back once it has recovered. Neurons
adapt to a repeated input exactly like this; conversation sounds alive for the same reason.

The second is explanation. "Why?" is answerable only if the chain that produced the answer can be recovered, so
`chain` walks the facts that were actually stated - through the episodes the map remembers, not by re-deriving
from position - and says them in order. An explanation that cannot be traced to stated facts is not given.
"""
import random

PHRASES = {
    "greet-empty": ["Hello. I don't know anything yet - tell me how things relate and I'll work out the rest.",
                    "Hi. I start from nothing: no facts, no pretrained anything. Teach me something.",
                    "Hello. Nothing in me yet. Tell me about a few things and ask me what follows."],
    "greet-known": ["Hello again. I'm holding {facts} fact(s) from you.",
                    "Hi. Still here, with {facts} fact(s) you've told me.",
                    "Hello. {facts} fact(s) so far - ask me something that follows from them."],
    "thanks": ["Any time.", "Glad it helped.", "You're welcome."],
    "sorry": ["No harm done.", "Nothing to apologise for.", "It's fine."],
    "bye": ["Goodbye. I'll keep what you told me.", "See you. Everything you taught me stays.",
            "Bye - I'll remember all of this."],
    "agree": ["Good.", "Noted.", "Glad that one landed."],
    "taught": ["Got it.", "Noted.", "Understood."],
    "unsure": ["I'm not certain: {why}.", "Not confident about that one - {why}.", "Only tentatively: {why}."],
    "cannot": ["I can't tell from what I have.", "Nothing I hold decides that.",
               "That one I can't answer from what you've told me."],
    "about-you": ["I'm a small brain that holds relations between things and works out the rest. "
                  "No training, no pretrained knowledge - only what you tell me.",
                  "There isn't much to me: sparse codes, a working memory, a map of what you've said, and a gate "
                  "that decides which of those answers. Everything I know came from this conversation.",
                  "I hold what you tell me as positions on a map and read answers off it. That's the whole of me."],
}


class Voice:
    """Picks a phrasing, then suppresses it for a while - adaptation, not randomness."""

    def __init__(self, seed=0, recovery=0.35):
        self.rng = random.Random(seed)
        self.fatigue = {}
        self.recovery = recovery

    def pick(self, kind, count):
        """Which of `count` phrasings to use for this situation, given what has just been used.

        Factored out of __call__ because the conversational openings live in another module and want the same
        adaptation: a marker that has just been said is harder to reach, and recovers over the next few turns.
        """
        if count <= 0:
            return 0
        weights = [1.0 / (1.0 + self.fatigue.get((kind, i), 0.0)) for i in range(count)]
        chosen = self.rng.choices(range(count), weights=weights)[0]
        for i in range(count):
            self.fatigue[(kind, i)] = self.fatigue.get((kind, i), 0.0) * (1 - self.recovery)
        self.fatigue[(kind, chosen)] = self.fatigue.get((kind, chosen), 0.0) + 2.0
        return chosen

    def __call__(self, kind, **slots):
        options = PHRASES.get(kind)
        if not options:
            return ""
        return options[self.pick(kind, len(options))].format(**slots)


def chain(maps, space, a, b):
    """The stated facts that link a and b, in order: what an explanation is made of.

    Breadth-first over the episodes the map actually stored, so every step is something that was said rather
    than something inferred from where things ended up.
    """
    m = maps.maps.get(space)
    if not m or not (m.knows(a) and m.knows(b)):
        return []
    ia, ib = m.index[a], m.index[b]
    edges = {}
    for ea, moves, eb in m.episodes:
        edges.setdefault(ea, []).append((eb, moves, +1))
        edges.setdefault(eb, []).append((ea, moves, -1))
    seen, queue = {ia: None}, [ia]
    while queue:
        node = queue.pop(0)
        if node == ib:
            break
        for nxt, moves, sign in edges.get(node, ()):
            if nxt not in seen:
                seen[nxt] = (node, moves, sign)
                queue.append(nxt)
    if ib not in seen:
        return []
    steps, node = [], ib
    while seen[node] is not None:
        prev, moves, sign = seen[node]
        steps.append((m.names[prev], moves, sign, m.names[node]))
        node = prev
    return list(reversed(steps))


def explain(maps, space, a, b, word, say_name):
    """'Why?' - the stated facts, in the order they chain, or an honest account of why there is no chain."""
    steps = chain(maps, space, a, b)
    if not steps:
        return (f"I can't show you a chain: nothing you've told me links {say_name(a)} and {say_name(b)}, "
                f"so that answer was not something I could derive.")
    told = []
    for src, moves, sign, dst in steps:
        for rel, count in moves:
            # `sign` is which way the stored fact was travelled to get here, so it decides who is named first
            hi, lo = (dst, src) if sign > 0 else (src, dst)
            told.append(f"{say_name(hi)} is {word or rel} than {say_name(lo)}" if space != "place"
                        else f"{say_name(dst)} is {abs(count)} {rel} of {say_name(src)}")
    if len(told) == 1:
        return f"Because you told me that {told[0]}."
    chainedanswer = f"{say_name(a)} and {say_name(b)}"
    return ("Because you told me that " + ", and that ".join(told)
            + f" - which is the whole chain between {chainedanswer}, and nothing in it was inferred.")


def confidence(steps, margin):
    """How sure, and why - the margin the answer rested on, said in words rather than as a number alone."""
    if margin is None:
        return "I'm going on what was stated directly."
    size = abs(margin)
    if size >= 2.5:
        return f"Fairly sure: they sit {size:.0f} steps apart on my scale, which is a wide gap."
    if size >= 1.2:
        return f"Reasonably sure: about {size:.1f} steps apart, over {steps} stated fact(s)."
    return (f"Not very: only {size:.1f} steps apart, which is close enough that one wrong fact would flip it.")

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/spelling.py
"""Hearing what was meant when it was not quite what was typed.

"what country is beijig in" and "cany ou write an essay" are perfectly clear to a person and were opaque to
this system, because recognition was exact: a word was either one it had heard or it was nothing. Real
recognition is not like that. The lexicon's codes are built from letter trigrams, so "beijig" and "beijing"
already drive overlapping units - the tissue has been saying they are nearly the same word all along, and
nothing downstream was listening.

Two repairs, both decided by statistics learned from reading rather than by rules:

  a near miss      a word that is rare or unheard, with a common word one or two letters away, is read as the
                   common word. Rarity is counted from the corpus: "beijig" occurs nowhere, "beijing" occurs
                   hundreds of times, and that asymmetry is the whole of the evidence. A word under discussion in
                   this conversation beats an equally close word from the corpus, which is how context resolves
                   a typo the way a listener does.
  a slipped space  two adjacent tokens that are rare as written but split into two common words at a
                   different point ("cany ou" -> "can you") are re-split there. Again the only evidence is how
                   much more common the repaired pair is than what was typed.

Every repair is reported, so the reasoning trace says "heard 'beijig' as 'beijing'" rather than quietly
answering a question nobody asked. A word that is rare but has nothing close to it is left alone: a new name
is not a typo, and treating it as one would be worse than not understanding it.
"""
import math
import os
import pickle
import re
from collections import Counter, defaultdict

WORD = re.compile(r"[a-z][a-z'\-]*")
CACHE = "E:/flybrain/unigrams.pkl" if os.path.isdir("E:/flybrain") else os.path.join(
    os.path.dirname(os.path.abspath(__file__)), "unigrams.pkl")
# contractions typed without the apostrophe are words, not slips - "whats" is "what's", not "what"
CONTRACTED = {"whats": "what's", "thats": "that's", "dont": "don't", "cant": "can't", "wont": "won't",
              "im": "I'm", "ive": "I've", "isnt": "isn't", "doesnt": "doesn't", "didnt": "didn't",
              "youre": "you're", "theyre": "they're", "lets": "let's", "hows": "how's", "wheres": "where's",
              "whos": "who's", "arent": "aren't", "wasnt": "wasn't", "couldnt": "couldn't", "wouldnt": "wouldn't",
              "shouldnt": "shouldn't", "ill": "I'll", "youve": "you've", "weve": "we've", "id": "I'd"}
CORPORA = ["E:/corpora/crawled.tsv", "E:/corpora/concept_prose.tsv", "E:/corpora/dialogue.tsv"]


def trigrams(word):
    w = f"<{word}>"
    return {w[i:i + 3] for i in range(len(w) - 2)}


def distance(a, b, cap=3):
    """Edit distance, with a transposition counting once - "teh" is one slip from "the", not two."""
    if abs(len(a) - len(b)) > cap:
        return cap + 1
    prev2, prev = None, list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cost = 0 if ca == cb else 1
            value = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
            if prev2 is not None and i > 1 and j > 1 and ca == b[j - 2] and a[i - 2] == cb:
                value = min(value, prev2[j - 2] + 1)
            cur.append(value)
        prev2, prev = prev, cur
    return prev[-1]


def count_corpus(paths=CORPORA, limit=None):
    """How often each word occurs in everything it has read. The only thing spelling repair trusts."""
    counts = Counter()
    for path in paths:
        if not os.path.exists(path):
            continue
        with open(path, encoding="utf-8", errors="ignore") as f:
            for n, line in enumerate(f):
                counts.update(WORD.findall(line.lower()))
                if limit and n >= limit:
                    break
    return counts


class Speller:
    def __init__(self, counts=None, vocabulary=None, speech=None):
        self.speech = speech          # the sequence memory: what the preceding words lead you to expect
        if counts is None:
            counts = self._load()
        if not counts and vocabulary:
            counts = Counter({w: 1 for w in vocabulary})
        self.counts = counts
        self.index = None

    @staticmethod
    def _load():
        if os.path.exists(CACHE):
            with open(CACHE, "rb") as f:
                return pickle.load(f)
        counts = count_corpus()
        if counts:
            counts = Counter({w: c for w, c in counts.items() if c >= 2 or len(w) <= 12})
            try:
                os.makedirs(os.path.dirname(CACHE), exist_ok=True)
                with open(CACHE, "wb") as f:
                    pickle.dump(counts, f)
            except OSError:
                pass
        return counts

    def _build(self):
        """Which words share which letter trigrams - built once, on the first word that needs it."""
        self.index = defaultdict(list)
        for word, count in self.counts.items():
            if count >= 3 and 3 <= len(word) <= 20:
                for g in trigrams(word):
                    self.index[g].append(word)

    def frequency(self, word):
        return self.counts.get(word, 0)

    # ---------------------------------------------------------------- near miss
    def suggest(self, word, context=(), before=(), speech=None, after=None):
        """The word most likely meant, or None if what was typed is already the likeliest reading.

        Ranked the way a listener ranks: fewest slips first - one wrong letter is far likelier than two, so
        "beijig" is "beijing" and not "being" however common "being" is - and among equally close words,
        the one this conversation is about, then the one the preceding words lead you to expect, then the
        commoner one. "write an esay" is "essay", not "say", because nothing that says "write an" goes on
        to say "say".
        """
        low = word.lower()
        if low in CONTRACTED:
            return None
        if len(low) < 3 or not low.isalpha() or (len(low) == 3 and self.frequency(low) > 0):
            return None
        typed = self.frequency(low)
        # a word read fifty times is a word. "what" is not a slip for "that" however much commoner "that" is,
        # and a repair that fires on real words is worse than one that never fires at all
        if typed >= 3:
            return None       # read three times is a word: "taller" is not a slip for "smaller"
        context = {c.lower() for c in context}
        if low in context:
            return None
        if self.index is None:
            self._build()
        grams = trigrams(low)
        votes = Counter()
        for g in grams:
            for candidate in self.index.get(g, ()):
                votes[candidate] += 1
        # words from this conversation are candidates whatever their corpus frequency
        for name in context:
            for part in name.split():
                if len(part) >= 4 and grams & trigrams(part):
                    votes[part] += len(grams & trigrams(part)) + 2
        history = [w.lower() for w in before][-3:]

        def expected(candidate):
            if speech is None or not history:
                return 0.0
            try:
                return speech.follows(history, candidate)
            except Exception:
                return 0.0

        best, best_key = None, None
        # two slips only for something never seen at all; a word met once or twice gets one
        allowed = 1 if (len(low) <= 5 or typed > 0) else 2
        pool = list(votes.most_common(300))
        # one-edit neighbours generated directly. A transposition wrecks trigram overlap - "abuot" shares one
        # trigram with "about" - so the index alone never proposes the likeliest correction of all.
        letters = "abcdefghijklmnopqrstuvwxyz"
        near = {low[:i] + low[i + 1:] for i in range(len(low))}
        near |= {low[:i] + low[i + 1] + low[i] + low[i + 2:] for i in range(len(low) - 1)}
        near |= {low[:i] + c + low[i + 1:] for i in range(len(low)) for c in letters}
        near |= {low[:i] + c + low[i:] for i in range(len(low) + 1) for c in letters}
        pool += [(w, 2) for w in near if self.counts.get(w, 0) >= 3]
        if len(low) == 3:                      # three letters share too few trigrams to be found by them
            pool += [(w, 2) for w in ("the", "and", "you", "are", "for", "not", "but", "how", "who", "why",
                                      "can", "was", "has", "had", "his", "her", "its", "our", "out", "now")]
        for candidate, shared in pool:
            if candidate == low or shared < 2:
                continue
            d = distance(low, candidate)
            if d > allowed:
                continue
            in_context = candidate in context or any(candidate in c.split() for c in context)
            # two slips away is only close enough for something this conversation is about. From the corpus
            # at large it is a guess - "beijig" two slips from "being" - and a wrong guess is worse than
            # leaving the word for the search engine, which has seen every misspelling of Beijing there is.
            if d > 1 and not in_context:
                continue
            freq = self.frequency(candidate)
            # evidence that this is a slip: the candidate is far commoner than what was typed, or it is the
            # thing we are talking about. Closeness alone is not enough - "cane" is one letter from "can".
            if not in_context and freq < 20 * max(1, typed):
                continue
            # log of how strongly the preceding words predict it: never-seen-here counts heavily against a
            # candidate, which is the difference between "an essay" (seen) and "an say" (never)
            fit = math.log(1e-6 + expected(candidate)) if history and speech is not None else 0.0
            if after and speech is not None:
                # and what comes next: "which city" is English and "with city" is not
                fit += math.log(1e-6 + speech.follows([candidate], after))
            within = (3.0 if in_context else 0.0) + 2.0 * fit + math.log1p(freq) \
                + (1.0 if candidate[0] == low[0] else 0.0)      # the first letter is rarely the slip
            key = (-d, within)
            if best_key is None or key > best_key:
                best, best_key = candidate, key
        return best

    # ------------------------------------------------------------ slipped space
    def resplit(self, first, second):
        """Two tokens that read better with the space somewhere else. None if they are fine as typed."""
        a, b = first.lower(), second.lower()
        if not (a.isalpha() and b.isalpha()):
            return None
        joined = a + b
        typed = min(self.frequency(a), self.frequency(b))
        best, best_score = None, 0
        for cut in range(1, len(joined)):
            x, y = joined[:cut], joined[cut:]
            if (x, y) == (a, b):
                continue
            score = min(self.frequency(x), self.frequency(y))
            if score > best_score:
                best, best_score = (x, y), score
        whole = self.frequency(joined)
        if whole > best_score and whole > 50 * max(1, typed):
            return (joined,)                 # "can not" -> no; "some thing" -> "something"
        if best and best_score > 50 * max(1, typed):
            return best
        return None

    # --------------------------------------------------------------------- both
    def repair(self, sentence, context=()):
        """The sentence as it was probably meant, and what was changed to get there."""
        tokens = re.findall(r"[A-Za-z][A-Za-z'\-]*|\d+|[^\sA-Za-z\d]+", sentence)
        notes, out, i = [], [], 0
        while i < len(tokens):
            tok = tokens[i]
            nxt = tokens[i + 1] if i + 1 < len(tokens) else None
            if nxt and tok.isalpha() and nxt.isalpha():
                fixed = self.resplit(tok, nxt)
                if fixed:
                    notes.append(f"'{tok} {nxt}' -> '{' '.join(fixed)}'")
                    out.extend(fixed)
                    i += 2
                    continue
            if tok.lower() in CONTRACTED:
                out.append(CONTRACTED[tok.lower()])
                i += 1
                continue
            # a capitalised word mid-sentence is a name, and a name nobody has heard is new rather than wrong:
            # "Cara" is not a slip for "Care". A name is only ever corrected toward a name under discussion.
            if tok.isalpha() and tok[:1].isupper() and out and not re.match(r"[.!?]", out[-1]):
                target = self.suggest(tok, context)
                if target and any(target == part.lower() for c in context for part in str(c).lower().split()):
                    notes.append(f"'{tok}' -> '{target.capitalize()}'")
                    out.append(target.capitalize())
                else:
                    out.append(tok)
                i += 1
                continue
            if tok.isalpha():
                after = next((t for t in tokens[i + 1:] if t.isalpha()), None)
                meant = self.suggest(tok, context, before=[t for t in out if t.isalpha()], speech=self.speech,
                                     after=after)
                if meant:
                    shown = meant.capitalize() if tok[:1].isupper() else meant
                    notes.append(f"'{tok}' -> '{shown}'")
                    out.append(shown)
                    i += 1
                    continue
            out.append(tok)
            i += 1
        text = ""
        for tok in out:
            text += tok if (not text or re.match(r"[^\sA-Za-z\d('\"]", tok)) else " " + tok
        return text, notes

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/selfmodel.py
"""Knowing that it is the one doing this, and that you are someone else.

A system that holds facts but not where they came from cannot tell "you told me" from "I worked it out" from
"I read it somewhere", and without that distinction there is no self to speak of - only content. Source
monitoring is the mechanism people actually use for this, and getting it wrong is what confabulation *is*: the
content is retrieved correctly and its origin is not.

So every belief here carries a source, and there are exactly three:

  told     you said it. The strongest thing it has, and not its own.
  read     it went and found it. Attributed to a page, weighted below testimony.
  worked   it inferred it from the map. Its own, and the only kind it can be blamed for.

On top of that sit two models rather than one:

  self     what it is, what it holds, what it has got wrong, what it cannot do. Built from its own structure
           and history - it can count its columns and its corrections, so it never has to be told about itself.
  other    what *you* have said, corrected and asked about. Separate, because "you said Max is older" and "Max
           is older" are different claims, and only one of them is the map's.

The test that matters is whether it can disagree with you and say which of you thinks what. That requires both
models, and it is the difference between a memory and a point of view.
"""
import time

TOLD, READ, WORKED = "told", "read", "worked"


class Other:
    """The person it is talking to: what they have said, and what they have corrected."""

    def __init__(self, name="you"):
        self.name = name
        self.said = []            # every claim they made, in order
        self.corrections = []     # times they said it was wrong
        self.asked = []           # what they wanted to know
        self.entities = {}        # things they introduced, and how they referred to them

    def claim(self, sentence, fact=None):
        self.said.append({"sentence": sentence, "fact": fact, "turn": len(self.said)})

    def correction(self, sentence, was, now):
        self.corrections.append({"sentence": sentence, "was": was, "now": now})

    def question(self, intent, terms):
        self.asked.append({"intent": intent, "terms": terms})

    def introduced(self, key, shown, possessive=False):
        self.entities[key] = {"shown": shown, "theirs": possessive}

    def theirs(self, key):
        return self.entities.get(key, {}).get("theirs", False)


class Self:
    """What it is, and what it has done - assembled from its own parts rather than described to it."""

    def __init__(self, agent):
        self.agent = agent
        self.born = time.time()
        self.sources = {}          # fact index -> where it came from
        self.mistakes = []         # things it said that were then corrected
        self.own_inferences = 0    # answers it worked out rather than recalled

    # ------------------------------------------------------------ attribution
    def attribute(self, key, source, detail=None):
        self.sources.setdefault(key, []).append({"source": source, "detail": detail})

    def origin(self, key):
        """Where what it holds about this came from, strongest claim first."""
        entries = self.sources.get(key, [])
        for kind in (TOLD, READ, WORKED):
            if any(e["source"] == kind for e in entries):
                return kind
        return None

    def credit(self, key):
        """How to say where something came from, in a clause."""
        return {TOLD: "you told me", READ: "I read it", WORKED: "I worked it out"}.get(self.origin(key),
                                                                                      "I am not sure where")

    def blame(self, sentence, was, now):
        self.mistakes.append({"said": was, "corrected_to": now, "sentence": sentence})

    # ------------------------------------------------------------- what it is
    def anatomy(self):
        """Its own structure, counted rather than declared."""
        a = self.agent
        return {"columns": [a.lex.column.spec(), a.epi.column.spec(), a.articulator.column.spec()],
                "gate_experts": len(a.gate.pool), "maps": list(a.maps.maps),
                "facts": sum(len(m.episodes) for m in a.maps.maps.values()),
                "episodes": len(a.epi.records), "words_it_can_say": len(a.articulator.words) - 1,
                "judgements": a.meta.stats()["outcomes_seen"], "synapses": a.footprint(),
                "alive_for": round(time.time() - self.born)}

    def account(self):
        """A first-person account of what it is. Every number is read off itself at the moment of asking."""
        a = self.anatomy()
        told = sum(1 for k in self.sources if self.origin(k) == TOLD)
        read = sum(1 for k in self.sources if self.origin(k) == READ)
        return {
            "what": "I am a map, a few stores and a gate. Nothing in me was trained and nothing was pretrained: "
                    "I started empty and everything I hold arrived in a conversation.",
            "size": f"{a['synapses']:,} synapses across {len(a['columns'])} columns, "
                    f"{a['gate_experts']} experts in the gate, grown from use",
            "holding": f"{a['facts']} fact(s) on {len(a['maps'])} map(s) and {a['episodes']} episode(s)",
            "provenance": f"{told} of them from you, {read} from things I read, the rest worked out",
            "fallible": f"{len(self.mistakes)} time(s) you have had to correct me",
            "cannot": "I cannot tell you anything I was not told or did not read, and I do not know what I do "
                      "not know until I go and look.",
        }

    def disagreement(self, key, mine, yours):
        """Saying which of us thinks what - the thing that needs both models to be possible at all."""
        return (f"We differ on {key}: you said {yours}, and what I have worked out from the rest is {mine}. "
                f"I am going with yours, because you were there and I only have the map.")

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/semantic.py
"""Semantic cortex: the slow store, and background knowledge if any has been built.

Complementary learning systems, which is the oldest well-supported division in memory research: the hippocampus
takes a fact in one exposure and keeps it apart from everything else; cortex learns slowly, overlapping, and ends
up with structure rather than instances. This is the cortical half.

Two things live here:

  what has been consolidated   facts the conversation established, replayed out of the episodic store during
                               sleep and written into a map that is allowed to blur them into structure.
  what was built beforehand    the ConceptNet/prose map, if it exists on this machine. It is background, not the
                               centre: the gate may consult it, and nothing forces an answer through it.

The prebuilt map is optional on purpose. Everything the system does about *this conversation* works on a machine
that has never seen a corpus.
"""
import os

import torch

PREBUILT = os.environ.get("CONCEPTS", "E:/corpora/concepts.pt")


class Semantic:
    def __init__(self, path=PREBUILT, top_terms=120000):
        self.pos = self.names = None
        self.rel = {}
        self.index = {}
        self.vocab = set()
        if path and os.path.exists(path):
            d = torch.load(path, weights_only=False)
            self.pos, self.names, self.rel = d["pos"], d["names"], d["rel"]
            self.index = {n: i for i, n in enumerate(self.names)}
            self.vocab = {n for n in self.names[:top_terms] if "_" not in n}

    @property
    def loaded(self):
        return self.pos is not None

    def knows(self, term):
        return term.lower() in self.index

    def neighbours(self, term, k=6):
        """Place-cell readout over concepts: what sits nearest, which is what this store is good at."""
        t = term.lower()
        if not self.loaded or t not in self.index:
            return []
        dist = (self.pos - self.pos[self.index[t]]).norm(dim=1)
        out = []
        for i in dist.argsort()[1:k * 8]:
            n = self.names[int(i)]
            if "_" in n or n.startswith(t[:max(3, len(t) - 1)]):
                continue      # morphology of the same word is a fact about English, not about the thing
            out.append((n, float(dist[i])))
            if len(out) >= k:
                break
        return out

    def along(self, term, relation, k=4):
        """One step of a relation from a term - inference by displacement, the map's own move."""
        t = term.lower()
        if not self.loaded or t not in self.index or relation not in self.rel:
            return []
        target = self.pos[self.index[t]] + self.rel[relation]
        dist = (self.pos - target).norm(dim=1)
        out = []
        for i in dist.argsort()[:k * 8]:
            n = self.names[int(i)]
            if "_" in n or n == t:
                continue
            out.append((n, float(dist[i])))
            if len(out) >= k:
                break
        return out

    def describe(self, term, relations=("IsA", "PartOf", "UsedFor", "AtLocation", "CapableOf", "HasProperty")):
        t = term.lower()
        if not self.knows(t):
            return None
        near = self.neighbours(t)
        out = {"term": t, "near": near, "relations": {}}
        here = {n for n, _ in near}
        for r in relations:
            hits = [h for h in self.along(t, r) if h[0] not in here]
            if hits:
                out["relations"][r] = hits[:3]
        return out

    def size(self):
        return 0 if not self.loaded else len(self.names)

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/speech.py
"""Putting an answer into words.

Deliberately thin. Every sentence here is composed from a structure some store produced - there is no template
bank standing in for knowing something, and nothing is said that no store supports. When the systems have
nothing, the honest sentence is the whole reply.
"""


def listify(items):
    items = [str(i) for i in items]
    if len(items) <= 1:
        return "".join(items)
    return ", ".join(items[:-1]) + " and " + items[-1]


def greeting(known, facts):
    if not facts:
        return ("Hello. I don't know anything yet - I have no pretrained knowledge at all. Tell me how things "
                "relate and I'll work out the rest.")
    return f"Hello again. I'm holding {facts} fact(s) from what you've told me."


def acknowledge(topic, recalled):
    """Someone said something about themselves or the world that is not a relation. Say what is actually held."""
    if recalled:
        return f"I hear you. You mentioned {topic} before - {recalled}."
    if topic:
        return f"I hear you. I don't have anything about {topic} beyond what you've just said."
    return "I hear you. I can't do much with that yet - I hold relations between things."


def clarify(reason, subject=None):
    if reason == "pronoun":
        return "Who or what do you mean? I've lost track of who that refers to."
    if reason == "unparsed":
        return ("I didn't follow that. I understand facts like \"Alice is taller than Bob\" or \"the cafe is "
                "north of the park\", and questions about them.")
    if reason == "unknown-term" and subject:
        return f"I don't know anything about {subject} yet. Tell me how it relates to something else."
    return "I'm not sure what you're asking."


def recalled(record, confidence, turn_gap):
    when = "just now" if turn_gap <= 2 else f"{turn_gap} turns ago"
    hedge = "" if confidence > 0.55 else " (though the memory is faint)"
    return f"You said {when}: \"{record['text']}\"{hedge}."


def about_semantic(term, detail):
    """What the background association store has, said as association rather than as fact.

    This store is a crowd-sourced graph and it is often wrong in ways that a list of relation names printed
    with colons makes look authoritative: "bonobo - IsA: oviparous animal; UsedFor: griffin" is nonsense
    wearing a database's clothes. It is still worth having, because it is where a word's neighbourhood comes
    from, so it is said as what it is - loose association that has not been checked against anything.
    """
    near = [n for n, _ in detail["near"][:3]]
    kinds = []
    for rel, hits in list(detail["relations"].items())[:3]:
        if rel.lower() in ("isa", "is a", "instanceof", "kindof"):
            kinds += [h[0] for h in hits[:2]]
    if kinds:
        line = f"I have {term} only by association, not from anything I have read: it is filed near "                f"{listify(kinds)}"
    elif near:
        line = f"I know {term} only by its company - it sits near {listify(near)}"
    else:
        return ""
    if near and kinds:
        line += f", alongside {listify(near)}"
    return line + ". None of that is a fact I can stand behind; ask me to go and read about it and I will."


def sleep_report(facts, moved, experts):
    return (f"Slept. I replayed {facts} fact(s) together, the biggest change to any position was {moved:.2f} "
            f"steps, and my gate is holding {experts} expert(s).")

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/timeline.py
"""When each region was busy, which is not a queue.

The first version of this recorded one tick per region and drew them in order, which is a diagram of a pipeline,
not of a brain. Nothing about cortex is one pass: a region starts as soon as its input arrives, keeps going
while others are going, fires again when something comes back to it, and the loops between regions are the
whole point. Hippocampus and prefrontal cortex in particular are a loop - prefrontal asks, the hippocampus
retrieves, what comes back changes what prefrontal asks next - and that traffic runs in both directions while
the sensory end is still busy with the same sentence.

So each region here has its own clock. A firing starts at the later of "my input arrived" and "I finished the
last thing", runs for as long as that operation takes, and emits to its targets after the conduction delay on
that pathway. Two regions with no dependency between them therefore overlap, and a region asked twice appears
twice. The record is a set of intervals, and the picture drawn from it has overlaps and return passes in it
because the computation did.
"""


# The regions, the pathway each sits on for conduction purposes, and the system it belongs to. Memory is not
# one structure: an item's familiarity, the context it arrived in, the binding that keeps them apart and the
# slow store that generalises them are different tissue doing different jobs, and on any real retrieval most of
# them are busy at once. Each name here is earned by a computation this system actually performs.
REGIONS = {
    # what came in
    "auditory / lexical cortex":  ("lexical", "sensory"),
    "Wernicke L4 (hearing)":      ("lexical", "sensory"),
    "Wernicke L2/3 (meaning)":    ("lexical", "sensory"),
    "temporal pole (concepts)":   ("semantic", "sensory"),
    # medial temporal lobe: the memory system proper, which is several structures
    "perirhinal (familiarity)":   ("episodic", "medial temporal"),
    "parahippocampal (context)":  ("episodic", "medial temporal"),
    "entorhinal grid modules":    ("map", "medial temporal"),
    "dentate gyrus (separation)": ("episodic", "medial temporal"),
    "CA3 (completion)":           ("episodic", "medial temporal"),
    "CA1 (readout)":              ("map", "medial temporal"),
    "hippocampal map":            ("map", "medial temporal"),
    # frontal: holding, judging, deciding
    "dlPFC (working memory)":     ("working", "frontal"),
    "frontopolar (task set)":     ("working", "frontal"),
    "ACC (conflict)":             ("gate", "frontal"),
    "dlPFC (evaluation)":         ("gate", "frontal"),
    "vmPFC (metamemory)":         ("gate", "frontal"),
    "mPFC (self / source)":       ("working", "frontal"),
    # subcortical: selection, relay, neuromodulation
    "thalamic gate (reticular)":  ("gate", "subcortical"),
    "mediodorsal (goal held)":    ("working", "subcortical"),
    "global workspace":           ("gate", "frontal"),
    "striatum (selection)":       ("gate", "subcortical"),
    "thalamus (relay)":           ("gate", "subcortical"),
    "VTA (surprise)":             ("gate", "subcortical"),
    # out
    "Broca L2/3 (assembling)":    ("map", "output"),
    "Broca L5 (sequencing)":      ("map", "output"),
    "Broca / motor speech":       ("map", "output"),
}
TRACT = {name: where for name, (where, _) in REGIONS.items()}
SYSTEM = {name: system for name, (_, system) in REGIONS.items()}


class Recorder:
    """Intervals of activity, per region, for one turn."""

    def __init__(self, tracts):
        self.tracts = tracts
        self.clock = {}        # region -> when it last finished
        self.events = []       # every firing, with when it started and how long it lasted
        self.messages = []     # who sent what to whom, and when it landed

    def ready(self, region):
        """When this region could start: after its own last job, and after its first input landed.

        The *first* input, not the last: a region does not wait for every signal that will ever reach it, it
        starts when it has something and is still going when more arrives. That is what puts two regions on the
        screen at once.
        """
        key = TRACT.get(region, region)
        arrived = [m["at"] for m in self.messages if m["to"] == key]
        return max(self.clock.get(region, 0), min(arrived) if arrived else 0)

    def fire(self, region, code=None, duration=1, because=None, at=None):
        """Record one firing. Returns when it finished, which is what the caller sends on."""
        start = at if at is not None else self.ready(region)
        end = start + max(1, duration)
        self.clock[region] = end
        self.events.append({"region": region, "from": start, "to": end, "because": because,
                            "code": code, "active": int((code > 0).sum()) if code is not None else 0})
        return end

    def send(self, src, dst, at, what=None):
        """A signal on its way: it lands after the conduction delay for that pathway."""
        delay = self.tracts.delay(src, dst) if src in self.tracts.layout and dst in self.tracts.layout else 1
        landing = at + delay
        self.messages.append({"from": src, "to": dst, "sent": at, "at": landing, "what": what})
        return landing

    # --------------------------------------------------------------- reporting
    def span(self):
        return max([e["to"] for e in self.events], default=0)

    def overlaps(self):
        """How many pairs of regions were active at the same time - the thing a pipeline cannot have."""
        n = 0
        for i, a in enumerate(self.events):
            for b in self.events[i + 1:]:
                if a["region"] != b["region"] and a["from"] < b["to"] and b["from"] < a["to"]:
                    n += 1
        return n

    def passes(self, region):
        return sum(1 for e in self.events if e["region"] == region)

    def busiest(self):
        """The most regions active at any one tick - how parallel this turn actually was."""
        best = 0
        for t in range(self.span() + 1):
            best = max(best, sum(1 for e in self.events if e["from"] <= t < e["to"]))
        return best

    def raster(self, bins=64):
        """Everything drawable: per firing, a binned code, its interval, and what set it off."""
        import torch
        out = []
        for e in self.events:
            code, row = e["code"], [0.0] * bins
            if code is not None:
                v = code.reshape(-1).float()
                pad = (-v.numel()) % bins
                binned = torch.cat([v, torch.zeros(pad)]).reshape(bins, -1).sum(1)
                top = float(binned.max()) or 1.0
                row = [round(float(x) / top, 3) for x in binned]
            out.append({"region": e["region"], "from": e["from"], "to": e["to"], "because": e["because"],
                        "active": e["active"], "bins": row, "system": SYSTEM.get(e["region"], "other")})
        return {"events": out, "span": self.span(), "overlaps": self.overlaps(),
                "busiest": self.busiest(), "regions": len({e["region"] for e in self.events}),
                "messages": [{"from": m["from"], "to": m["to"], "sent": m["sent"], "at": m["at"]}
                             for m in self.messages]}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/tracts.py
"""Conduction delay, which is where sequence comes from.

Nothing in a brain happens "next" because a program said so. A spike leaves a cell and takes time to arrive -
about a millisecond per millimetre in unmyelinated cortex - so the order in which regions hear about an event is
set by how far apart they are. Fast, short pathways deliver a rough answer first; long ones deliver a better one
later. That is why there are reflexes at all, and why deciding early and deciding well are different things.

Until now the order of this project's turn loop was the order the Python happened to be written in. Here it is
distance:

    lexical --1--> working memory --1--> gate
        \\--2--> episodic --1-->/
         \\--------4-------> map ---1--->/

The gate decides at a deadline. Evidence that has not arrived by then cannot affect the choice - it is not
ignored, it is *not there yet*. Shorten the deadline and the same question is answered by whichever store is
nearest; lengthen it and the slow, careful store gets its say. Both are the same circuit run under different
urgency, which is what the two-pathway story in the brain actually is.
"""
import math

# where each region sits, in conduction units. Only the distances matter.
LAYOUT = {"lexical": 0.0, "working": 1.0, "gate": 2.0, "episodic": 2.5, "semantic": 3.0, "map": 4.5}
DEADLINE = 6            # ticks the gate will wait before it has to choose


class Tracts:
    def __init__(self, layout=None, speed=1.0, deadline=DEADLINE):
        self.layout = dict(layout or LAYOUT)
        self.speed = speed
        self.deadline = deadline

    def delay(self, src, dst):
        """Ticks for a signal to travel, never less than one: nothing is instantaneous."""
        d = abs(self.layout[dst] - self.layout[src]) / self.speed
        return max(1, int(math.ceil(d)))

    def arrival(self, src, dst, via=None):
        """When a signal sent at tick 0 from `src` reaches `dst`, optionally relayed through other regions."""
        t, here = 0, src
        for hop in list(via or []) + [dst]:
            t += self.delay(here, hop)
            here = hop
        return t

    def in_time(self, src, dst, via=None, deadline=None):
        return self.arrival(src, dst, via) <= (self.deadline if deadline is None else deadline)

    def timeline(self, src="lexical"):
        """What arrives when, for the panel and for the record of each turn."""
        return sorted(((self.arrival(src, r), r) for r in self.layout if r != src))

    def spec(self):
        return {"layout": self.layout, "speed": self.speed, "deadline": self.deadline,
                "arrivals": {r: self.arrival("lexical", r) for r in self.layout if r != "lexical"}}

In [ ]:
%%writefile /kaggle/working/neuromoe/brain/working.py
"""Prefrontal working memory: what is being talked about right now.

This is the piece whose absence made the old portal unable to hold a conversation. It is not a dictionary of
"context variables" - it is the same one-shot associative memory the benchmarks showed recalls a thousand pairs
with no training (M += beta (v - M k) k^T / |k|), used the way prefrontal cortex is thought to use persistent
activity: a few things held active, decaying unless refreshed, addressable by content.

  slots        topic, who was mentioned, the last relation, the last answer. Each slot is a fixed random key
               code; writing a binding is one local update, reading it is one matrix-vector product.
  decay        bindings fade unless rewritten, so "she" means whoever was last talked about rather than whoever
               was talked about an hour ago. This is a leak, not a queue - nothing is deleted on a schedule.
  time cells   every turn gets its own sparse code, so an utterance can be bound to *when* it was said and the
               conversation can be replayed in order.

Reading a slot gives a pattern, not a string. Turning it back into a word is a population readout over the terms
the conversation has actually used - so the memory can be wrong, and says how strongly it is bound.
"""
import torch

from neuromoe import TimeCells, WorkingMemory

SLOTS = ("topic", "subject", "object", "relation", "answer", "question", "unresolved")


class Prefrontal:
    def __init__(self, lexicon, decay=0.94, max_turns=4096, seed=3):
        self.lex = lexicon
        self.n = lexicon.n
        # bindings of slot -> content, and of turn -> what was said then
        self.slots = WorkingMemory(self.n, self.n, beta=1.0, decay=decay)
        self.episodes = WorkingMemory(self.n, self.n, beta=1.0, decay=1.0)
        self.time = TimeCells(n=self.n, k=lexicon.k, seed=seed, max_steps=max_turns)
        self.slot_key = {s: lexicon.code("\x00slot:" + s) for s in SLOTS}
        self.turn = 0
        self.mentioned = []      # terms this conversation has used, most recent first: the readout's candidates
        self.history = []        # [(turn, text, kind)] - the transcript, for grounding what it claims to recall

    # ------------------------------------------------------------------ holding
    def hold(self, slot, term):
        """Bind a slot to a term. One write, no training."""
        if term is None:
            return
        self.slots.write(self.slot_key[slot], self.lex.code(term))
        self.note_mention(term)

    def note_mention(self, term):
        if term in self.mentioned:
            self.mentioned.remove(term)
        self.mentioned.insert(0, term)
        del self.mentioned[64:]

    def recall(self, slot, top=1, floor=0.25):
        """What is in a slot: the pattern read out, then named by whichever mentioned term it overlaps most."""
        pattern = self.slots.read(self.slot_key[slot])
        hits = [h for h in self.lex.nearest(pattern, self.mentioned, top=top) if h[1] >= floor]
        return hits[0] if top == 1 and hits else (hits if top != 1 else None)

    # -------------------------------------------------------------------- turns
    def advance(self, text, kind="said"):
        """A new turn: bind what was said to this moment, so it can be recalled by time as well as by content."""
        self.turn += 1
        code = self.lex.bag(text.lower().split()) if text else torch.zeros(self.n)
        self.episodes.write(self.time(self.turn), code)
        self.history.append((self.turn, text, kind))
        return code

    def at_turn(self, t, top=3):
        """What was said at a given turn, named from the pattern - memory, not a transcript lookup."""
        if t < 1 or t > self.turn:
            return []
        return self.lex.nearest(self.episodes.read(self.time(t)), self.mentioned, top=top)

    def last_turns(self, n=2):
        return self.history[-n:]

    def clear(self):
        self.slots.clear()
        self.mentioned.clear()

    def synapses(self):
        return self.slots.synapses() + self.episodes.synapses()

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_brain.py
"""Checks that the brain mechanisms are *in the loop*, not sitting beside it.

Each check names the mechanism it is evidence for. If a mechanism were replaced by a lookup table or an if-else,
the corresponding check should fail - that is what they are for.
"""
import os
import sys

import torch

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, ROOT)

from brain.agent import Agent  # noqa: E402
from brain.codes import Lexicon  # noqa: E402

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name}{'  -> ' + str(detail)[:140] if detail else ''}")


def reply(a, text):
    return a.say(text)[-1]["reply"]


def step(a, text):
    return a.say(text)[-1]


# 1. lexical cortex: sparse codes, similarity by overlap, unbounded vocabulary -----------------------------
lex = Lexicon(seed=0)
dog, dogs, violin = lex.code("dog"), lex.code("dogs"), lex.code("violin")
check("lexical code is sparse and exactly k units", float(dog.sum()) == lex.k and dog.max() == 1.0)
check("morphology overlaps, unrelated words do not", float(dog @ dogs) > float(dog @ violin),
      f"dog.dogs={float(dog @ dogs):.0f} vs dog.violin={float(dog @ violin):.0f}")
check("a word never seen before still has a code", float(lex.code("zzxqv").sum()) == lex.k)

a = Agent(background=False)

# 2. cognitive map: one-shot facts, inference never stated -------------------------------------------------
a.say("Dan is taller than Eve. Alice is taller than Bob. Cara is taller than Dan. Bob is taller than Cara.")
r = reply(a, "Is Alice taller than Eve?")
check("map infers a 4-step chain told out of order", r.startswith("Yes"), r)

# 3. prefrontal working memory: a pronoun resolved from a bound slot, not from word order -------------------
s = step(a, "Is she taller than Cara?")
check("pronoun resolved from working memory", s["reply"].startswith("Yes") and "resolved_ref" in s["signals"],
      s["reply"])
a2 = Agent(background=False)
s = step(a2, "Is she taller than Bob?")
check("pronoun with nothing bound asks instead of guessing", "mean" in s["reply"].lower(), s["reply"])

# 4. episodic store: pattern completion over what was said -------------------------------------------------
s = step(a, "what did we just talk about")
check("episodic recall returns something actually said", s["action"] == "recall" and "said" in s["reply"],
      s["reply"])
rec, conf = a.epi.recall("Cara taller")
check("a partial cue completes to the right episode", rec is not None and "Cara" in rec["text"],
      f"{rec and rec['text']} ({conf:.2f})")

# 5. basal-ganglia gate: grows experts from use and learns to select ----------------------------------------
before = a.gate.stats()["experts"]
for q in ("Who is the tallest?", "Is Bob taller than Dan?", "Rank everyone by height"):
    a.say(q)
after = a.gate.stats()
check("gate recruits experts from use (neurogenesis)", after["experts"] > 0 and after["born"] > 0, after)
check("gate takes over from the reflexes", any(t["via"] == "learned" for t in a.trace),
      [t["via"] for t in a.trace][-6:])
check("gate selection is recorded per turn", set(a.trace[-1]) >= {"action", "via", "signals", "grounded"})

# 6. the gate corrects itself when a system comes back empty ------------------------------------------------
a3 = Agent(background=False)
a3.say("Alice is taller than Bob.")
s = step(a3, "Is Zog taller than Alice?")
check("no store has it: says so rather than inventing", "don't know" in s["reply"].lower(), s["reply"])

# 7. neuromodulation: surprise gates how strongly a fact is written ------------------------------------------
agree = step(a3, "Alice is taller than Bob.")["evidence"]
clash = step(a3, "Bob is taller than Alice.")["evidence"]
check("a contradiction is more surprising than a repetition", clash["surprise"] > agree["surprise"] + 0.3,
      f"{agree['surprise']:.2f} vs {clash['surprise']:.2f}")
check("surprise scales how hard the fact is written", clash["gain"] > agree["gain"],
      f"gain {agree['gain']:.2f} -> {clash['gain']:.2f}")
check("surprise does not buy a louder vote in consolidation",
      set(a3.maps.maps["height"].weights) == {1.0}, a3.maps.maps["height"].weights)

# 8. sleep: replay settles the map and outvotes the lone contradiction ---------------------------------------
a.say("Eve is taller than Alice.")
a.sleep()
answer = reply(a, "Is Alice taller than Eve?")
check("one contradicted fact is outvoted after replay",
      "alice is taller than eve" in answer.lower() and "no -" not in answer.lower(), answer)
check("but the answer says it is less sure, because the two are no longer pinned",
      "not lean on it" in answer or "floating" in answer, answer)
spreads = {n: a.maps.maps["height"].spread_of(n) for n in ("alice", "eve", "cara")}
check("the contradicted pair floats more than an item nobody argued about",
      min(spreads["alice"], spreads["eve"]) > spreads["cara"],
      {k: round(v, 2) for k, v in spreads.items()})

# 9. the memory grows with use: synaptogenesis, not a fixed parameter count ----------------------------------
small = Agent(background=False)
base = small.footprint()
small.say("Ann is older than Ben. Ben is older than Cal.")
grown = small.footprint()
check("footprint grows as it is used", grown > base, f"{base:,} -> {grown:,}")

# 10. nothing anywhere is trained by gradient ----------------------------------------------------------------
tensors = [small.gate.pool.P, small.gate.pool.O, small.wm.slots.M, small.epi.codes]
tensors += [m.pos for m in small.maps.maps.values()]
check("no tensor requires a gradient", not any(t.requires_grad for t in tensors))

print("\nALL PASS" if ok else "\nFAILURES ABOVE")
sys.exit(0 if ok else 1)

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_chat.py
"""End-to-end checks for the chat interface: controlled English in, answers read off the cognitive maps out."""
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "chat"))
import language as L  # noqa: E402
from mapsystem import Brain  # noqa: E402

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name}{'  -> ' + detail if detail else ''}")


def ask(b, text):
    return b.say(text)[-1]["reply"]


# parser ------------------------------------------------------------------------------------------------
P = L.parse
check("fact: comparative", P("Alice is taller than Bob.")["moves"] == [("height", 1)])
check("fact: opposite comparative", P("Bob is shorter than Alice")["moves"] == [("height", -1)])
check("fact: more X than", P("Cara is more careful than Dan")["moves"] == [("careful", 1)])
check("fact: compound direction", P("The bank is north-east of the park")["moves"] == [("north", 1), ("east", 1)])
check("fact: counted direction", P("Dan is 2 blocks east of Eve")["moves"] == [("east", 2)])
check("fact: left of", P("The lamp is left of the bed")["moves"] == [("east", -1)])
check("names drop articles", P("The Cafe is north of the park")["a"] == ("cafe", "The Cafe"))
check("question: compare", P("Is Alice taller than Cara?")["q"] == "compare")
check("question: which", P("Who is taller, Alice or Dan?")["q"] == "which")
check("question: superlative", P("Who is the tallest?")["axis"] == "height")
check("question: superlative -iest", P("Who is the happiest?")["axis"] == "happiness")
check("question: rank", P("Rank everyone by height")["q"] == "rank")
check("question: at", P("What is 2 blocks north of the park?")["moves"] == [("north", 2)])
check("question: direction", P("Is the bank north-east of the park?")["q"] == "direction")
check("unknown sentence refused", P("The weather is lovely today")["kind"] == "unknown")
check("sentence splitting", len(L.sentences("A is taller than B. B is taller than C! Is A taller than C?")) == 3)

# transitive inference over a shuffled chain, told once each --------------------------------------------
b = Brain()
b.say("Dan is taller than Eve. Alice is taller than Bob. Cara is taller than Dan. Bob is taller than Cara.")
r = ask(b, "Is Alice taller than Eve?")
check("transitive: 4-step inference", r.startswith("Yes"), r)
r = ask(b, "Is Eve taller than Bob?")
check("transitive: reversed", r.startswith("No"), r)
r = ask(b, "Who is the tallest?")
check("superlative", r.startswith("Alice"), r)
r = ask(b, "Who is the shortest?")
check("superlative, other direction", r.startswith("Eve"), r)
r = ask(b, "Rank everyone by height")
check("ranking", "Alice > Bob > Cara > Dan > Eve" in r, r)
r = ask(b, "Who is between Bob and Eve?")
check("between", "Cara" in r and "Dan" in r and "Alice" not in r, r)
r = ask(b, "How much taller is Alice than Eve?")
check("how much", "4 steps" in r, r)
r = ask(b, "Is Alice older than Bob?")
check("different dimension: doesn't know", "don't know" in r.lower(), r)
r = ask(b, "Is Zed taller than Bob?")
check("unknown item", "don't know anything about Zed" in r, r)

# islands: unknown until linked, then known ---------------------------------------------------------------
b.say("Xavier is taller than Yuri.")
r = ask(b, "Is Xavier taller than Alice?")
check("unlinked islands -> I don't know", "don't know" in r.lower(), r)
b.say("Eve is taller than Xavier.")
r = ask(b, "Is Alice taller than Yuri?")
check("islands linked by one fact", r.startswith("Yes"), r)

# contradiction is flagged and outvoted --------------------------------------------------------------------
r = ask(b, "Eve is taller than Alice.")
check("contradiction noticed", "contradicts" in r, r)
b.say("sleep")
r = ask(b, "Is Alice taller than Eve?")
check("lone contradiction outvoted after sleep", r.startswith("Yes"), r)

# space: routes and retrieval by location ------------------------------------------------------------------
s = Brain()
s.say("The cafe is north of the park. The bank is east of the cafe. The school is 2 blocks east of the park. "
      "The library is north of the bank.")
r = ask(s, "What is 1 block north and 1 east of the park?")
check("retrieval by location (never told directly)", r.startswith("the bank") or r.startswith("The bank"), r)
r = ask(s, "Where is the library relative to the park?")
check("path integration", "2 north" in r and "1 east" in r, r)
r = ask(s, "Is the school south-east of the cafe?")
check("direction check", r.startswith("Yes"), r)
r = ask(s, "Is the park north of the bank?")
check("direction check (no)", r.startswith("No"), r)
r = ask(s, "What is 5 blocks west of the park?")
check("empty location", "Nothing I know of" in r, r)
r = ask(s, "Who is between the park and the school?")
check("spatial between (nothing there)", "Nothing" in r, r)
s.say("The shop is east of the park.")
r = ask(s, "What is between the park and the school?")
check("spatial between", "shop" in r, r)

r = ask(s, "what do you know")
check("summary", "Places" in r, r.replace("\n", " | "))
check("view has points", s.view()[0]["kind"] == "place" and len(s.view()[0]["points"]) == 6)

print("\nALL PASS" if ok else "\nFAILURES ABOVE")
sys.exit(0 if ok else 1)

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_pfc.py
"""Checks the division of labour: the hippocampus retrieves, prefrontal cortex decides.

What makes this more than a naming convention is that the decisions are visible and separable. The plan says
which stores were worth recruiting for this question and what was run against them; the verdict comes from
comparing retrievals, not from any one of them; and a pair too close to call is a conclusion the planner
reaches when two retrievals disagree, which no single retrieval could report.
"""
import os
import sys

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, ROOT)

from brain import hippocampus, judgement, pfc  # noqa: E402
from brain.agent import Agent  # noqa: E402

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name}{'  -> ' + str(detail)[:160] if detail else ''}")


# 1. how much the place cells are worth using depends on the question -------------------------------------
check("a comparison is a map question", pfc.recruitment("compare")["map"] == 1.0)
check("remembering what was said is not", pfc.recruitment("recall")["map"] < 0.2
      and pfc.recruitment("recall")["episodic"] == 1.0, pfc.recruitment("recall"))
check("chatting recruits the map not at all", pfc.recruitment("social")["map"] == 0.0)
check("an unfamiliar question gets a middling default", pfc.recruitment("gibberish") == pfc.DEFAULT)

a = Agent(background=False)
a.say("Alice is taller than Bob. Bob is taller than Cara. Cara is taller than Dan.")
height = a.maps.maps["height"]

# 2. the verdict is reached by composing retrievals, not by taking one -------------------------------------
out = pfc.compare(height, "alice", "dan", "height")
check("it decides the comparison", out["verdict"] is True, out["why"])
check("it walked the relation in both directions before deciding",
      len(out["plan"]["steps"]) == 2 and all(s["did"].startswith("walk") for s in out["plan"]["steps"]),
      [s["did"] for s in out["plan"]["steps"]])
check("the reason is which walk arrived, and in how many steps",
      "reaches" in out["why"] and "step" in out["why"] and out["steps"] == 3, out["why"])
back = pfc.compare(height, "dan", "alice", "height")
check("and it reverses when the question does", back["verdict"] is False, back["why"])

# 3. the plan records what was recruited -------------------------------------------------------------------
check("the plan carries the recruitment for this question",
      out["plan"]["recruited"]["map"] == 1.0 and "episodic" in out["plan"]["recruited"],
      out["plan"]["recruited"])
plan = pfc.Plan("recall", ["alice"], None)
quiet = pfc.compare(height, "alice", "dan", "height", plan=plan)
check("a question the map is not for is refused rather than answered from it",
      quiet["verdict"] is True or plan.recruit["map"] < 0.2, plan.recruit)

# 4. an unclear retrieval is judged unclear, by the planner ------------------------------------------------
s = Agent(background=False)
s.say("The cafe is north of the park. The bank is east of the cafe.")
place = s.maps.maps["place"]
meta = judgement.Metamemory(s.lex)
near = pfc.locate(place, "park", [("north", 1)], meta=meta)
far = pfc.locate(place, "park", [("north", 9)], meta=meta)
check("with no history it acts on both and says it has no history",
      near["plan"]["result"]["judgement"]["how"] == "unlearned"
      and far["plan"]["result"]["judgement"]["how"] == "unlearned"
      and far["plan"]["result"]["item"] is not None,
      far["plan"]["result"]["why"][:60])
check("the two retrievals do look different, which is what there is to learn",
      near["presence"] > far["presence"], f"{near['presence']} vs {far['presence']}")

# what happened next: routes like the first held up, routes like the second did not
for _ in range(4):
    meta.learn(pfc.locate(place, "park", [("north", 1)], meta=meta)["meta_code"], True)
    meta.learn(pfc.locate(place, "cafe", [("east", 1)], meta=meta)["meta_code"], True)
    meta.learn(pfc.locate(place, "park", [("north", 9)], meta=meta)["meta_code"], False)
    meta.learn(pfc.locate(place, "bank", [("north", 7)], meta=meta)["meta_code"], False)

after_far = pfc.locate(place, "park", [("north", 8)], meta=meta)
after_near = pfc.locate(place, "park", [("north", 1)], meta=meta)
check("after seeing outcomes it withholds on the kind that never held up",
      after_far["plan"]["result"]["item"] is None
      and after_far["plan"]["result"]["judgement"]["how"] == "learned",
      after_far["plan"]["result"])
check("and still acts on the kind that did", after_near["plan"]["result"]["item"] == "cafe",
      after_near["plan"]["result"])
check("the boundary came from experience, not from a number in the code",
      meta.stats()["outcomes_seen"] == 16 and meta.stats()["experts"] >= 2, meta.stats())

# 5. the hippocampus offers operations, and holds no goal of its own ---------------------------------------
check("the hippocampus exposes retrieval operations only",
      all(hasattr(hippocampus.Thought, op) for op in ("at", "travel", "settle", "read", "confidence")))
check("it has no planner, no goal and no verdict",
      not any(hasattr(hippocampus.Thought, x) for x in ("decide", "goal", "verdict", "plan")))

print("\nALL PASS" if ok else "\nFAILURES ABOVE")
sys.exit(0 if ok else 1)

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_nonverbal.py
"""Checks that reasoning happens in activity, and that words exist only at the two ends.

The claim being tested is narrow and checkable: the gate selects on patterns rather than on a parsed label, the
answer leaves the map as a population state, and the sentence is produced by reading that state out. If any of
those were still a string being formatted, the corresponding check fails.
"""
import os
import sys

import torch

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, ROOT)

from brain import motor  # noqa: E402
from brain.agent import Agent  # noqa: E402
from brain.codes import Lexicon  # noqa: E402

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name}{'  -> ' + str(detail)[:150] if detail else ''}")


lex = Lexicon(seed=0)

# 1. an answer is a pattern, and roles survive superposition --------------------------------------------
s1 = motor.state_code(lex, {"subject": "alice", "object": "eve", "relation": "height", "sign": True})
s2 = motor.state_code(lex, {"subject": "eve", "object": "alice", "relation": "height", "sign": True})
check("the answer state is activity, not text", isinstance(s1, torch.Tensor) and s1.numel() == lex.n)
check("swapping the roles changes the pattern", float((s1 - s2).abs().sum()) > 0,
      f"difference {float((s1 - s2).abs().sum()):.0f}")
s3 = motor.state_code(lex, {"subject": "alice", "object": "eve", "relation": "height", "sign": False})
check("the verdict is part of the pattern", float((s1 - s3).abs().sum()) > 0)

# 2. speech is learned by imitation and produced by readout -----------------------------------------------
art = motor.Articulator(lex, seed=3)
check("it can say nothing before it has heard anything", art.speak(s1, {"subject": "Alice"}) == [])
art.imitate(s1, "Yes , Alice is taller than Eve".split(), roles={"subject": "alice", "object": "eve"})
said = art.speak(s1, {"subject": "Alice", "object": "Eve"})
check("what it heard, it can say from the state alone", said[:2] == ["Yes", ","] and "Alice" in said, " ".join(said))

# the same shape with people it has never heard in those positions: the slots are filled by readout
s4 = motor.state_code(lex, {"subject": "milo", "object": "rex", "relation": "height", "sign": True})
novel = art.speak(s4, {"subject": "Milo", "object": "Rex"})
check("a sentence shape generalises to items it never heard in it",
      "Milo" in novel and "Rex" in novel and "taller" in novel, " ".join(novel))
check("no word unit exists for those names", "milo" not in art.index and "Milo" not in art.index,
      sorted(w for w in art.index if not w.startswith("\x00"))[:8])

# 3. the gate decides on activity, not on a parsed label ---------------------------------------------------
a = Agent(background=False)
a.say("Alice is taller than Bob. Bob is taller than Cara.")
p = a._understand("Is Alice taller than Cara?")
activity = a._activity(p, "Is Alice taller than Cara?")
check("the gate is given patterns", all(isinstance(v, torch.Tensor) for v in activity.values()),
      list(activity))
signals, _ = a._signals(p)
with_activity = a.gate.situation(p["intent"], signals, activity)
without = a.gate.situation(p["intent"], signals, None)
check("the situation code changes when the circuit's state changes",
      float((with_activity - without).abs().sum()) > 0)
same_words_other_state = a.gate.situation("compare", {k: False for k in signals}, activity)
check("two turns with the same words but different store states get different codes",
      float((with_activity - same_words_other_state).abs().sum()) > 0)

# 4. the reply the user sees was produced by the readout, at least once ------------------------------------
a2 = Agent(background=False)
a2.say("Ann is older than Ben. Ben is older than Cal. Cal is older than Dee.")
spoken = [a2.say("Is Ann older than Cal?")[-1] for _ in range(3)]
check("after rehearsal the answer is spoken from the state",
      any(s["evidence"].get("said_from_state") for s in spoken),
      [s["reply"] for s in spoken])
check("and it is still the right answer", all(s["reply"].lower().startswith("yes") for s in spoken),
      [s["reply"] for s in spoken])
check("the articulator grew its vocabulary from use", a2.articulator.spec()["words"] > 0,
      a2.articulator.spec())

print("\nALL PASS" if ok else "\nFAILURES ABOVE")
sys.exit(0 if ok else 1)

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_extract.py
"""What the learned reader and the learned openings must not get wrong.

Both of these learn from text nobody vetted, so the tests here are mostly about refusal: a phrasing that
means nothing, a span that is half a clause, an opening that is really the first word of an answer. Yield is
measured in the training run; what is checked here is that nothing false gets through cheaply.
"""
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "chat"))

from brain import answer, extract  # noqa: E402

FAILED = []


def check(name, condition, detail=""):
    print(f"{'PASS' if condition else 'FAIL'}  {name}" + (f"   {detail}" if detail and not condition else ""))
    if not condition:
        FAILED.append(name)


def taught(pairs, relation="kind", support=2):
    p = extract.Patterns(support=support, purity=0.75)
    for sentence, a, b in pairs:
        p.observe(sentence, a, b, relation)
    p.settle()
    return p


# ---------------------------------------------------------------- learning phrasings
p = taught([("Paris is the capital city of France", "Paris", "France"),
            ("Tokyo is the capital city of Japan", "Tokyo", "Japan"),
            ("Lima is the capital city of Peru", "Lima", "Peru")], relation="origin")
check("a phrasing seen with several pairs is kept", "is the capital city of" in p.kept)
check("and it keeps the relation it was seen with", p.kept.get("is the capital city of") == "origin")

p = taught([("Paris is the capital city of France", "Paris", "France")], support=3)
check("a phrasing seen with one pair is not kept", not p.kept)

p = taught([("The amendments to the act were passed", "amendments", "act"),
            ("The changes to the law were passed", "changes", "law"),
            ("The revisions to the code were passed", "revisions", "code")])
check("a phrasing made only of glue is refused", "to the" not in p.kept,
      f"kept: {sorted(p.kept)}")

p = taught([("A dog is a mammal", "dog", "mammal"), ("A cat is a mammal", "cat", "mammal"),
            ("A whale is a mammal", "whale", "mammal")])
check("a copula is kept even though it is all glue", p.kept.get("is a") == "kind")

# a phrasing that means two different things in the corpus is not a phrasing
p = extract.Patterns(support=2, purity=0.75)
for a, b in (("Rome", "Italy"), ("Oslo", "Norway"), ("Cairo", "Egypt")):
    p.observe(f"{a} in {b} is old", a, b, "origin")
for a, b in (("wood", "houses"), ("stone", "walls"), ("iron", "tools")):
    p.observe(f"{a} in {b} is old", a, b, "kind")
p.settle()
check("a phrasing that means two things is refused", "in" not in p.kept)

# ---------------------------------------------------------------- reading with them
p = taught([("Paris is the capital city of France", "Paris", "France"),
            ("Tokyo is the capital city of Japan", "Tokyo", "Japan"),
            ("Lima is the capital city of Peru", "Lima", "Peru")], relation="origin")
got = p.read("Kuala Lumpur is the capital city of Malaysia.")
check("it reads a sentence of the shape it learned", bool(got), f"{got}")
if got:
    check("with the right two ends", got[0]["a"][0] == "kuala lumpur" and got[0]["b"][0] == "malaysia",
          f"{got[0]['a'][0]} | {got[0]['b'][0]}")

got = p.read("The report is the capital city of Malaysia was founded in 1857.")
check("a wh-clause is not mistaken for a thing",
      all("what" not in f["a"][0] and "which" not in f["a"][0] for f in got))

check("a fragment ending in a preposition is not a thing", not extract.usable_span("a central role in"))
check("a wh-word inside a span disqualifies it", not extract.usable_span("a central role in what"))
check("an adverb is not a thing", not extract.usable_span("historically"))
check("a proper name is a thing", extract.usable_span("Kuala Lumpur"))

check("the subject before a copula wins over the complement",
      extract._left_of("Category theory is another formalization that ") == "Category theory",
      extract._left_of("Category theory is another formalization that "))
check("an appositive is stepped over to reach the subject",
      "Kuala Lumpur" in extract._left_of("Kuala Lumpur, officially the Federal Territory, "),
      extract._left_of("Kuala Lumpur, officially the Federal Territory, "))
check("a connective is not stepped over",
      extract._left_of("monarchs played a central role in what ") == "",
      repr(extract._left_of("monarchs played a central role in what ")))

# ---------------------------------------------------------------- learned openings
m = answer.Markers()
check("a discourse word before a comma is an opening", m.hear("Well, I think it rained.", True) == "well")
check("a two-word opening is kept whole", m.hear("Good question - the answer is yes.", True) == "good question")
check("a pronoun is not an opening", m.hear("I went to the shops yesterday.", True) is None)
check("a turn that is only markers is not an opening", m.hear("Well, yes.", True) is None)

m = answer.Markers()
m.hear_many(["Is it raining?", "Well, I think it is raining now.",
             "Is it cold?", "Well, it is rather cold today.",
             "It rained all day yesterday.", "Actually, it stopped before lunchtime.",
             "It was cold too.", "Actually, it was warmer than Tuesday."])
after_q = m.best("after_question", floor=2)
after_s = m.best("after_statement", floor=2)
check("openings are counted after a question", any("Well" in x for x in after_q), f"{after_q}")
check("and separately after a statement", any("Actually" in x for x in after_s), f"{after_s}")
check("what it says after a question is not what it says after a statement", after_q != after_s)
check("saying nothing stays an option", "" in answer.openings_for("answer", m))
check("with nothing heard it falls back to what it was given",
      answer.openings_for("answer", answer.Markers()) == answer.OPENINGS["answer"])

check("an encyclopedia's asides come out of an answer",
      answer.gloss("Smith (born September 25, 1968) is an actor.") == "Smith is an actor.",
      answer.gloss("Smith (born September 25, 1968) is an actor."))

print("\n" + ("ALL PASS" if not FAILED else f"{len(FAILED)} FAILED: " + ", ".join(FAILED)))
sys.exit(1 if FAILED else 0)

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_layers.py
"""The layering, and what must remain true inside it.

Three claims are being made by the reshape and all three are cheap to falsify:

  layering    drive enters at L4, settles at L2/3, leaves at L5, and feedback from above changes gain rather
              than content. A caller who tries to skip a layer gets an exception, not a plausible answer.
  plasticity  the recurrent weights change with use, the dendrites still adapt, and the dentate still grows
              new units under crowding. Enforcing structure must not have frozen anything.
  language    input is a function of Wernicke's pattern and output is a readout of Broca's activity, which
              means both degrade when their tissue does. A system where damage changes nothing was not using
              the tissue in the first place.
"""
import os
import sys

import torch

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "chat"))

from brain.broca import Broca  # noqa: E402
from brain.codes import Lexicon  # noqa: E402
from brain.cortex import Laminar  # noqa: E402
from brain.episodic import Episodes  # noqa: E402
from brain.wernicke import Wernicke  # noqa: E402

FAILED = []


def check(name, condition, detail=""):
    print(f"{'PASS' if condition else 'FAIL'}  {name}" + (f"   {detail}" if detail and not condition else ""))
    if not condition:
        FAILED.append(name)


def raises(fn):
    try:
        fn()
    except RuntimeError:
        return True
    except Exception:
        return False
    return False


# ------------------------------------------------------------------ layering
area = Laminar("test", 256, n4=512, k4=16, n23=256, k23=12, n_out=8, seed=1)
check("L2/3 refuses to settle on nothing", raises(lambda: area.settle()))
check("L5 refuses to read an unsettled L2/3", raises(lambda: area.project()))

x = torch.zeros(256)
x[[3, 17, 88, 140]] = 1.0
code4 = area.drive(x)
check("drive enters at L4", int((code4 > 0).sum()) == area.k4, f"{int((code4 > 0).sum())}")
check("and settling is still required after a drive", raises(lambda: area.project()))
pattern = area.settle()
check("L2/3 settles to exactly k units", int((pattern > 0).sum()) == area.k23, f"{int((pattern > 0).sum())}")
check("L5 reads once L2/3 has settled", area.project() is not None)
check("a new drive un-settles the layer above it", raises(lambda: (area.drive(x), area.project())[1]))

# feedback modulates, it does not speak
above = Laminar("above", 256, n4=256, k4=12, n23=128, k23=8, seed=2)
above.drive(x)
above.settle()
before = area.gain.clone()
above.feedback(area)
check("feedback changes the gain of the area below", not torch.equal(before, area.gain))
check("and only ever multiplies it", bool((area.gain >= 1.0).all()))
# gain only bites where inputs compete, so this is tested on an input with something to choose between
mixed = torch.rand(256, generator=torch.Generator().manual_seed(7))
above.release(area)
area.drive(mixed, learn=False)
plain = area.settle(learn=False)
above.feedback(area, strength=2.0)
area.drive(mixed, learn=False)
biased = area.settle(learn=False)
check("which changes what the lower area hears", not torch.equal(biased, plain),
      "feedback made no difference to the settled pattern")
check("but the content still comes from below, not from above",
      float((area.state4 > 0).sum()) == area.k4)
above.release(area)

# ---------------------------------------------------------------- plasticity
area = Laminar("plastic", 256, n4=512, k4=16, n23=256, k23=12, seed=3)
area.drive(x)
area.settle()
first = int((area.recurrent != 0).sum())
for _ in range(3):
    area.drive(x)
    area.settle()
check("recurrent weights grow with use", int((area.recurrent != 0).sum()) >= first > 0,
      f"{first} -> {int((area.recurrent != 0).sum())}")

y = torch.zeros(256)
y[[5, 19, 90, 142]] = 1.0
a1 = area.assembly(x)
a2 = area.assembly(y)
area.sequence(a1, a2)
check("a sequence weight is asymmetric",
      float(area.recurrent[(a1 > 0).nonzero()[0, 0], (a2 > 0).nonzero()[0, 0]])
      != float(area.recurrent[(a2 > 0).nonzero()[0, 0], (a1 > 0).nonzero()[0, 0]]))
area.drive(x)
held = area.settle().clone()
area.assembly(y)
check("asking for an assembly leaves the area holding what it was holding",
      torch.equal(area.state23, held) and area.settled)

# pattern completion: a fragment of a learned pattern pulls the rest of it back
# at the default recurrent gain: it was tuned against this very trade-off, since the recurrent input is
# divided by how much each cell already receives and a weaker gain no longer completes anything
area = Laminar("complete", 256, n4=512, k4=16, n23=256, k23=12, seed=4)
full = area.assembly(x, learn=True)
for _ in range(6):
    area.drive(x)
    area.settle()
whole = area.settle(learn=False)
half = x.clone()
half[[88, 140]] = 0.0
area.drive(half)
fragment = area.settle(learn=False)
overlap = float((whole > 0).float() @ (fragment > 0).float()) / area.k23
check("a fragment completes toward the whole pattern", overlap > 0.3, f"overlap {overlap:.2f}")

# and the other side of the same trade: two different inputs must not settle into one pattern. Before the
# recurrent input was normalised, the first assembly this layer bound won every competition afterwards.
other = torch.zeros(256)
other[[11, 29, 101, 200]] = 1.0
area.drive(other)
apart = area.settle(learn=False)
collision = float((whole > 0).float() @ (apart > 0).float()) / area.k23
check("but two unrelated inputs do not collapse into one", collision < 0.7, f"overlap {collision:.2f}")

# ------------------------------------------------------------- neurogenesis
lex = Lexicon(seed=0)
epi = Episodes(lex)
started = epi.column.dendrites.n
for i in range(60):
    epi.store(f"the cat sat on the mat number {i % 3}", i, "fact", payload={})
check("the dentate grew new granule cells under crowding", epi.column.dendrites.n > started,
      f"{started} -> {epi.column.dendrites.n}, recruitments {len(epi.recruited)}")
check("and it recorded when and why", len(epi.recruited) > 0 and "overlap" in epi.recruited[0],
      f"{epi.recruited[:1]}")
check("the population each memory was written into is remembered", len(epi.widths) == 60)

# ------------------------------------------------------------------ language
lex = Lexicon(seed=0)
w = Wernicke(lex)
w.hear("Kuala Lumpur is the capital city of Malaysia")
check("a clean pattern gives back what was said", w.fidelity() == 1.0, f"{w.fidelity()} {w.heard_text()!r}")
check("recognition works in Wernicke's own units, not the lexicon's",
      all("local" in v and "lexical" in v for v in w.heard.values()))

clean = w.heard_text()
w.hear("Kuala Lumpur is the capital city of Malaysia", noise=0.35)
degraded = w.heard_text()
check("a degraded pattern gives back less", len(degraded) < len(clean), f"{degraded!r}")
check("and comprehension is downstream of it, not of the text",
      w.comprehend("Rex is older than Max", lambda t: {"saw": t}, noise=0.9)[0]["saw"] != "Rex is older than Max")

fresh = Wernicke(Lexicon(seed=0))
check("a word never heard cannot be recovered", fresh.recognise() == [])

b = Broca(lex)
said = b.say("Kuala Lumpur", "is the capital city of", "Malaysia")
check("production puts the parts in order", said == "Kuala Lumpur is the capital city of Malaysia", repr(said))
check("and every word of it was decoded from an assembly", b.fidelity() == 1.0, f"{b.last_trace}")
check("the trace says which assembly said what",
      [t["role"] for t in b.last_trace] == ["subject", "frame", "object"], f"{[t['role'] for t in b.last_trace]}")

b.load([("Rex", "subject", 1.0), ("is a", "frame", 0.75), ("dog", "object", 0.55)])
b.items[1]["assembly"] = torch.zeros_like(b.items[1]["assembly"])      # lesion one assembly
words = b.observe()
check("a lesioned assembly costs a word rather than being papered over",
      b.fidelity() < 1.0 or "is a" not in words, f"{words} {b.fidelity()}")

print("\n" + ("ALL PASS" if not FAILED else f"{len(FAILED)} FAILED: " + ", ".join(FAILED)))
sys.exit(1 if FAILED else 0)

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_conversation.py
"""What a conversation needs that a single question does not: pointers, slips, and the shape of an answer.

Every case here is one that failed in a real chat with the portal:

  "compare the three cities"            read a book about the history of technology
  "an essay on just kl"                 read the Kullback-Leibler divergence
  "cany ou write an essay about it"     was not understood at all
  "what country is beijig in"           was not understood at all
  "where is washington dc"              gave a definition instead of a place
"""
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "chat"))

from collections import Counter  # noqa: E402

from brain import answer, prose  # noqa: E402
from brain.referents import Referents  # noqa: E402
from brain.spelling import Speller  # noqa: E402

FAILED = []


def check(name, condition, detail=""):
    print(f"{'PASS' if condition else 'FAIL'}  {name}" + (f"   {detail}" if detail and not condition else ""))
    if not condition:
        FAILED.append(name)


KL = "Kuala Lumpur (KL), officially the Federal Territory of Kuala Lumpur, is the capital city and a federal " \
     "territory of Malaysia."
DC = "Washington, D.C., officially the District of Columbia and commonly known as Washington, is the federal " \
     "capital city of the United States, and its only federal district."
DELHI = "Delhi, officially the National Capital Territory (NCT) of Delhi, is a megacity and a union territory " \
        "of India containing New Delhi, the capital of India."

# ------------------------------------------------------------------ pointers
r = Referents()
r.mention("Kuala Lumpur", KL)
r.advance()
r.mention("Washington, D.C.", DC)
r.advance()
r.mention("Delhi", DELHI)
r.advance()

text, notes = r.resolve("write an essay that compares and contrasts the three cities")
check("'the three cities' binds to the three cities discussed",
      all(c in text for c in ("Kuala Lumpur", "Washington, D.C.", "Delhi")), text)
check("and keeps the group whole, commas and all", r.group and len(r.group) == 3, f"{r.group}")
text, _ = r.resolve("can you write an essay on just kl?")
check("an acronym binds to the thing it abbreviates in this conversation", "Kuala Lumpur" in text, text)
text, _ = r.resolve("write an essay about it")
check("'it' binds to the most present thing", "Delhi" in text, text)
text, _ = r.resolve("is it bigger than Delhi?")
check("'it' is never the thing the sentence already names", "Delhi bigger than Delhi" not in text, text)
text, _ = r.resolve("what is the capital city of India")
check("a description is not a pointer", text == "what is the capital city of India", text)
check("it knows what kind of thing each is", "city" in r.entries["kuala lumpur"]["kinds"])

r.mention("washington dc", "")
r.merge("washington dc", "Washington, D.C.")
check("what the person called it and what the page called it merge", "washington dc" not in r.entries)

# ------------------------------------------------------------------- slips
counts = Counter({"can": 5000, "you": 9000, "write": 800, "an": 7000, "essay": 300, "about": 6000, "it": 9000,
                  "what": 5000, "country": 900, "is": 20000, "in": 20000, "beijing": 200, "being": 4000,
                  "the": 50000, "capital": 700, "of": 40000, "taller": 34, "smaller": 900, "tell": 900,
                  "me": 3000, "abbot": 40})
s = Speller(counts=counts)
check("a slipped space is re-split", s.repair("cany ou write an essay about it")[0].startswith("can you"))
check("one slip beats a commoner word two slips away", "beijing" in s.repair("what country is beijig in")[0])
check("a transposition is one slip", "about" in s.repair("tell me abuot it")[0], s.repair("tell me abuot it")[0])
check("a real word is never 'corrected'", s.repair("Alice is taller than Bob")[0] == "Alice is taller than Bob")
check("a new name is not a typo", "Cara" in s.repair("Bob is taller than Cara")[0])
check("every repair is reported", s.repair("what country is beijig in")[1] != [])

# ------------------------------------------------------------ shape of answer
check("a where-question finds the place in a definition", answer.locate([KL]) == "Malaysia")
check("with its article", answer.locate([DC]) == "the United States", answer.locate([DC]))
check("even when the place is mid-sentence", answer.locate([DELHI]) == "India")
check("and says nothing when the reading never says", answer.locate(["Jazz is a music genre."]) is None)

check("a figure is read, not estimated",
      prose.figure(["It has a population of 2,111,536 as of 2026."], "population") == "2,111,536")
check("and none is invented", prose.figure(["It is a city."], "population") is None)
check("figures can be put in order", prose.magnitude("33.8 million") > prose.magnitude("2,111,536"))

piece = prose.compare([
    {"name": "Kuala Lumpur", "material": [KL], "definition": KL, "place": "Malaysia", "kinds": ["capital", "city"],
     "population": "2,111,536", "area": None},
    {"name": "Delhi", "material": [DELHI], "definition": DELHI, "place": "India", "kinds": ["capital", "city"],
     "population": "33.8 million", "area": None}])
check("a comparison says what they share", "capital cities" in piece, piece[:120])
check("and ranks only on figures it read", "Delhi is the most populous" in piece, piece)

print("\n" + ("ALL PASS" if not FAILED else f"{len(FAILED)} FAILED: " + ", ".join(FAILED)))
sys.exit(1 if FAILED else 0)

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_workspace.py
"""What the workspace has to do to be worth having.

The risk with a global workspace is that it becomes a for-loop with a grand name: if every bid is broadcast
and nothing is ever excluded, no area is better off than before and the architecture has only gained a word.
So the tests here are mostly about exclusion and about consequence:

  exclusion     a bid below threshold must leave every other area exactly as it was. If suppressed content
                and broadcast content have the same effect, there is no workspace.
  gain          arousal is one knob, and it works by changing what can ignite. Raising it must let contents
                through that were previously suppressed, and that must change how far the system can reason.
  globality     the measure used clinically on people - perturb, then see how widely and how variously the
                response spreads - applied to a running program. Ignited content should move several areas
                in different ways; suppressed content should move none.
  reasoning     the thing none of this was possible without: taking the answer to one retrieval and making it
                the question for the next, with a goal that survives both.
"""
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "chat"))

from brain import deliberate, workspace as ws  # noqa: E402
from brain.agent import Agent  # noqa: E402
from brain.thalamus import Thalamus  # noqa: E402

FAILED = []


def check(name, condition, detail=""):
    print(f"{'PASS' if condition else 'FAIL'}  {name}" + (f"   {detail}" if detail and not condition else ""))
    if not condition:
        FAILED.append(name)


class Area:
    """A subscriber that records what it was told, so a broadcast has something visible to change."""

    def __init__(self):
        self.heard = []

    def receive(self, content):
        self.heard.append(content.label)

    def signature(self):
        return ws.digest(*self.heard)


# ------------------------------------------------------------------- exclusion
board = ws.Workspace(gain=1.0, threshold=0.5)
a, b = board.subscribe("a", Area()), board.subscribe("b", Area())
board.post(ws.Content("a", "quiet thought", 0.2))
check("a bid below threshold does not ignite", board.cycle() is None)
check("and reaches no other area", b.heard == [], f"{b.heard}")
check("but is kept, so the exclusion can be inspected", len(board.suppressed) == 1)

board.post(ws.Content("a", "loud thought", 0.9))
winner = board.cycle()
check("a bid above threshold ignites", winner is not None and winner.label == "loud thought")
check("and reaches the other areas", b.heard == ["loud thought"], f"{b.heard}")
check("but not the area that thought it", a.heard == [], f"{a.heard}")
check("the stream is what became globally available", board.report() == ["loud thought"], f"{board.report()}")

board.post(ws.Content("a", "first", 0.9))
board.post(ws.Content("a", "second", 0.8))
board.cycle()
check("only one content wins a cycle", b.heard == ["loud thought", "first"], f"{b.heard}")
check("the rest are suppressed, not queued", len(board.bids) == 0 and len(board.suppressed) == 2)

# ------------------------------------------------------------------------ gain
board = ws.Workspace(gain=0.5, threshold=0.5)
board.subscribe("b", Area())
board.post(ws.Content("a", "middling", 0.6))
check("at low gain a middling bid stays local", board.cycle() is None)
board.gain = 1.5
board.post(ws.Content("a", "middling", 0.6))
check("at high gain the same bid ignites", board.cycle() is not None)

# ------------------------------------------------------------------ globality
board = ws.Workspace(gain=1.0, threshold=0.4)
areas = {n: board.subscribe(n, Area()) for n in ("map", "episodic", "reader")}
quiet = ws.Content("map", "quiet", 0.1)
board.post(quiet)
board.cycle()
after_quiet = board.index(ws.Content("nobody", "", 0.0))
check("content that never ignited leaves the areas alone", after_quiet["spread"] == 0.0, f"{after_quiet}")
loud = board.index(ws.Content("map", "loud", 0.9))
check("ignited content spreads to the other areas", loud["spread"] > 0.5, f"{loud}")
check("and the areas are changed differently, not identically", loud["differentiation"] > 0.0, f"{loud}")

# ------------------------------------------------------------------- the gate
board = ws.Workspace(gain=1.0, threshold=0.3)
thal = Thalamus(board, relevance=0.3)
thal.hold(ws.Content("prefrontal", "is rex a mammal", 1.0))
kept = thal.gate([ws.Content("map", "rex kind dog", 0.5, {"focus": "rex"}),
                  ws.Content("map", "paris kind city", 0.5, {"focus": "paris"})], focus="rex")
check("the gate stops what has nothing to do with the goal", len(kept) == 1 and "rex" in kept[0].label,
      f"{[c.label for c in kept]}")
check("and amplifies what does", kept[0].salience > 0.5, f"{kept[0].salience}")
check("what it stopped is recorded", len(thal.blocked) == 1)

for _ in range(4):
    thal.sustain()
    thal.progressed(False)
check("a goal that gets nowhere is given up on", thal.sustain() is False)
check("and the thalamus says how uncertain it had become", thal.uncertainty() >= 1.0, f"{thal.uncertainty()}")

before = board.gain
thal.arouse(1.0)
check("arousal changes the gain the workspace competes under", board.gain > before, f"{board.gain}")

# -------------------------------------------------------------------- reasoning
agent = Agent(background=False)
for said in ("Rex is a dog", "dogs are mammals", "mammals are animals"):
    agent.say(said)
out = deliberate.is_a(agent, "rex", "animal")
check("it chains across separate facts to an answer", out["found"], f"{out['trail']}")
check("and the trail is every hop it took", len(out["trail"]) >= 3, f"{out['trail']}")
check("the stream is the reasoning, in order", len(out["stream"]) >= 3, f"{out['stream']}")
check("a chain that does not exist is not invented", not deliberate.is_a(agent, "rex", "fish")["found"])

turn = agent.say("is Rex an animal")[-1]
check("and the turn says so in words", "yes" in turn["reply"].lower() and "animal" in turn["reply"].lower(),
      turn["reply"][:120])
check("with the workspace's own account in the reasoning channel",
      any(line.startswith("stream:") for line in turn["reasoning"]), f"{turn['reasoning']}")

fresh = Agent(background=False)
fresh.say("Rex is a dog")
check("without the middle fact it will not claim the chain", not deliberate.is_a(fresh, "rex", "animal")["found"])

print("\n" + ("ALL PASS" if not FAILED else f"{len(FAILED)} FAILED: " + ", ".join(FAILED)))
sys.exit(1 if FAILED else 0)

### Does it still work at all

In [ ]:
import subprocess, sys
for test in ("tests/check_brain.py", "tests/check_chat.py", "tests/check_pfc.py",
             "tests/check_nonverbal.py", "tests/check_extract.py",
             "tests/check_layers.py", "tests/check_conversation.py",
             "tests/check_workspace.py"):
    r = subprocess.run([sys.executable, test], capture_output=True, text=True, cwd="/kaggle/working/neuromoe")
    print(test, r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-400:])
    assert r.returncode == 0, f"{test} failed - not training on a broken brain"

### Stage 1 - read the world

In [ ]:
%%time
# Stage 1: go and read. Nothing was shipped with this notebook.
import sys, time
sys.path[:0] = ["/kaggle/working/neuromoe", "/kaggle/working/neuromoe/chat"]
import web

TOPICS = ['Hip hop', 'Dance', 'Actor', 'Music', 'Biology', 'City', 'Language', 'Science', 'History', 'Film', 'Malaysia', 'Kuala Lumpur', 'Will Smith', 'Bonobo', 'Philadelphia', 'Jazz', 'Architecture', 'Economics', 'Psychology', 'Physics', 'Chemistry', 'Geography', 'Literature', 'Theatre', 'Painting', 'Sculpture', 'Anthropology', 'Sociology', 'Migration', 'Colonialism', 'Trade', 'Religion', 'Education', 'Medicine', 'Agriculture', 'Climate', 'Ocean', 'Mountain', 'River', 'Bird', 'Mammal', 'Insect', 'Tree', 'Fungus', 'Bacterium', 'Virus', 'Galaxy', 'Planet', 'Star', 'Computer', 'Internet', 'Mathematics', 'Logic', 'Philosophy', 'Ethics', 'Law', 'Politics', 'Africa', 'Asia', 'Europe', 'South America', 'North America', 'Australia', 'Antarctica', 'India', 'China', 'Japan', 'Brazil', 'Nigeria', 'Indonesia', 'Germany', 'France', 'Egypt', 'Singapore', 'Thailand', 'Vietnam', 'Philippines', 'Mexico', 'Kenya', 'Morocco', 'Peru', 'London', 'Tokyo', 'Lagos', 'Jakarta', 'Mumbai', 'Istanbul', 'Cairo', 'Paris', 'New York City', 'Rap music', 'Blues', 'Reggae', 'Salsa music', 'Gamelan', 'Bharatanatyam', 'Ballet', 'Opera', 'Hip hop dance', 'Breakdancing', 'Choreography', 'Folk dance', 'Popular music', 'Rock music', 'Cinema of India', 'Hollywood', 'Television', 'Radio', 'Journalism', 'Advertising', 'Evolution', 'Genetics', 'Ecology', 'Neuroscience', 'Immunology', 'Cell biology', 'Botany', 'Primate', 'Chimpanzee', 'Gorilla', 'Orangutan', 'Elephant', 'Whale', 'Dolphin', 'Octopus', 'Ant', 'Bee', 'Butterfly', 'Spider', 'Coral reef', 'Rainforest', 'Desert', 'Savanna', 'Tundra', 'Volcano', 'Earthquake', 'Glacier', 'Monsoon', 'Hurricane', 'Photosynthesis', 'Metabolism', 'Quantum mechanics', 'Relativity', 'Thermodynamics', 'Electricity', 'Magnetism', 'Optics', 'Astronomy', 'Cosmology', 'Black hole', 'Solar System', 'Moon', 'Mars', 'Jupiter', 'Sun', 'Algebra', 'Geometry', 'Calculus', 'Probability', 'Statistics', 'Topology', 'Number theory', 'Algorithm', 'Artificial intelligence', 'Machine learning', 'Cryptography', 'Database', 'Operating system', 'Programming language', 'Software engineering', 'Robotics', 'Semiconductor', 'Industrial Revolution', 'World War II', 'Cold War', 'Renaissance', 'Enlightenment', 'Silk Road', 'Ottoman Empire', 'Roman Empire', 'Ancient Egypt', 'Maya civilization', 'Democracy', 'Capitalism', 'Socialism', 'Constitution', 'Human rights', 'United Nations', 'Public health', 'Vaccine', 'Antibiotic', 'Cancer', 'Diabetes', 'Mental health', 'Nutrition', 'Architecture of Malaysia', 'Southeast Asia', 'Islam', 'Buddhism', 'Hinduism', 'Christianity', 'Poetry', 'Novel', 'Drama', 'Mythology', 'Folklore', 'Linguistics', 'Writing system', 'Agricultural economics', 'International trade', 'Urbanization', 'Transport', 'Railway', 'Aviation', 'Shipping', 'Energy', 'Renewable energy', 'Nuclear power', 'Water supply', 'Photography', 'Design', 'Fashion', 'Cuisine', 'Coffee', 'Tea', 'Rice', 'Wheat', 'Maize']
lines, seen = [], set()
t0 = time.time()
for topic in TOPICS:
    page = web.look_up(topic, sentences=400)
    got = [s for s in page.get("sentences", []) if 30 < len(s) < 300]
    for s in got:
        if s not in seen:
            seen.add(s)
            lines.append(s)
    print(f"  {page.get('title')}: {len(got)} sentences ({len(lines):,} total)", flush=True)
wiki = len(lines)
print(f"{wiki:,} encyclopedia sentences in {time.time() - t0:.0f}s")

# textbooks and novels: how an explanation unfolds, and how English sounds when nobody is summarising
import books
t1 = time.time()
counts = {"encyclopedia": wiki}
expository = list(lines)          # what explains things: the encyclopedia and the textbooks, not the novels
for register, sentence in books.harvest():
    if sentence not in seen:
        seen.add(sentence)
        lines.append(sentence)
        counts[register] = counts.get(register, 0) + 1
        if register != "novel":
            expository.append(sentence)
# reading patterns are learned from explanatory writing only. v13 learned them from novels too, and the
# hand-written reader run over fiction pairs "I" with whatever came next - "not", "would" and "well i" became
# ways of saying a thing is involved with another. Novels still teach the speech memory how English sounds.
with open("/kaggle/working/expository.tsv", "w", encoding="utf-8") as f:
    for s in expository:
        f.write(s.replace(chr(9), " ") + chr(10))
print(f"{len(expository):,} expository sentences -> /kaggle/working/expository.tsv")
print(f"books: {counts} in {time.time() - t1:.0f}s")
with open("/kaggle/working/corpus.tsv", "w", encoding="utf-8") as f:
    for s in lines:
        f.write(s.replace(chr(9), " ") + chr(10))
print(f"{len(lines):,} sentences in {time.time() - t0:.0f}s -> /kaggle/working/corpus.tsv")

### Stage 2 - learn to speak

In [ ]:
%%time
# Stage 2: hear it. One pass, no objective, counts only.
import subprocess, sys
r = subprocess.run([sys.executable, "-u", "brain/learn_to_speak.py", "--corpus", "/kaggle/working/corpus.tsv",
                    "--sentences", "600000", "--cells", str(1 << 23),
                    "--out", "/kaggle/working/artifacts/speech.pt"],
                   capture_output=True, text=True, cwd="/kaggle/working/neuromoe")
print(r.stdout[-6000:], r.stderr[-2000:])
assert r.returncode == 0

### Stage 3 - learn to read

In [ ]:
%%time
# Stage 3: learn the shapes English states a fact in, from the pairs the hand-written reader is sure about.
import subprocess, sys
r = subprocess.run([sys.executable, "-u", "brain/learn_to_extract.py", "--corpus", "/kaggle/working/expository.tsv",
                    "--sentences", "600000", "--rounds", "3", "--holdout", "600",
                    "--out", "/kaggle/working/artifacts/patterns.pt"],
                   capture_output=True, text=True, cwd="/kaggle/working/neuromoe")
print(r.stdout[-12000:], r.stderr[-2000:])
assert r.returncode == 0

### Stage 4 - learn to talk

In [ ]:
%%time
# Stage 4: go and find people talking, and learn how a turn opens.
import subprocess, sys
r = subprocess.run([sys.executable, "-u", "brain/learn_to_converse.py", "--turns", "40000",
                    "--per-query", "12", "--cache", "/kaggle/working/dialogue.tsv",
                    "--markers", "/kaggle/working/artifacts/markers.pt",
                    "--speech", "/kaggle/working/artifacts/speech.pt"],
                   capture_output=True, text=True, cwd="/kaggle/working/neuromoe")
print(r.stdout[-8000:], r.stderr[-2000:])

### Word frequencies, for hearing past typos

In [ ]:
%%time
# Word frequencies, counted off everything it read: the only evidence spelling repair trusts.
import sys, pickle
sys.path[:0] = ["/kaggle/working/neuromoe", "/kaggle/working/neuromoe/chat"]
from brain import spelling
counts = spelling.count_corpus(["/kaggle/working/corpus.tsv", "/kaggle/working/dialogue.tsv"])
counts = type(counts)({w: c for w, c in counts.items() if c >= 2})
pickle.dump(counts, open("/kaggle/working/artifacts/unigrams.pkl", "wb"))
print(f"{len(counts):,} words counted")
from brain.lang import Speech
speller = spelling.Speller(counts=counts, speech=Speech.load("/kaggle/working/artifacts/speech.pt"))
for typed in ["cany ou write an essay about it", "what country is beijig in", "whats teh capitl of malaysia",
              "tell me abuot the goverment", "Alice is taller than Cara"]:
    print(f"  {typed!r} -> {speller.repair(typed)}")

### What the learned phrasings are worth

In [ ]:
%%time
# What the learned phrasings are actually worth: the same pages read with them and without them.
import sys, json
sys.path[:0] = ["/kaggle/working/neuromoe", "/kaggle/working/neuromoe/chat"]
import language as L
from brain import extract
import web

patterns = extract.Patterns.load("/kaggle/working/artifacts/patterns.pt")
rows = []
for topic in ["Kuala Lumpur", "Bonobo", "Jazz", "Philadelphia", "Malaysia", "Will Smith"]:
    page = web.look_up(topic, sentences=30)
    hand = learned = 0
    for s in page.get("sentences", []):
        got = L.prose_facts(s, limit=3)
        hand += len(got)
        if not got:
            learned += len(patterns.read(s))
    rows.append({"topic": topic, "sentences": len(page.get("sentences", [])),
                 "hand": hand, "learned": learned})
    print(f"  {topic:16} {len(page.get('sentences', [])):3} sentences  hand {hand:3}  +learned {learned:3}")
total_hand = sum(r["hand"] for r in rows)
total_learned = sum(r["learned"] for r in rows)
print(f"\nacross {len(rows)} pages: {total_hand} facts by hand-written rules, "
      f"{total_learned} more from learned phrasings "
      f"({(total_hand + total_learned) / max(1, total_hand):.2f}x)")
json.dump(rows, open("/kaggle/working/artifacts/extraction_ablation.json", "w"), indent=1)

### Does the workspace earn its place

In [ ]:
%%time
# Does the workspace earn its place? Chains it can follow, and how global an ignited content actually is.
import sys, json, time
sys.path[:0] = ["/kaggle/working/neuromoe", "/kaggle/working/neuromoe/chat"]
from brain.agent import Agent
from brain import deliberate, workspace as ws

a = Agent(background=False)
chain = ["Rex is a dog", "dogs are mammals", "mammals are animals", "animals are living things"]
for said in chain:
    a.say(said)
depth = {}
for target in ("dog", "mammal", "animal", "living thing"):
    out = deliberate.is_a(a, "rex", target)
    depth[target] = {"found": out["found"], "hops": len(out["trail"]), "cycles": out["cycles"],
                     "gain": out["gain"], "stream": out["stream"]}
    print(f"  is Rex a {target}? {out['found']} in {len(out['trail'])} hop(s), {out['cycles']} cycle(s)")
print("  invented chain refused:", not deliberate.is_a(a, "rex", "fish")["found"])

# the perturbational measure: ignited content should move several areas differently, suppressed content none
loud = a.workspace.index(ws.Content("map", "rex kind dog", 0.9, {"focus": "rex", "moves_to": "dog"}))
quiet = a.workspace.index(ws.Content("nobody", "", 0.0))
print(f"  globality of an ignited content: {loud}")
print(f"  globality of nothing:            {quiet}")

# and what gain does to it: low arousal should reason less far than high
reach = {}
for difficulty in (0.0, 0.5, 1.0):
    out = deliberate.is_a(a, "rex", "living thing", budget=4)
    a.thalamus.arouse(difficulty)
    reach[difficulty] = {"gain": a.workspace.gain, "found": out["found"], "hops": len(out["trail"])}
print("  reach by arousal:", reach)
json.dump({"depth": depth, "globality": {"ignited": loud, "suppressed": quiet}, "reach": reach},
          open("/kaggle/working/artifacts/workspace.json", "w"), indent=1)

### Development: growth, death, then acquisition

In [ ]:
%%time
# Development, then acquisition. Four arms, five seeds: does a critical period of growth and death on
# near-identical sentences leave a system that separates better - and what does the dentate then cost?
import subprocess, sys, json
r = subprocess.run([sys.executable, "-u", "brain/learn_to_discriminate.py", "--grades", "1", "2", "3", "4", "5",
                    "--pairs", "32", "--steps", "400", "--seeds", "5", "--arms", "all",
                    "--out", "/kaggle/working/artifacts/discriminate.json"],
                   capture_output=True, text=True, cwd="/kaggle/working/neuromoe")
print(r.stdout[-9000:], r.stderr[-2500:])
rows = json.load(open("/kaggle/working/artifacts/discriminate.json"))
by_arm = {}
for row in rows:
    arm = by_arm.setdefault(row["arm"], {"recall": [], "separation": [], "forgetting": [], "born": []})
    grades = list(row["after"])
    arm["recall"].append(sum(row["after"][g]["recall"] for g in grades) / len(grades))
    arm["separation"].append(sum(row["after"][g]["separation"] for g in grades) / len(grades))
    arm["forgetting"].append(sum(row["forgetting"].values()) / max(1, len(row["forgetting"])))
    arm["born"].append((row["development"] or {"born": {}})["born"].get("wernicke", 0))
print()
for arm, v in by_arm.items():
    mean = lambda xs: sum(xs) / max(1, len(xs))
    print(f"  {arm:12} recall {mean(v['recall']):.3f}  separation {mean(v['separation']):.3f}  "
          f"forgetting {mean(v['forgetting']):+.3f}  cortex born {mean(v['born']):.0f}")

# the pre-registered tests (PREREGISTRATION_development.md, amendment): paired t over seeds, Holm-Bonferroni
import math, statistics as st
from scipy import stats
def paired(a, b, key):
    d = [x - y for x, y in zip(by_arm[a][key], by_arm[b][key])]
    t = st.mean(d) / (st.stdev(d) / math.sqrt(len(d))) if st.stdev(d) else float("inf")
    p = 2 * stats.t.sf(abs(t), df=len(d) - 1)
    return st.mean(d), t, p
tests = [("H1 critical > frozen, separation", "critical", "frozen", "separation"),
         ("H2 tune > frozen, separation", "tune", "frozen", "separation"),
         ("H3 critical > tune, separation", "critical", "tune", "separation"),
         ("H4 critical > dentate-too, recall", "critical", "dentate-too", "recall")]
results = [(name, *paired(a, b, k)) for name, a, b, k in tests]
order = sorted(range(len(results)), key=lambda i: results[i][3])
verdict, alive = {}, True
for rank, i in enumerate(order):
    name, diff, t, p = results[i]
    alive = alive and p <= 0.05 / (len(results) - rank) and diff > 0
    verdict[name] = "SUPPORTED" if alive else "not supported"
print()
for name, diff, t, p in results:
    print(f"  {name:36} diff {diff:+.3f}  t {t:+.2f}  p {p:.4f}  -> {verdict[name]}")
json.dump({"tests": [dict(zip(("name", "diff", "t", "p"), r)) for r in results], "verdict": verdict},
          open("/kaggle/working/artifacts/preregistered.json", "w"), indent=1)

### The whole thing, end to end

In [ ]:
%%time
# The whole thing, with everything it just learned loaded, and the internet on.
import importlib, os, shutil, sys, json
for name in ("speech.pt", "patterns.pt", "markers.pt", "unigrams.pkl"):
    src = f"/kaggle/working/artifacts/{name}"
    if os.path.exists(src):
        shutil.copy(src, f"/kaggle/working/neuromoe/brain/{name}")
sys.path[:0] = ["/kaggle/working/neuromoe", "/kaggle/working/neuromoe/chat"]
os.chdir("/kaggle/working/neuromoe")
from brain.agent import Agent

a = Agent(background=False)
a.allow_web = True
print("loaded:", {"speech": a.speech is not None,
                 "patterns": len(a.patterns.kept) if a.patterns else 0,
                 "openings": a.markers.stats() if a.markers else {}})

transcript = []
# the conversation that failed in the portal, replayed: pointers, acronyms, typos, where-questions
a.say("Rex is a dog"); a.say("dogs are mammals"); a.say("mammals are animals")
for q in ["is Rex an animal", "is Rex a fish",
          "hello, how is your day today", "what is Kuala Lumpur", "where is washington dc",
          "what is the capital city of india", "write an essay that compares and contrasts the three cities",
          "can you write an essay on just kl?", "cany ou write an essay about it", "what country is beijig in",
          "is it bigger than Kuala Lumpur?", "who is Will Smith", "how does Will Smith bear on Malaysia"]:
    turn = a.say(q)[-1]
    transcript.append({"asked": q, "action": turn["action"], "reply": turn["reply"],
                       "reasoning": turn.get("reasoning", [])})
    print("=" * 96)
    print(">", q, f"[{turn['intent']} -> {turn['action']}]")
    print(turn["reply"])
    for line in turn.get("reasoning", []):
        print("   .", line)
json.dump(transcript, open("/kaggle/working/artifacts/transcript.json", "w"), indent=1)
print("\nfacts held:", len(a.maps.log), "| pages read:", len(a.read_pages))

Artifacts in `/kaggle/working/artifacts`: the speech memory, the learned phrasings, the openings, the extraction ablation and the transcript.